<a href="https://colab.research.google.com/github/Qalani/Dissertation/blob/main/winam_wh_spatial_panel_predictive_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Qalani/Dissertation/blob/main/winam_wh_spatial_panel_predictive_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Winam Gulf water hyacinth spatial-panel — predictive ML model (Track A)

> **One of a pair.** This notebook was split from `winam_wh_spatial_panel_test_model.ipynb` so the two model families can run in parallel and save wall-clock time. It carries the **shared panel build** (Drive → classified GeoTIFFs → grid → Earth Engine covariates → feature engineering, §1–§12) followed by the **Track A predictive workhorse only**. The inferential driver GAM (Track B) lives in the sibling notebook `winam_wh_spatial_panel_driver_gam.ipynb`. Build the panel once (both notebooks share the same Drive checkpoints, e.g. `PANEL_RAW_FROM_CHECKPOINT`), then run the two model sections concurrently.

This notebook builds a **spatially explicit monthly panel dataset** from already-classified water hyacinth GeoTIFFs in Google Drive, then fits the **accuracy** model.

It is designed as a **test-period prototype**, not the final dissertation model. It:

1. mounts Google Drive;
2. finds classified WH GeoTIFFs for a chosen period;
3. creates fixed 500 m or 1 km grid cells over Winam Gulf;
4. aggregates each classified raster to `grid cell × month`;
5. pulls environmental driver covariates directly from **Google Earth Engine** — rainfall (CHIRPS), wind and air temperature (ERA5 / ERA5-Land), water-surface temperature (MODIS LST), turbidity and chlorophyll-a proxies (Sentinel-2 / Sentinel-3 / MODIS), distance to rivers and shore, and catchment pressure (ESA WorldCover / WorldPop / GHSL) — and optionally merges CSV covariates for layers that have no EE asset (lake level, ENSO/IOD, bathymetry, in-situ nutrients);
6. creates lagged WH-cover and lagged environmental terms;
7. fits the **Track A predictive workhorse** — a two-stage hurdle (P(present) × E(cover | present)) plus a single Tweedie-objective gradient booster — on the autoregressive / neighbour / seasonal structure, evaluated on **spatial and temporal block cross-validation** against persistence and seasonal-climatology baselines;
8. exports panel tables, grid outputs and the ML skill / feature-importance tables.

The intended response variable is **monthly WH proportional cover per spatial unit**, not pixel-level WH occurrence.

The Earth Engine layers, assets and project (`ee-bmillwardsadler1`) match those already used in `Batch_Export.ipynb` and `Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb`, so the same authenticated Colab runtime can run all three notebooks.

## 1. Install packages

Run this once at the start of a fresh Colab runtime.

In [ ]:
# In Colab, uncomment and run this if packages are missing.
# This can take a few minutes.

# earthengine-api and geemap are needed for the Earth Engine covariate extraction
# in section 8. They match the dependencies of the classifier/export notebooks.
!pip -q install rasterio geopandas pyproj shapely fiona pyogrio statsmodels scikit-learn pygam tqdm earthengine-api geemap libpysal esda lightgbm shap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 99.6 MB/s eta 0:00:00


## 2. Imports and Google Drive mount

In [ ]:
from pathlib import Path
import re
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely.geometry import box
from tqdm.auto import tqdm

import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.windows import from_bounds, Window

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
)

import matplotlib.pyplot as plt

# Import the earthengine-api library
import ee

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive was not mounted automatically. If running outside Colab, this is expected.")
    print(e)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)

EE_PROJECT = 'ee-bmillwardsadler1'
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

Mounted at /content/drive


## 3. User configuration

Edit this cell first.

Important assumptions:

- By default this notebook now reads the **classified GeoTIFFs and run logs produced by `Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb`**.
- `WH_CLASS_VALUES` is set to `[2]`, matching the classifier notebook's `FLOATING_CLASS_CODE` for floating plants / water hyacinth.
- Classified rasters are selected from **one exact classifier run**, `ACTIVE_CLASSIFIER_VERSION`, via the run log named for it (`CLASSIFIER_RUN_LOG_PATH`). The run log is never chosen by modification time or by a `winam_full_stack_run_log_*.csv` wildcard.
- The **batch-export token** (`REQUIRED_EXPORT_TOKEN_BY_SENSOR`) and the **classifier version** are separate provenance fields. The token identifies the predictor schema a snapshot was exported with, so on its own it is not evidence that the raster came from the active classifier run; both must hold.
- `REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR` is matched with an exact `startswith()`, so rasters from superseded export schemas are excluded rather than averaged in.
- `SUPPLEMENT_RUN_LOG_WITH_FOLDER = False` and there is **no directory-scan fallback**. A missing or incomplete run-log record raises rather than being filled in from `classified_geotiffs/`.
- `CLASSIFIER_SENSOR_FILTER` chooses the sensors; `CLASSIFIER_PRODUCT_FILTER` must stay `"model"`, so rule, probability, diagnostic and tiled-intermediate rasters can never become the response.
- The default grid CRS is **EPSG:32736**, UTM Zone 36S, which is suitable for the Winam Gulf area south of the equator.
- The default AOI is the same broad Winam Gulf rectangle used in your previous GEE workflow.


In [ ]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

# These defaults match Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb:
#   DRIVE_ROOT / "outputs" / "full_stack_batch"
CLASSIFIER_BATCH_OUTPUT_DIR = Path("/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch")
CLASSIFIER_TABLE_DIR = CLASSIFIER_BATCH_OUTPUT_DIR / "tables"
CLASSIFIED_TIF_DIR = CLASSIFIER_BATCH_OUTPUT_DIR / "classified_geotiffs"

# ---------------------------------------------------------------------
# Classifier provenance: one exact classifier run, explicit export tokens
# ---------------------------------------------------------------------

# The single classifier run this panel is built from. It is the CLASSIFIER_VERSION
# assembled in Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb:
# 'route_b_s1_scc_spatialcv_proba_v4_whlev_temporal' plus the manual-correction
# suffix '_corr_' + MANUAL_CORRECTION_VERSION ('v1'). Outputs of any other run
# must not enter the response.
ACTIVE_CLASSIFIER_VERSION = (
    "route_b_s1_scc_spatialcv_proba_v4_whlev_temporal_corr_v1"
)

# The run log of that exact run, addressed by name. Never selected by
# modification time and never through a 'winam_full_stack_run_log_*.csv'
# wildcard: "newest on disk" quietly promotes a partial rerun of a different
# classifier version into the panel response.
CLASSIFIER_RUN_LOG_PATH = (
    CLASSIFIER_TABLE_DIR
    / f"winam_full_stack_run_log_{ACTIVE_CLASSIFIER_VERSION}.csv"
)

# Batch-export schema tokens, from S1_EXPORT_SCHEMA_VERSION /
# S2_EXPORT_SCHEMA_VERSION in Batch_Export.ipynb. These identify the PREDICTOR
# SCHEMA a snapshot was exported with -- which bands exist and in what order --
# and say nothing about which classifier later consumed it. Export token and
# classifier version are therefore two SEPARATE provenance fields: a matching
# token is never treated as proof of the classifier version on its own.
REQUIRED_EXPORT_TOKEN_BY_SENSOR = {
    "S1": "s1_scc_temporal_v1",
    "S2": "s2_whlev_temporal_v1",
}

# Batch_Export.ipynb (section 7) builds each snapshot prefix as
#   S1: f"winam_{S1_EXPORT_SCHEMA_VERSION}_{start}_to_{end}"
#   S2: f"winam_s2_predictors_{S2_EXPORT_SCHEMA_VERSION}_{start}_to_{end}"
# so a classified raster from the current schema starts with exactly these.
# Superseded exports (winam_s1_scc_predictors_..., s2_whlev_texture_v1) fail the
# startswith() test and are excluded instead of being averaged in unnoticed.
REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR = {
    "S1": "winam_s1_scc_temporal_v1_",
    "S2": "winam_s2_predictors_s2_whlev_temporal_v1_",
}

OUTPUT_DIR = Path("/content/drive/MyDrive/WH_spatial_panel_test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The run log named for ACTIVE_CLASSIFIER_VERSION is the ONLY inventory source.
USE_CLASSIFIER_RUN_LOG = True
# Directory supplementation is off and the folder-scan fallback has been removed.
# Scanning CLASSIFIED_TIF_DIR pulled in rasters from earlier classifier runs and
# earlier batch-export schemas -- exactly the mixed inventory this configuration
# exists to prevent. A missing or incomplete run-log record now raises a clear
# error so the classifier notebook is rerun, instead of being papered over with
# unrelated files.
SUPPLEMENT_RUN_LOG_WITH_FOLDER = False

# Classifier-output selection. To use Sentinel-1 as a cloud-gap-fill for the
# Sentinel-2 response (Task 1), BOTH sensors are discovered here; the S1/S2
# fusion (prefer-S2, gap-fill-S1) happens in Section 7b, NOT by raw concatenation.
# Set CLASSIFIER_SENSOR_FILTER = ["S2"] to restore the single-sensor (S2) panel.
CLASSIFIER_SENSOR_FILTER = ["S2"]       # None, ["S2"], ["S1"], or ["S2", "S1"]
# Must stay "model": the panel response is defined as the final
# model-classification product. Rule rasters are a different product and are not
# interchangeable with it, so they are never mixed into the same response.
CLASSIFIER_PRODUCT_FILTER = "model"
# The S1 patch filter writes '<model raster>_patch_cleaned.tif' beside its input.
# Where both exist for one acquisition the patch-cleaned raster is the final
# product and wins; competing outputs that are NOT a raw/cleaned pair raise.
PREFER_PATCH_CLEANED_S1 = True

# --- Sentinel-1 cloud-gap-fill fusion (Task 1) -------------------------------
# Sentinel-2 is cloud-limited: cloudy months drop out of the classified response,
# and that missingness correlates with rainfall (MNAR). Sentinel-1 SAR sees
# through cloud, so it is harmonised onto the S2 measurement scale and used to
# GAP-FILL S2 (never concatenated raw). S1 cover is calibrated to S2 from their
# overlap, then used ONLY where S2 is missing for a given cell-month. A `sensor`
# indicator is carried into the model so any residual cross-sensor offset is
# absorbed rather than attributed to ecology. See Section 7b / 9b.
ENABLE_S1_GAPFILL = False                 # S1 gap-fill REMOVED: the S1->S2 cover calibration collapsed to a near-constant floor (slope ~2e-4 on Pearson r~0.30), so the ~18% S1-derived cell-months carried no usable cover magnitude and fed a flat, non-informative response. S2-only panel; single sensor also removes the sensor-alternation lag corruption. Set True (and add "S1" to CLASSIFIER_SENSOR_FILTER) to restore.
PRIMARY_SENSOR = "S2"                     # preferred sensor; the other fills its gaps
S1_CALIBRATION_METHOD = "robust_linear"  # "robust_linear" (Theil-Sen), "isotonic", or "none"
SENSOR_FEATURE_ENABLED = False           # no S1 in the panel -> sensor_is_s1 is constant, so keep it out of the feature set
S1_CALIBRATION_MAX_POINTS = 50000        # cap on S1/S2 overlap points used to FIT the S1->S2 calibration. TheilSen RAM/time scale with the point count, so fitting the full ~10^6-row overlap at 500 m exhausts memory; None or 0 = use all.

# Optional monthly environmental covariates.
# Expected format: one row per month, with a column called 'month'.
# Example columns:
# month,rainfall_mm,wind_speed_ms,lake_level_m,turbidity,chl_a
# Lake level from Schwatke, C., Dettmering, D., Bosch, W., and Seitz, F.:
# DAHITI - an innovative approach for estimating water level time series over inland waters using multi-mission satellite altimetry:
# Hydrol. Earth Syst. Sci., 19, 4345-4364, doi:10.5194/hess-19-4345-2015, 2015

ENV_MONTHLY_CSV = None
ENV_MONTHLY_CSV = Path("/content/drive/MyDrive/WH_drivers/lake_level_monthly.csv")

# Optional spatial covariates by grid_id.
# Expected format: one row per grid_id, with a column called 'grid_id'.
# Example columns:
# grid_id,dist_river_m,fetch_m,depth_m,shelter_index
SPATIAL_COVARIATES_CSV = None
# SPATIAL_COVARIATES_CSV = Path("/content/drive/MyDrive/WH_drivers/grid_spatial_covariates.csv")

# ---------------------------------------------------------------------
# Test period
# ---------------------------------------------------------------------

TEST_START = "2017-01-01"
TEST_END = "2026-12-31"

# ---------------------------------------------------------------------
# Spatial design
# ---------------------------------------------------------------------

# Use 500 for a more detailed panel, or 1000 for a simpler/coarser test.
# 1000 m is recommended when USE_EARTH_ENGINE is True, because the per-cell
# Earth Engine reductions scale with the number of grid cells.
CELL_SIZE_M = 500

# UTM Zone 36S. Suitable for Winam Gulf because it is just south of the equator.
PANEL_CRS = "EPSG:32736"

# Winam Gulf AOI in WGS84: min_lon, min_lat, max_lon, max_lat.
# Updated to the agreed main-lake AOI (aoi/winam_gulf_main_lake_aoi.geojson): the
# western edge is clipped east to lon 34.2046673170698 so the panel covers only the
# main Winam Gulf / Lake Victoria water body (the small western ponds are dropped),
# matching the classifier's main-lake water mask (commit 566ea23). The bathymetry
# water mask below further restricts the rectangular grid to lake cells.
AOI_BBOX_WGS84 = (34.2046673170698, -0.55, 34.9, 0.0)

# ---------------------------------------------------------------------
# Classified raster settings
# ---------------------------------------------------------------------

# Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb writes floating plants /
# WH-like vegetation as class code 2 (FLOATING_CLASS_CODE = 2). This is the
# response class aggregated into WH cover by the panel model.
WH_CLASS_VALUES = [2]

# The classifier notebook writes NODATA_VALUE = 255. Keep it here even if a
# GeoTIFF's nodata metadata is missing.
EXTRA_NODATA_VALUES = [255]

# Valid classifier classes. S2 uses 0-3; S1/SCC uses 0-2. Including 3 is safe for
# S1 because it simply will not occur.
VALID_CLASS_VALUES = [0, 1, 2, 3]

# Minimum valid pixels in a cell-month for that observation to be retained.
MIN_VALID_PIXELS_PER_CELL_MONTH = 10

# How to combine duplicate classified GeoTIFFs in the same month.
# With the default classifier filters, duplicates usually mean multiple snapshots
# in the same month. Options: "mean", "max"
DUPLICATE_MONTH_METHOD = "mean"

# ---------------------------------------------------------------------
# Model settings
# ---------------------------------------------------------------------

TRAIN_FRACTION = 0.75
RANDOM_STATE = 42

# Fraction of the TRAIN months held out as a validation block used only to tune
# the presence probability threshold. The final TEST months are never used here.
VALIDATION_FRACTION = 0.25

# --- WH presence (Stage 1) target definition ---------------------------------
# Define WH "present" using an ecologically meaningful threshold rather than any
# non-zero pixel. If PRESENCE_AREA_HA_THRESHOLD is set it takes precedence;
# otherwise a fractional-cover threshold is used.
PRESENCE_COVER_THRESHOLD = 0.02      # >= 2% of valid cell area (~0.5 ha at 500 m); raised from 0.01 to cut label noise from near-zero cells flipping on classifier noise. See the presence-threshold sensitivity grid below.
PRESENCE_AREA_HA_THRESHOLD = None    # e.g. 0.5 to require >= 0.5 ha of WH instead

# Presence probability-threshold tuning. With "prevalence_match" the threshold
# whose predicted positive rate is closest to the observed WH prevalence on the
# validation block is selected; combined with calibrated probabilities this keeps
# total predicted WH extent roughly unbiased (rather than the F1-optimal point,
# which over-predicts presence). Other options: "f1", "precision", "recall".
PRESENCE_THRESHOLD_METRIC = "prevalence_match"
# A higher-recall threshold is also reported if its precision stays above this.
PRESENCE_MIN_PRECISION_FOR_RECALL = 0.5

# Probability calibration for the presence model. The base logistic regression is
# fit WITHOUT class balancing and its probabilities are calibrated on the held-out
# validation months so their sum matches the true number of present cells.
# "isotonic" (flexible, needs plenty of data) or "sigmoid" (Platt; safer when the
# validation block is small).
PRESENCE_CALIBRATION_METHOD = "isotonic"

# --- Habitat / valid-cell filters (applied before modelling) -----------------
# MIN_VALID_PIXELS_PER_CELL_MONTH is defined above with the raster settings.
MIN_VALID_FRACTION_PER_CELL_MONTH = None   # e.g. 0.5 to require >=50% valid cell area
REQUIRE_EVER_WET_OR_EVER_OBSERVED = True   # keep only cells ever validly observed

# ---------------------------------------------------------------------
# Earth Engine environmental covariates
# ---------------------------------------------------------------------

# Master switch. When True, section 8 pulls environmental driver covariates
# directly from Google Earth Engine for each grid cell / month. When False, the
# notebook keeps its original behaviour and uses only the optional CSVs above.
USE_EARTH_ENGINE = True

# Cloud project used for ee.Initialize. Matches Batch_Export.ipynb and
# Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb.
EE_PROJECT = "ee-bmillwardsadler1"

# Run ee.Authenticate() automatically if ee.Initialize fails the first time on a
# fresh runtime. Set False if you authenticate Earth Engine some other way.
EE_AUTHENTICATE = True

# Per-driver switches. Turn heavy collections (Sentinel-2/3) off for a quick
# test while keeping the cheap climate layers. Each maps to a row of the
# recommended environmental-driver table.
EE_LAYERS = {
    "rainfall_chirps":    True,   # CHIRPS daily -> monthly total + antecedent sums
    "rainfall_era5":      False,  # ERA5 hourly total precipitation (secondary)
    "wind_era5":          True,   # ERA5 hourly 10 m u/v wind -> speed/direction/axis
    "air_temp_era5":      True,   # ERA5 hourly 2 m air temperature
    "water_temp_modis":   True,   # MODIS LST (MOD11A2) day LST over water
    "chl_modis":          False,  # OFF: chlorophyll-a is sourced from Sentinel-3 OLCI
    "waterquality_s2":    True,   # Sentinel-2 turbidity (NDTI + red); chl-a -> Sentinel-3
    "waterquality_s3":    True,   # Sentinel-3 OLCI chl-a (MCI/MPH; Kravitz et al. 2020)
    "river_distance":     True,   # distance to HydroSHEDS rivers
    "shore_distance":     True,   # distance to shore + openness (JRC GSW)
    "catchment_pressure": True,   # ESA WorldCover fractions, WorldPop, GHSL built-up
    "gsw_water_fraction": True,   # JRC GSW water fraction per cell (for the "gsw"/"both" water mask)
}

# Which Sentinel-2 water-quality proxies to produce (subset of {"chla", "turbidity"}).
# Chlorophyll-a is retrieved from Sentinel-3 OLCI (see below), so the Sentinel-2 layer
# is limited to turbidity. Add "chla" to also produce the Sentinel-2 NDCI chl-a proxy.
EE_S2_PRODUCTS = {"turbidity"}

# Antecedent rainfall windows (days) summed up to the end of each month.
EE_RAIN_ANTECEDENT_DAYS = [30, 90]

# Rainfall-intensity ("spike") metrics from the CHIRPS daily series, added to
# the rainfall covariates when True. Mechanism: heavy downpours drive
# wave action and runoff pulses that can fragment and disperse floating hyacinth
# mats, so the FREQUENCY and INTENSITY of downpours carry signal a monthly total
# hides.
#   rain_spikes_<X>mm_cnt : days in the month with >= X mm (heavy-rain "spikes")
#   rain_max_1d_mm        : wettest single day (peak daily intensity, Rx1day)
#   rain_wet_days         : days with >= EE_RAIN_WET_DAY_MM (a wet day)
#   rain_sdii_mm          : mean rain per wet day (rainfall concentration; derived)
# CHIRPS is DAILY, so a "spike" is a heavy-rain day; for true sub-daily intensity
# (mm/hour) swap in an hourly product (GPM IMERG ~0.1 deg, or ERA5 precip).
EE_RAIN_INTENSITY = True
EE_RAIN_SPIKE_THRESHOLDS_MM = [20, 40]
EE_RAIN_WET_DAY_MM = 1.0

# CHIRPS rainfall resolution. CHIRPS daily is ~5.5 km, so over the ~100x61 km
# gulf an AOI mean discards a genuine within-gulf rainfall gradient (~18x11
# native pixels). With this True, ALL rainfall bands above (total, antecedent
# sums, and the intensity metrics) are reduced per grid cell and month instead --
# the same per-cell path as MODIS/Sentinel -- so they carry local structure and
# sharpen the rain-driven interactions (e.g. nutrient_pulse_idx). False = AOI mean.
EE_RAIN_PER_CELL = True

# Bay-axis bearing (degrees clockwise from north) used to resolve the monthly
# wind vector into along-axis and cross-axis components. Winam Gulf opens to the
# main lake roughly to the west/south-west, so its long axis is approximately
# E-W; positive along-axis wind points eastward, into the gulf.
EE_BAY_AXIS_BEARING_DEG = 90.0

# JRC Global Surface Water occurrence threshold (%) used to define water for the
# water-quality reductions and the shoreline geometry. Higher = more permanent
# water. (Batch_Export.ipynb uses 5 for an inclusive export mask; a higher value
# here keeps water-quality proxies over genuinely open water.)
EE_WATER_OCCURRENCE_THRESHOLD = 30

# Sentinel-2 granule cloud prefilter (CLOUDY_PIXEL_PERCENTAGE) before SCL masking.
EE_S2_CLOUD_PCT = 70

# Single scale (m) for the AOI-mean monthly climate reductions. The climate
# datasets are coarse (CHIRPS ~5 km, ERA5 ~28 km, MODIS ~1-4 km) relative to the
# gulf, so an AOI mean at 1 km is appropriate and fast.
EE_CLIMATE_AOI_SCALE = 1000

# Air-temperature source for the AOI-mean climate series. ERA5-Land
# (ECMWF/ERA5_LAND/HOURLY, 0.1 deg ~ 11 km) is ~2.5x finer than ERA5
# (0.25 deg ~ 28 km) and applies an elevation/lapse-rate correction to the 2 m
# temperature field, so it adds genuine spatial detail (mainly near the
# shoreline). ERA5-Land masks open ocean but resolves large lakes, so it covers
# Lake Victoria; extract_ee_covariates verifies AOI coverage at run time and
# falls back to ERA5 if the gulf is masked. Wind deliberately stays on ERA5:
# ERA5-Land does NOT downscale wind (its 10 m u/v is ERA5 bilinearly
# interpolated, no new information), so switching would imply spatial precision
# the data does not have.
EE_AIR_TEMP_USE_ERA5_LAND = True

# Resolution for the MODIS LST water-surface-temperature covariate. MOD11A2 is
# a 1 km product, so over the ~100x61 km gulf an AOI mean collapses ~100x60
# native pixels of real within-gulf temperature structure to one monthly value.
# With this True, water-surface temperature is reduced per grid cell and month
# (like the Sentinel-2/3 water-quality proxies), preserving that structure; the
# covariate then lands in the per-cell/month table instead of the AOI-mean
# monthly table. Set False to keep the AOI-mean monthly series.
EE_WATER_TEMP_PER_CELL = True

# Buffer (m) around each grid cell used for the catchment-pressure layers
# (land cover, population, built-up), representing local catchment influence.
EE_CATCHMENT_BUFFER_M = 2000

# Discharge-weighted river proximity. When True, add dist_majriver_m = distance
# to MAJOR rivers only (HydroSHEDS RIV_ORD <= EE_RIVER_MAJOR_MAX_ORD; RIV_ORD is
# the discharge-based river order, so LOWER = LARGER river). Water hyacinth is
# N/P-limited and the big inflows (Nzoia, Sondu-Miriu, Nyando, Yala) carry most
# of the nutrient load, so proximity to a MAJOR river is a better nutrient-
# delivery proxy than distance to any stream (kept as dist_river_m). Check the
# major set is non-empty for your AOI; raise the cutoff to include smaller rivers.
EE_RIVER_DISCHARGE_WEIGHTED = True
EE_RIVER_MAJOR_MAX_ORD = 7

# Representative year for WorldPop (annual; available 2000-2021).
EE_STATIC_YEAR = 2020

# reduceRegions tiling. Larger tileScale reduces "User memory limit exceeded".
EE_TILE_SCALE = 4

# Upper bound for the adaptive tileScale escalation in reduce_image_over_cells.
# When a per-cell reduceRegions chunk hits a capacity error ("User memory limit
# exceeded" / computation timed out), the reducer retries with a progressively
# doubled tileScale up to this cap and then splits the chunk, so a heavy group
# (e.g. Sentinel-3 OLCI chl-a) completes instead of failing the whole group.
EE_TILE_SCALE_MAX = 16

# Grid cells per getInfo request for the per-cell reductions. Lower this if
# requests time out; raise it to issue fewer, larger requests.
EE_MAX_CELLS_PER_REQUEST = 1500

# Sentinel-3 OLCI chl-a is by far the heaviest per-cell reduction: each granule
# is converted to TOA reflectance with a per-pixel solar-geometry term and is
# reprojected before the monthly median, so the default reduceRegions budget is
# exhausted ("User memory limit exceeded"). It therefore starts from a higher
# tileScale and fewer cells per request than the other per-cell groups; the
# adaptive fallback in reduce_image_over_cells escalates/splits further if even
# these are not enough. These are pure performance knobs (not part of the cache
# signature), so changing them re-attempts only Sentinel-3 without rebuilding the
# groups that already cached.
EE_S3_TILE_SCALE = 8
EE_S3_MAX_CELLS_PER_REQUEST = 300

# --- Sentinel-3 OLCI chlorophyll-a (Kravitz et al. 2020) ---
# Kravitz, J., Matthews, M., Bernard, S., Griffith, D. (2020). Application of
# Sentinel 3 OLCI for chl-a retrieval over small inland water targets: successes
# and challenges. Remote Sensing of Environment, 237, 111562.
# https://doi.org/10.1016/j.rse.2019.111562
#
# GEE's COPERNICUS/S3/OLCI carries only TOA radiances + quality_flags; the
# per-pixel observation geometry and meteorology are absent (Warren et al. 2021,
# Remote Sensing 13:1098), so the paper's BRR and 6SV1 atmospheric corrections
# cannot be run inside GEE. Kravitz et al. show their MCI/MPH band-difference
# algorithms are robust to atmospheric correction and give TOA-reflectance
# calibrations (their Table B.1). The notebook computes OLCI TOA reflectance and
# applies:
#   - MCI on TOA reflectance (their best TOA method, R2 = 0.53);
#   - MPH (best overall, R2 = 0.55; only a BRR calibration exists, applied to TOA
#     reflectance here as a documented approximation).
# Use {"mci"} for the single most defensible product, or {"mci", "mph"} for both.
EE_S3_PRODUCTS = {"mci", "mph"}

# Quadratic chl-a calibrations chl = a0*X^2 + a1*X + a2 (Kravitz et al. 2020, Table B.1).
EE_S3_CALIB = {
    "mci": (-93927.30, 7700.85, 16.62),    # Source "TOA Ref", model MCI
    "mph": (-169671.35, 10815.89, 7.54),   # Source "BRR", model MPH (best overall)
}

# Plausible range (mg/m3) used to clamp the retrieved chl-a.
EE_S3_CHLA_CLAMP = (0.0, 1000.0)

# Nominal OLCI top-of-atmosphere solar irradiance E0 (W m-2 um-1; Thuillier 2003)
# for the radiance->reflectance conversion (GEE lacks per-pixel solar flux).
EE_OLCI_E0 = {"Oa08": 1523.6, "Oa10": 1480.0, "Oa11": 1403.0, "Oa12": 1294.0, "Oa18": 935.0}

# Per-band OLCI radiance scale factors (gee:scale in the GEE COPERNICUS/S3/OLCI
# catalogue). The OLCI images carry no per-band "*_radiance_scale" property, so
# these published constants convert the stored DN bands to TOA radiance.
EE_OLCI_RADIANCE_SCALE = {"Oa08": 0.00876539, "Oa10": 0.00773378, "Oa11": 0.00675523,
                          "Oa12": 0.0071996, "Oa18": 0.00549962}

# ---------------------------------------------------------------------
# Earth Engine covariate caching (Drive)
# ---------------------------------------------------------------------

# Earth Engine extraction (section 8) is the slow part of this notebook. After
# the first successful run the covariate tables are written to Drive, and later
# runs reload them from disk instead of re-querying Earth Engine.
#
#   USE_EE_CACHE     - reload cached covariate tables when they are available.
#   EE_FORCE_REFRESH - ignore any cache and re-query Earth Engine, then rewrite
#                      the cache (use after editing the cached tables by hand).
#   EE_CACHE_DIR     - folder holding the cached tables and their manifest.
#                      Defaults to OUTPUT_DIR, so they are the same files that
#                      section 17 exports.
#
# The cache is keyed by a fingerprint of the settings that change the covariate
# values (cell size, AOI, test period, the panel's month set and the EE_*
# parameters below), so changing any of them rebuilds the cache automatically.
USE_EE_CACHE = True
EE_FORCE_REFRESH = False
EE_CACHE_DIR = OUTPUT_DIR

# ---------------------------------------------------------------------
# Raster-to-grid cache (Drive)
# ---------------------------------------------------------------------

# The classified GeoTIFF -> grid-cell reduction is deterministic for a given
# raster, grid, CRS and class-code setup, but it is expensive at 500 m because
# every notebook run otherwise reprojects/rasterizes each source GeoTIFF again.
# With this cache enabled, each reduced raster is saved once as a gridded
# 500 m table in Drive and later runs reload it directly. Delete the cache
# folder or set GRIDDED_RASTER_FORCE_REFRESH = True after changing class-code,
# nodata, grid, probability-response or raster-selection settings.
USE_GRIDDED_RASTER_CACHE = True
GRIDDED_RASTER_FORCE_REFRESH = True   # forced once: the per-raster grid cache is keyed by raster (not grid geometry), so the AOI/grid change above requires re-reducing rasters onto the new grid. Set back to False after the first successful run on the new AOI.
GRIDDED_RASTER_CACHE_DIR = OUTPUT_DIR / f"gridded_raster_cache_{CELL_SIZE_M}m"

# =====================================================================
# Methodological additions: bathymetry, water mask, probabilistic response,
# neighbour spatial-lag terms, and structure-aware cross-validation.
# =====================================================================

# --- Bathymetry (Lake Victoria analytical bathymetry raster) -----------------
# Per-cell mean depth (m) as a static habitat covariate. The supplied raster is
# depth-positive-down (0 at the shoreline, deeper offshore), 100 m, ESRI:102024,
# and is clipped to the lake shoreline so it also defines the water mask below.
# Upload Lake_Victoria_Analytical_ras.tif to Drive and point this at it.
BATHYMETRY_RASTER = Path("/content/drive/MyDrive/WH_drivers/Lake_Victoria_Analytical_ras.tif")
BATHYMETRY_DEPTH_POSITIVE_DOWN = True   # True for this raster (values increase with depth)
BATHYMETRY_RESAMPLING = "bilinear"      # continuous surface -> bilinear

# --- Water / littoral habitat mask (Priority 1) ------------------------------
# Restrict the panel to plausibly-wettable cells instead of the full bounding
# box, so "absence" means open water without WH rather than dry land.
#   "bathymetry" : fraction of the cell covered by valid lake-bathymetry pixels
#                  (no Earth Engine needed);
#   "gsw"        : JRC Global Surface Water occurrence fraction per cell (computed
#                  in the Earth Engine static extraction; needs USE_EARTH_ENGINE);
#   "both"       : require both; "none" : disable the mask.
WATER_MASK_SOURCE = "bathymetry"
MIN_WATER_FRACTION = 0.5

# --- Probabilistic (confidence-weighted) response (Priority 2) ---------------
# Use the classifier's per-pixel winning-class confidence raster to build a
# confidence-weighted WH cover that propagates classification uncertainty into
# the response. Falls back to hard-class cover automatically when no probability
# raster is available. The hard-class cover is always retained as wh_cover_hard.
USE_PROBABILITY_RESPONSE = True
PROBA_NODATA_VALUE = 255
PROBA_SCALE = 100.0                      # rasters store 0-100 -> divide to 0-1
WEIGHT_COVER_BY_CONFIDENCE = True        # weight Stage-2 cover rows by mean cell confidence

# --- Neighbourhood spatial-lag terms (Priority 3) ----------------------------
# Last month's mean WH cover/presence in adjacent cells. WH drifts with wind and
# spreads by fragmentation, so neighbour state is a mechanistic colonisation
# predictor (an autologistic-style term; Augustin et al. 1996).
NEIGHBOUR_LAG_ENABLED = True
NEIGHBOUR_CONTIGUITY = "queen"           # "queen" (8) or "rook" (4)

# --- Wind-driven advection & bloom memory (#4, predictive workhorse) ----------
# WH mats float and drift downwind, so a cell gains biomass from its UPWIND
# neighbours -- a directional colonisation term the isotropic neighbour-lag
# cannot express. The multi-month lags/rolling mean capture bloom build-up rather
# than one-step persistence, and the interaction terms encode wind-driven leeward
# accumulation on open fetch. All are leakage-free (only t-1 information).
WIND_ADVECTION_ENABLED = True
WH_MEMORY_LAGS = (2, 3)                   # extra per-cell wh_cover lags (months)
WH_MEMORY_ROLL = 3                        # trailing rolling-mean window over past cover
DRIVER_INTERACTIONS_ENABLED = True        # wind x fetch/exposure interaction features

# --- Structure-aware cross-validation (Priority 4) ---------------------------
CV_TEMPORAL_ENABLED = True
CV_TEMPORAL_N_FOLDS = 4                   # rolling-origin (expanding-window) folds
CV_TEMPORAL_MIN_TRAIN_MONTHS = 6
CV_SPATIAL_ENABLED = True
CV_SPATIAL_BLOCK_KM = 10.0                # block size for leave-block-out spatial CV
CV_SPATIAL_N_FOLDS = 5
BOYCE_N_BINS = 10


## 4. Helper functions

In [ ]:
def parse_month_from_filename(path):
    """
    Extract a monthly timestamp from a GeoTIFF filename.

    Handles common patterns:
    - YYYY-MM-DD
    - YYYY_MM_DD
    - YYYY-MM
    - YYYY_MM
    - YYYYMMDD
    - YYYYMM

    If the name contains a date range, the first date is used.
    """
    name = Path(path).stem

    patterns = [
        r"(?P<year>20\d{2})[-_](?P<month>\d{2})[-_](?P<day>\d{2})",
        r"(?P<year>20\d{2})(?P<month>\d{2})(?P<day>\d{2})",
        r"(?P<year>20\d{2})[-_](?P<month>\d{2})",
        r"(?P<year>20\d{2})(?P<month>\d{2})",
    ]

    for pat in patterns:
        m = re.search(pat, name)
        if m:
            year = int(m.group("year"))
            month = int(m.group("month"))
            return pd.Timestamp(year=year, month=month, day=1)

    return pd.NaT


def _normalise_classifier_sensor(sensor):
    if sensor is None:
        return None
    if isinstance(sensor, float) and pd.isna(sensor):
        return None
    sensor = str(sensor).strip().upper()
    if sensor == "":
        return None
    if sensor in {"S1", "S1_SCC", "SCC"}:
        return "S1"
    if sensor == "S2":
        return "S2"
    return sensor


# =====================================================================
# Classified-raster selection: ONE exact classifier run, EXPLICIT tokens
#
# Two provenance fields are tracked SEPARATELY, and both must hold for a raster
# to enter the panel response:
#
#   1. ACTIVE_CLASSIFIER_VERSION -- which classifier RUN produced the raster.
#      It is carried by the run-log filename. The run log is opened by that
#      exact name (never by mtime, never through a wildcard), so every record
#      read out of it belongs to that run by construction.
#
#   2. REQUIRED_EXPORT_TOKEN_BY_SENSOR -- which Batch_Export.ipynb predictor
#      SCHEMA the underlying snapshot was exported with. It is carried by the
#      filename prefix. It states which predictor bands went in, NOT which
#      classifier came out, so it is never accepted as evidence of the
#      classifier version on its own.
#
# A file that satisfies only one of the two is excluded, not merged.
# =====================================================================

_PROBABILITY_SUFFIX = "_proba"
_RULE_SUFFIX = "_local_rules"
_PATCH_CLEANED_SUFFIX = "_patch_cleaned"
_MODEL_SEGMENT = "_local_"
# Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb writes per-tile
# intermediates as '<final stem>_tile_000.tif' before mosaicking them.
_TILE_INTERMEDIATE_RE = re.compile(r"_tile_\d+$")
_GEOTIFF_SUFFIXES = {".tif", ".tiff"}


def required_classified_prefix(sensor):
    """Required filename prefix for `sensor`, or None when it is unconfigured."""
    return REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR.get(_normalise_classifier_sensor(sensor))


def required_export_token(sensor):
    """Required batch-export schema token for `sensor`, or None."""
    return REQUIRED_EXPORT_TOKEN_BY_SENSOR.get(_normalise_classifier_sensor(sensor))


def _check_prefix_token_consistency():
    """Fail fast if a required prefix does not carry its sensor's export token.

    Batch_Export.ipynb builds every snapshot prefix around the schema token, so
    the two settings must agree. Checking it here means the token test below can
    be a parsed comparison against the prefix rather than a substring scan of
    the whole filename.
    """
    for sensor, prefix in REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR.items():
        token = REQUIRED_EXPORT_TOKEN_BY_SENSOR.get(sensor)
        if token is None:
            raise ValueError(
                f"REQUIRED_EXPORT_TOKEN_BY_SENSOR has no entry for sensor {sensor!r}, "
                f"but REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR does ({prefix!r})."
            )
        if f"_{token}_" not in f"_{prefix}":
            raise ValueError(
                f"Configuration mismatch for {sensor}: required export token {token!r} is not "
                f"a token of the required prefix {prefix!r}. Check S1_EXPORT_SCHEMA_VERSION / "
                "S2_EXPORT_SCHEMA_VERSION and the section 7 prefixes in Batch_Export.ipynb."
            )


def _classifier_sensor_from_path(path):
    """Sensor implied by an EXACT prefix match, or None.

    Uses startswith() against the configured prefixes only. A loose substring
    search would let, for example, a re-exported file that merely mentions
    's1_scc_temporal_v1' somewhere in its name claim to be a current S1 product.
    """
    name = Path(path).name
    for sensor, prefix in REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR.items():
        if name.startswith(prefix):
            return sensor
    return None


def _export_token_from_path(path, sensor):
    """Parse the batch-export token `path` actually carries, or None.

    The token is read back out of the prefix the file is required to start with,
    so this is a parsed-token comparison rather than an 'is this string anywhere
    in the name' test.
    """
    sensor = _normalise_classifier_sensor(sensor)
    prefix = REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR.get(sensor)
    token = REQUIRED_EXPORT_TOKEN_BY_SENSOR.get(sensor)
    if prefix is None or token is None:
        return None
    if not Path(path).name.startswith(prefix):
        return None
    return token


def _classifier_product_from_path(path):
    """Classify a filename into the product it represents.

    Returns 'model', 'probability', 'rules', 'tile_intermediate' or 'unknown'.
    Only 'model' is the intended final model-classification product; everything
    else is a probability surface, a paper-rule raster, a pre-mosaic tile or an
    unrecognised diagnostic, and must stay out of the hard-class response.
    """
    stem = Path(path).stem
    if _TILE_INTERMEDIATE_RE.search(stem):
        return "tile_intermediate"
    base = stem[: -len(_PATCH_CLEANED_SUFFIX)] if stem.endswith(_PATCH_CLEANED_SUFFIX) else stem
    if base.endswith(_PROBABILITY_SUFFIX):
        return "probability"
    if base.endswith(_RULE_SUFFIX):
        return "rules"
    if _MODEL_SEGMENT in base:
        return "model"
    return "unknown"


def _path_is_final_model_classification(path):
    """True only for the intended final model-classification GeoTIFF."""
    return (
        Path(path).suffix.lower() in _GEOTIFF_SUFFIXES
        and _classifier_product_from_path(path) == "model"
    )


def _patch_clean_base_stem(path):
    """Stem with the S1 patch-filter suffix removed, so raw and cleaned pair up."""
    stem = Path(path).stem
    if stem.endswith(_PATCH_CLEANED_SUFFIX):
        return stem[: -len(_PATCH_CLEANED_SUFFIX)]
    return stem


def _blank_run_log_value(value):
    if value is None:
        return True
    if isinstance(value, float) and pd.isna(value):
        return True
    return str(value).strip() in {"", "nan", "None"}


def read_active_classifier_run_log(run_log_path=None):
    """Load THE run log of ACTIVE_CLASSIFIER_VERSION, addressed by exact name.

    Selecting a run log by modification time or through a
    'winam_full_stack_run_log_*.csv' wildcard silently promotes whichever run
    happened to finish last -- including a partial rerun of a different
    classifier version -- into the panel response. The run log is therefore
    pinned to one filename, and a missing one is an error with no fallback.
    """
    run_log_path = Path(CLASSIFIER_RUN_LOG_PATH if run_log_path is None else run_log_path)
    expected_name = f"winam_full_stack_run_log_{ACTIVE_CLASSIFIER_VERSION}.csv"

    if run_log_path.name != expected_name:
        raise ValueError(
            "The classifier run log must be addressed by the exact name of the active run.\n"
            f"  ACTIVE_CLASSIFIER_VERSION : {ACTIVE_CLASSIFIER_VERSION}\n"
            f"  expected filename         : {expected_name}\n"
            f"  configured filename       : {run_log_path.name}"
        )
    if not run_log_path.exists():
        raise FileNotFoundError(
            "The classifier run log for the active classifier run was not found:\n"
            f"  {run_log_path}\n"
            f"  ACTIVE_CLASSIFIER_VERSION = {ACTIVE_CLASSIFIER_VERSION!r}\n"
            "Run Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb for this exact version, "
            "or correct ACTIVE_CLASSIFIER_VERSION. There is deliberately no directory-scan "
            f"fallback: supplementing from {CLASSIFIED_TIF_DIR} would mix other classifier "
            "runs and other batch-export schemas into a single response."
        )

    log = pd.read_csv(run_log_path)
    required = {"sensor", "start_date", "end_date", "status", "model_classification_tif"}
    missing = required.difference(log.columns)
    if missing:
        raise ValueError(
            f"Classifier run log is missing required columns {sorted(missing)}: {run_log_path}"
        )
    return run_log_path, log.reset_index(drop=True)


def _records_from_classifier_run_log(run_log_path, log):
    """Strictly filtered classification records from the active run log.

    Returns (records, excluded). A record is retained only when it
    (a) is a completed classification of the active run, (b) is a sensor the
    panel asked for, (c) starts with that sensor's required prefix, (d) carries
    that sensor's required batch-export token, and (e) is the intended final
    model-classification product -- not a rule, probability, diagnostic or
    tiled-intermediate raster.
    """
    records = []
    excluded = []

    def drop(reason, path="", sensor=None, start_date="", end_date=""):
        excluded.append({
            "reason": reason,
            "path": "" if _blank_run_log_value(path) else str(path),
            "file_name": "" if _blank_run_log_value(path) else Path(str(path)).name,
            "sensor": "" if sensor is None else str(sensor),
            "start_date": "" if _blank_run_log_value(start_date) else str(start_date),
            "end_date": "" if _blank_run_log_value(end_date) else str(end_date),
        })

    requested_sensors = None
    if CLASSIFIER_SENSOR_FILTER is not None:
        requested_sensors = {_normalise_classifier_sensor(s) for s in CLASSIFIER_SENSOR_FILTER}

    for _, row in log.iterrows():
        sensor = _normalise_classifier_sensor(row.get("sensor"))
        start_date = row.get("start_date", "")
        end_date = row.get("end_date", "")
        raw_path = row.get("model_classification_tif", "")
        status = "" if _blank_run_log_value(row.get("status")) else str(row.get("status")).strip().lower()

        # (a) completed classifications only.
        if status != "completed":
            drop(f"status_{status or 'blank'}", raw_path, sensor, start_date, end_date)
            continue

        # (b) sensors this panel asked for.
        if requested_sensors is not None and sensor not in requested_sensors:
            drop("sensor_not_requested", raw_path, sensor, start_date, end_date)
            continue

        # A completed row with no model-classification path is an incomplete
        # record, not an invitation to go and find the file on disk.
        if _blank_run_log_value(raw_path):
            drop("completed_row_without_model_classification_path", "", sensor, start_date, end_date)
            continue
        path = Path(str(raw_path).strip())

        # (c) exact sensor-specific prefix.
        prefix = REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR.get(sensor)
        if prefix is None:
            drop("sensor_has_no_configured_prefix", path, sensor, start_date, end_date)
            continue
        if not path.name.startswith(prefix):
            drop(f"prefix_mismatch_expected_{prefix}", path, sensor, start_date, end_date)
            continue

        # (d) required batch-export token, parsed from that prefix.
        token = _export_token_from_path(path, sensor)
        if token is None or token != REQUIRED_EXPORT_TOKEN_BY_SENSOR.get(sensor):
            drop("export_token_mismatch", path, sensor, start_date, end_date)
            continue

        # (e) the intended final model-classification product only.
        if path.suffix.lower() not in _GEOTIFF_SUFFIXES:
            drop("not_a_geotiff", path, sensor, start_date, end_date)
            continue
        product = _classifier_product_from_path(path)
        if product != "model":
            drop(f"not_final_model_product_{product}", path, sensor, start_date, end_date)
            continue

        if _blank_run_log_value(start_date):
            drop("missing_start_date", path, sensor, start_date, end_date)
            continue
        month = pd.Timestamp(start_date).to_period("M").to_timestamp()

        # The probability raster is a SEPARATE confidence field used by the
        # confidence-weighted response. It is never a classification, so it is
        # held in its own column and validated on its own terms.
        proba_path = None
        raw_proba = row.get("model_probability_tif", "") if "model_probability_tif" in log.columns else ""
        if not _blank_run_log_value(raw_proba):
            cand = Path(str(raw_proba).strip())
            if (
                cand.name.startswith(prefix)
                and _classifier_product_from_path(cand) == "probability"
                and cand.exists()
            ):
                proba_path = str(cand)
            else:
                drop("probability_raster_rejected_as_confidence_field", cand, sensor, start_date, end_date)

        records.append({
            "path": path,
            "month": month,
            "sensor": sensor,
            "product": "model",
            "proba_path": proba_path,
            "start_date": str(start_date),
            "end_date": "" if _blank_run_log_value(end_date) else str(end_date),
            "export_token": token,
            "classifier_version": ACTIVE_CLASSIFIER_VERSION,
            "source_run_log": str(run_log_path),
            "from_directory_scan": False,
            "is_patch_cleaned": Path(path).stem.endswith(_PATCH_CLEANED_SUFFIX),
        })

    return records, excluded


def _resolve_canonical_records(records):
    """Reduce to ONE canonical classified raster per sensor and acquisition.

    An acquisition is the (sensor, start_date, end_date) export window, so two
    genuine acquisitions inside the same month both survive here and are averaged
    later by the Section 7 duplicate-month step. What must not survive is two
    competing rasters of the SAME acquisition. The only competition that has a
    defined winner is the S1 patch filter, which writes
    '<model raster>_patch_cleaned.tif' beside its input: the cleaned raster is
    the final product. Anything else is ambiguous and raises.
    """
    deduped = {}
    for rec in records:
        deduped[(rec["sensor"], rec["start_date"], rec["end_date"], str(rec["path"]))] = rec

    grouped = {}
    for rec in deduped.values():
        grouped.setdefault((rec["sensor"], rec["start_date"], rec["end_date"]), []).append(rec)

    kept = []
    superseded = []
    ambiguous = []

    for key, group in sorted(grouped.items()):
        if len(group) == 1:
            kept.append(group[0])
            continue

        by_base = {}
        for rec in group:
            by_base.setdefault(_patch_clean_base_stem(rec["path"]), []).append(rec)

        cleaned = [r for r in group if r["is_patch_cleaned"]]
        raw = [r for r in group if not r["is_patch_cleaned"]]

        if len(by_base) > 1 or len(cleaned) > 1 or len(raw) > 1:
            ambiguous.append((key, group))
            continue

        if PREFER_PATCH_CLEANED_S1 and cleaned:
            chosen, dropped = cleaned[0], raw
            reason = "superseded_by_patch_cleaned_raster"
        elif raw:
            chosen, dropped = raw[0], cleaned
            reason = "patch_cleaned_raster_disabled_by_PREFER_PATCH_CLEANED_S1"
        else:
            chosen, dropped = cleaned[0], []
            reason = "superseded_by_patch_cleaned_raster"

        kept.append(chosen)
        for rec in dropped:
            superseded.append({
                "reason": reason,
                "path": str(rec["path"]),
                "file_name": Path(rec["path"]).name,
                "sensor": rec["sensor"],
                "start_date": rec["start_date"],
                "end_date": rec["end_date"],
            })

    if ambiguous:
        lines = []
        for (sensor, start_date, end_date), group in ambiguous:
            lines.append(f"  {sensor} {start_date} to {end_date}:")
            for rec in sorted(group, key=lambda r: str(r["path"])):
                lines.append(f"    - {rec['path']}")
        raise ValueError(
            "Competing classified rasters remain for the same sensor and acquisition after "
            "provenance filtering, so the canonical product is ambiguous:\n"
            + "\n".join(lines)
            + "\n\nEvery sensor/acquisition must resolve to exactly one classified raster. "
            "Only a raw/patch-cleaned S1 pair is resolved automatically (the patch-cleaned "
            "raster wins). Remove or re-run the duplicates in "
            f"{CLASSIFIER_RUN_LOG_PATH.name} before rebuilding the panel."
        )

    return kept, superseded


def find_classified_tifs(folder, start, end):
    """Select the classified GeoTIFFs that make up the panel response.

    Selection comes from ONE run log -- the one named for
    ACTIVE_CLASSIFIER_VERSION -- and from nothing else. `folder` is reported for
    context only; it is never scanned, because supplementing the run log from
    the classified_geotiffs directory is what previously mixed several classifier
    runs and export schemas into one response.

    Also populates CLASSIFIED_SELECTION_AUDIT with the provenance audit.
    """
    global CLASSIFIED_SELECTION_AUDIT

    if CLASSIFIER_PRODUCT_FILTER != "model":
        raise ValueError(
            "CLASSIFIER_PRODUCT_FILTER must be 'model'. The panel response is defined as the "
            "final model-classification product; rule rasters are a separate product and are "
            f"not interchangeable with it (got {CLASSIFIER_PRODUCT_FILTER!r})."
        )
    if globals().get("SUPPLEMENT_RUN_LOG_WITH_FOLDER", False):
        raise ValueError(
            "SUPPLEMENT_RUN_LOG_WITH_FOLDER must be False. Directory supplementation was "
            "removed: it added classified rasters from other classifier runs and other "
            "batch-export schemas that the active run log deliberately excludes."
        )
    if not USE_CLASSIFIER_RUN_LOG:
        raise ValueError(
            "USE_CLASSIFIER_RUN_LOG must be True. The exact run log for "
            f"{ACTIVE_CLASSIFIER_VERSION!r} is the only supported inventory source."
        )
    _check_prefix_token_consistency()

    folder = Path(folder)
    start = pd.Timestamp(start).to_period("M").to_timestamp()
    end = pd.Timestamp(end).to_period("M").to_timestamp()

    run_log_path, log = read_active_classifier_run_log()
    print(f"Classifier run           : {ACTIVE_CLASSIFIER_VERSION}")
    print(f"Classifier run log       : {run_log_path}")
    print(f"Run-log rows             : {len(log):,}")

    records, excluded = _records_from_classifier_run_log(run_log_path, log)

    in_window = []
    for rec in records:
        if start <= rec["month"] <= end:
            in_window.append(rec)
        else:
            excluded.append({
                "reason": "outside_test_period",
                "path": str(rec["path"]),
                "file_name": Path(rec["path"]).name,
                "sensor": rec["sensor"],
                "start_date": rec["start_date"],
                "end_date": rec["end_date"],
            })

    kept, superseded = _resolve_canonical_records(in_window)
    excluded.extend(superseded)

    # Every retained record must point at a file that is actually there. A
    # dangling run-log record is an incomplete record and fails loudly instead of
    # being replaced by whatever else happens to be in the directory.
    missing = [rec for rec in kept if not Path(rec["path"]).exists()]
    if missing:
        listing = "\n".join(
            f"  {rec['sensor']} {rec['start_date']} to {rec['end_date']}: {rec['path']}"
            for rec in sorted(missing, key=lambda r: (r["sensor"], r["start_date"]))
        )
        raise FileNotFoundError(
            f"{len(missing)} classified raster(s) recorded as completed in {run_log_path.name} "
            f"are missing from disk:\n{listing}\n\n"
            "Re-run those datasets in Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb. The "
            "panel will not substitute other files for them."
        )

    out = pd.DataFrame(kept)
    if len(out) == 0:
        reason_counts = pd.Series([e["reason"] for e in excluded]).value_counts().to_dict()
        raise FileNotFoundError(
            f"No classified GeoTIFFs from classifier run {ACTIVE_CLASSIFIER_VERSION!r} survived "
            f"selection between {start.date()} and {end.date()} for sensor filter "
            f"{CLASSIFIER_SENSOR_FILTER}.\n"
            f"  run log        : {run_log_path}\n"
            f"  rasters live in: {folder}\n"
            f"  exclusions     : {reason_counts}"
        )

    out = out.sort_values(["month", "sensor", "start_date", "path"]).reset_index(drop=True)

    CLASSIFIED_SELECTION_AUDIT = _build_selection_audit(out, run_log_path, log, excluded, start, end, folder)
    return out


def _build_selection_audit(tif_index, run_log_path, log, excluded, start, end, folder):
    """Provenance audit for the selected classified rasters.

    Every value is JSON-serialisable except 'excluded_records', which is a
    DataFrame the Section 5 audit cell writes out as its own CSV.
    """
    excluded_df = pd.DataFrame(
        excluded, columns=["reason", "path", "file_name", "sensor", "start_date", "end_date"]
    )
    excluded_reasons = (
        excluded_df["reason"].value_counts().sort_index().to_dict() if len(excluded_df) else {}
    )

    by_sensor = {}
    for sensor, grp in tif_index.groupby("sensor"):
        months = pd.to_datetime(grp["month"])
        by_sensor[str(sensor)] = {
            "n_classified_rasters": int(len(grp)),
            "n_acquisitions": int(grp[["start_date", "end_date"]].drop_duplicates().shape[0]),
            "n_months": int(months.dt.to_period("M").nunique()),
            "first_month": months.min().strftime("%Y-%m"),
            "last_month": months.max().strftime("%Y-%m"),
            "n_patch_cleaned": int(grp["is_patch_cleaned"].sum()),
            "n_with_probability_raster": int(grp["proba_path"].notna().sum()),
            "required_prefix": REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR.get(str(sensor)),
            "required_export_token": REQUIRED_EXPORT_TOKEN_BY_SENSOR.get(str(sensor)),
        }

    months = pd.to_datetime(tif_index["month"])
    covered = sorted({m.strftime("%Y-%m") for m in months})
    expected = pd.period_range(start.to_period("M"), end.to_period("M"), freq="M")
    expected = [str(p) for p in expected]
    missing_months = [m for m in expected if m not in set(covered)]

    dup_keys = (
        tif_index.groupby(["sensor", "start_date", "end_date"]).size().rename("n").reset_index()
    )
    duplicate_acquisitions = dup_keys[dup_keys["n"] > 1]

    month_counts = tif_index.groupby(["sensor", tif_index["month"]]).size()
    multi_acq_months = {
        f"{s}:{pd.Timestamp(m).strftime('%Y-%m')}": int(n)
        for (s, m), n in month_counts.items() if n > 1
    }

    return {
        "classifier_version": ACTIVE_CLASSIFIER_VERSION,
        "classifier_run_log_path": str(run_log_path),
        "classifier_run_log_name": run_log_path.name,
        "classified_tif_dir": str(folder),
        "run_log_rows_total": int(len(log)),
        "required_export_token_by_sensor": dict(REQUIRED_EXPORT_TOKEN_BY_SENSOR),
        "required_classified_prefix_by_sensor": dict(REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR),
        "product_filter": CLASSIFIER_PRODUCT_FILTER,
        "sensor_filter": list(CLASSIFIER_SENSOR_FILTER) if CLASSIFIER_SENSOR_FILTER else None,
        "prefer_patch_cleaned_s1": bool(PREFER_PATCH_CLEANED_S1),
        "supplement_run_log_with_folder": bool(globals().get("SUPPLEMENT_RUN_LOG_WITH_FOLDER", False)),
        "n_from_directory_supplementation": int(tif_index["from_directory_scan"].astype(bool).sum()),
        "test_period_months": {"start": start.strftime("%Y-%m"),
                               "end": end.strftime("%Y-%m")},
        "n_selected_total": int(len(tif_index)),
        "n_selected_by_sensor": {str(k): int(v) for k, v in tif_index["sensor"].value_counts().items()},
        "by_sensor": by_sensor,
        "date_coverage": {
            "months_covered": len(covered),
            "first_month": covered[0] if covered else None,
            "last_month": covered[-1] if covered else None,
            "months_without_any_raster": missing_months,
        },
        "months_with_multiple_acquisitions": multi_acq_months,
        "duplicate_sensor_acquisitions": duplicate_acquisitions.to_dict("records"),
        "n_excluded_total": int(len(excluded_df)),
        "excluded_reasons": excluded_reasons,
        "excluded_records": excluded_df,
    }


# Provenance and metadata carried from each selected raster onto its panel rows.
# Names are 'source_' + the tif_index column, EXCEPT for columns that are already
# prefixed (source_run_log), which would otherwise become source_source_run_log.
PANEL_SOURCE_METADATA_COLUMNS = [
    "sensor", "product", "start_date", "end_date", "source_run_log",
    "classifier_version", "export_token", "from_directory_scan",
]


def panel_source_column(column):
    """Panel column name for a tif_index metadata column."""
    return column if column.startswith("source_") else f"source_{column}"


def stamp_source_metadata(df, row_dict, columns=None):
    """Copy the selected raster's metadata onto every row it produced."""
    columns = PANEL_SOURCE_METADATA_COLUMNS if columns is None else columns
    for column in columns:
        if column in row_dict:
            df[panel_source_column(column)] = row_dict[column]
    return df


def validate_panel_provenance(panel_df, context="panel"):
    """Re-validate the provenance fields carried on an already-built panel.

    Section 7 stamps each panel row with source_classifier_version,
    source_export_token, source_run_log and source_from_directory_scan. This
    check runs both when the panel is built and when a prebuilt panel_raw is
    reloaded from its Drive checkpoint, so a panel produced by an older, mixed
    inventory cannot be consumed as if it were the corrected one.
    """
    required_cols = [
        "sensor",
        "source_classifier_version",
        "source_export_token",
        "source_run_log",
        "source_from_directory_scan",
    ]
    missing_cols = [c for c in required_cols if c not in panel_df.columns]
    if missing_cols:
        raise ValueError(
            f"{context}: missing provenance column(s) {missing_cols}.\n"
            f"  columns present: {sorted(panel_df.columns)}\n"
            "A panel RELOADED from a checkpoint that is missing these predates the "
            "pinned-classifier selection: rebuild it with PANEL_RAW_FORCE_REFRESH = True. "
            "A panel just BUILT from the rasters should already carry them via "
            "stamp_source_metadata(), so this means Section 7 and "
            "PANEL_SOURCE_METADATA_COLUMNS have drifted apart."
        )

    versions = set(panel_df["source_classifier_version"].astype(str).unique())
    if versions != {ACTIVE_CLASSIFIER_VERSION}:
        raise ValueError(
            f"{context}: classifier version(s) {sorted(versions)} do not match "
            f"ACTIVE_CLASSIFIER_VERSION {ACTIVE_CLASSIFIER_VERSION!r}."
        )

    logs = set(panel_df["source_run_log"].astype(str).unique())
    expected_log = str(CLASSIFIER_RUN_LOG_PATH)
    if logs != {expected_log}:
        raise ValueError(
            f"{context}: rows came from run log(s) {sorted(logs)}; expected only {expected_log!r}."
        )

    scanned = panel_df["source_from_directory_scan"].astype(str).str.lower()
    if not scanned.isin({"false", "0", "0.0"}).all():
        raise ValueError(
            f"{context}: contains rows sourced by scanning the classified_geotiffs directory."
        )

    bad_tokens = {}
    for sensor, grp in panel_df.groupby("sensor"):
        expected_token = REQUIRED_EXPORT_TOKEN_BY_SENSOR.get(_normalise_classifier_sensor(sensor))
        found = set(grp["source_export_token"].astype(str).unique())
        if found != {str(expected_token)}:
            bad_tokens[str(sensor)] = (expected_token, sorted(found))
    if bad_tokens:
        raise ValueError(f"{context}: export-token mismatch by sensor: {bad_tokens}")

    return True


# Populated by find_classified_tifs; consumed by the Section 5 audit cell.
CLASSIFIED_SELECTION_AUDIT = None


def create_grid_from_bbox(aoi_bbox_wgs84, cell_size_m, panel_crs):
    """
    Create square grid cells over the AOI bounding box.

    The grid is generated in a projected CRS so each cell has a meaningful metric area.
    """
    aoi_wgs84 = gpd.GeoDataFrame(
        {"name": ["aoi"]},
        geometry=[box(*aoi_bbox_wgs84)],
        crs="EPSG:4326",
    )

    aoi_proj = aoi_wgs84.to_crs(panel_crs)
    xmin, ymin, xmax, ymax = aoi_proj.total_bounds

    xmin = np.floor(xmin / cell_size_m) * cell_size_m
    ymin = np.floor(ymin / cell_size_m) * cell_size_m
    xmax = np.ceil(xmax / cell_size_m) * cell_size_m
    ymax = np.ceil(ymax / cell_size_m) * cell_size_m

    xs = np.arange(xmin, xmax, cell_size_m)
    ys = np.arange(ymin, ymax, cell_size_m)

    cells = []
    for x in xs:
        for y in ys:
            geom = box(x, y, x + cell_size_m, y + cell_size_m)
            if geom.intersects(aoi_proj.geometry.iloc[0]):
                cells.append(geom)

    grid = gpd.GeoDataFrame(
        {"grid_id": np.arange(len(cells), dtype=np.int32)},
        geometry=cells,
        crs=panel_crs,
    )

    # Retain simple centroid-based spatial controls.
    cent = grid.geometry.centroid
    grid["x_km"] = cent.x / 1000.0
    grid["y_km"] = cent.y / 1000.0
    grid["cell_area_m2"] = grid.geometry.area

    return grid


def clamp_window(win, width, height):
    """
    Clamp a rasterio window to raster bounds.
    """
    col_off = max(0, int(np.floor(win.col_off)))
    row_off = max(0, int(np.floor(win.row_off)))
    col_max = min(width, int(np.ceil(win.col_off + win.width)))
    row_max = min(height, int(np.ceil(win.row_off + win.height)))

    if col_max <= col_off or row_max <= row_off:
        return None

    return Window(col_off, row_off, col_max - col_off, row_max - row_off)


def aggregate_one_raster_to_grid(
    tif_path,
    month,
    grid_gdf,
    panel_crs,
    wh_class_values,
    extra_nodata_values=None,
    valid_class_values=None,
):
    """
    Aggregate one classified WH GeoTIFF to the fixed grid.

    The raster is read through a WarpedVRT in the panel CRS, so pixel area is
    approximately metric and consistent across rasters.
    """
    extra_nodata_values = [] if extra_nodata_values is None else list(extra_nodata_values)
    wh_class_values = np.array(wh_class_values)

    with rasterio.open(tif_path) as src:
        vrt_kwargs = {
            "crs": panel_crs,
            "resampling": Resampling.nearest,
        }

        if src.nodata is not None:
            vrt_kwargs["nodata"] = src.nodata

        with WarpedVRT(src, **vrt_kwargs) as vrt:
            grid_bounds = grid_gdf.total_bounds
            raw_win = from_bounds(*grid_bounds, transform=vrt.transform)
            win = clamp_window(raw_win, vrt.width, vrt.height)

            if win is None:
                return pd.DataFrame()

            arr = vrt.read(1, window=win)
            transform = vrt.window_transform(win)

            # Rasterize grid IDs onto the same raster grid.
            shapes = ((geom, int(gid)) for geom, gid in zip(grid_gdf.geometry, grid_gdf["grid_id"]))
            id_arr = rasterize(
                shapes=shapes,
                out_shape=arr.shape,
                transform=transform,
                fill=-1,
                dtype="int32",
            )

            valid = id_arr >= 0

            if src.nodata is not None:
                valid &= arr != src.nodata

            if len(extra_nodata_values) > 0:
                valid &= ~np.isin(arr, np.array(extra_nodata_values))

            if valid_class_values is not None:
                valid &= np.isin(arr, np.array(valid_class_values))

            wh = valid & np.isin(arr, wh_class_values)

            valid_ids = id_arr[valid].astype(np.int32)
            wh_ids = id_arr[wh].astype(np.int32)

            n_cells = int(grid_gdf["grid_id"].max()) + 1
            valid_counts = np.bincount(valid_ids, minlength=n_cells)
            wh_counts = np.bincount(wh_ids, minlength=n_cells)

            pixel_area_m2 = abs(transform.a * transform.e)

            df = pd.DataFrame({
                "grid_id": np.arange(n_cells, dtype=np.int32),
                "month": pd.Timestamp(month),
                "valid_pixels": valid_counts.astype(np.int64),
                "wh_pixels": wh_counts.astype(np.int64),
            })

            df = df[df["valid_pixels"] > 0].copy()
            df["valid_area_m2"] = df["valid_pixels"] * pixel_area_m2
            df["wh_area_m2"] = df["wh_pixels"] * pixel_area_m2
            df["wh_cover"] = df["wh_area_m2"] / df["valid_area_m2"]
            df["source_file"] = Path(tif_path).name

            return df


def _gridded_raster_cache_key(tif_path, proba_path=None):
    """Stable cache key for one classified raster reduced onto the current grid."""
    tif_path = Path(tif_path)
    payload = {
        "tif_path": str(tif_path),
        "tif_mtime_ns": tif_path.stat().st_mtime_ns if tif_path.exists() else None,
        "tif_size": tif_path.stat().st_size if tif_path.exists() else None,
        "proba_path": None if proba_path in (None, "", "None", "nan") else str(proba_path),
        "proba_mtime_ns": None,
        "proba_size": None,
        "cell_size_m": CELL_SIZE_M,
        "panel_crs": PANEL_CRS,
        "aoi_bbox_wgs84": list(AOI_BBOX_WGS84),
        "n_grid_cells": int(grid["grid_id"].nunique()) if "grid" in globals() else None,
        "grid_bounds": [float(x) for x in grid.total_bounds] if "grid" in globals() else None,
        "wh_class_values": list(WH_CLASS_VALUES),
        "extra_nodata_values": list(EXTRA_NODATA_VALUES),
        "valid_class_values": list(VALID_CLASS_VALUES),
        "use_probability_response": bool(USE_PROBABILITY_RESPONSE),
        "proba_nodata_value": PROBA_NODATA_VALUE,
        "proba_scale": PROBA_SCALE,
    }
    if payload["proba_path"]:
        pp = Path(payload["proba_path"])
        if pp.exists():
            payload["proba_mtime_ns"] = pp.stat().st_mtime_ns
            payload["proba_size"] = pp.stat().st_size
    blob = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:20]


def _gridded_raster_cache_path(tif_path, month, sensor=None, proba_path=None):
    """CSV path for the cached 500 m grid reduction of one source raster."""
    tag = _gridded_raster_cache_key(tif_path, proba_path=proba_path)
    sensor_part = _normalise_classifier_sensor(sensor) or "unknown"
    month_part = pd.Timestamp(month).strftime("%Y%m")
    stem = Path(tif_path).stem.replace("/", "_")
    return Path(GRIDDED_RASTER_CACHE_DIR) / f"{month_part}_{sensor_part}_{stem}_{tag}.csv"


def load_cached_gridded_raster(cache_path):
    """Load a cached raster-to-grid reduction table, if available."""
    cache_path = Path(cache_path)
    if not cache_path.exists():
        return None
    df = pd.read_csv(cache_path)
    if "month" in df.columns:
        df["month"] = pd.to_datetime(df["month"]).dt.to_period("M").dt.to_timestamp()
    return df


def save_cached_gridded_raster(df, cache_path):
    """Persist one reduced classified raster as a 500 m gridded table."""
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(cache_path, index=False)
    return cache_path



def load_monthly_covariates(csv_path):
    """
    Load monthly covariates from CSV.

    Required column:
    - month

    The month column can be YYYY-MM, YYYY-MM-DD, or another pandas-parseable date.
    """
    df = pd.read_csv(csv_path)
    if "month" not in df.columns:
        raise ValueError("ENV_MONTHLY_CSV must contain a 'month' column.")

    df["month"] = pd.to_datetime(df["month"]).dt.to_period("M").dt.to_timestamp()
    return df


def clean_feature_columns(df, candidate_cols):
    """
    Keep numeric feature columns that are not entirely missing and not constant.
    """
    keep = []
    for col in candidate_cols:
        if col not in df.columns:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        non_missing = df[col].dropna()
        if len(non_missing) == 0:
            continue
        if non_missing.nunique() <= 1:
            continue
        keep.append(col)
    return keep


In [ ]:
# =====================================================================
# Earth Engine covariate helpers
# =====================================================================
# These functions pull the environmental driver layers that have usable
# Google Earth Engine assets and reduce them onto the spatial-panel grid.
#
# Coverage of the recommended environmental-driver table:
#   Rainfall ............ CHIRPS daily (UCSB-CHG/CHIRPS/DAILY), per cell when
#                         EE_RAIN_PER_CELL else AOI mean; optional ERA5
#   Wind ................ ERA5 hourly 10 m u/v (ECMWF/ERA5/HOURLY)
#   Air temperature ..... ERA5-Land 2 m where it covers the AOI (ECMWF/ERA5_LAND/HOURLY,
#                         ~11 km, lapse-rate corrected), else ERA5 (ECMWF/ERA5/HOURLY)
#   Water temperature ... MODIS LST 8-day day (MODIS/061/MOD11A2), water-masked;
#                         per grid cell when EE_WATER_TEMP_PER_CELL, else AOI mean
#   Turbidity/TSS ....... Sentinel-2 NDTI + red reflectance (water-masked)
#   Chlorophyll-a ....... Sentinel-2 NDCI (water-masked) and MODIS-Aqua chlor_a;
#                         optional Sentinel-3 OLCI MCI bloom index
#   River influence ..... distance to HydroSHEDS rivers (WWF/HydroSHEDS/v1/FreeFlowingRivers)
#   Shoreline retention . distance to shore + openness (JRC/GSW1_4/GlobalSurfaceWater)
#   Catchment pressure .. ESA WorldCover v200, WorldPop GP 100m, GHSL P2023A built-up
#
# Layers WITHOUT a standard EE asset are still handled by the optional CSVs in
# section 9: lake level (DAHITI / Hydroweb / G-REALM altimetry), ENSO/IOD
# indices, bathymetry/depth, and in-situ nutrients.
# =====================================================================

try:
    import ee
except Exception:
    ee = None

try:
    from shapely.geometry import mapping as _shapely_mapping
except Exception:
    _shapely_mapping = None

_EE_STATE = {"initialised": False}


def initialize_earth_engine(project=None, authenticate=None):
    """Initialise Earth Engine, authenticating on first use if needed."""
    if ee is None:
        raise ImportError("earthengine-api is not installed. Run the install cell first.")
    project = project or EE_PROJECT
    if authenticate is None:
        authenticate = EE_AUTHENTICATE
    if _EE_STATE["initialised"]:
        return ee
    try:
        ee.Initialize(project=project)
    except Exception:
        if authenticate:
            ee.Authenticate()
        ee.Initialize(project=project)
    _EE_STATE["initialised"] = True
    print(f"Earth Engine initialised on project: {project}")
    return ee


def ee_aoi_geometry(bbox=None):
    """AOI as an ee.Geometry rectangle in EPSG:4326 (matches AOI_BBOX_WGS84)."""
    bbox = AOI_BBOX_WGS84 if bbox is None else bbox
    return ee.Geometry.Rectangle(list(bbox), proj="EPSG:4326", geodesic=False)


def ee_water_mask01(aoi):
    """Mask that keeps water (JRC GSW occurrence >= threshold) and drops land."""
    return (
        ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
        .select("occurrence")
        .gte(EE_WATER_OCCURRENCE_THRESHOLD)
        .selfMask()
        .clip(aoi)
    )


def mask_s2_sr(img):
    """Sentinel-2 SCL cloud/shadow/snow mask (same logic as Batch_Export.ipynb)."""
    scl = img.select("SCL")
    mask = (
        scl.neq(3)     # cloud shadow
        .And(scl.neq(8))    # medium-probability cloud
        .And(scl.neq(9))    # high-probability cloud
        .And(scl.neq(10))   # thin cirrus
        .And(scl.neq(11))   # snow/ice
    )
    return img.updateMask(mask)


def _masked_band_template(band_names):
    """A fully masked multi-band image, used so monthly composites always carry
    their band schema even when a month has no imagery (keeps reduceRegions from
    failing on a zero-band image)."""
    n = len(band_names)
    return (
        ee.Image.constant([0] * n).rename(band_names).toFloat()
        .updateMask(ee.Image.constant(0))
    )


# ---------------------------------------------------------------------
# AOI-mean monthly climate covariates (rainfall, wind, temperature, chl-a)
# ---------------------------------------------------------------------

# Resolved by _resolve_air_temp_collection() at extraction time (coverage-checked).
_EE_AIR_TEMP_COLLECTION = "ECMWF/ERA5/HOURLY"


def _resolve_air_temp_collection(aoi):
    """Pick the 2 m air-temperature source, preferring ERA5-Land when it covers
    the AOI.

    ERA5-Land (0.1 deg ~ 11 km) is finer than ERA5 (0.25 deg ~ 28 km) and
    lapse-rate corrected, but it masks open water. Large lakes are resolved, so
    this checks that ERA5-Land actually has unmasked pixels over the AOI at the
    reduction scale and falls back to ERA5 if not. Sets and returns the module
    global _EE_AIR_TEMP_COLLECTION that climate_image_for_month() reads.
    """
    global _EE_AIR_TEMP_COLLECTION
    if not globals().get("EE_AIR_TEMP_USE_ERA5_LAND", False):
        _EE_AIR_TEMP_COLLECTION = "ECMWF/ERA5/HOURLY"
        return _EE_AIR_TEMP_COLLECTION
    n = None
    try:
        sample = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY").select("temperature_2m").first()
        n = sample.reduceRegion(
            reducer=ee.Reducer.count(), geometry=aoi, scale=EE_CLIMATE_AOI_SCALE,
            maxPixels=int(1e13), tileScale=EE_TILE_SCALE, bestEffort=True,
        ).get("temperature_2m").getInfo()
    except Exception as exc:
        print(f"  Air temperature: ERA5-Land coverage check failed "
              f"({type(exc).__name__}: {exc}); using ERA5.")
    if n:
        _EE_AIR_TEMP_COLLECTION = "ECMWF/ERA5_LAND/HOURLY"
        print(f"  Air temperature: ERA5-Land 0.1 deg (~11 km), {int(n)} valid AOI "
              f"pixels at {EE_CLIMATE_AOI_SCALE} m.")
    else:
        _EE_AIR_TEMP_COLLECTION = "ECMWF/ERA5/HOURLY"
        if n is not None:
            print("  Air temperature: ERA5 0.25 deg (~28 km); ERA5-Land has no AOI coverage.")
    return _EE_AIR_TEMP_COLLECTION


def climate_image_for_month(month_start):
    """Multi-band monthly climate image for an ee.Date month start."""
    start = ee.Date(month_start)
    end = start.advance(1, "month")
    aoi = ee_aoi_geometry()
    bands = []

    if EE_LAYERS.get("rainfall_chirps") and not globals().get("EE_RAIN_PER_CELL", False):
        # AOI-mean path (EE_RAIN_PER_CELL is False). When True, the same rainfall
        # bands are built per cell/month by chirps_rain_image (see extract_ee_covariates).
        chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")
        bands.append(chirps.filterDate(start, end).sum().rename("rain_chirps_mm"))
        for d in EE_RAIN_ANTECEDENT_DAYS:
            d = int(d)
            bands.append(
                chirps.filterDate(end.advance(-d, "day"), end).sum().rename(f"rain_chirps_{d}d_mm")
            )
        if globals().get("EE_RAIN_INTENSITY", False):
            # Daily-intensity / "spike" metrics within the month. CHIRPS is DAILY,
            # so a "spike" is a heavy-rain DAY; for sub-daily downpour intensity
            # (mm/hour) use an hourly product (GPM IMERG / ERA5). Heavy days (wave
            # action, runoff pulses) and rainfall concentration can fragment and
            # disperse hyacinth mats in ways a monthly total cannot express.
            month_chirps = chirps.filterDate(start, end)
            for x in EE_RAIN_SPIKE_THRESHOLDS_MM:
                bands.append(
                    month_chirps.map(lambda im, t=float(x): im.gte(t)).sum()
                    .rename(f"rain_spikes_{int(x)}mm_cnt")
                )
            bands.append(month_chirps.max().rename("rain_max_1d_mm"))
            bands.append(
                month_chirps.map(lambda im, t=float(EE_RAIN_WET_DAY_MM): im.gte(t)).sum()
                .rename("rain_wet_days")
            )

    if EE_LAYERS.get("rainfall_era5"):
        era5p = ee.ImageCollection("ECMWF/ERA5/HOURLY").select("total_precipitation_hourly")
        bands.append(
            era5p.filterDate(start, end).sum().multiply(1000.0).rename("rain_era5_mm")
        )

    if EE_LAYERS.get("wind_era5"):
        era5 = ee.ImageCollection("ECMWF/ERA5/HOURLY").filterDate(start, end)
        u = era5.select("u_component_of_wind_10m").mean().rename("wind_u_ms")
        v = era5.select("v_component_of_wind_10m").mean().rename("wind_v_ms")
        spd = (
            era5.map(lambda im: im.expression(
                "sqrt(u*u + v*v)",
                {"u": im.select("u_component_of_wind_10m"),
                 "v": im.select("v_component_of_wind_10m")},
            ).rename("s")).mean().rename("wind_speed_ms")
        )
        bands.extend([u, v, spd])

    if EE_LAYERS.get("air_temp_era5"):
        # Source resolved by _resolve_air_temp_collection(): ERA5-Land if it covers
        # the AOI (finer, lapse-rate corrected), else ERA5. Both expose temperature_2m
        # in kelvin with the same hourly semantics, so the reduction is unchanged.
        era5t = ee.ImageCollection(_EE_AIR_TEMP_COLLECTION).filterDate(start, end).select("temperature_2m")
        bands.append(era5t.mean().subtract(273.15).rename("air_temp_c"))

    if EE_LAYERS.get("water_temp_modis") and not globals().get("EE_WATER_TEMP_PER_CELL", False):
        # AOI-mean path, used only when EE_WATER_TEMP_PER_CELL is False. When True,
        # water-surface temperature is extracted per grid cell/month instead (see
        # modis_water_temp_image and extract_ee_covariates), keeping the 1 km signal.
        lst = (
            ee.ImageCollection("MODIS/061/MOD11A2").filterDate(start, end)
            .select("LST_Day_1km").mean().multiply(0.02).subtract(273.15)
            .updateMask(ee_water_mask01(aoi)).rename("water_temp_c")
        )
        bands.append(lst)

    if EE_LAYERS.get("chl_modis"):
        # MODIS-Aqua ocean colour ends 2022-02-28; covers the default 2021 test.
        chl = (
            ee.ImageCollection("NASA/OCEANDATA/MODIS-Aqua/L3SMI").filterDate(start, end)
            .select("chlor_a").mean().rename("chl_modis_mg_m3")
        )
        bands.append(chl)

    return ee.Image.cat(bands)


def aoi_monthly_dataframe(image_for_month, months, scale, aoi):
    """AOI-mean of a per-month image, returned as a month-indexed DataFrame."""
    month_millis = [int(pd.Timestamp(m).value // 10**6) for m in months]

    def per_month(ms):
        ms = ee.Number(ms)
        img = image_for_month(ee.Date(ms))
        stats = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi,
            scale=scale,
            maxPixels=1e13,
            tileScale=EE_TILE_SCALE,
            bestEffort=True,
        )
        return ee.Feature(None, stats).set("month_millis", ms)

    fc = ee.FeatureCollection(ee.List(month_millis).map(per_month))
    info = fc.getInfo()
    rows = []
    for feat in info["features"]:
        props = dict(feat["properties"])
        ms = props.pop("month_millis")
        props["month"] = pd.Timestamp(int(ms), unit="ms").to_period("M").to_timestamp()
        rows.append(props)
    return pd.DataFrame(rows).sort_values("month").reset_index(drop=True)


def add_wind_components(monthly_df):
    """Derive wind direction and bay-axis components from monthly mean u/v."""
    if monthly_df is None or not {"wind_u_ms", "wind_v_ms"}.issubset(monthly_df.columns):
        return monthly_df
    theta = np.radians(EE_BAY_AXIS_BEARING_DEG)
    u = monthly_df["wind_u_ms"]
    v = monthly_df["wind_v_ms"]
    # Direction the wind blows TOWARDS, degrees clockwise from north.
    monthly_df["wind_dir_to_deg"] = np.degrees(np.arctan2(u, v)) % 360
    # Components relative to the bay axis (positive along-axis = towards bearing).
    monthly_df["wind_axis_comp_ms"] = u * np.sin(theta) + v * np.cos(theta)
    monthly_df["wind_cross_comp_ms"] = u * np.cos(theta) - v * np.sin(theta)
    return monthly_df


def add_rain_intensity(monthly_df):
    """Derive rainfall-concentration indices that need division (kept out of EE).

    SDII (simple daily intensity index) = monthly total / wet days = the mean
    rainfall on rainy days. It separates a month whose rain fell in a few intense
    bursts (high SDII -- more mat-breaking wave/runoff energy) from one of equal
    total spread as steady drizzle (low SDII).
    """
    if monthly_df is None or not {"rain_chirps_mm", "rain_wet_days"}.issubset(monthly_df.columns):
        return monthly_df
    wet = monthly_df["rain_wet_days"].astype(float)
    monthly_df["rain_sdii_mm"] = np.where(wet > 0, monthly_df["rain_chirps_mm"] / wet, 0.0)
    return monthly_df


# ---------------------------------------------------------------------
# Per grid cell / month water-quality covariates (Sentinel-2 / Sentinel-3)
# ---------------------------------------------------------------------

def gdf_to_ee_fc(gdf_wgs84):
    """Build an ee.FeatureCollection from a WGS84 GeoDataFrame chunk."""
    feats = []
    for row in gdf_wgs84.itertuples(index=False):
        geom = ee.Geometry(_shapely_mapping(row.geometry), proj="EPSG:4326", geodesic=False)
        feats.append(ee.Feature(geom, {"grid_id": int(row.grid_id)}))
    return ee.FeatureCollection(feats)


def _is_ee_capacity_error(exc):
    """True for the Earth Engine errors a smaller / more finely tiled request can
    fix: the per-query memory limit, too many concurrent aggregations, or a
    computation timeout. Other errors (missing band, bad geometry) are NOT
    retried, because shrinking the request would not help and we want them to
    surface."""
    msg = str(exc).lower()
    return any(s in msg for s in (
        "user memory limit", "memory limit exceeded", "out of memory",
        "too many", "computation timed out", "timed out",
    ))


def _reduce_fc_chunk(image, chunk, reducer, scale, buffer_m, extra_props,
                     tile_scale, max_tile_scale):
    """reduceRegions one chunk of cells, self-recovering from capacity errors.

    On an Earth Engine memory/timeout error the same chunk is retried with a
    doubled tileScale up to ``max_tile_scale``; once that is reached the chunk is
    split in half and each half retried, down to a single cell. This lets a heavy
    group (Sentinel-3 OLCI chl-a) finish instead of aborting the whole group. A
    genuine (non-capacity) error, or a single cell still failing at the maximum
    tileScale, is re-raised."""
    fc = gdf_to_ee_fc(chunk)
    if buffer_m and buffer_m > 0:
        fc = fc.map(lambda f: f.buffer(buffer_m))
    try:
        reduced = image.reduceRegions(
            collection=fc, reducer=reducer, scale=scale, crs="EPSG:4326",
            tileScale=tile_scale,
        )
        info = reduced.getInfo()
    except Exception as exc:
        if not _is_ee_capacity_error(exc):
            raise
        if tile_scale < max_tile_scale:
            new_tile_scale = min(max_tile_scale, tile_scale * 2)
            print(f"    capacity error at tileScale={tile_scale} for {len(chunk)} "
                  f"cell(s); retrying at tileScale={new_tile_scale}.")
            return _reduce_fc_chunk(image, chunk, reducer, scale, buffer_m,
                                    extra_props, new_tile_scale, max_tile_scale)
        if len(chunk) > 1:
            mid = len(chunk) // 2
            print(f"    capacity error at tileScale={tile_scale}; splitting "
                  f"{len(chunk)} cells into {mid}+{len(chunk) - mid} and retrying.")
            left = _reduce_fc_chunk(image, chunk.iloc[:mid], reducer, scale,
                                    buffer_m, extra_props, tile_scale, max_tile_scale)
            right = _reduce_fc_chunk(image, chunk.iloc[mid:], reducer, scale,
                                     buffer_m, extra_props, tile_scale, max_tile_scale)
            return left + right
        raise

    rows = []
    for feat in info["features"]:
        props = dict(feat["properties"])
        props.pop("system:index", None)
        if extra_props:
            props.update(extra_props)
        rows.append(props)
    return rows


def reduce_image_over_cells(image, grid_wgs84, reducer, scale, buffer_m=0,
                            extra_props=None, tile_scale=None, max_cells=None,
                            max_tile_scale=None):
    """reduceRegions an image over grid cells, chunked, returned as a DataFrame.

    ``tile_scale`` / ``max_cells`` default to EE_TILE_SCALE / EE_MAX_CELLS_PER_REQUEST
    but can be overridden per group (the Sentinel-3 chl-a group uses heavier
    settings). Each chunk is reduced through _reduce_fc_chunk, which escalates the
    tileScale and finally splits the chunk on an Earth Engine capacity error, so a
    memory-hungry group completes rather than failing outright."""
    tile_scale = EE_TILE_SCALE if tile_scale is None else tile_scale
    max_cells = EE_MAX_CELLS_PER_REQUEST if max_cells is None else max_cells
    if max_tile_scale is None:
        max_tile_scale = max(tile_scale, globals().get("EE_TILE_SCALE_MAX", 16))
    rows = []
    n = len(grid_wgs84)
    for start in range(0, n, max_cells):
        chunk = grid_wgs84.iloc[start:start + max_cells]
        rows.extend(_reduce_fc_chunk(image, chunk, reducer, scale, buffer_m,
                                     extra_props, tile_scale, max_tile_scale))
    df = pd.DataFrame(rows)
    # A SINGLE-band image reduced by a named reducer (mean/median/sum/...) comes
    # back named after the REDUCER (e.g. "mean"), not the band, so two single-band
    # layers later collide into pandas' 'mean_x'/'mean_y' on merge (see the
    # feature-engineering repair). Rename the lone value column back to the band
    # name here so every covariate keeps its identity at the source.
    try:
        _bnames = list(image.bandNames().getInfo())
    except Exception:
        _bnames = None
    if _bnames is not None and len(_bnames) == 1 and len(df):
        _keys = {"grid_id"} | set((extra_props or {}).keys())
        _valcols = [c for c in df.columns if c not in _keys]
        if len(_valcols) == 1 and _valcols[0] != _bnames[0]:
            df = df.rename(columns={_valcols[0]: _bnames[0]})
    return df


def s2_waterquality_image(month_start, aoi):
    """Monthly median Sentinel-2 water-quality proxies, masked to water.

    EE_S2_PRODUCTS selects which proxies are produced:
      - "turbidity": NDTI + red reflectance
      - "chla":      NDCI (red-edge chlorophyll-a proxy)
    Chlorophyll-a is sourced from Sentinel-3 OLCI by default, so "chla" is off.
    """
    start = pd.Timestamp(month_start)
    end = start + pd.offsets.MonthBegin(1)
    bands = []
    if "chla" in EE_S2_PRODUCTS:
        bands.append("chl_ndci_s2")
    if "turbidity" in EE_S2_PRODUCTS:
        bands += ["turb_ndti_s2", "red_reflectance_s2"]
    coll = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(str(start.date()), str(end.date()))
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", EE_S2_CLOUD_PCT))
        .map(mask_s2_sr)
    )

    def add_wq(img):
        b = img.select(["B3", "B4", "B5"]).multiply(0.0001)
        out = []
        if "chla" in EE_S2_PRODUCTS:
            out.append(b.normalizedDifference(["B5", "B4"]).rename("chl_ndci_s2"))     # red-edge chl-a proxy
        if "turbidity" in EE_S2_PRODUCTS:
            out.append(b.normalizedDifference(["B4", "B3"]).rename("turb_ndti_s2"))     # turbidity proxy
            out.append(b.select("B4").rename("red_reflectance_s2"))                     # turbidity magnitude
        return ee.Image.cat(out)

    # Prepend a fully masked template so the composite always carries its band
    # schema, even for a month with no usable Sentinel-2 imagery.
    wq_coll = ee.ImageCollection([_masked_band_template(bands)]).merge(coll.map(add_wq))
    return wq_coll.median().updateMask(ee_water_mask01(aoi)).clip(aoi)


def _olci_cos_solar_zenith(img):
    """Per-pixel cos(solar zenith) for an OLCI image, from acquisition time and
    pixel location (NOAA approximation). GEE's OLCI asset has no geometry
    tie-points, so the solar geometry needed to convert radiance to reflectance
    is reconstructed here."""
    date = ee.Date(img.get("system:time_start"))
    doy = ee.Number(date.getRelative("day", "year")).add(1)
    hour = ee.Number(date.get("hour")).add(ee.Number(date.get("minute")).divide(60.0))
    gamma = doy.subtract(1).add(hour.subtract(12).divide(24.0)).multiply(2.0 * np.pi / 365.0)
    cg, sg = gamma.cos(), gamma.sin()
    c2g, s2g = gamma.multiply(2).cos(), gamma.multiply(2).sin()
    c3g, s3g = gamma.multiply(3).cos(), gamma.multiply(3).sin()
    decl = (ee.Number(0.006918)
            .subtract(cg.multiply(0.399912)).add(sg.multiply(0.070257))
            .subtract(c2g.multiply(0.006758)).add(s2g.multiply(0.000907))
            .subtract(c3g.multiply(0.002697)).add(s3g.multiply(0.00148)))
    eqtime = ee.Number(229.18).multiply(
        ee.Number(0.000075).add(cg.multiply(0.001868)).subtract(sg.multiply(0.032077))
        .subtract(c2g.multiply(0.014615)).subtract(s2g.multiply(0.040849)))
    lonlat = ee.Image.pixelLonLat()
    lat = lonlat.select("latitude").multiply(np.pi / 180.0)
    lon = lonlat.select("longitude")
    tst = lon.multiply(4).add(hour.multiply(60)).add(eqtime)            # true solar time (min)
    ha = tst.divide(4).subtract(180).multiply(np.pi / 180.0)            # hour angle (rad)
    cosz = (lat.sin().multiply(decl.sin())
            .add(lat.cos().multiply(ha.cos()).multiply(decl.cos())))
    return cosz.clamp(0.05, 1.0)


def s3_chla_image(month_start, aoi):
    """Monthly Sentinel-3 OLCI chlorophyll-a (mg/m3) from band-difference
    algorithms, following Kravitz et al. (2020), Remote Sensing of Environment
    237:111562, https://doi.org/10.1016/j.rse.2019.111562.

    GEE's COPERNICUS/S3/OLCI provides only TOA radiances + quality_flags (no
    observation geometry or meteorology), so the paper's best atmospheric
    corrections (BRR, 6SV1) cannot be reproduced here. Kravitz et al. show the
    MCI and MPH band-difference algorithms are robust to atmospheric correction
    (baseline subtraction removes the smooth atmospheric signal) and publish
    TOA-reflectance calibrations (their Table B.1). We therefore compute OLCI TOA
    reflectance and apply MCI (their best TOA method) and MPH (best overall; its
    only calibration is BRR, applied here to TOA reflectance as an approximation).
    The MPH peak-switching index originates with Matthews et al. (2012), RSE
    124:637-652. Per-band GEE radiance scaling is used as published, which is
    known to deviate slightly from the original product (Warren et al. 2021,
    Remote Sensing 13:1098), so the mg/m3 values are indicative.
    """
    start = pd.Timestamp(month_start)
    end = start + pd.offsets.MonthBegin(1)
    # OLCI bands used: Oa08=665, Oa10=681.25, Oa11=708.75, Oa12=753.75, Oa18=885 nm.
    bands = ["Oa08", "Oa10", "Oa11", "Oa12", "Oa18"]

    def to_reflectance(img):
        # OLCI granules are full ~1270 km swaths; reprojecting and median-
        # compositing them at full extent (below) is what exhausts the Earth
        # Engine per-query memory budget ("User memory limit exceeded"), even
        # for a single cell. Clip to the small AOI first so the heavy resample/
        # reproject/cos(SZA)/median steps run on the AOI only; values inside the
        # AOI are unchanged (the composite is clipped to the same AOI at the end).
        img = img.clip(aoi)
        # Put each granule on a fixed WGS84 grid BEFORE the cos(SZA) division: cosz
        # comes from pixelLonLat() (abstract EPSG:4326), and dividing the native
        # UTM-projected radiance by it triggers the projection-intersection error.
        img = img.resample("bilinear").reproject(crs="EPSG:4326", scale=300)
        cosz = _olci_cos_solar_zenith(img)
        out = []
        for b in bands:
            rad = img.select(b + "_radiance").multiply(ee.Number(EE_OLCI_RADIANCE_SCALE[b]))
            refl = rad.multiply(np.pi).divide(ee.Number(EE_OLCI_E0[b])).divide(cosz).rename(b)
            out.append(refl)
        return ee.Image.cat(out)

    coll = (
        ee.ImageCollection("COPERNICUS/S3/OLCI")
        .filterBounds(aoi)
        .filterDate(str(start.date()), str(end.date()))
        .map(to_reflectance)
    )
    # Monthly median reflectance; masked template keeps the band schema if empty.
    refl = ee.ImageCollection([_masked_band_template(bands)]).merge(coll).median()
    R665, R681, R709, R754, R885 = (refl.select(b) for b in bands)

    out_bands = []
    if "mci" in EE_S3_PRODUCTS:
        # MCI: peak at 709, baseline 681<->754 (Kravitz et al. 2020, eq. 3).
        mci = R709.subtract(R681).subtract(
            R754.subtract(R681).multiply((708.75 - 681.25) / (753.75 - 681.25)))
        a0, a1, a2 = EE_S3_CALIB["mci"]
        out_bands.append(
            mci.pow(2).multiply(a0).add(mci.multiply(a1)).add(a2)
            .clamp(EE_S3_CHLA_CLAMP[0], EE_S3_CHLA_CLAMP[1]).rename("chl_mci_s3"))
    if "mph" in EE_S3_PRODUCTS:
        # MPH: peak switches among 681/709/754, baseline 665<->885 (Matthews et al. 2012).
        peak = R681.max(R709).max(R754)
        lam = ee.Image(753.75).where(peak.eq(R709), 708.75).where(peak.eq(R681), 681.25)
        mph = peak.subtract(R665).subtract(
            R885.subtract(R665).multiply(lam.subtract(665.0).divide(885.0 - 665.0)))
        a0, a1, a2 = EE_S3_CALIB["mph"]
        out_bands.append(
            mph.pow(2).multiply(a0).add(mph.multiply(a1)).add(a2)
            .clamp(EE_S3_CHLA_CLAMP[0], EE_S3_CHLA_CLAMP[1]).rename("chl_mph_s3"))

    # Reproject the OLCI monthly composite before intersecting it with the WGS84
    # water mask/AOI. Some OLCI granules expose native swath projections far from
    # Winam (for example EPSG:32621), and clipping/masking those directly against
    # EPSG:4326 geometries can raise an Earth Engine projection-intersection error.
    # For the panel we only need 300 m cell means, so force a stable WGS84 grid and
    # let reduceRegions use the same CRS.
    img = ee.Image.cat(out_bands).resample("bilinear").reproject(crs="EPSG:4326", scale=300)
    water = ee_water_mask01(aoi).reproject(crs="EPSG:4326", scale=300)
    return img.updateMask(water).clip(aoi)


def chirps_rain_image(month_start, aoi):
    """All CHIRPS rainfall bands for one month, built as a single image for
    reduce_image_over_cells so CHIRPS's ~5.5 km within-gulf gradient is kept per
    cell instead of collapsed to an AOI mean. Mirrors the rainfall_chirps bands in
    climate_image_for_month: monthly total, antecedent sums, and (when
    EE_RAIN_INTENSITY) the heavy-day spike counts, Rx1day and wet-day count.
    """
    start = pd.Timestamp(month_start)
    end = start + pd.offsets.MonthBegin(1)
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")
    month_chirps = chirps.filterDate(str(start.date()), str(end.date()))
    bands = [month_chirps.sum().rename("rain_chirps_mm")]
    for d in EE_RAIN_ANTECEDENT_DAYS:
        d = int(d)
        ante_start = (end - pd.Timedelta(days=d)).date()
        bands.append(
            chirps.filterDate(str(ante_start), str(end.date())).sum().rename(f"rain_chirps_{d}d_mm")
        )
    if globals().get("EE_RAIN_INTENSITY", False):
        for x in EE_RAIN_SPIKE_THRESHOLDS_MM:
            bands.append(
                month_chirps.map(lambda im, t=float(x): im.gte(t)).sum()
                .rename(f"rain_spikes_{int(x)}mm_cnt")
            )
        bands.append(month_chirps.max().rename("rain_max_1d_mm"))
        bands.append(
            month_chirps.map(lambda im, t=float(EE_RAIN_WET_DAY_MM): im.gte(t)).sum()
            .rename("rain_wet_days")
        )
    return ee.Image.cat(bands).clip(aoi)


def modis_water_temp_image(month_start, aoi):
    """Monthly mean MODIS LST day water-surface temperature (deg C), water-masked.

    MOD11A2 is an 8-day day-LST composite at 1 km. This image is built for
    reduce_image_over_cells so the ~1 km within-gulf temperature structure is kept
    per cell instead of collapsed to an AOI mean. A fully masked template is merged
    first so the image always carries the water_temp_c band, even for a month with
    no MODIS imagery.
    """
    start = pd.Timestamp(month_start)
    end = start + pd.offsets.MonthBegin(1)
    coll = (
        ee.ImageCollection("MODIS/061/MOD11A2")
        .filterDate(str(start.date()), str(end.date()))
        .select("LST_Day_1km")
        .map(lambda im: im.multiply(0.02).subtract(273.15).rename("water_temp_c"))
    )
    lst = ee.ImageCollection([_masked_band_template(["water_temp_c"])]).merge(coll).mean()
    return lst.updateMask(ee_water_mask01(aoi)).clip(aoi)


# ---------------------------------------------------------------------
# Static per grid cell covariates (river/shore geometry, catchment pressure)
# ---------------------------------------------------------------------

def static_cell_mean_image(aoi):
    """Image of cell-mean static layers: distances and shoreline openness."""
    bands = []
    if EE_LAYERS.get("river_distance"):
        rivers = ee.FeatureCollection("WWF/HydroSHEDS/v1/FreeFlowingRivers").filterBounds(aoi.buffer(50000))
        bands.append(rivers.distance(50000).rename("dist_river_m"))
        if globals().get("EE_RIVER_DISCHARGE_WEIGHTED", False):
            # Discharge-weighted river proximity: distance to MAJOR rivers only.
            # RIV_ORD is HydroSHEDS' discharge-based order (lower = larger river),
            # so RIV_ORD <= EE_RIVER_MAJOR_MAX_ORD keeps the high-discharge inflows
            # that carry most of the N/P load and drops the minor streams a flat
            # distance-to-any-river weights equally.
            major = rivers.filter(ee.Filter.lte("RIV_ORD", EE_RIVER_MAJOR_MAX_ORD))
            bands.append(major.distance(50000).rename("dist_majriver_m"))
    if EE_LAYERS.get("shore_distance"):
        occ = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").unmask(0)
        land01 = occ.gte(EE_WATER_OCCURRENCE_THRESHOLD).Not()
        bands.append(land01.distance(ee.Kernel.euclidean(20000, "meters")).rename("dist_shore_m"))
        bands.append(
            occ.reduceNeighborhood(reducer=ee.Reducer.mean(), kernel=ee.Kernel.circle(5000, "meters"))
            .rename("openness_index")
        )
    if not bands:
        return None
    return ee.Image.cat(bands).clip(aoi)


def gsw_water_fraction_image(aoi):
    """Binary water(1)/non-water(0) from JRC GSW occurrence, so a cell-mean
    reduction gives the fraction of the cell that is water. Land / never-water
    pixels are unmasked to 0 (matching the shore-distance layer), so every cell
    gets a defined 0-1 value used by the "gsw"/"both" water mask."""
    occ = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").unmask(0)
    return (occ.gte(EE_WATER_OCCURRENCE_THRESHOLD)
            .rename("gsw_water_fraction").toFloat().clip(aoi))


def landcover_fraction_image(aoi):
    """ESA WorldCover binary class layers; reduced with mean over a buffer -> fraction."""
    wc = ee.ImageCollection("ESA/WorldCover/v200").filterBounds(aoi).mosaic().select("Map")
    crop = wc.eq(40).rename("frac_cropland")
    urban = wc.eq(50).rename("frac_urban")
    wetland = wc.eq(90).rename("frac_wetland")
    return ee.Image.cat([crop, urban, wetland]).toFloat().clip(aoi)


def population_builtup_image(aoi):
    """WorldPop population and GHSL built-up surface; reduced with sum over a buffer."""
    pop = (
        ee.ImageCollection("WorldPop/GP/100m/pop")
        .filter(ee.Filter.eq("year", int(EE_STATIC_YEAR)))
        .filterBounds(aoi).mosaic().select("population").rename("pop_count")
    )
    built = (
        ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_S")
        .filterDate("2020-01-01", "2021-01-01").first().select("built_surface").rename("ghsl_built_m2")
    )
    return ee.Image.cat([pop, built]).clip(aoi)


# ---------------------------------------------------------------------
# Orchestrator
# ---------------------------------------------------------------------


# Covariate groups are the units this orchestrator builds and the cache layer
# stores / re-attempts independently. Each group is extracted inside its own
# try/except, so a failure or transient Earth Engine error in one group (for
# example a Sentinel-3 projection error after Sentinel-2 has already been built)
# no longer aborts the whole extraction and discards the groups that succeeded.
EE_COVARIATE_GROUPS = (
    "monthly_climate",   # AOI-mean climate     -> monthly table
    "waterquality_s2",   # Sentinel-2 NDCI/NDTI  -> cell/month table
    "waterquality_s3",   # Sentinel-3 OLCI chl-a -> cell/month table
    "water_temp_cell",   # MODIS LST per cell    -> cell/month table
    "rainfall_cell",     # CHIRPS per cell       -> cell/month table
    "static",            # river/shore/catchment -> static table
)


def _ee_enabled_groups():
    """(enabled, climate_keys, water_temp_per_cell, rain_per_cell) for the current
    EE_LAYERS / product configuration. ``enabled`` is the set of covariate-group
    keys (from EE_COVARIATE_GROUPS) the configuration asks the extractor to build."""
    water_temp_per_cell = bool(EE_LAYERS.get("water_temp_modis")) and bool(
        globals().get("EE_WATER_TEMP_PER_CELL", False))
    rain_per_cell = bool(EE_LAYERS.get("rainfall_chirps")) and bool(
        globals().get("EE_RAIN_PER_CELL", False))
    climate_keys = ["rainfall_chirps", "rainfall_era5", "wind_era5",
                    "air_temp_era5", "water_temp_modis", "chl_modis"]
    if water_temp_per_cell:
        climate_keys = [k for k in climate_keys if k != "water_temp_modis"]
    if rain_per_cell:
        climate_keys = [k for k in climate_keys if k != "rainfall_chirps"]
    enabled = set()
    if any(EE_LAYERS.get(k) for k in climate_keys):
        enabled.add("monthly_climate")
    if EE_LAYERS.get("waterquality_s2") and EE_S2_PRODUCTS:
        enabled.add("waterquality_s2")
    if EE_LAYERS.get("waterquality_s3") and EE_S3_PRODUCTS:
        enabled.add("waterquality_s3")
    if water_temp_per_cell:
        enabled.add("water_temp_cell")
    if rain_per_cell:
        enabled.add("rainfall_cell")
    if any(EE_LAYERS.get(k) for k in ("river_distance", "shore_distance",
                                      "gsw_water_fraction", "catchment_pressure")):
        enabled.add("static")
    return enabled, climate_keys, water_temp_per_cell, rain_per_cell


def extract_ee_covariates(grid_gdf, months, only_groups=None):
    """Extract the enabled Earth Engine covariates onto the panel grid, group by group.

    Returns (monthly_df, cellmonth_df, static_df, registry, status) where:
      - monthly_df is keyed by 'month' (AOI-mean climate),
      - cellmonth_df is keyed by ['grid_id', 'month'] (S2/S3 water quality, and
        per-cell MODIS temperature / CHIRPS rainfall when enabled),
      - static_df is keyed by 'grid_id' (river/shore/catchment),
      - registry lists the covariate columns in each group, and
      - status is {"enabled","attempted","present","failed"} sets of group keys.

    Each covariate group builds inside its own try/except: a failure in one group
    is recorded in status["failed"] while the remaining groups still build and
    return, so the caller can cache everything that succeeded and re-attempt only
    the failed groups on a later run instead of rebuilding all of them.
    ``only_groups`` restricts the build to a subset of EE_COVARIATE_GROUPS (used by
    the cache layer to fill in groups missing from an earlier partial cache).
    """
    initialize_earth_engine()
    aoi = ee_aoi_geometry()
    grid_wgs84 = grid_gdf[["grid_id", "geometry"]].to_crs("EPSG:4326")
    months = sorted(
        pd.to_datetime(pd.Series(list(months))).dt.to_period("M").dt.to_timestamp().unique()
    )

    enabled, climate_keys, water_temp_per_cell, rain_per_cell = _ee_enabled_groups()
    attempted = enabled if only_groups is None else (enabled & set(only_groups))
    present, failed = set(), set()

    def _run_group(name, build):
        """Build one covariate group, recording success/failure without aborting."""
        if name not in attempted:
            return None
        try:
            result = build()
            present.add(name)
            return result
        except Exception as exc:
            failed.add(name)
            print(f"  [{name}] Earth Engine extraction failed; keeping the other "
                  f"covariate groups and retrying this one on a later run.")
            print(f"    Reason: {type(exc).__name__}: {exc}")
            return None

    monthly_df = cellmonth_df = static_df = None

    # --- AOI-mean monthly climate ---
    def _build_climate():
        if EE_LAYERS.get("air_temp_era5"):
            _resolve_air_temp_collection(aoi)
        print("Extracting AOI-mean monthly climate covariates (CHIRPS / ERA5 / MODIS) ...")
        df = aoi_monthly_dataframe(climate_image_for_month, months, EE_CLIMATE_AOI_SCALE, aoi)
        df = add_wind_components(df)
        df = add_rain_intensity(df)
        print("  monthly climate columns:", [c for c in df.columns if c != "month"])
        return df
    monthly_df = _run_group("monthly_climate", _build_climate)

    # --- Per cell / month water quality, temperature and rainfall ---
    def _cellmonth_source(desc, image_for_month, reducer, scale,
                          tile_scale=None, max_cells=None):
        print(f"Extracting {desc} ...")
        frames = []
        for m in tqdm(months, desc=desc):
            img = image_for_month(m, aoi)
            frames.append(
                reduce_image_over_cells(img, grid_wgs84, reducer, scale,
                                        extra_props={"month": m},
                                        tile_scale=tile_scale, max_cells=max_cells))
        return pd.concat(frames, ignore_index=True)

    cm_parts = []
    s2 = _run_group("waterquality_s2", lambda: _cellmonth_source(
        "Sentinel-2 water-quality proxies per cell and month",
        s2_waterquality_image, ee.Reducer.median(), 20))
    if s2 is not None:
        cm_parts.append(s2)
    s3 = _run_group("waterquality_s3", lambda: _cellmonth_source(
        "Sentinel-3 OLCI chl-a (MCI/MPH; Kravitz et al. 2020) per cell and month",
        s3_chla_image, ee.Reducer.mean(), 300,
        tile_scale=globals().get("EE_S3_TILE_SCALE", 8),
        max_cells=globals().get("EE_S3_MAX_CELLS_PER_REQUEST", 300)))
    if s3 is not None:
        cm_parts.append(s3)
    wt = _run_group("water_temp_cell", lambda: _cellmonth_source(
        "MODIS LST water-surface temperature per cell and month",
        modis_water_temp_image, ee.Reducer.mean(), 1000))
    if wt is not None:
        cm_parts.append(wt)
    rn = _run_group("rainfall_cell", lambda: _cellmonth_source(
        "CHIRPS rainfall (total / antecedent / intensity) per cell and month",
        chirps_rain_image, ee.Reducer.mean(), 5566))
    if rn is not None:
        cm_parts.append(rn)
    if cm_parts:
        cellmonth_df = cm_parts[0]
        for part in cm_parts[1:]:
            cellmonth_df = cellmonth_df.merge(part, on=["grid_id", "month"], how="outer")
        cellmonth_df["month"] = pd.to_datetime(cellmonth_df["month"]).dt.to_period("M").dt.to_timestamp()
        cellmonth_df = add_rain_intensity(cellmonth_df)
        print("  per-cell/month columns:", [c for c in cellmonth_df.columns if c not in ("grid_id", "month")])

    # --- Static per cell ---
    def _build_static():
        static_parts = []
        cell_img = static_cell_mean_image(aoi)
        if cell_img is not None:
            print("Extracting static distance/openness covariates per cell ...")
            static_parts.append(reduce_image_over_cells(cell_img, grid_wgs84, ee.Reducer.mean(), 90))
        if EE_LAYERS.get("gsw_water_fraction"):
            print("Extracting JRC GSW water fraction per cell (water mask) ...")
            gsw_img = gsw_water_fraction_image(aoi)
            static_parts.append(reduce_image_over_cells(gsw_img, grid_wgs84, ee.Reducer.mean(), 30))
        if EE_LAYERS.get("catchment_pressure"):
            print("Extracting catchment-pressure covariates per cell (WorldCover / WorldPop / GHSL) ...")
            frac_img = landcover_fraction_image(aoi)
            static_parts.append(
                reduce_image_over_cells(frac_img, grid_wgs84, ee.Reducer.mean(), 10, buffer_m=EE_CATCHMENT_BUFFER_M))
            count_img = population_builtup_image(aoi)
            static_parts.append(
                reduce_image_over_cells(count_img, grid_wgs84, ee.Reducer.sum(), 100, buffer_m=EE_CATCHMENT_BUFFER_M))
        if not static_parts:
            return None
        df = static_parts[0]
        for part in static_parts[1:]:
            df = df.merge(part, on="grid_id", how="outer")
        df = df.groupby("grid_id", as_index=False).first()
        print("  static columns:", [c for c in df.columns if c != "grid_id"])
        return df
    static_df = _run_group("static", _build_static)

    registry = {
        "monthly":   [] if monthly_df is None else [c for c in monthly_df.columns if c != "month"],
        "cellmonth": [] if cellmonth_df is None else [c for c in cellmonth_df.columns if c not in ("grid_id", "month")],
        "static":    [] if static_df is None else [c for c in static_df.columns if c != "grid_id"],
    }
    status = {"enabled": set(enabled), "attempted": set(attempted),
              "present": set(present), "failed": set(failed)}
    return monthly_df, cellmonth_df, static_df, registry, status


## 4b. Helpers for the methodological additions

In [ ]:
# =====================================================================
# Helpers for the methodological additions
#   - confidence-weighted (probabilistic) WH response   (Priority 2)
#   - per-cell bathymetry sampling + water/littoral mask (bathymetry + Priority 1)
#   - queen/rook neighbour graph + neighbour spatial-lag (Priority 3)
#   - Boyce index for presence evaluation                (Priority 4)
# These are additive; the original hard-classification path is unchanged.
# =====================================================================
try:
    from scipy.stats import spearmanr as _spearmanr
except Exception:
    _spearmanr = None


def aggregate_one_proba_raster_to_grid(
    class_path,
    proba_path,
    month,
    grid_gdf,
    panel_crs,
    wh_class_values,
    extra_nodata_values=None,
    valid_class_values=None,
    proba_nodata_value=255,
    proba_scale=100.0,
):
    """Confidence-weighted WH cover from a classification + max-class proba raster.

    The classifier writes a per-pixel *winning-class* confidence (0-100, nodata
    255). Where the winning class is WH that confidence IS P(WH), so a
    confidence-weighted WH cover -- sum(P(WH)) over WH pixels / valid pixels --
    down-weights low-confidence WH pixels and propagates classification
    uncertainty into the response. The class and proba rasters share the
    classifier's profile, so they are read through identical WarpedVRTs.

    Returns a DataFrame[grid_id, wh_cover_soft, wh_conf_mean] for cells with at
    least one valid pixel (no month column; the caller attaches it).
    """
    extra_nodata_values = [] if extra_nodata_values is None else list(extra_nodata_values)
    wh_class_values = np.array(wh_class_values)

    with rasterio.open(class_path) as csrc, rasterio.open(proba_path) as psrc:
        ckw = {"crs": panel_crs, "resampling": Resampling.nearest}
        if csrc.nodata is not None:
            ckw["nodata"] = csrc.nodata
        with WarpedVRT(csrc, **ckw) as cvrt, WarpedVRT(psrc, crs=panel_crs,
                                                       resampling=Resampling.nearest) as pvrt:
            grid_bounds = grid_gdf.total_bounds
            win = clamp_window(from_bounds(*grid_bounds, transform=cvrt.transform),
                               cvrt.width, cvrt.height)
            if win is None:
                return pd.DataFrame()

            carr = cvrt.read(1, window=win)
            transform = cvrt.window_transform(win)
            # Force the proba read onto the SAME output grid as the classification.
            pwin = clamp_window(from_bounds(*grid_bounds, transform=pvrt.transform),
                                pvrt.width, pvrt.height)
            parr = pvrt.read(1, window=pwin, out_shape=carr.shape,
                             resampling=Resampling.nearest).astype("float64")

            shapes = ((geom, int(gid)) for geom, gid in zip(grid_gdf.geometry, grid_gdf["grid_id"]))
            id_arr = rasterize(shapes=shapes, out_shape=carr.shape, transform=transform,
                               fill=-1, dtype="int32")

            valid = id_arr >= 0
            if csrc.nodata is not None:
                valid &= carr != csrc.nodata
            if len(extra_nodata_values) > 0:
                valid &= ~np.isin(carr, np.array(extra_nodata_values))
            if valid_class_values is not None:
                valid &= np.isin(carr, np.array(valid_class_values))
            wh = valid & np.isin(carr, wh_class_values)

            # Confidence (0-1); proba nodata -> treat as full confidence for an
            # already-classified WH pixel (so soft never undercounts a hard WH px).
            conf = np.where(parr == proba_nodata_value, np.nan, parr / proba_scale)
            conf = np.clip(conf, 0.0, 1.0)

            n_cells = int(grid_gdf["grid_id"].max()) + 1
            valid_counts = np.bincount(id_arr[valid].astype(np.int64), minlength=n_cells)
            wh_idx = id_arr[wh].astype(np.int64)
            wh_conf = conf[wh]
            wh_conf = np.where(np.isfinite(wh_conf), wh_conf, 1.0)
            conf_sum = np.bincount(wh_idx, weights=wh_conf, minlength=n_cells)
            wh_counts = np.bincount(wh_idx, minlength=n_cells)

            df = pd.DataFrame({
                "grid_id": np.arange(n_cells, dtype=np.int32),
                "wh_soft_pixels": conf_sum,
                "wh_pixels_hard": wh_counts.astype(np.int64),
                "valid_pixels_p": valid_counts.astype(np.int64),
            })
            df = df[df["valid_pixels_p"] > 0].copy()
            df["wh_cover_soft"] = df["wh_soft_pixels"] / df["valid_pixels_p"]
            with np.errstate(invalid="ignore", divide="ignore"):
                df["wh_conf_mean"] = np.where(
                    df["wh_pixels_hard"] > 0, df["wh_soft_pixels"] / df["wh_pixels_hard"], np.nan)
            return df[["grid_id", "wh_cover_soft", "wh_conf_mean"]]


def sample_bathymetry_to_grid(grid_gdf, raster_path, panel_crs, depth_positive_down=True,
                              resampling="bilinear"):
    """Mean depth (m) and valid-pixel fraction per grid cell from a bathymetry raster.

    The raster is read through a WarpedVRT in the panel CRS. Because the supplied
    Lake Victoria bathymetry is clipped to the lake shoreline, the fraction of a
    cell covered by valid depth pixels doubles as a water/littoral indicator
    (``bathy_water_fraction``). Returns DataFrame[grid_id, depth_m,
    bathy_water_fraction, n_depth_px].
    """
    raster_path = Path(raster_path)
    rs = getattr(Resampling, str(resampling), Resampling.bilinear)
    with rasterio.open(raster_path) as src:
        with WarpedVRT(src, crs=panel_crs, resampling=rs) as vrt:
            grid_bounds = grid_gdf.total_bounds
            win = clamp_window(from_bounds(*grid_bounds, transform=vrt.transform),
                               vrt.width, vrt.height)
            if win is None:
                raise ValueError("Bathymetry raster does not overlap the grid bounds.")
            arr = vrt.read(1, window=win).astype("float64")
            transform = vrt.window_transform(win)

            bad = ~np.isfinite(arr)
            if src.nodata is not None:
                bad |= np.isclose(arr, src.nodata)
            bad |= (arr < -1e30)            # float32 nodata sentinel
            arr[bad] = np.nan
            if not depth_positive_down:
                arr = -arr                  # store depth as positive-down metres

            shapes = ((geom, int(gid)) for geom, gid in zip(grid_gdf.geometry, grid_gdf["grid_id"]))
            id_arr = rasterize(shapes=shapes, out_shape=arr.shape, transform=transform,
                               fill=-1, dtype="int32")

            in_grid = id_arr >= 0
            has_depth = in_grid & np.isfinite(arr)
            n_cells = int(grid_gdf["grid_id"].max()) + 1

            depth_sum = np.bincount(id_arr[has_depth].astype(np.int64),
                                    weights=arr[has_depth], minlength=n_cells)
            depth_cnt = np.bincount(id_arr[has_depth].astype(np.int64), minlength=n_cells)
            total_cnt = np.bincount(id_arr[in_grid].astype(np.int64), minlength=n_cells)
            with np.errstate(invalid="ignore", divide="ignore"):
                depth_mean = np.where(depth_cnt > 0, depth_sum / depth_cnt, np.nan)
                water_frac = np.where(total_cnt > 0, depth_cnt / total_cnt, np.nan)

    out = pd.DataFrame({
        "grid_id": np.arange(n_cells, dtype=np.int32),
        "depth_m": depth_mean,
        "bathy_water_fraction": water_frac,
        "n_depth_px": depth_cnt.astype(np.int64),
    })
    return out[out["grid_id"].isin(grid_gdf["grid_id"])].reset_index(drop=True)


def build_grid_neighbours(grid_gdf, cell_size_m, contiguity="queen"):
    """Edge list (grid_id, neighbor_id) for a regular grid via integer cell indices.

    O(n) and exact for a regular grid, so it scales to tens of thousands of cells
    without building a full contiguity matrix. ``queen`` = 8-neighbour, ``rook`` =
    4-neighbour.
    """
    cent = grid_gdf.geometry.centroid
    ix = np.round((cent.x.values - cell_size_m / 2.0) / cell_size_m).astype(np.int64)
    iy = np.round((cent.y.values - cell_size_m / 2.0) / cell_size_m).astype(np.int64)
    ids = grid_gdf["grid_id"].values
    coord_to_id = {(int(a), int(b)): int(g) for a, b, g in zip(ix, iy, ids)}
    if contiguity == "rook":
        offsets = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    else:
        offsets = [(dx, dy) for dx in (-1, 0, 1) for dy in (-1, 0, 1) if not (dx == 0 and dy == 0)]
    edges = []
    for a, b, g in zip(ix, iy, ids):
        for dx, dy in offsets:
            nb = coord_to_id.get((int(a) + dx, int(b) + dy))
            if nb is not None:
                edges.append((int(g), nb))
    return pd.DataFrame(edges, columns=["grid_id", "neighbor_id"])


def add_neighbour_lag(panel_df, edges, value_cols=("wh_cover", "wh_present")):
    """Add the PREVIOUS month's neighbour-mean of each value column.

    For each (cell, month) the neighbour-mean of the current month is computed by
    joining the edge list to the panel, then shifted one month within the cell so
    the feature only uses information available at t-1 (no contemporaneous leakage).
    """
    value_cols = list(value_cols)
    panel_df = panel_df.sort_values(["grid_id", "month"]).reset_index(drop=True)
    base = panel_df[["grid_id", "month"] + value_cols].copy()
    for c in value_cols:
        base[c] = base[c].astype(float)
    merged = edges.merge(
        base.rename(columns={"grid_id": "neighbor_id"}),
        on="neighbor_id", how="left")
    aggd = (merged.groupby(["grid_id", "month"], as_index=False)[value_cols].mean()
            .rename(columns={c: f"{c}_neigh" for c in value_cols}))
    panel_df = panel_df.merge(aggd, on=["grid_id", "month"], how="left")
    for c in value_cols:
        panel_df[f"{c}_neigh_lag1"] = panel_df.groupby("grid_id")[f"{c}_neigh"].shift(1)
    panel_df = panel_df.drop(columns=[f"{c}_neigh" for c in value_cols])
    return panel_df


def add_wind_advection_lag(panel_df, edges, wind_cols=("wind_u_ms", "wind_v_ms"),
                           value_col="wh_cover", out_prefix="wh_adv_upwind"):
    """Wind-driven advective inflow of WH cover from UPWIND neighbours (lag-1).

    WH mats float and are pushed downwind, so a cell gains biomass from whichever
    neighbours lie upwind of it. For a target cell g and neighbour n let u be the
    unit vector from n to g; if the monthly wind w has a positive component along u
    (w . u > 0) the wind blows from n toward g, so n is upwind and contributes
    max(0, w . u) * cover(n). Two contemporaneous summaries are formed and then
    shifted one month within the cell (mirroring add_neighbour_lag), so the feature
    uses only information available at t-1 -- no contemporaneous leakage:
      <prefix>_flux_lag1 : sum_n max(0, w.u) * cover(n)   (wind-speed-weighted flux)
      <prefix>_mean_lag1 : that flux / sum_n max(0, w.u)  (directional-mean cover)
    Wind is read at the TARGET cell, so a spatially-uniform lake wind works too --
    the directionality then comes entirely from the static neighbour geometry.
    """
    wu_col, wv_col = wind_cols
    need = {"grid_id", "month", "x_km", "y_km", value_col, wu_col, wv_col}
    if not need.issubset(panel_df.columns) or edges is None or not len(edges):
        return panel_df
    panel_df = panel_df.sort_values(["grid_id", "month"]).reset_index(drop=True)

    # Static unit displacement from neighbour n -> target g (time-invariant grid).
    pos = panel_df.groupby("grid_id")[["x_km", "y_km"]].first()
    e = edges.rename(columns={"grid_id": "g", "neighbor_id": "n"})[["g", "n"]].copy()
    e = e.merge(pos.rename(columns={"x_km": "gx", "y_km": "gy"}), left_on="g", right_index=True, how="left")
    e = e.merge(pos.rename(columns={"x_km": "nx", "y_km": "ny"}), left_on="n", right_index=True, how="left")
    dx = (e["gx"] - e["nx"]).to_numpy(); dy = (e["gy"] - e["ny"]).to_numpy()
    dn = np.hypot(dx, dy)
    keep = np.isfinite(dn) & (dn > 0)
    e = e.loc[keep, ["g", "n"]].reset_index(drop=True)
    e["ux"] = dx[keep] / dn[keep]
    e["uy"] = dy[keep] / dn[keep]
    if not len(e):
        return panel_df
    g_arr, n_arr = e["g"].to_numpy(), e["n"].to_numpy()
    ux_arr, uy_arr = e["ux"].to_numpy(), e["uy"].to_numpy()

    rows = []
    for m, sub in panel_df.groupby("month"):
        cov = sub.set_index("grid_id")[value_col]
        wu = sub.set_index("grid_id")[wu_col]
        wv = sub.set_index("grid_id")[wv_col]
        w = np.maximum(0.0, e["g"].map(wu).to_numpy() * ux_arr
                            + e["g"].map(wv).to_numpy() * uy_arr)
        flux = w * e["n"].map(cov).to_numpy()
        d = pd.DataFrame({"grid_id": g_arr, "w": w, "flux": flux})
        agg = d.groupby("grid_id", as_index=False).agg(w=("w", "sum"), flux=("flux", "sum"))
        agg["month"] = m
        rows.append(agg)
    adv = pd.concat(rows, ignore_index=True)
    adv[f"{out_prefix}_flux"] = adv["flux"]
    adv[f"{out_prefix}_mean"] = np.where(adv["w"] > 0, adv["flux"] / adv["w"], np.nan)

    panel_df = panel_df.merge(
        adv[["grid_id", "month", f"{out_prefix}_flux", f"{out_prefix}_mean"]],
        on=["grid_id", "month"], how="left")
    for c in (f"{out_prefix}_flux", f"{out_prefix}_mean"):
        panel_df[f"{c}_lag1"] = panel_df.groupby("grid_id")[c].shift(1)
    panel_df = panel_df.drop(columns=[f"{out_prefix}_flux", f"{out_prefix}_mean"])
    return panel_df



def boyce_index(y_true, y_prob, n_bins=10, window=0.1):
    """Continuous Boyce index (Hirzel et al. 2006): Spearman correlation between the
    predicted-probability bin midpoint and the presence-enrichment ratio
    (observed positive rate in bin / overall positive rate). ~+1 = well ranked,
    ~0 = no better than random. Robust to prevalence, unlike AUC alone."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    pos_rate = y_true.mean()
    if pos_rate <= 0 or pos_rate >= 1:
        return np.nan
    lo = np.linspace(0.0, 1.0 - window, n_bins)
    hi = lo + window
    mids = (lo + hi) / 2.0
    ratios = []
    for a, b in zip(lo, hi):
        inb = (y_prob >= a) & (y_prob <= b) if b >= 1.0 else (y_prob >= a) & (y_prob < b)
        ratios.append(y_true[inb].mean() / pos_rate if inb.sum() > 0 else np.nan)
    ratios = np.array(ratios)
    ok = np.isfinite(ratios)
    if ok.sum() < 3:
        return np.nan
    if _spearmanr is not None:
        return float(_spearmanr(mids[ok], ratios[ok]).correlation)
    # Fallback: Spearman via rank-Pearson.
    xr = pd.Series(mids[ok]).rank().values
    yr = pd.Series(ratios[ok]).rank().values
    return float(np.corrcoef(xr, yr)[0, 1])


## 4c. Helpers for Sentinel-1 cloud-gap-fill fusion (Task 1)

Sentinel-2 is cloud-limited, so cloudy months drop out of the classified WH response and the gaps correlate with rainfall (an MNAR pattern that biases any rainfall effect). Sentinel-1 SAR is cloud-penetrating. The helpers below harmonise the S1 WH-cover panel onto the S2 scale and use it **only to fill S2 cloud gaps** (prefer-S2, gap-fill-S1), rather than concatenating the two sensors:

- `wh_presence_from_cover` — the single source of truth for the WH-present definition (reused by Section 7b and the cross-sensor confusion matrix and the merged panel).
- `sensor_overlap_frame` / `sensor_agreement_stats` — pair the two sensors on shared cell-months and quantify bias and rank/linear agreement.
- `fit_secondary_to_primary_calibration` — calibrate S1 cover onto the S2 scale by **Theil–Sen** robust linear regression (default; outlier-robust, deterministic given `RANDOM_STATE`) or **isotonic** regression (monotone, non-parametric). Falls back to identity when the overlap is too small.

The raster-to-grid aggregation itself is unchanged: the S1 GeoTIFFs are discovered from the same run log (`CLASSIFIER_SENSOR_FILTER` now includes `"S1"`) and aggregated by the existing `aggregate_one_raster_to_grid` / `aggregate_one_proba_raster_to_grid`, so the confidence-weighted (probabilistic) response works for S1 too.


In [ ]:
# =====================================================================
# Helpers for Sentinel-1 cloud-gap-fill fusion (Task 1)
# ---------------------------------------------------------------------
# Harmonise the Sentinel-1 (SAR) WH-cover panel onto the Sentinel-2 measurement
# scale and use it ONLY to fill S2 cloud gaps (prefer-S2, gap-fill-S1). These
# helpers add the cross-sensor agreement, calibration and merge logic; the
# raster-to-grid aggregation itself is reused unchanged from Section 7.
# Libraries: Theil-Sen (sklearn.linear_model.TheilSenRegressor) for an
# outlier-robust linear calibration, IsotonicRegression as a monotone
# non-parametric alternative. Both are deterministic given RANDOM_STATE.
# =====================================================================
from sklearn.linear_model import TheilSenRegressor
from sklearn.isotonic import IsotonicRegression
try:
    from scipy.stats import spearmanr as _spearmanr_fusion
except Exception:
    _spearmanr_fusion = globals().get("_spearmanr", None)


def wh_presence_from_cover(cover, area_ha=None):
    """Apply the configured WH-present definition to a cover (and optional area)
    series. Mirrors Section 7 so presence is defined identically wherever it is
    needed (the cross-sensor confusion matrix and the merged panel). Returns a
    boolean numpy array aligned by position."""
    cover = pd.to_numeric(pd.Series(cover).reset_index(drop=True), errors="coerce")
    if PRESENCE_AREA_HA_THRESHOLD is not None and area_ha is not None:
        area_ha = pd.to_numeric(pd.Series(area_ha).reset_index(drop=True), errors="coerce")
        return (area_ha >= PRESENCE_AREA_HA_THRESHOLD).to_numpy()
    return (cover >= PRESENCE_COVER_THRESHOLD).to_numpy()


def sensor_overlap_frame(panel_sensor, cover_col="wh_cover", primary="S2", secondary="S1"):
    """Wide frame of primary/secondary response on shared (grid_id, month) cells.

    Returns one row per cell-month observed by BOTH sensors, carrying each
    sensor's cover, WH area and valid area, for the agreement and calibration
    steps.
    """
    keep = ["grid_id", "month", cover_col, "wh_area_ha", "valid_area_m2"]
    ren = lambda s: {cover_col: f"{cover_col}_{s}", "wh_area_ha": f"wh_area_ha_{s}",
                     "valid_area_m2": f"valid_area_m2_{s}"}
    a = panel_sensor.loc[panel_sensor["sensor"] == primary, keep].rename(columns=ren(primary))
    b = panel_sensor.loc[panel_sensor["sensor"] == secondary, keep].rename(columns=ren(secondary))
    return a.merge(b, on=["grid_id", "month"], how="inner")


def sensor_agreement_stats(primary_cover, secondary_cover):
    """Mean/median bias and Pearson/Spearman correlation on paired cover values
    (non-finite pairs dropped). bias is secondary - primary (i.e. S1 - S2)."""
    p = np.asarray(primary_cover, float)
    s = np.asarray(secondary_cover, float)
    ok = np.isfinite(p) & np.isfinite(s)
    p, s = p[ok], s[ok]
    out = {"n_pairs": int(ok.sum()),
           "mean_sec_minus_prim": float(np.mean(s - p)) if ok.sum() else np.nan,
           "median_sec_minus_prim": float(np.median(s - p)) if ok.sum() else np.nan,
           "pearson_r": np.nan, "spearman_r": np.nan}
    if ok.sum() >= 3 and np.std(p) > 0 and np.std(s) > 0:
        out["pearson_r"] = float(np.corrcoef(p, s)[0, 1])
        if _spearmanr_fusion is not None:
            out["spearman_r"] = float(_spearmanr_fusion(p, s).correlation)
    return out


def fit_secondary_to_primary_calibration(secondary_cover, primary_cover,
                                         method="robust_linear", random_state=None):
    """Calibrate secondary (S1) cover onto the primary (S2) scale.

    Returns (f, info) where f maps S1 cover -> S2-equivalent cover, clipped to
    [0, 1]. method:
      - "robust_linear": Theil-Sen regression of S2 ~ S1 (outlier-robust).
      - "isotonic":      monotone non-parametric fit (order-preserving).
      - "none":          identity (use S1 as-is).
    Falls back to identity when the overlap is too small (< 5 pairs) or S1 is
    constant, so the merge never fails.
    """
    if random_state is None:
        random_state = globals().get("RANDOM_STATE", 42)
    s = np.asarray(secondary_cover, float)
    p = np.asarray(primary_cover, float)
    ok = np.isfinite(s) & np.isfinite(p)
    s, p = s[ok], p[ok]
    if method == "none" or ok.sum() < 5 or np.std(s) == 0:
        return (lambda x: np.clip(np.asarray(x, float), 0.0, 1.0),
                {"method": "identity", "n_overlap": int(ok.sum())})
    if method == "isotonic":
        ir = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(s, p)
        lo, hi = float(s.min()), float(s.max())
        return (lambda x: np.clip(ir.predict(np.clip(np.asarray(x, float), lo, hi)), 0.0, 1.0),
                {"method": "isotonic", "n_overlap": int(ok.sum()), "s1_range": (lo, hi)})
    _n = int(np.size(s))
    _cap = int(S1_CALIBRATION_MAX_POINTS or 0)
    if _cap and _n > _cap:
        # TheilSenRegressor memory and time scale with the number of points
        # (~0.08 GB per 1,000 here), so the full ~10^6-row S1/S2 overlap at
        # 500 m exhausts RAM; the slope+intercept is recovered to within ~1%
        # from tens of thousands of points. Deterministic given random_state.
        _sel = np.random.default_rng(random_state).choice(_n, size=_cap, replace=False)
        s, p = s[_sel], p[_sel]
    ts = TheilSenRegressor(random_state=random_state).fit(s.reshape(-1, 1), p)
    slope, intercept = float(ts.coef_[0]), float(ts.intercept_)
    return (lambda x: np.clip(intercept + slope * np.asarray(x, float), 0.0, 1.0),
            {"method": "robust_linear", "slope": slope, "intercept": intercept,
             "n_overlap": int(ok.sum())})


## 5. Select classifier-output GeoTIFFs for the test period

Selection is driven by **one** run log — `winam_full_stack_run_log_{ACTIVE_CLASSIFIER_VERSION}.csv` — and by nothing else. `classified_geotiffs/` is never scanned.

A run-log record is retained only when it:

1. belongs to the exact classifier run `ACTIVE_CLASSIFIER_VERSION` (guaranteed by the run-log filename);
2. is a **completed** classification;
3. starts with the required sensor-specific prefix (exact `startswith()`);
4. carries the required batch-export token for that sensor (parsed back out of that prefix);
5. is the intended **final model-classification** product — not a rule, probability, diagnostic, tiled-intermediate (`_tile_000`) or superseded-schema raster.

Each sensor/acquisition must then resolve to exactly one canonical raster. A raw/patch-cleaned S1 pair resolves to the patch-cleaned raster; any other competing pair raises an error listing the files. Several *genuine* acquisitions in one month are kept and averaged later in Section 7.

The cell finishes with hard assertions on all of the above plus file existence, and writes a provenance audit (classifier version, tokens, prefixes, run-log path, per-sensor counts, exclusion reasons, date coverage and duplicate checks) next to the panel outputs.


In [ ]:
tif_index = find_classified_tifs(CLASSIFIED_TIF_DIR, TEST_START, TEST_END)
audit = CLASSIFIED_SELECTION_AUDIT

print(f"\nSelected {len(tif_index)} classified GeoTIFF(s) from classifier run "
      f"{ACTIVE_CLASSIFIER_VERSION}.")
display(tif_index.head(20))

# =====================================================================
# Provenance gates. Each of these has failed silently before, so each is an
# assertion rather than a printed warning.
# =====================================================================

# 1. The run log is the one named for the active classifier run -- not the most
#    recently modified winam_full_stack_run_log_*.csv on Drive.
_expected_run_log_name = f"winam_full_stack_run_log_{ACTIVE_CLASSIFIER_VERSION}.csv"
assert audit["classifier_run_log_name"] == _expected_run_log_name, (
    f"Run log {audit['classifier_run_log_name']!r} does not match "
    f"ACTIVE_CLASSIFIER_VERSION ({_expected_run_log_name!r})."
)
assert audit["classifier_version"] == ACTIVE_CLASSIFIER_VERSION

# 2/3. Every path starts with its sensor's configured prefix (exact startswith).
for _sensor, _prefix in REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR.items():
    _names = [Path(p).name for p, s in zip(tif_index["path"], tif_index["sensor"]) if s == _sensor]
    _bad = [n for n in _names if not n.startswith(_prefix)]
    assert not _bad, f"{_sensor} rasters do not start with {_prefix!r}: {_bad[:10]}"

_unconfigured = sorted(set(tif_index["sensor"]) - set(REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR))
assert not _unconfigured, f"Selected rasters for unconfigured sensor(s): {_unconfigured}"

# 4. Every selected record carries the required batch-export token. The token is
#    tracked as its own provenance field: it identifies the PREDICTOR SCHEMA, and
#    on its own it is not evidence of the classifier version (asserted in 1).
_bad_token = [
    f"{Path(r.path).name} (token={r.export_token!r}, expected="
    f"{REQUIRED_EXPORT_TOKEN_BY_SENSOR.get(r.sensor)!r})"
    for r in tif_index.itertuples(index=False)
    if r.export_token != REQUIRED_EXPORT_TOKEN_BY_SENSOR.get(r.sensor)
]
assert not _bad_token, f"Records without the required export token: {_bad_token[:10]}"
assert set(tif_index["classifier_version"]) == {ACTIVE_CLASSIFIER_VERSION}, (
    "Selected records mix classifier versions: "
    f"{sorted(set(tif_index['classifier_version']))}"
)

# 5. Nothing came from directory supplementation.
assert SUPPLEMENT_RUN_LOG_WITH_FOLDER is False, "SUPPLEMENT_RUN_LOG_WITH_FOLDER must be False."
assert not tif_index["from_directory_scan"].astype(bool).any(), (
    "Some classified rasters were added by scanning the classified_geotiffs directory."
)
assert tif_index["source_run_log"].nunique() == 1, (
    f"Records came from more than one run log: {sorted(set(tif_index['source_run_log']))}"
)

# 6. No probability, rule, tiled-intermediate or unrecognised rasters.
_products = {Path(p).name: _classifier_product_from_path(p) for p in tif_index["path"]}
_not_model = {n: k for n, k in _products.items() if k != "model"}
assert not _not_model, f"Non-final-model rasters selected: {list(_not_model.items())[:10]}"
assert set(tif_index["product"]) == {"model"}, sorted(set(tif_index["product"]))

# 7. One canonical raster per sensor and acquisition.
_dupes = tif_index.groupby(["sensor", "start_date", "end_date"]).size()
_dupes = _dupes[_dupes > 1]
assert _dupes.empty, f"Duplicate sensor/acquisition records remain:\n{_dupes}"
assert not tif_index["path"].astype(str).duplicated().any(), "The same raster was selected twice."

# 8. Every selected file exists.
_missing = [str(p) for p in tif_index["path"] if not Path(p).exists()]
assert not _missing, f"Selected classified rasters missing from disk: {_missing[:10]}"

print("Provenance assertions passed.")

# =====================================================================
# Provenance audit -- printed and saved next to the panel outputs.
# =====================================================================
_excluded_records = audit["excluded_records"]
_audit_json = {k: v for k, v in audit.items() if k != "excluded_records"}

print("\n" + "=" * 72)
print("CLASSIFIED-RASTER PROVENANCE AUDIT")
print("=" * 72)
print(f"Classifier version : {audit['classifier_version']}")
print(f"Run log            : {audit['classifier_run_log_path']}")
print(f"                     ({audit['run_log_rows_total']:,} rows; directory supplementation OFF)")
print(f"Raster directory   : {audit['classified_tif_dir']} (never scanned; run log is authoritative)")
print(f"Test period        : {audit['test_period_months']['start']} to "
      f"{audit['test_period_months']['end']} (inclusive months)")
print("\nExport token / prefix by sensor:")
for _sensor in sorted(REQUIRED_EXPORT_TOKEN_BY_SENSOR):
    print(f"  {_sensor}: token={REQUIRED_EXPORT_TOKEN_BY_SENSOR[_sensor]!r}  "
          f"prefix={REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR.get(_sensor)!r}")

print(f"\nIncluded classified rasters: {audit['n_selected_total']}")
for _sensor, _info in sorted(audit["by_sensor"].items()):
    print(f"  {_sensor}: {_info['n_classified_rasters']} raster(s), "
          f"{_info['n_acquisitions']} acquisition(s), {_info['n_months']} month(s), "
          f"{_info['first_month']} to {_info['last_month']}, "
          f"{_info['n_patch_cleaned']} patch-cleaned, "
          f"{_info['n_with_probability_raster']} with a probability raster")

print(f"\nExcluded run-log records: {audit['n_excluded_total']}")
for _reason, _n in sorted(audit["excluded_reasons"].items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"  {_n:>5}  {_reason}")

_cov = audit["date_coverage"]
print(f"\nDate coverage: {_cov['months_covered']} month(s) with at least one raster, "
      f"{_cov['first_month']} to {_cov['last_month']}")
if _cov["months_without_any_raster"]:
    _gap = _cov["months_without_any_raster"]
    print(f"  Months in the test period with NO classified raster: {len(_gap)}")
    print(f"  {_gap[:24]}{' ...' if len(_gap) > 24 else ''}")

print("\nDuplicate checks:")
print(f"  duplicate sensor/acquisition records : {len(audit['duplicate_sensor_acquisitions'])}")
print(f"  rasters added by directory scan      : {audit['n_from_directory_supplementation']}")
if audit["months_with_multiple_acquisitions"]:
    print("  months holding several genuine acquisitions (averaged in Section 7 by "
          f"DUPLICATE_MONTH_METHOD={DUPLICATE_MONTH_METHOD!r}):")
    for _k, _n in sorted(audit["months_with_multiple_acquisitions"].items()):
        print(f"    {_k}: {_n}")
else:
    print("  months holding several genuine acquisitions: 0")
print("=" * 72)

_audit_stem = f"classified_provenance_audit_{ACTIVE_CLASSIFIER_VERSION}"
PROVENANCE_AUDIT_JSON = OUTPUT_DIR / f"{_audit_stem}.json"
PROVENANCE_AUDIT_INCLUDED_CSV = OUTPUT_DIR / f"{_audit_stem}_included.csv"
PROVENANCE_AUDIT_EXCLUDED_CSV = OUTPUT_DIR / f"{_audit_stem}_excluded.csv"

PROVENANCE_AUDIT_JSON.write_text(json.dumps(_audit_json, indent=2, default=str))
tif_index.assign(path=tif_index["path"].astype(str)).to_csv(PROVENANCE_AUDIT_INCLUDED_CSV, index=False)
_excluded_records.to_csv(PROVENANCE_AUDIT_EXCLUDED_CSV, index=False)

print("\nSaved provenance audit:")
for _p in (PROVENANCE_AUDIT_JSON, PROVENANCE_AUDIT_INCLUDED_CSV, PROVENANCE_AUDIT_EXCLUDED_CSV):
    print(" ", _p)

if {"sensor", "product"}.issubset(tif_index.columns):
    print("\nClassifier outputs selected for the panel:")
    display(tif_index.groupby(["sensor", "product"]).size().rename("n_geotiffs").reset_index())

dupes = tif_index.groupby("month").size()
dupes = dupes[dupes > 1]
if len(dupes) > 0:
    print("Months with multiple genuine acquisitions. These are combined later using:",
          DUPLICATE_MONTH_METHOD)
    display(dupes)


## 6. Create the spatial grid

This creates the fixed spatial units used in the panel. For a quick test, `CELL_SIZE_M = 1000` is faster. For a more detailed prototype, use `CELL_SIZE_M = 500`.

In [ ]:
grid = create_grid_from_bbox(AOI_BBOX_WGS84, CELL_SIZE_M, PANEL_CRS)

print(f"Created {len(grid):,} grid cells at {CELL_SIZE_M} m resolution.")
display(grid.head())

grid_path_gpkg = OUTPUT_DIR / f"winam_grid_{CELL_SIZE_M}m.gpkg"
grid_path_geojson = OUTPUT_DIR / f"winam_grid_{CELL_SIZE_M}m.geojson"

grid.to_file(grid_path_gpkg, layer="grid", driver="GPKG")
grid.to_crs("EPSG:4326").to_file(grid_path_geojson, driver="GeoJSON")

print("Saved grid to:")
print(grid_path_gpkg)
print(grid_path_geojson)

ax = grid.to_crs("EPSG:4326").plot(figsize=(8, 6), facecolor="none", edgecolor="black", linewidth=0.2)
ax.set_title(f"Winam Gulf spatial panel grid: {CELL_SIZE_M} m")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

## 6b. Bathymetry, water mask, and neighbour graph

Samples per-cell **depth** from the Lake Victoria analytical bathymetry raster (also used as the water/littoral mask, since it is clipped to the lake shoreline) and builds the **queen/rook neighbour graph** for the spatial-lag features.

In [ ]:
# ---------------------------------------------------------------------
# Bathymetry covariate, water/littoral mask inputs, and neighbour graph
# ---------------------------------------------------------------------
# Adds three grid-level structures used downstream:
#   - depth_m / bathy_water_fraction : sampled from the Lake Victoria analytical
#     bathymetry raster. Depth is a static habitat covariate; because the raster
#     is clipped to the lake shoreline, the per-cell valid-pixel fraction is also
#     used as the water/littoral mask (Priority 1).
#   - grid_neighbours : queen/rook contiguity edge list for the neighbour
#     spatial-lag features (Priority 3).
# depth_m and bathy_water_fraction are attached to `grid`, so they flow into the
# panel through the existing grid-attribute merge in section 7.

grid_neighbours = None
if NEIGHBOUR_LAG_ENABLED:
    grid_neighbours = build_grid_neighbours(grid, CELL_SIZE_M, contiguity=NEIGHBOUR_CONTIGUITY)
    _deg = grid_neighbours.groupby("grid_id").size()
    print(f"Neighbour graph ({NEIGHBOUR_CONTIGUITY}): {len(grid_neighbours):,} directed edges, "
          f"mean degree {_deg.mean():.2f} over {grid['grid_id'].nunique():,} cells.")
else:
    print("Neighbour spatial-lag features disabled (NEIGHBOUR_LAG_ENABLED = False).")

# Bathymetry sampling (graceful if the raster is missing, so the notebook still runs).
for _col in ("depth_m", "bathy_water_fraction"):
    if _col in grid.columns:
        grid = grid.drop(columns=_col)

if BATHYMETRY_RASTER is not None and Path(BATHYMETRY_RASTER).exists():
    try:
        bathy_cov = sample_bathymetry_to_grid(
            grid, BATHYMETRY_RASTER, PANEL_CRS,
            depth_positive_down=BATHYMETRY_DEPTH_POSITIVE_DOWN,
            resampling=BATHYMETRY_RESAMPLING,
        )
        grid = grid.merge(bathy_cov[["grid_id", "depth_m", "bathy_water_fraction"]],
                          on="grid_id", how="left")
        _have = grid["depth_m"].notna()
        print(f"\nBathymetry sampled for {int(_have.sum()):,}/{len(grid):,} cells "
              f"({100 * _have.mean():.0f}% contain lake pixels).")
        print(f"  depth_m (m):            min {grid.loc[_have,'depth_m'].min():.1f}, "
              f"median {grid.loc[_have,'depth_m'].median():.1f}, "
              f"max {grid.loc[_have,'depth_m'].max():.1f}")
        print(f"  cells >= MIN_WATER_FRACTION ({MIN_WATER_FRACTION:g}): "
              f"{int((grid['bathy_water_fraction'] >= MIN_WATER_FRACTION).sum()):,}")
    except Exception as exc:
        grid["depth_m"] = np.nan
        grid["bathy_water_fraction"] = np.nan
        print(f"\nBathymetry sampling failed ({type(exc).__name__}: {exc}); "
              "depth_m / bathy_water_fraction set to NaN.")
else:
    grid["depth_m"] = np.nan
    grid["bathy_water_fraction"] = np.nan
    print(f"\nNo bathymetry raster found at BATHYMETRY_RASTER={BATHYMETRY_RASTER}.")
    print("Upload Lake_Victoria_Analytical_ras.tif and point BATHYMETRY_RASTER at it to "
          "enable the depth covariate and the bathymetry water mask.")

display(grid[["grid_id", "x_km", "y_km", "depth_m", "bathy_water_fraction"]].head())


## 7. Aggregate classifier-output WH rasters to the grid

This is the main preprocessing step.

For every selected classifier GeoTIFF, the notebook calculates, for each grid cell:

- valid classified pixel count;
- WH / floating-plant pixel count using `WH_CLASS_VALUES`;
- valid classified area;
- WH area;
- WH proportional cover.

The result is a `grid_id × month` panel with source classifier metadata retained for auditability.


In [ ]:
# =====================================================================
# Consolidated panel_raw checkpoint (Drive). Collapses the Section 7
# per-raster reduction loop into one Parquet read on later runs / fresh
# runtimes. Place immediately ABOVE the "## 7. Aggregate ... grid" cell.
# =====================================================================
PANEL_RAW_CHECKPOINT_DIR = OUTPUT_DIR / "panel_checkpoints"
PANEL_RAW_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PANEL_RAW_FORCE_REFRESH = False   # set True to rebuild panel_raw from the rasters

def _panel_raw_fingerprint():
    """Hash everything that defines panel_raw -- and nothing about model tuning."""
    recs = []
    for r in tif_index.itertuples(index=False):
        p = Path(str(r.path))
        pp = getattr(r, "proba_path", None)
        pp = None if (pp is None or (isinstance(pp, float) and pd.isna(pp))
                      or str(pp).strip() in ("", "None", "nan")) else str(pp)
        rec = {"path": str(p),
               "mtime": p.stat().st_mtime_ns if p.exists() else None,
               "size": p.stat().st_size if p.exists() else None,
               "proba": pp}
        if pp and Path(pp).exists():
            rec["proba_mtime"] = Path(pp).stat().st_mtime_ns
            rec["proba_size"] = Path(pp).stat().st_size
        recs.append(rec)
    recs.sort(key=lambda d: d["path"])
    payload = {
        "rasters": recs,
        "cell_size_m": CELL_SIZE_M, "panel_crs": PANEL_CRS,
        "aoi_bbox_wgs84": list(AOI_BBOX_WGS84),
        "n_grid_cells": int(grid["grid_id"].nunique()),
        "grid_bounds": [float(x) for x in grid.total_bounds],
        "wh_class_values": list(WH_CLASS_VALUES),
        "extra_nodata_values": list(EXTRA_NODATA_VALUES),
        "valid_class_values": list(VALID_CLASS_VALUES),
        "use_probability_response": bool(USE_PROBABILITY_RESPONSE),
        "proba_nodata_value": PROBA_NODATA_VALUE, "proba_scale": PROBA_SCALE,
        "sensor_filter": sorted(CLASSIFIER_SENSOR_FILTER) if CLASSIFIER_SENSOR_FILTER else None,
        # Provenance is part of the identity of panel_raw: a checkpoint built
        # from a different classifier run, export schema or run log must never be
        # reloaded in place of this one.
        "classifier_version": ACTIVE_CLASSIFIER_VERSION,
        "classifier_run_log": str(CLASSIFIER_RUN_LOG_PATH),
        "export_tokens": dict(sorted(REQUIRED_EXPORT_TOKEN_BY_SENSOR.items())),
        "classified_prefixes": dict(sorted(REQUIRED_CLASSIFIED_PREFIX_BY_SENSOR.items())),
        "product_filter": CLASSIFIER_PRODUCT_FILTER,
        "prefer_patch_cleaned_s1": bool(PREFER_PATCH_CLEANED_S1),
        "supplement_run_log_with_folder": bool(SUPPLEMENT_RUN_LOG_WITH_FOLDER),
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:20]

PANEL_RAW_CKPT_PATH = PANEL_RAW_CHECKPOINT_DIR / f"panel_raw_{_panel_raw_fingerprint()}.parquet"

PANEL_RAW_FROM_CHECKPOINT = False
if PANEL_RAW_CKPT_PATH.exists() and not PANEL_RAW_FORCE_REFRESH:
    panel_raw = pd.read_parquet(PANEL_RAW_CKPT_PATH)
    panel_raw["month"] = pd.to_datetime(panel_raw["month"])
    validate_panel_provenance(panel_raw, context=f"panel_raw checkpoint {PANEL_RAW_CKPT_PATH.name}")
    PANEL_RAW_FROM_CHECKPOINT = True
    print(f"Loaded panel_raw from checkpoint ({len(panel_raw):,} rows): {PANEL_RAW_CKPT_PATH.name}")
    print(f"-> Provenance validated: {ACTIVE_CLASSIFIER_VERSION}")
    print("-> Skipping the Section 7 raster-to-grid reduction loop.")
else:
    print("No matching panel_raw checkpoint -- the reduction loop will run, then cache its result.")

In [ ]:
if not PANEL_RAW_FROM_CHECKPOINT:
    panel_parts = []
    # Provenance carried onto every panel row as source_* columns, so a built
    # panel can be re-validated without the run log (validate_panel_provenance).
    metadata_cols = PANEL_SOURCE_METADATA_COLUMNS

    for row in tqdm(tif_index.itertuples(index=False), total=len(tif_index)):
        proba_path = getattr(row, "proba_path", None)
        use_proba = (USE_PROBABILITY_RESPONSE and proba_path is not None
                    and not (isinstance(proba_path, float) and pd.isna(proba_path))
                    and str(proba_path).strip() not in ("", "None", "nan")
                    and Path(str(proba_path)).exists())
        cache_path = None
        df = None
        if USE_GRIDDED_RASTER_CACHE and not GRIDDED_RASTER_FORCE_REFRESH:
            cache_path = _gridded_raster_cache_path(
                row.path, row.month, sensor=getattr(row, "sensor", None),
                proba_path=str(proba_path) if use_proba else None,
            )
            df = load_cached_gridded_raster(cache_path)
        if df is None:
            df = aggregate_one_raster_to_grid(
                tif_path=row.path,
                month=row.month,
                grid_gdf=grid,
                panel_crs=PANEL_CRS,
                wh_class_values=WH_CLASS_VALUES,
                extra_nodata_values=EXTRA_NODATA_VALUES,
                valid_class_values=VALID_CLASS_VALUES,
            )
            if len(df) > 0 and use_proba:
                try:
                    soft = aggregate_one_proba_raster_to_grid(
                        class_path=row.path,
                        proba_path=str(proba_path),
                        month=row.month,
                        grid_gdf=grid,
                        panel_crs=PANEL_CRS,
                        wh_class_values=WH_CLASS_VALUES,
                        extra_nodata_values=EXTRA_NODATA_VALUES,
                        valid_class_values=VALID_CLASS_VALUES,
                        proba_nodata_value=PROBA_NODATA_VALUE,
                        proba_scale=PROBA_SCALE,
                    )
                    if len(soft) > 0:
                        df = df.merge(soft, on="grid_id", how="left")
                except Exception as exc:
                    print(f"  proba aggregation skipped for {Path(row.path).name}: "
                          f"{type(exc).__name__}: {exc}")
            if len(df) > 0 and USE_GRIDDED_RASTER_CACHE:
                cache_path = _gridded_raster_cache_path(
                    row.path, row.month, sensor=getattr(row, "sensor", None),
                    proba_path=str(proba_path) if use_proba else None,
                )
                save_cached_gridded_raster(df, cache_path)
        if len(df) > 0:
            row_dict = row._asdict()
            stamp_source_metadata(df, row_dict, metadata_cols)
            # Clean per-observation sensor tag (S2/S1) for the Section 7b fusion; the
            # joined-string source_sensor is kept for provenance. Defaults to the
            # primary sensor if discovery could not determine it (folder fallback).
            df["sensor"] = _normalise_classifier_sensor(row_dict.get("sensor")) or PRIMARY_SENSOR
            panel_parts.append(df)

    if len(panel_parts) == 0:
        raise RuntimeError("No panel rows were created. Check raster CRS, AOI, class codes and no-data settings.")

    panel_raw = pd.concat(panel_parts, ignore_index=True)
    validate_panel_provenance(panel_raw, context="panel_raw built from the classified rasters")

    print(f"Raw panel rows before duplicate-month handling: {len(panel_raw):,}")
display(panel_raw.head())


In [ ]:
if not PANEL_RAW_FROM_CHECKPOINT:
    panel_raw.to_parquet(PANEL_RAW_CKPT_PATH, index=False)
    print(f"Saved panel_raw checkpoint: {PANEL_RAW_CKPT_PATH}")

### Combine duplicate observations within the same month

If the classifier output has only one selected raster per month, this step will not change much.

If there are multiple selected snapshots in the same month, the default behaviour is to take the **mean** grid-cell WH cover/area across those rasters. You can change this to `max` in the configuration cell.


In [ ]:
if DUPLICATE_MONTH_METHOD not in {"mean", "max"}:
    raise ValueError("DUPLICATE_MONTH_METHOD must be either 'mean' or 'max'.")

source_agg = lambda x: "; ".join(sorted(set(map(str, x.dropna()))))
source_metadata_cols = [c for c in panel_raw.columns if c.startswith("source_")]

# Reduce duplicates WITHIN each sensor (e.g. several S2 snapshots in one month),
# but keep S1 and S2 as SEPARATE observations of the same cell-month so they can
# be harmonised -- not blended -- in the Section 7b cloud-gap-fill fusion.
agg_funcs = {
    "valid_pixels": DUPLICATE_MONTH_METHOD,
    "wh_pixels": DUPLICATE_MONTH_METHOD,
    "valid_area_m2": DUPLICATE_MONTH_METHOD,
    "wh_area_m2": DUPLICATE_MONTH_METHOD,
    "wh_cover": DUPLICATE_MONTH_METHOD,
    "source_file": source_agg,
}
for col in source_metadata_cols:
    agg_funcs[col] = source_agg

# Carry the confidence-weighted response columns through the monthly aggregation.
for _soft_col in ["wh_cover_soft", "wh_conf_mean"]:
    if _soft_col in panel_raw.columns:
        agg_funcs[_soft_col] = "mean"

# Ensure every observation carries a sensor tag (defensive for folder fallback).
if "sensor" not in panel_raw.columns:
    panel_raw["sensor"] = PRIMARY_SENSOR
panel_raw["sensor"] = panel_raw["sensor"].fillna(PRIMARY_SENSOR)

panel_sensor = (
    panel_raw
    .groupby(["grid_id", "month", "sensor"], as_index=False)
    .agg(agg_funcs)
)

# Retain only minimally valid cell-months (per sensor).
panel_sensor = panel_sensor[panel_sensor["valid_pixels"] >= MIN_VALID_PIXELS_PER_CELL_MONTH].copy()

# Basic response variables (per sensor).
panel_sensor["wh_area_ha"] = panel_sensor["wh_area_m2"] / 10_000
panel_sensor["valid_area_ha"] = panel_sensor["valid_area_m2"] / 10_000
# Fractional valid coverage of the nominal cell area (1.0 == fully observed).
panel_sensor["valid_fraction"] = (panel_sensor["valid_area_m2"] / (CELL_SIZE_M ** 2)).clip(upper=1.0)

# --- Confidence-weighted (probabilistic) response per sensor (Priority 2) -----
# Keep hard-class cover/area for reference, then -- if enabled and available --
# swap the modelling response to the confidence-weighted cover (same rule as
# before, now applied within each sensor so S1 keeps its own soft response).
panel_sensor["wh_cover_hard"] = panel_sensor["wh_cover"]
panel_sensor["wh_area_ha_hard"] = panel_sensor["wh_area_ha"]
if (USE_PROBABILITY_RESPONSE and "wh_cover_soft" in panel_sensor.columns
        and panel_sensor["wh_cover_soft"].notna().any()):
    panel_sensor["wh_cover"] = panel_sensor["wh_cover_soft"].where(
        panel_sensor["wh_cover_soft"].notna(), panel_sensor["wh_cover_hard"])
    panel_sensor["wh_area_m2"] = panel_sensor["wh_cover"] * panel_sensor["valid_area_m2"]
    panel_sensor["wh_area_ha"] = panel_sensor["wh_area_m2"] / 10_000
    print("Response: CONFIDENCE-WEIGHTED WH cover per sensor "
          "(USE_PROBABILITY_RESPONSE=True); hard-class cover retained as wh_cover_hard.")
else:
    print("Response: HARD-CLASS WH cover per sensor (probability rasters unavailable or disabled).")

# Presence-definition label. Presence itself is computed ONCE on the merged panel
# in Section 7b (after the S1 gap-fill), so it is defined in a single place.
if PRESENCE_AREA_HA_THRESHOLD is not None:
    presence_definition = f"wh_area_ha >= {PRESENCE_AREA_HA_THRESHOLD} ha"
else:
    presence_definition = f"wh_cover >= {PRESENCE_COVER_THRESHOLD:g} (fractional cover)"

print(f"\nPer-sensor panel rows (before fusion): {len(panel_sensor):,}")
_per_sensor = panel_sensor.groupby("sensor").agg(
    cell_months=("grid_id", "size"),
    months=("month", "nunique"),
    cells=("grid_id", "nunique"),
    mean_cover=("wh_cover", "mean"),
).reset_index()
display(_per_sensor)
display(panel_sensor.head())

## 7b. Sentinel-1 cloud-gap-fill fusion (prefer-S2, gap-fill-S1) — Task 1

This section turns the per-sensor panel into a single harmonised response:

1. **Agreement (1b):** on cell-months observed by *both* sensors, report mean/median bias `S1 − S2`, Pearson & Spearman correlation, a Bland–Altman plot, and a presence confusion matrix at the current presence definition.
2. **Calibration (1c):** fit `S2_cover ~ S1_cover` on the overlap (Theil–Sen robust-linear by default; isotonic optional) and apply it to S1-only cell-months, so S1 enters on the S2 measurement scale.
3. **Merge (1d):** build `panel` by **preferring S2** and gap-filling with calibrated S1 *only where S2 is missing* for that cell-month. A `sensor` label and a numeric `sensor_is_s1` indicator are carried so any residual cross-sensor offset is absorbed by the model rather than attributed to a driver.

The MNAR payoff (1e) — whether the S1 fills concentrate in high-rainfall months — is reported in Section 9b, once the CHIRPS rainfall covariate has been merged.


In [ ]:
# Harmonise the per-sensor panel into a single cloud-gap-filled response.
PRIMARY = PRIMARY_SENSOR
SECONDARY = "S1" if PRIMARY == "S2" else "S2"

available_sensors = sorted(s for s in panel_sensor["sensor"].dropna().unique().tolist())
print("Sensors present in the per-sensor panel:", available_sensors)

prim = panel_sensor[panel_sensor["sensor"] == PRIMARY].copy()
sec = panel_sensor[panel_sensor["sensor"] == SECONDARY].copy()

# Month-coverage BEFORE fusion (primary sensor only) -- the "before" acceptance.
cov_before_cellmonths = int(len(prim))
cov_before_months = int(prim["month"].nunique())

do_fusion = bool(ENABLE_S1_GAPFILL and len(prim) > 0 and len(sec) > 0)
calib_info = {"method": "disabled"}

if not do_fusion:
    base = prim if len(prim) > 0 else panel_sensor.copy()
    panel = base.copy()
    if "sensor" not in panel.columns:
        panel["sensor"] = PRIMARY
    panel["sensor_is_s1"] = (panel["sensor"] == "S1").astype(int)
    panel["wh_cover_raw_sensor"] = panel["wh_cover"]
    print(f"\nS1 gap-fill disabled or unavailable -- using {PRIMARY} only "
          f"({cov_before_cellmonths:,} cell-months, {cov_before_months} months).")
else:
    # ---- (1b) Cross-sensor agreement on overlap cell-months -----------------
    overlap = sensor_overlap_frame(panel_sensor, "wh_cover", PRIMARY, SECONDARY)
    prim_c = overlap[f"wh_cover_{PRIMARY}"]
    sec_c = overlap[f"wh_cover_{SECONDARY}"]
    stats = sensor_agreement_stats(prim_c, sec_c)
    print("\n--- (1b) S1 vs S2 agreement on overlap cell-months ---")
    print(f"Overlap cell-months (both sensors valid): {stats['n_pairs']:,}")
    print(f"mean(S1 - S2) cover = {stats['mean_sec_minus_prim']:+.4f}   "
          f"median = {stats['median_sec_minus_prim']:+.4f}")
    print(f"Pearson r = {stats['pearson_r']:.3f}   Spearman r = {stats['spearman_r']:.3f}")

    if stats["n_pairs"] > 0:
        prim_pres = wh_presence_from_cover(overlap[f"wh_cover_{PRIMARY}"],
                                           overlap[f"wh_area_ha_{PRIMARY}"]).astype(int)
        sec_pres = wh_presence_from_cover(overlap[f"wh_cover_{SECONDARY}"],
                                          overlap[f"wh_area_ha_{SECONDARY}"]).astype(int)
        cm = confusion_matrix(prim_pres, sec_pres, labels=[0, 1])
        print(f"\nPresence confusion (rows = S2 obs, cols = S1 obs; {presence_definition}):")
        display(pd.DataFrame(cm, index=["S2_absent", "S2_present"],
                             columns=["S1_absent", "S1_present"]))
        _agree = float(np.trace(cm) / cm.sum()) if cm.sum() else np.nan
        print(f"Overall presence agreement: {_agree:.3f}")

        # Bland-Altman: difference vs mean of the two sensors' cover.
        m = (prim_c.to_numpy() + sec_c.to_numpy()) / 2.0
        d = sec_c.to_numpy() - prim_c.to_numpy()
        md, sd = float(np.nanmean(d)), float(np.nanstd(d))
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.scatter(m, d, s=8, alpha=0.3)
        ax.axhline(md, color="red", label=f"mean {md:+.3f}")
        ax.axhline(md + 1.96 * sd, color="grey", ls="--", label=f"+1.96sd {md + 1.96 * sd:+.3f}")
        ax.axhline(md - 1.96 * sd, color="grey", ls="--", label=f"-1.96sd {md - 1.96 * sd:+.3f}")
        ax.set_xlabel("Mean of S1 and S2 WH cover")
        ax.set_ylabel("S1 - S2 WH cover")
        ax.set_title("Bland-Altman: S1 vs S2 WH cover (overlap cell-months)")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.show()

    # ---- (1c) Calibrate S1 cover onto the S2 scale --------------------------
    calib_f, calib_info = fit_secondary_to_primary_calibration(
        sec_c.to_numpy(), prim_c.to_numpy(),
        method=S1_CALIBRATION_METHOD, random_state=RANDOM_STATE)
    print("\n--- (1c) S1 -> S2 cover calibration ---")
    print(f"Calibration: {calib_info}")

    # ---- (1d) Merge: prefer S2, gap-fill with calibrated S1 -----------------
    prim_keep = prim.copy()
    prim_keep["sensor"] = PRIMARY
    prim_keep["sensor_is_s1"] = 0
    prim_keep["wh_cover_raw_sensor"] = prim_keep["wh_cover"]

    sec_idx = sec.set_index(["grid_id", "month"])
    prim_index = prim_keep.set_index(["grid_id", "month"]).index
    sec_fill = sec_idx.loc[~sec_idx.index.isin(prim_index)].reset_index()
    sec_fill["sensor"] = SECONDARY
    sec_fill["sensor_is_s1"] = 1
    sec_fill["wh_cover_raw_sensor"] = sec_fill["wh_cover"]
    # Put S1 cover on the S2 scale and recompute its WH area accordingly.
    sec_fill["wh_cover"] = calib_f(sec_fill["wh_cover"].to_numpy())
    sec_fill["wh_area_m2"] = sec_fill["wh_cover"] * sec_fill["valid_area_m2"]
    sec_fill["wh_area_ha"] = sec_fill["wh_area_m2"] / 10_000

    panel = pd.concat([prim_keep, sec_fill], ignore_index=True)
    print("\n--- (1d) Merged response (prefer S2, gap-fill calibrated S1) ---")
    print(f"S2 cell-months kept:            {len(prim_keep):,}")
    print(f"S1 gap-fill cell-months added:  {len(sec_fill):,} "
          f"(S2-missing cell-months recovered by S1)")

# ---- Finalise the merged panel: grid attributes, presence, coverage ---------
# Re-attach the static grid attributes (coords, depth, water fraction, ...).
panel = panel.drop(
    columns=[c for c in grid.columns if c not in ("grid_id", "geometry") and c in panel.columns],
    errors="ignore")
panel = panel.merge(grid.drop(columns="geometry"), on="grid_id", how="left")
panel = panel.sort_values(["grid_id", "month"]).reset_index(drop=True)
if "sensor_is_s1" not in panel.columns:
    panel["sensor_is_s1"] = 0
panel["wh_present"] = wh_presence_from_cover(panel["wh_cover"], panel.get("wh_area_ha"))

# Month-coverage AFTER fusion -- the "after" acceptance.
cov_after_cellmonths = int(len(panel))
cov_after_months = int(panel["month"].nunique())
print("\n--- Panel month-coverage: BEFORE vs AFTER S1 gap-fill ---")
print(f"Before (S2 only):      {cov_before_cellmonths:,} cell-months, {cov_before_months} months")
print(f"After  (S2 + S1 fill): {cov_after_cellmonths:,} cell-months, {cov_after_months} months")
_gain = cov_after_cellmonths - cov_before_cellmonths
if cov_before_cellmonths:
    print(f"Net cell-months recovered by S1: {_gain:,} "
          f"({_gain / cov_before_cellmonths * 100:.1f}% of the S2-only panel)")
if "sensor" in panel.columns:
    print("\nCell-months by response sensor:")
    display(panel.groupby("sensor").size().rename("n_cell_months").reset_index())

# Per-month coverage comparison (S2 cells vs S1-filled cells vs merged cells).
_cov_by_month = (panel.groupby("month")
                 .agg(merged_cells=("grid_id", "nunique"),
                      s1_filled_cells=("sensor_is_s1", "sum")).reset_index())
_s2_by_month = prim.groupby("month")["grid_id"].nunique().rename("s2_cells")
_cov_by_month = _cov_by_month.merge(_s2_by_month, on="month", how="left")
_cov_by_month["s2_cells"] = _cov_by_month["s2_cells"].fillna(0).astype(int)
_cov_by_month = _cov_by_month[["month", "s2_cells", "s1_filled_cells", "merged_cells"]]
print("\nPer-month coverage (S2 cells vs S1-filled cells vs merged cells):")
display(_cov_by_month)

print(f"\nWH presence defined as: {presence_definition}")
print(f"Overall WH-present prevalence (merged panel): {panel['wh_present'].mean():.4f} "
      f"({int(panel['wh_present'].sum()):,} of {len(panel):,} cell-months)")
display(panel.head())


### Sensitivity of the WH-present definition to the chosen threshold

The response variable depends strongly on how "present" is defined. The table
below shows how the number of positive cell-months, the positive rate, the mean
WH signal among positives, and the number of months with at least one positive
cell change as the fractional-cover presence threshold is varied. Threshold `0`
reproduces the original "any non-zero WH pixel" definition.


In [ ]:
presence_threshold_grid = [0, 0.001, 0.005, 0.01, 0.02, 0.05]

sensitivity_rows = []
for thr in presence_threshold_grid:
    # thr == 0 reproduces the original "any non-zero WH" definition.
    present = panel["wh_cover"] > 0 if thr == 0 else panel["wh_cover"] >= thr
    n_pos = int(present.sum())
    sensitivity_rows.append({
        "cover_threshold": thr,
        "n_positive_cell_months": n_pos,
        "pct_positive": round(100 * present.mean(), 3),
        "mean_cover_among_pos": round(panel.loc[present, "wh_cover"].mean(), 4) if n_pos else np.nan,
        "mean_area_ha_among_pos": round(panel.loc[present, "wh_area_ha"].mean(), 4) if n_pos else np.nan,
        "n_months_with_positive": int(panel.loc[present, "month"].nunique()),
    })

presence_sensitivity = pd.DataFrame(sensitivity_rows)
print("Sensitivity of WH-present prevalence to the cover threshold:")
display(presence_sensitivity)


## Classifier accuracy and response measurement error

The response is a classifier output, so its accuracy and per-cell confidence are reported here rather than assumed perfect (Priority 2).

In [ ]:
# ---------------------------------------------------------------------
# Classifier accuracy and response measurement-error (Priority 2)
# ---------------------------------------------------------------------
# The panel response is itself a classifier output, so its error should be
# surfaced rather than treated as ground truth. This cell (a) reports any
# cross-validation accuracy metrics the classifier notebook logged, and (b)
# summarises the per-cell classification confidence and the hard-vs-soft cover
# agreement, which quantify the measurement error entering the response.

# (a) Classifier CV accuracy metrics, if the classifier notebook saved a table.
_metric_tables = []
try:
    for _patt in ("*model_selection*metrics*.csv", "*metrics*.csv", "*cv*metrics*.csv",
                  "*_metrics_*.csv"):
        _metric_tables += list(Path(CLASSIFIER_TABLE_DIR).glob(_patt))
except Exception:
    pass
_metric_tables = sorted(set(_metric_tables), key=lambda p: p.stat().st_mtime if p.exists() else 0)
if _metric_tables:
    _mt = _metric_tables[-1]
    print(f"Classifier accuracy metrics from: {_mt.name}")
    try:
        _mdf = pd.read_csv(_mt)
        _show = [c for c in ["sensor", "model", "method", "accuracy", "balanced_accuracy",
                             "kappa", "macro_f1", "weighted_f1"] if c in _mdf.columns]
        display(_mdf[_show] if _show else _mdf.head())
    except Exception as exc:
        print("  could not read metrics table:", exc)
else:
    print("No classifier CV-metrics CSV found in CLASSIFIER_TABLE_DIR.")
    print("  (Look for the model-selection metrics exported by "
          "Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb; accuracy/kappa/macro-F1 "
          "should be quoted alongside these panel results.)")

# (b) Response confidence + hard-vs-soft agreement, from the panel.
if "wh_conf_mean" in panel.columns and panel["wh_conf_mean"].notna().any():
    _c = panel["wh_conf_mean"].dropna()
    print(f"\nPer-cell WH classification confidence (winning-class P over WH pixels), "
          f"n={len(_c):,}:")
    display(_c.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(3))
    _low = (_c < 0.6).mean()
    print(f"Share of WH-bearing cell-months with mean confidence < 0.60: {_low:.1%} "
          "(these contribute most of the response measurement error).")
else:
    print("\nNo per-cell confidence available (probability rasters not aggregated).")

if {"wh_cover_hard", "wh_cover_soft"}.issubset(panel.columns) and panel["wh_cover_soft"].notna().any():
    _m = panel[["wh_cover_hard", "wh_cover_soft"]].dropna()
    if len(_m) > 2:
        _corr = _m["wh_cover_hard"].corr(_m["wh_cover_soft"])
        _bias = (_m["wh_cover_soft"] - _m["wh_cover_hard"]).mean()
        print(f"\nHard vs confidence-weighted cover: r={_corr:.3f}, "
              f"mean(soft - hard)={_bias:+.4f} "
              f"(soft is typically slightly lower, having down-weighted low-confidence WH pixels).")
        _ha = panel["valid_area_m2"]
        _area_hard = float((panel["wh_cover_hard"] * _ha / 1e4).sum())
        _area_soft = float((panel["wh_cover_soft"].fillna(panel["wh_cover_hard"]) * _ha / 1e4).sum())
        print(f"Panel-total WH area: hard={_area_hard:,.0f} ha, "
              f"confidence-weighted={_area_soft:,.0f} ha "
              f"({_area_soft / _area_hard:.0%} of hard).")
        print(f"Modelling response in use: "
              f"{'confidence-weighted (soft)' if USE_PROBABILITY_RESPONSE else 'hard-class'} cover.")
else:
    print("\nConfidence-weighted cover not available; the response uses hard-class cover.")


## 8. Earth Engine environmental covariate extraction

This step pulls the environmental driver layers that have usable Google Earth Engine assets and reduces them onto the panel. It runs when `USE_EARTH_ENGINE = True` (configured in section 3) and is wrapped in a `try/except` so the notebook still completes if Earth Engine is unavailable.

Three tables are produced:

| Output | Keyed by | Layers |
| --- | --- | --- |
| `ee_monthly_cov` | `month` | Wind (ERA5 u/v, speed, direction, bay-axis components), air temperature (ERA5-Land where it covers the AOI, else ERA5), optional MODIS ocean-colour chlorophyll-a (off by default) |
| `ee_cellmonth_cov` | `grid_id`, `month` | Rainfall (CHIRPS ~5 km, per cell by default: total, antecedent sums, spike counts, Rx1day, wet days, SDII); MODIS LST **water-surface temperature** (~1 km, per cell by default); Sentinel-2 **turbidity** (NDTI + red reflectance) proxy; **Sentinel-3 OLCI chlorophyll-a** (mg/m³) via MCI and MPH band-difference algorithms calibrated per Kravitz et al. (2020); all water-masked |
| `ee_static_cov` | `grid_id` | Distance to rivers and to major (discharge-weighted) rivers (HydroSHEDS), distance to shore and openness (JRC GSW), catchment pressure (ESA WorldCover fractions, WorldPop population, GHSL built-up) |

**Resolution strategy.** Coarse reanalysis climate layers (ERA5 wind & air temperature, ~28 km) are extracted as **AOI-mean monthly series**, so they enter as month-level covariates. Air temperature defaults to **ERA5-Land** (~11 km, lapse-rate corrected) where it covers the AOI, falling back to ERA5; wind stays on ERA5, whose 10 m field ERA5-Land only interpolates. MODIS LST water-surface temperature (~1 km) and CHIRPS rainfall (~5 km) -- both per cell by default (`EE_WATER_TEMP_PER_CELL`, `EE_RAIN_PER_CELL`) -- and the Sentinel-2 water-quality proxies (~20 m) are extracted **per grid cell and month**, because they carry real within-gulf spatial signal. Static layers are extracted **once per cell**.

**Derived mechanistic interactions.** `wave_exposure_idx` (openness x wind^2, a local physical-disturbance force) and `nutrient_pulse_idx` (antecedent rain x cropland, the storm-fertilisation pulse, entered lagged) are computed from the merged covariates so the disturbance and nutrient mechanisms are encoded explicitly, not left to the main effects alone.

**Sentinel-3 OLCI chlorophyll-a.** GEE's OLCI asset provides only TOA radiances (no geometry/meteorology tie-points), so the best atmospheric corrections in Kravitz et al. (2020) — BRR and 6SV1 — cannot be reproduced inside GEE. Their key result is that the **MCI and MPH band-difference algorithms are robust to atmospheric correction**, and they publish TOA-reflectance calibrations (Table B.1). The notebook therefore computes OLCI TOA reflectance (nominal solar irradiance + analytically reconstructed solar zenith) and retrieves chl-a with **MCI** (their best TOA method, R²=0.53) and **MPH** (best overall, R²=0.55; its BRR calibration is applied to TOA reflectance as a documented approximation). For publication-grade chl-a, process BRR/6SV externally (SNAP / Py6S) and ingest via the CSV hook in section 9.

**Layers that need a CSV instead (no standard EE asset):** lake level (DAHITI / Hydroweb / G-REALM altimetry), ENSO/IOD indices, bathymetry/depth, and in-situ nutrients (TN, TP, DIN, SRP). Supply these through `ENV_MONTHLY_CSV` / `SPATIAL_COVARIATES_CSV` in section 9.

**Caching to Drive.** Earth Engine extraction is the slow step here, so the covariate tables are cached. The first successful run writes them to `EE_CACHE_DIR` (defaults to `OUTPUT_DIR`) alongside a small JSON manifest, and every later run reloads them from Drive instead of querying Earth Engine — so the per-cell reductions only run once per configuration. The cache is keyed by a fingerprint of the settings that change the covariates (cell size, AOI, test period, the panel's month set and the `EE_*` parameters), so it rebuilds automatically when any of those change. Set `EE_FORCE_REFRESH = True` (or delete the `ee_*_covariates_*` files) to force a fresh pull, or `USE_EE_CACHE = False` to bypass the cache entirely.

> The first run on a fresh Colab runtime triggers `ee.Authenticate()`. Per-cell reductions scale with the number of grid cells, so prefer `CELL_SIZE_M = 1000` for Earth Engine test runs, or turn off `waterquality_s2`/`waterquality_s3` in `EE_LAYERS` for the quickest pass.

In [ ]:
# Initialise the EE covariate containers so later cells work even when
# USE_EARTH_ENGINE is False or extraction fails.
ee_monthly_cov = None
ee_cellmonth_cov = None
ee_static_cov = None
ee_registry = {"monthly": [], "cellmonth": [], "static": []}

# ---------------------------------------------------------------------
# Drive cache helpers for the Earth Engine covariates
# ---------------------------------------------------------------------
# Earth Engine extraction is the slow step in this notebook. The covariate
# tables are written to EE_CACHE_DIR after the first successful run and reloaded
# on later runs (see USE_EE_CACHE / EE_FORCE_REFRESH in section 3), so the
# per-cell Earth Engine reductions only happen once per configuration. A small
# JSON manifest records which tables exist plus a fingerprint of the settings
# that change the covariates, so the cache is rebuilt automatically when any of
# those settings change.
import hashlib
import json

_EE_CACHE_TAG = f"{CELL_SIZE_M}m_{TEST_START}_to_{TEST_END}"
EE_CACHE_PATHS = {
    "monthly":   Path(EE_CACHE_DIR) / f"ee_monthly_covariates_{_EE_CACHE_TAG}.csv",
    "cellmonth": Path(EE_CACHE_DIR) / f"ee_cellmonth_covariates_{_EE_CACHE_TAG}.csv",
    "static":    Path(EE_CACHE_DIR) / f"ee_static_covariates_{_EE_CACHE_TAG}.csv",
}
EE_CACHE_MANIFEST = Path(EE_CACHE_DIR) / f"ee_covariates_cache_manifest_{_EE_CACHE_TAG}.json"


def _ee_cache_signature(months):
    """Fingerprint the configuration that determines the EE covariate values."""
    payload = {
        "cell_size_m": CELL_SIZE_M,
        "panel_crs": PANEL_CRS,
        "aoi_bbox_wgs84": list(AOI_BBOX_WGS84),
        "n_grid_cells": int(grid["grid_id"].nunique()),
        "months": [str(pd.Timestamp(m).date()) for m in months],
        "ee_layers": {k: bool(v) for k, v in EE_LAYERS.items()},
        "ee_s2_products": sorted(EE_S2_PRODUCTS),
        "ee_s3_products": sorted(EE_S3_PRODUCTS),
        "ee_rain_antecedent_days": list(EE_RAIN_ANTECEDENT_DAYS),
        "ee_rain_intensity": bool(EE_RAIN_INTENSITY),
        "ee_rain_spike_thresholds_mm": list(EE_RAIN_SPIKE_THRESHOLDS_MM),
        "ee_rain_wet_day_mm": EE_RAIN_WET_DAY_MM,
        "ee_rain_per_cell": bool(EE_RAIN_PER_CELL),
        "ee_bay_axis_bearing_deg": EE_BAY_AXIS_BEARING_DEG,
        "ee_water_occurrence_threshold": EE_WATER_OCCURRENCE_THRESHOLD,
        "ee_s2_cloud_pct": EE_S2_CLOUD_PCT,
        "ee_climate_aoi_scale": EE_CLIMATE_AOI_SCALE,
        "ee_air_temp_use_era5_land": bool(EE_AIR_TEMP_USE_ERA5_LAND),
        "ee_water_temp_per_cell": bool(EE_WATER_TEMP_PER_CELL),
        "ee_catchment_buffer_m": EE_CATCHMENT_BUFFER_M,
        "ee_river_discharge_weighted": bool(EE_RIVER_DISCHARGE_WEIGHTED),
        "ee_river_major_max_ord": EE_RIVER_MAJOR_MAX_ORD,
        "ee_static_year": EE_STATIC_YEAR,
        "ee_s3_calib": {k: list(v) for k, v in EE_S3_CALIB.items()},
    }
    blob = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()


def _registry_from_tables(monthly_df, cellmonth_df, static_df):
    """Rebuild the covariate-group registry from cached table columns."""
    return {
        "monthly":   [] if monthly_df is None else [c for c in monthly_df.columns if c != "month"],
        "cellmonth": [] if cellmonth_df is None else [c for c in cellmonth_df.columns if c not in ("grid_id", "month")],
        "static":    [] if static_df is None else [c for c in static_df.columns if c != "grid_id"],
    }


def _read_cached_table(path):
    df = pd.read_csv(path)
    if "month" in df.columns:
        df["month"] = pd.to_datetime(df["month"]).dt.to_period("M").dt.to_timestamp()
    return df


def save_ee_cache(monthly_df, cellmonth_df, static_df, registry, signature,
                  present_groups=None, enabled_groups=None):
    """Write the EE covariate tables and a manifest to EE_CACHE_DIR.

    ``present_groups`` records which covariate groups this cache actually holds
    and ``enabled_groups`` which the configuration asks for. A group that failed
    to build is simply absent from ``present_groups``, so a later run reloads the
    cached groups and re-attempts only the missing ones (see load_ee_cache) rather
    than rebuilding every group. They are omitted from the manifest when unknown
    (legacy caches), which load treats as a complete cache."""
    Path(EE_CACHE_DIR).mkdir(parents=True, exist_ok=True)
    written = {}
    for key, table in (("monthly", monthly_df),
                       ("cellmonth", cellmonth_df),
                       ("static", static_df)):
        path = EE_CACHE_PATHS[key]
        if table is not None and len(table) > 0:
            table.to_csv(path, index=False)
            written[key] = path.name
        elif path.exists():
            path.unlink()  # drop a stale table that no longer applies
    manifest = {
        "signature": signature,
        "tag": _EE_CACHE_TAG,
        "registry": registry,
        "tables": written,
        "created_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    }
    if present_groups is not None:
        manifest["present_groups"] = sorted(present_groups)
    if enabled_groups is not None:
        manifest["enabled_groups"] = sorted(enabled_groups)
    EE_CACHE_MANIFEST.write_text(json.dumps(manifest, indent=2))
    return written


def load_ee_cache(signature):
    """Return (monthly, cellmonth, static, registry, present_groups) or None.

    None means there is no usable cache (no manifest, the configuration changed,
    or a listed file is missing), so the caller re-queries Earth Engine.
    ``present_groups`` is the list of covariate groups the cache holds, or None
    when it is not tracked (legacy caches), in which case the caller treats the
    cache as complete. When an older run exported the CSV tables but not the
    manifest, load those legacy tables instead of re-fetching them; the filenames
    are already scoped by cell size and test period.
    """
    if not EE_CACHE_MANIFEST.exists():
        legacy = {key: (_read_cached_table(path) if path.exists() else None)
                  for key, path in EE_CACHE_PATHS.items()}
        if any(table is not None for table in legacy.values()):
            print("Found cached EE covariate CSVs without a manifest; reusing them and writing a manifest.")
            registry = _registry_from_tables(legacy["monthly"], legacy["cellmonth"], legacy["static"])
            save_ee_cache(legacy["monthly"], legacy["cellmonth"], legacy["static"], registry, signature)
            return legacy["monthly"], legacy["cellmonth"], legacy["static"], registry, None
        return None
    try:
        manifest = json.loads(EE_CACHE_MANIFEST.read_text())
    except (ValueError, OSError):
        return None
    if manifest.get("signature") != signature:
        print("Cached Earth Engine covariates are stale (configuration changed); rebuilding.")
        return None
    tables = manifest.get("tables", {})
    loaded = {"monthly": None, "cellmonth": None, "static": None}
    for key in loaded:
        if key in tables:
            path = EE_CACHE_PATHS[key]
            if not path.exists():
                print(f"Cache manifest lists {path.name} but the file is missing; rebuilding.")
                return None
            loaded[key] = _read_cached_table(path)
    registry = manifest.get("registry") or _registry_from_tables(
        loaded["monthly"], loaded["cellmonth"], loaded["static"])
    present_groups = manifest.get("present_groups")
    return loaded["monthly"], loaded["cellmonth"], loaded["static"], registry, present_groups


def _merge_cached(base, extra, keys):
    """Merge freshly-extracted columns into a cached table without losing rows.

    Used when an earlier run cached some covariate groups but not others: the
    newly built group(s) are joined onto the cached table by ``keys`` (outer), with
    the fresh values winning for any overlapping non-key column."""
    if extra is None or len(extra) == 0:
        return base
    if base is None or len(base) == 0:
        return extra
    keys = list(keys)
    overlap = [c for c in extra.columns if c in base.columns and c not in keys]
    if overlap:
        base = base.drop(columns=overlap)
    return base.merge(extra, on=keys, how="outer")


if USE_EARTH_ENGINE:
    panel_months = sorted(panel["month"].unique())
    ee_cache_signature = _ee_cache_signature(panel_months)
    enabled_groups = _ee_enabled_groups()[0]

    cached = load_ee_cache(ee_cache_signature) if (USE_EE_CACHE and not EE_FORCE_REFRESH) else None

    if cached is not None:
        ee_monthly_cov, ee_cellmonth_cov, ee_static_cov, ee_registry, present_groups = cached
        missing = (enabled_groups - set(present_groups)) if present_groups is not None else set()
        if missing:
            # The cache holds some covariate groups but not all the enabled ones
            # (an earlier run failed partway -- e.g. Sentinel-3 errored after
            # Sentinel-2 was already built). Reuse the cached groups instead of
            # rebuilding them, and re-attempt ONLY the missing groups.
            print(f"Loaded {len(present_groups)} cached Earth Engine covariate group(s) from "
                  f"{EE_CACHE_DIR}; re-attempting missing group(s): {sorted(missing)}.")
            try:
                m_new, cm_new, st_new, _, status = extract_ee_covariates(
                    grid, panel_months, only_groups=missing)
                ee_monthly_cov = _merge_cached(ee_monthly_cov, m_new, ["month"])
                ee_cellmonth_cov = _merge_cached(ee_cellmonth_cov, cm_new, ["grid_id", "month"])
                ee_static_cov = _merge_cached(ee_static_cov, st_new, ["grid_id"])
                # Re-derive the idempotent cross-source columns after merging.
                ee_monthly_cov = add_rain_intensity(add_wind_components(ee_monthly_cov))
                ee_cellmonth_cov = add_rain_intensity(ee_cellmonth_cov)
                ee_registry = _registry_from_tables(ee_monthly_cov, ee_cellmonth_cov, ee_static_cov)
                present_groups = sorted(set(present_groups) | status["present"])
                if USE_EE_CACHE:
                    save_ee_cache(ee_monthly_cov, ee_cellmonth_cov, ee_static_cov, ee_registry,
                                  ee_cache_signature, present_groups=present_groups,
                                  enabled_groups=enabled_groups)
                still_missing = enabled_groups - set(present_groups)
                if still_missing:
                    print(f"Still unable to build {sorted(still_missing)}; will retry on the next run.")
                else:
                    print("All enabled Earth Engine covariate groups are now cached.")
            except Exception as exc:
                print("Re-attempt of the missing Earth Engine covariate groups failed; "
                      "using the cached groups only.")
                print(f"Reason: {type(exc).__name__}: {exc}")
        else:
            print(f"Loaded Earth Engine covariates from the Drive cache in {EE_CACHE_DIR}.")
            print("Earth Engine was not queried. Set EE_FORCE_REFRESH = True (or delete the "
                  "ee_*_covariates_* files) to rebuild from Earth Engine.")
        for label, table in (("Monthly climate", ee_monthly_cov),
                             ("Per-cell/month water-quality", ee_cellmonth_cov),
                             ("Static per-cell", ee_static_cov)):
            if table is not None:
                print(f"\n{label} covariates: {table.shape}")
                display(table.head())
    else:
        print(f"Requesting Earth Engine covariates for {len(panel_months)} month(s) "
              f"and {grid['grid_id'].nunique():,} grid cells.")
        if CELL_SIZE_M < 1000:
            print("Note: per-cell Earth Engine reductions scale with cell count. "
                  "If requests are slow or time out, use CELL_SIZE_M = 1000, lower "
                  "EE_MAX_CELLS_PER_REQUEST, or disable the Sentinel-2/3 layers.")
        try:
            ee_monthly_cov, ee_cellmonth_cov, ee_static_cov, ee_registry, status = extract_ee_covariates(
                grid, panel_months
            )
            print("\nEarth Engine covariate extraction complete.")
            if status["failed"]:
                print(f"  Note: these covariate group(s) failed and will be retried on the next "
                      f"run (the groups that succeeded are cached): {sorted(status['failed'])}.")
            if USE_EE_CACHE:
                written = save_ee_cache(ee_monthly_cov, ee_cellmonth_cov, ee_static_cov,
                                        ee_registry, ee_cache_signature,
                                        present_groups=status["present"],
                                        enabled_groups=status["enabled"])
                kinds = ", ".join(written) if written else "no tables"
                print(f"Cached Earth Engine covariates to {EE_CACHE_DIR} ({kinds}); "
                      "later runs reload them instead of re-querying Earth Engine.")
            if ee_monthly_cov is not None:
                print(f"\nMonthly climate covariates: {ee_monthly_cov.shape}")
                display(ee_monthly_cov.head())
            if ee_cellmonth_cov is not None:
                print(f"\nPer-cell/month water-quality covariates: {ee_cellmonth_cov.shape}")
                display(ee_cellmonth_cov.head())
            if ee_static_cov is not None:
                print(f"\nStatic per-cell covariates: {ee_static_cov.shape}")
                display(ee_static_cov.head())
        except Exception as exc:
            print("Earth Engine covariate extraction failed; continuing without EE covariates.")
            print(f"Reason: {type(exc).__name__}: {exc}")
            print("Check ee.Authenticate()/EE_PROJECT, your internet connection, or disable "
                  "individual layers in EE_LAYERS, then re-run this cell.")
            ee_monthly_cov = ee_cellmonth_cov = ee_static_cov = None
            ee_registry = {"monthly": [], "cellmonth": [], "static": []}
else:
    print("USE_EARTH_ENGINE is False. Skipping Earth Engine extraction.")
    print("The notebook will use only the optional CSV covariates (if provided) in section 9.")


## 9. Merge covariates (Earth Engine + optional CSVs)

This is where the environmental-driver variables enter the panel. The classified GeoTIFFs alone produce the WH response variable, but they cannot explain environmental controls unless these covariates are merged in.

Three things are merged here:

1. **Earth Engine covariates** from section 8 (`ee_monthly_cov` on `month`, `ee_static_cov` on `grid_id`, `ee_cellmonth_cov` on `grid_id` + `month`).
2. **Optional monthly CSV** (`ENV_MONTHLY_CSV`) for month-level drivers without an EE asset — lake level, ENSO/IOD, in-situ nutrients.
3. **Optional spatial CSV** (`SPATIAL_COVARIATES_CSV`) for static per-cell drivers without an EE asset — bathymetry/depth, manually derived fetch/shelter.

The merge also builds three column registries used by the feature-engineering step:

- `MONTHLY_COVARIATE_COLS` and `CELLMONTH_COVARIATE_COLS` — **time-varying**, so they receive one-month lags;
- `STATIC_COVARIATE_COLS` — **time-invariant**, so they are used directly without lags.

CSV columns that duplicate an Earth Engine column are dropped (the CSV is only for drivers EE cannot supply), so providing a CSV never silently overwrites an EE covariate.

In [ ]:
# Track covariate groups so feature engineering can lag the time-varying
# covariates but treat static spatial covariates as time-invariant.
MONTHLY_COVARIATE_COLS = []
CELLMONTH_COVARIATE_COLS = []
STATIC_COVARIATE_COLS = []


def _merge_and_record(panel_df, cov_df, on, group_cols):
    """Left-merge a covariate table into the panel and record its new columns."""
    if cov_df is None or len(cov_df) == 0:
        return panel_df
    cov_df = cov_df.copy()
    on_cols = [on] if isinstance(on, str) else list(on)
    if "month" in on_cols and "month" in cov_df.columns:
        cov_df["month"] = pd.to_datetime(cov_df["month"]).dt.to_period("M").dt.to_timestamp()
    new_cols = [c for c in cov_df.columns if c not in on_cols]
    overlap = [c for c in new_cols if c in panel_df.columns]
    if overlap:
        print("  dropping covariate columns already present in the panel:", overlap)
        cov_df = cov_df.drop(columns=overlap)
        new_cols = [c for c in new_cols if c not in overlap]
    panel_df = panel_df.merge(cov_df, on=on_cols, how="left")
    for c in new_cols:
        if c not in group_cols:
            group_cols.append(c)
    return panel_df


# 1) Earth Engine covariates (from section 8).
print("Merging Earth Engine covariates ...")
panel = _merge_and_record(panel, ee_monthly_cov, "month", MONTHLY_COVARIATE_COLS)
panel = _merge_and_record(panel, ee_static_cov, "grid_id", STATIC_COVARIATE_COLS)
panel = _merge_and_record(panel, ee_cellmonth_cov, ["grid_id", "month"], CELLMONTH_COVARIATE_COLS)

# 2) Optional monthly covariates CSV (lake level, ENSO/IOD, nutrients, ...).
if ENV_MONTHLY_CSV is not None:
    env = load_monthly_covariates(ENV_MONTHLY_CSV)
    print("Loaded monthly environmental covariates from CSV:")
    display(env.head())
    panel = _merge_and_record(panel, env, "month", MONTHLY_COVARIATE_COLS)
else:
    print("No ENV_MONTHLY_CSV supplied (use it for lake level, ENSO/IOD, nutrients, etc.).")

# 3) Optional static spatial covariates CSV (bathymetry/depth, fetch, ...).
if SPATIAL_COVARIATES_CSV is not None:
    spatial_cov = pd.read_csv(SPATIAL_COVARIATES_CSV)
    if "grid_id" not in spatial_cov.columns:
        raise ValueError("SPATIAL_COVARIATES_CSV must contain a 'grid_id' column.")
    print("Loaded spatial covariates from CSV:")
    display(spatial_cov.head())
    panel = _merge_and_record(panel, spatial_cov, "grid_id", STATIC_COVARIATE_COLS)
else:
    print("No SPATIAL_COVARIATES_CSV supplied (use it for bathymetry/depth, fetch, etc.).")

# Derived mechanistic interactions, built from the already-merged covariates so
# each one encodes a stated hypothesis (with an a-priori sign) rather than adding
# a new data source. Both vary by cell and month, so they join the per-cell/month
# group and are lagged like the other time-varying covariates.
# 1) Wave-exposure index (disturbance, expected sign -). Wave energy that
#    fragments and piles floating mats scales with fetch x wind^2; openness_index
#    (per-cell open-water fetch proxy) x wind_speed_ms^2 turns static fetch +
#    regional wind into a local, time-varying disturbance force -- the spatial
#    partner to the regional rainfall spike metrics.
if {"openness_index", "wind_speed_ms"}.issubset(panel.columns):
    panel["wave_exposure_idx"] = panel["openness_index"] * panel["wind_speed_ms"] ** 2
    if "wave_exposure_idx" not in CELLMONTH_COVARIATE_COLS:
        CELLMONTH_COVARIATE_COLS.append("wave_exposure_idx")
    print("  derived wave_exposure_idx = openness_index x wind_speed_ms^2")

# 2) Storm nutrient-pulse (growth, expected sign +, lagged). Heavy rain over
#    fertilised cropland flushes N/P into the gulf and fuels growth weeks later;
#    antecedent rain x catchment cropland fraction, with the one-month lag added
#    in feature engineering supplying the delay.
_ante_rain = "rain_chirps_30d_mm" if "rain_chirps_30d_mm" in panel.columns else "rain_chirps_mm"
if _ante_rain in panel.columns and "frac_cropland" in panel.columns:
    panel["nutrient_pulse_idx"] = panel[_ante_rain] * panel["frac_cropland"]
    if "nutrient_pulse_idx" not in CELLMONTH_COVARIATE_COLS:
        CELLMONTH_COVARIATE_COLS.append("nutrient_pulse_idx")
    print(f"  derived nutrient_pulse_idx = {_ante_rain} x frac_cropland")

if not (MONTHLY_COVARIATE_COLS or CELLMONTH_COVARIATE_COLS or STATIC_COVARIATE_COLS):
    print("\nNo environmental covariates were merged.")
    print("The test model will use seasonality, grid coordinates and lagged WH cover only.")

print("\nCovariate groups assembled:")
print("  Monthly (time-varying):       ", MONTHLY_COVARIATE_COLS)
print("  Per-cell/month (time-varying):", CELLMONTH_COVARIATE_COLS)
print("  Static (time-invariant):      ", STATIC_COVARIATE_COLS)
print(f"\nPanel now has {panel.shape[0]:,} rows and {panel.shape[1]} columns.")
display(panel.head())

## 9b. MNAR check: does S1 gap-fill reduce the rainfall confound? — Task 1 (1e)

Sentinel-2 gaps are cloud-driven, and cloud correlates with rainfall, so the S2-only missingness is **missing-not-at-random (MNAR)** with respect to rainfall. If the Sentinel-1 fills concentrate in high-rainfall months, S1 is recovering exactly the rain-correlated observations S2 drops — reducing the confound. This cross-tabulates the S1 fill rate against monthly CHIRPS-rainfall tertiles (now that `rain_chirps_mm` has been merged onto the panel).


In [ ]:
# (1e) Are the Sentinel-1 gap-fills concentrated in high-rainfall months?
_rain_col = "rain_chirps_mm"
if not ("sensor_is_s1" in panel.columns and _rain_col in panel.columns
        and int(panel["sensor_is_s1"].sum()) > 0 and panel[_rain_col].notna().any()):
    print("MNAR check skipped: no S1 gap-fills present, or rain_chirps_mm is unavailable "
          "(Earth Engine rainfall layer off, or EE extraction failed/cached without it).")
else:
    mnar = panel[["grid_id", "month", "sensor_is_s1", _rain_col]].dropna(subset=[_rain_col]).copy()
    month_rain = mnar.groupby("month")[_rain_col].first()
    n_fill = int(mnar["sensor_is_s1"].sum())

    if month_rain.nunique() < 3:
        # Too few distinct monthly-rainfall values for tertiles; report counts only.
        print(f"S2-missing cell-months recovered by S1: {n_fill:,}")
        print("MNAR tertile cross-tab skipped: fewer than 3 distinct monthly-rainfall "
              "values in this panel.")
    else:
        # Month-level rainfall tertiles (one rainfall value per month).
        try:
            month_tertile = pd.qcut(month_rain, 3, labels=["low", "mid", "high"])
        except ValueError:
            # Ties in the rainfall distribution: rank-then-cut as a fallback.
            month_tertile = pd.qcut(month_rain.rank(method="first"), 3,
                                    labels=["low", "mid", "high"])
        mnar["rain_tertile"] = mnar["month"].map(month_tertile)

        tab = mnar.groupby("rain_tertile", observed=True).agg(
            cell_months=("sensor_is_s1", "size"),
            n_s1_filled=("sensor_is_s1", "sum"),
            mean_rain_mm=(_rain_col, "mean"),
        )
        tab["s1_fill_rate"] = tab["n_s1_filled"] / tab["cell_months"]
        print("S1 gap-fill rate by monthly-rainfall tertile:")
        display(tab.round({"mean_rain_mm": 1, "s1_fill_rate": 3}))

        r_s2 = mnar.loc[mnar["sensor_is_s1"] == 0, _rain_col].mean()
        r_s1 = mnar.loc[mnar["sensor_is_s1"] == 1, _rain_col].mean()
        print(f"\nS2-missing cell-months recovered by S1: {n_fill:,}")
        print(f"Mean monthly rainfall: S2-retained cell-months = {r_s2:.1f} mm; "
              f"S1-filled cell-months = {r_s1:.1f} mm")

        hi = tab.loc["high", "s1_fill_rate"] if "high" in tab.index else np.nan
        lo = tab.loc["low", "s1_fill_rate"] if "low" in tab.index else np.nan
        if np.isfinite(hi) and np.isfinite(lo) and hi > lo and r_s1 > r_s2:
            print("\nVerdict: S1 fills concentrate in higher-rainfall months and add wetter-month "
                  "observations, so S1 gap-fill REDUCES the rainfall-driven (MNAR) missingness in "
                  "the S2 response.")
        elif np.isfinite(hi) and np.isfinite(lo) and hi > lo:
            print("\nVerdict: S1 fills lean toward higher-rainfall months (partial MNAR "
                  "mitigation); the mean-rainfall shift is small.")
        else:
            print("\nVerdict: S1 fills are NOT concentrated in high-rainfall months in this panel, "
                  "so gap-fill does not strongly reduce the rainfall confound here.")


## 10. Feature engineering

This creates:

- cyclic seasonal terms;
- linear time index;
- lagged WH cover and presence per grid cell;
- one-month lags for the **time-varying** environmental covariates (monthly climate + per-cell/month water quality + any monthly CSV).

Static spatial covariates (distances, land-cover fractions, population, built-up) are time-invariant, so they are kept as-is and not lagged.

Lagging helps because WH extent in a given month may respond to antecedent rainfall, wind, temperature or water-quality conditions, rather than only same-month conditions.

In [ ]:
panel = panel.sort_values(["grid_id", "month"]).reset_index(drop=True)

# --- Repair mislabeled single-band reducer columns ---------------------------
# reduceRegions on a SINGLE-band image names its output after the REDUCER ("mean"),
# not the band, so single-band layers arrive as a generic "mean" and, when two of
# them meet on a merge, collide into pandas' 'mean_x'/'mean_y' suffixes. Three such
# columns leaked into this panel:
#   * bare 'mean'  = the JRC GSW water-fraction STATIC layer (a water mask). It is
#     meant to be excluded from the model features (see the mask-only static set
#     below), but under the name 'mean' that exclusion misses it and the mask leaks
#     in as a predictor. Rename -> gsw_water_fraction so it is correctly excluded.
#   * 'mean_x'/'mean_y' = two per-cell/month layers (MODIS water-surface temperature
#     plus a second single-band layer carried in an older Drive cache) fed to the
#     model as anonymous features. Their physical identity is not recoverable from
#     code, so drop them by default; set ML_KEEP_MEAN_REDUCER_COLS=True to keep them
#     (renaming mean_x -> water_temp_c on the positional inference that it is the
#     MODIS water-temperature slot).
# reduce_image_over_cells now renames single-band outputs at the source (S4 EE
# helpers), so a fresh cache (EE_FORCE_REFRESH=True) carries every layer under its
# real name (gsw_water_fraction, water_temp_c, ...) and never produces these columns.
if "mean" in panel.columns and "gsw_water_fraction" not in panel.columns:
    panel = panel.rename(columns={"mean": "gsw_water_fraction"})
    STATIC_COVARIATE_COLS = ["gsw_water_fraction" if c == "mean" else c
                             for c in STATIC_COVARIATE_COLS]
    print("Repaired static reducer column: mean -> gsw_water_fraction "
          "(JRC GSW water mask; kept out of the model features).")
ML_KEEP_MEAN_REDUCER_COLS = False
_mean_artifacts = [c for c in ("mean_x", "mean_y") if c in panel.columns]
if _mean_artifacts and "water_temp_c" not in panel.columns:
    if ML_KEEP_MEAN_REDUCER_COLS and "mean_x" in panel.columns:
        panel = panel.rename(columns={"mean_x": "water_temp_c"})
        CELLMONTH_COVARIATE_COLS = ["water_temp_c" if c == "mean_x" else c
                                    for c in CELLMONTH_COVARIATE_COLS]
        panel = panel.drop(columns=[c for c in ["mean_y"] if c in panel.columns])
        CELLMONTH_COVARIATE_COLS = [c for c in CELLMONTH_COVARIATE_COLS if c != "mean_y"]
        print("Kept mean_x -> water_temp_c (positional inference); dropped mean_y.")
    else:
        panel = panel.drop(columns=_mean_artifacts)
        CELLMONTH_COVARIATE_COLS = [c for c in CELLMONTH_COVARIATE_COLS
                                    if c not in _mean_artifacts]
        print(f"Dropped un-provenanced per-cell reducer artifacts {_mean_artifacts} "
              f"(stale-cache 'mean_x'/'mean_y'); rebuild with EE_FORCE_REFRESH=True to "
              f"restore per-cell layers (e.g. water_temp_c) under their real names.")

# Seasonal and time terms.
panel["month_num"] = panel["month"].dt.month
panel["month_sin"] = np.sin(2 * np.pi * panel["month_num"] / 12)
panel["month_cos"] = np.cos(2 * np.pi * panel["month_num"] / 12)

month_min = panel["month"].min()
panel["time_index"] = (
    (panel["month"].dt.year - month_min.year) * 12
    + (panel["month"].dt.month - month_min.month)
)

# Lagged WH terms by spatial unit (#5, this round). Build TRUE t-1 calendar-month
# lags rather than a positional groupby-shift: a positional shift silently reaches
# ACROSS missing months (Sentinel-2 cloud gaps; and, in the old S1 gap-filled panel,
# it also landed on flat S1-derived months -- corrupting the autoregressive signal
# so wh_cover_lag1 barely registered in tree gain). With S1 gap-fill now removed the
# panel is single-sensor; here we additionally require the lag source to be the
# immediately preceding calendar month, else NaN (imputed by the model; the
# persistence baseline treats a missing lag as 0).
LAG_CALENDAR_AWARE = True
if LAG_CALENDAR_AWARE:
    _key_now = list(zip(panel["grid_id"].to_numpy(), panel["month"].to_numpy()))
    _cov_map = dict(zip(_key_now, panel["wh_cover"].to_numpy()))
    _pres_map = dict(zip(_key_now, panel["wh_present"].to_numpy()))
    _prev_month = (panel["month"].dt.to_period("M") - 1).dt.to_timestamp().to_numpy()
    _prev_key = list(zip(panel["grid_id"].to_numpy(), _prev_month))
    panel["wh_cover_lag1"] = [_cov_map.get(k, np.nan) for k in _prev_key]
    panel["wh_present_lag1"] = np.array([_pres_map.get(k, np.nan) for k in _prev_key],
                                        dtype="float")
    _have = int(panel["wh_cover_lag1"].notna().sum()); _n_rows = len(panel)
    print(f"Calendar-aware t-1 lags: {_have:,}/{_n_rows:,} "
          f"({100 * _have / max(_n_rows, 1):.1f}%) rows have a true previous-calendar-month "
          f"lag; {_n_rows - _have:,} first-month/post-gap rows -> NaN (imputed).")
else:
    panel["wh_cover_lag1"] = panel.groupby("grid_id")["wh_cover"].shift(1)
    panel["wh_present_lag1"] = panel.groupby("grid_id")["wh_present"].shift(1).astype("float")

# Neighbour spatial-lag terms (Priority 3): previous month's mean cover/presence
# in adjacent cells (queen/rook contiguity). Already lagged -- not re-lagged.
if NEIGHBOUR_LAG_ENABLED and "grid_neighbours" in globals() and grid_neighbours is not None and len(grid_neighbours):
    panel = add_neighbour_lag(panel, grid_neighbours, value_cols=["wh_cover", "wh_present"])
    neighbour_lag_cols = [c for c in ["wh_cover_neigh_lag1", "wh_present_neigh_lag1"]
                          if c in panel.columns]
    print(f"Neighbour spatial-lag features added: {neighbour_lag_cols}")
else:
    neighbour_lag_cols = []
    print("Neighbour spatial-lag features disabled or no neighbour graph available.")

# Time-varying environmental covariates (monthly climate + per-cell/month water
# quality + any monthly CSV) receive one-month lags. Static spatial covariates do not.
time_varying_cols = [
    c for c in (MONTHLY_COVARIATE_COLS + CELLMONTH_COVARIATE_COLS)
    if c in panel.columns and pd.api.types.is_numeric_dtype(panel[c])
]
# gsw_water_fraction is a water/littoral mask used by the habitat filter, not an
# ecological driver, so it is kept out of the model feature set.
_mask_only_static = {"gsw_water_fraction"}
static_cols = [
    c for c in STATIC_COVARIATE_COLS
    if c in panel.columns and pd.api.types.is_numeric_dtype(panel[c])
    and c not in _mask_only_static
]

for col in time_varying_cols:
    panel[f"{col}_lag1"] = panel.groupby("grid_id")[col].shift(1)

# Backwards-compatible alias for the original variable name.
env_candidate_cols = time_varying_cols

print(f"Time-varying covariates lagged ({len(time_varying_cols)}):")
print(time_varying_cols)
print(f"\nStatic spatial covariates ({len(static_cols)}):")
print(static_cols)

display(panel.head())

## 11. Exploratory checks

Before modelling, check whether WH observations are mostly zero, whether the time series looks plausible, and whether some months have poor coverage.

In [ ]:
monthly_summary = (
    panel.groupby("month", as_index=False)
    .agg(
        wh_area_ha=("wh_area_ha", "sum"),
        valid_area_ha=("valid_area_ha", "sum"),
        mean_cover=("wh_cover", "mean"),
        occurrence_rate=("wh_present", "mean"),
        n_cells=("grid_id", "nunique"),
    )
)

display(monthly_summary)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly_summary["month"], monthly_summary["wh_area_ha"], marker="o")
ax.set_title("Mapped WH area by month")
ax.set_xlabel("Month")
ax.set_ylabel("WH area aggregated across grid cells (ha)")
ax.grid(True, alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly_summary["month"], monthly_summary["occurrence_rate"], marker="o")
ax.set_title("Grid-cell WH occurrence rate by month")
ax.set_xlabel("Month")
ax.set_ylabel("Share of valid grid cells with WH present")
ax.grid(True, alpha=0.3)
plt.show()

print("Overall share of cell-months with WH present:", round(panel["wh_present"].mean(), 4))

## Methodological note: presence definition and probability threshold

The previous model used any non-zero WH area to define presence and used a fixed
0.5 probability threshold. This caused the presence model to be highly
conservative: precision was high but recall was very low, so the model missed
most true WH-present cells and underestimated total WH area. The updated workflow
tests a more ecologically meaningful presence threshold and tunes the probability
threshold using validation data.

Concretely, the changes below:

- define presence with a configurable cover/area threshold instead of "any pixel";
- split the existing training period into fitting and validation months and keep
  the original test months untouched;
- tune the presence probability threshold on the validation months (maximising F1
  by default, with a higher-recall option reported);
- form the final predicted WH area as a two-stage hurdle: the positive-cover model
  is applied only to cells predicted present at the tuned threshold, and zero cover
  is assigned to predicted-absent cells.


## Habitat / valid-cell filter

The panel can contain background cells and cell-months with very few valid
pixels, which dilute presence and push the classifier toward always predicting
absence. The filters below are applied conservatively before modelling:

- the existing minimum valid-pixel filter is retained;
- an optional minimum valid-fraction filter (off by default);
- an optional "ever validly observed" cell filter that keeps only cells with at
  least one valid observation across the panel.

No shoreline/littoral mask is invented here. The number of rows and unique cells
removed by each filter is printed.


In [ ]:
n_rows_before_filter = len(panel)
n_cells_before_filter = panel["grid_id"].nunique()
print(f"Before habitat/valid-cell filters: {n_rows_before_filter:,} rows, "
      f"{n_cells_before_filter:,} unique cells")

# 1. Minimum valid pixels per cell-month (already applied in section 7).
#    Re-assert here so the filter is explicit and its effect is reported.
before_rows, before_cells = len(panel), panel["grid_id"].nunique()
panel = panel[panel["valid_pixels"] >= MIN_VALID_PIXELS_PER_CELL_MONTH].copy()
print(f"[valid-pixel >= {MIN_VALID_PIXELS_PER_CELL_MONTH}] removed "
      f"{before_rows - len(panel):,} rows, {before_cells - panel['grid_id'].nunique():,} cells")

# 2. Optional minimum valid fraction per cell-month.
if MIN_VALID_FRACTION_PER_CELL_MONTH is not None and "valid_fraction" in panel.columns:
    before_rows, before_cells = len(panel), panel["grid_id"].nunique()
    panel = panel[panel["valid_fraction"] >= MIN_VALID_FRACTION_PER_CELL_MONTH].copy()
    print(f"[valid-fraction >= {MIN_VALID_FRACTION_PER_CELL_MONTH}] removed "
          f"{before_rows - len(panel):,} rows, {before_cells - panel['grid_id'].nunique():,} cells")
else:
    print("[valid-fraction] filter disabled (MIN_VALID_FRACTION_PER_CELL_MONTH is None)")

# 3. Optional "ever validly observed" cell filter.
#    Conservative: keep cells with at least one valid observation across the panel.
#    With only a classified/no-data validity flag available this is near-universal;
#    it is a scaffold for a stronger habitat mask if an ever-wet indicator is added.
if REQUIRE_EVER_WET_OR_EVER_OBSERVED:
    before_rows, before_cells = len(panel), panel["grid_id"].nunique()
    ever_valid_cells = (
        panel.loc[panel["valid_pixels"] >= MIN_VALID_PIXELS_PER_CELL_MONTH, "grid_id"].unique()
    )
    panel = panel[panel["grid_id"].isin(ever_valid_cells)].copy()
    print(f"[ever validly observed] removed "
          f"{before_rows - len(panel):,} rows, {before_cells - panel['grid_id'].nunique():,} cells")
else:
    print("[ever validly observed] filter disabled (REQUIRE_EVER_WET_OR_EVER_OBSERVED is False)")

# 4. Water / littoral habitat mask (Priority 1): keep cells that are mostly water.
if WATER_MASK_SOURCE != "none":
    before_rows, before_cells = len(panel), panel["grid_id"].nunique()
    wf = None
    if WATER_MASK_SOURCE in ("bathymetry", "both") and "bathy_water_fraction" in panel.columns:
        wf = panel["bathy_water_fraction"].fillna(0.0)
    if WATER_MASK_SOURCE in ("gsw", "both") and "gsw_water_fraction" in panel.columns:
        g = panel["gsw_water_fraction"].fillna(0.0)
        wf = g if wf is None else (np.minimum(wf, g) if WATER_MASK_SOURCE == "both" else g)
    if wf is None:
        print(f"[water mask] source '{WATER_MASK_SOURCE}' requested but no water-fraction "
              "column is available; skipping (provide bathymetry or a GSW fraction).")
    else:
        panel = panel[wf >= MIN_WATER_FRACTION].copy()
        print(f"[water mask: {WATER_MASK_SOURCE} >= {MIN_WATER_FRACTION:g}] removed "
              f"{before_rows - len(panel):,} rows, "
              f"{before_cells - panel['grid_id'].nunique():,} cells "
              f"(kept {panel['grid_id'].nunique():,} water cells)")
else:
    print("[water mask] disabled (WATER_MASK_SOURCE = 'none')")

print(f"After habitat/valid-cell filters: {len(panel):,} rows, "
      f"{panel['grid_id'].nunique():,} unique cells "
      f"(removed {n_rows_before_filter - len(panel):,} rows, "
      f"{n_cells_before_filter - panel['grid_id'].nunique():,} cells overall)")


## 11b. Exogenous driver covariates: effective depth & onshore wind

The cell below adds two **response-free** exogenous drivers to `panel` (so they are valid with `GAM_INCLUDE_AR=False`) and registers them in §12 so the §49 GAM-frame builder turns each into a forcing `s()` smooth automatically — **no change to the R formula strings**:

- **`effective_depth_m`** = `depth_m` + `lake_level_anom_m`, where `lake_level_anom_m` = `lake_level_m` − its panel mean. `depth_m` is positive-down (larger = deeper); a rising lake (positive anomaly) deepens the water column. Under the per-cell random intercept `s(grid_id, bs='re')` in §13 the *static* `depth_m` is absorbed by the cell effect (concurvity ≈ 1), so this lake-level interaction is the **only** route by which bathymetry enters — and static `depth_m` is dropped from the candidate features whenever the cell RE is on.
- **`wind_onshore_ms`** = the monthly wind `(wind_u_ms, wind_v_ms)` projected onto each cell's shore-normal unit vector and negated. The shore normal is the least-squares gradient of `dist_shore_m` over the cell's contiguity neighbours (`grid_neighbours`); `dist_shore_m` grows away from land, so its gradient points **lakeward**. Negating the projection makes **`wind_onshore_ms` > 0 ⇒ wind blowing toward the cell's nearest shore ⇒ leeward mat accumulation** (expected sign +).
- One-month lags `effective_depth_m_lag1` and `wind_onshore_ms_lag1` (per-cell `shift(1)`, mirroring §10).

Each block is guarded and skips (with a printed notice) when its inputs are missing: `effective_depth_m` needs a monthly `lake_level_m` (supply via `ENV_MONTHLY_CSV`); `wind_onshore_ms` needs EE `dist_shore_m`, the monthly wind components, and the `grid_neighbours` graph.

### Checks to run in Colab after fitting §13

1. **Partial-dependence signs (§13 effect plots).** `effective_depth_m`: **low** effective depth ⇒ **higher** cover (shallow-water preference). `wind_onshore_ms`: **higher** onshore wind ⇒ **higher** cover (leeward pile-up). A flipped `wind_onshore_ms` curve means the shore gradient points landward instead of lakeward — re-check `shore_gx`/`shore_gy` on `grid` (sign of the `dist_shore_m` gradient).
2. **Worst-case concurvity (§13 concurvity table).** Inspect `effective_depth_m` and `wind_onshore_ms` against **`wave_exposure_idx`** and **`dist_shore_m`** (and `depth_m` if you keep it): worst-case concurvity should stay clear of ~0.8, otherwise the new smooths are not separately identified from the existing exposure/shore geometry.
3. **§49 VIF pre-screen printout.** Confirm neither new driver nor its lag is dropped by the VIF screen. A lag dropped for collinearity with its own level is expected and harmless.
4. **Smooth EDFs (§13 fit summary).** Check the new smooths have EDF > 1 (genuinely used / non-linear) and that `select=TRUE` has not shrunk them toward 0.
5. **Spatial-block vs temporal CV delta (§53–55) — the decider.** A driver that helps temporal CV but collapses under spatial-block CV is fitting spatial structure, not transferable forcing. Keep a new covariate only if spatial-block CV skill does not degrade materially relative to temporal CV.


In [ ]:
# =====================================================================
# Exogenous driver covariates (built before §12) -- response-free, so they are
# valid even with GAM_INCLUDE_AR=False. Each block is guarded and skips when its
# target column already exists, so this cell is safe to re-run.
#
#   A) effective_depth_m  -- lake-level x bathymetry interaction
#   B) wind_onshore_ms    -- shore-normal wind component (leeward accumulation)
#
# They become forcing s() smooths automatically once §12 appends them to
# candidate_features -- the R formula strings in §49 are NOT touched.
# =====================================================================
import numpy as np
from collections import defaultdict

# ---------------------------------------------------------------------
# A) effective_depth_m = depth_m + lake-level anomaly.
# ---------------------------------------------------------------------
# Under the per-cell random intercept s(grid_id, bs='re') used in §13, the static
# per-cell depth_m is fully absorbed by the cell effect (concurvity ~1) and so
# carries no driver signal on its own. Interacting bathymetry with the
# time-varying lake level is the ONLY route by which depth enters the model:
# a rising lake (positive level anomaly) deepens the water column, and water
# hyacinth favours shallow water, so a LOW effective_depth_m should map to
# HIGHER cover. depth_m is stored positive-down (larger = deeper).
LAKE_LEVEL_COL = "lake_level_m"

if {LAKE_LEVEL_COL, "depth_m"}.issubset(panel.columns):
    if "lake_level_anom_m" not in panel.columns:
        _lvl = panel[LAKE_LEVEL_COL].astype(float)
        panel["lake_level_anom_m"] = _lvl - _lvl.mean()
    if "effective_depth_m" not in panel.columns:
        panel["effective_depth_m"] = panel["depth_m"] + panel["lake_level_anom_m"]
        print(f"[effective_depth_m] built = depth_m + (lake level - mean); range "
              f"{panel['effective_depth_m'].min():.2f} .. {panel['effective_depth_m'].max():.2f} m")
    else:
        print("[effective_depth_m] already present; skipping.")
else:
    print(f"[effective_depth_m] needs both '{LAKE_LEVEL_COL}' and 'depth_m' in panel; "
          f"supply lake level via ENV_MONTHLY_CSV (a monthly column named "
          f"'{LAKE_LEVEL_COL}'). Skipping.")

# ---------------------------------------------------------------------
# B) wind_onshore_ms = wind component blowing toward each cell's nearest shore.
# ---------------------------------------------------------------------
# Per-cell shore-normal unit vector (shore_gx, shore_gy) = least-squares gradient
# of dist_shore_m over the cell's contiguity neighbours (grid_neighbours).
# dist_shore_m increases away from land, so its gradient points LAKEWARD. Project
# the monthly wind (wind_u_ms eastward, wind_v_ms northward) onto that vector and
# NEGATE: wind_onshore_ms > 0 means the wind blows toward the nearest shore,
# piling floating mats on the leeward side (directional drift), expected sign +.
_have_wind  = {"wind_u_ms", "wind_v_ms"}.issubset(panel.columns)
_have_shore = "dist_shore_m" in panel.columns
_have_graph = ("grid_neighbours" in globals() and grid_neighbours is not None
               and len(grid_neighbours) and "grid" in globals())

if "wind_onshore_ms" in panel.columns:
    print("[wind_onshore_ms] already present; skipping.")
elif not (_have_wind and _have_shore and _have_graph):
    _missing = []
    if not _have_wind:  _missing.append("wind_u_ms/wind_v_ms")
    if not _have_shore: _missing.append("dist_shore_m")
    if not _have_graph: _missing.append("grid_neighbours/grid")
    print(f"[wind_onshore_ms] missing prerequisites ({', '.join(_missing)}); skipping.")
else:
    # One static dist_shore_m + centroid per cell (dist_shore_m is time-invariant).
    _cell = (panel[["grid_id", "x_km", "y_km", "dist_shore_m"]]
             .dropna(subset=["dist_shore_m"])
             .groupby("grid_id", as_index=False).first())
    _pos = {int(r.grid_id): (float(r.x_km), float(r.y_km)) for r in _cell.itertuples()}
    _ds  = {int(r.grid_id): float(r.dist_shore_m) for r in _cell.itertuples()}

    _adj = defaultdict(list)
    for _g, _n in grid_neighbours[["grid_id", "neighbor_id"]].itertuples(index=False):
        _adj[int(_g)].append(int(_n))

    _gx, _gy = {}, {}
    for _gid in _pos:
        _nbrs = [n for n in _adj.get(_gid, []) if n in _pos]   # neighbour has dist_shore
        if len(_nbrs) < 2:                                     # need >= 2 valid neighbours
            continue
        _A = np.array([[_pos[n][0] - _pos[_gid][0], _pos[n][1] - _pos[_gid][1]]
                       for n in _nbrs], dtype=float)
        _b = np.array([_ds[n] - _ds[_gid] for n in _nbrs], dtype=float)
        _grad, *_ = np.linalg.lstsq(_A, _b, rcond=None)        # lakeward gradient
        _norm = float(np.hypot(_grad[0], _grad[1]))
        if not np.isfinite(_norm) or _norm == 0.0:             # degenerate / zero-norm
            continue
        _gx[_gid], _gy[_gid] = _grad[0] / _norm, _grad[1] / _norm

    # Attach the shore-normal unit vector to `grid` (NaN where undefined).
    for _c in ("shore_gx", "shore_gy"):
        if _c in grid.columns:
            grid = grid.drop(columns=_c)
    grid["shore_gx"] = grid["grid_id"].map(_gx)
    grid["shore_gy"] = grid["grid_id"].map(_gy)

    # Merge onto the panel and project the monthly wind onto the shore normal.
    panel = panel.merge(grid[["grid_id", "shore_gx", "shore_gy"]], on="grid_id", how="left")
    panel["wind_onshore_ms"] = -(panel["wind_u_ms"] * panel["shore_gx"]
                                 + panel["wind_v_ms"] * panel["shore_gy"])
    _n_ok = int(panel["wind_onshore_ms"].notna().sum())
    print(f"[wind_onshore_ms] built for {_n_ok:,} cell-months "
          f"({len(_gx):,}/{grid['grid_id'].nunique():,} cells have a shore normal); "
          f"NaNs (sparse/degenerate neighbours) are median-imputed in §49.")

# ---------------------------------------------------------------------
# One-month lags (mirror §10: sort by [grid_id, month], then per-cell shift(1)).
# ---------------------------------------------------------------------
panel = panel.sort_values(["grid_id", "month"]).reset_index(drop=True)
for _col in ["effective_depth_m", "wind_onshore_ms"]:
    _lag = f"{_col}_lag1"
    if _col in panel.columns and _lag not in panel.columns:
        panel[_lag] = panel.groupby("grid_id")[_col].shift(1)
        print(f"[{_lag}] created.")


In [ ]:
# =====================================================================
# 11c. #4 -- Wind-driven advection + multi-month memory + driver interactions
# =====================================================================
# Track A read: wh_cover_lag1 barely registered in tree gain even though the
# persistence baseline is strong -- the trees were not being handed the right
# autoregressive structure. WH is a free-floating, wind-drifted, fragmenting
# invasive, so three families of physically-motivated, leakage-free features are
# added here (all use only information available at t-1) and appended to
# candidate_features in the next cell:
#   A) wind-advective inflow from UPWIND neighbours  (directional colonisation);
#   B) multi-month cover lags + a rolling mean       (bloom build-up, not 1-step);
#   C) wind x fetch/exposure interactions            (leeward accumulation).
import numpy as np

panel = panel.sort_values(["grid_id", "month"]).reset_index(drop=True)
_new_feat_task4 = []

# A) Wind advection inflow (see add_wind_advection_lag; both summaries are lag-1).
if (globals().get("WIND_ADVECTION_ENABLED", True)
        and "grid_neighbours" in globals() and grid_neighbours is not None
        and len(grid_neighbours)
        and {"wind_u_ms", "wind_v_ms"}.issubset(panel.columns)):
    _before = set(panel.columns)
    panel = add_wind_advection_lag(panel, grid_neighbours,
                                   wind_cols=("wind_u_ms", "wind_v_ms"),
                                   value_col="wh_cover", out_prefix="wh_adv_upwind")
    _adv_cols = [c for c in panel.columns if c.startswith("wh_adv_upwind") and c not in _before]
    _new_feat_task4 += _adv_cols
    if _adv_cols and "wh_adv_upwind_flux_lag1" in panel.columns:
        _cov = float(panel["wh_adv_upwind_flux_lag1"].notna().mean())
        print(f"[advection] added {_adv_cols} (non-null on {100 * _cov:.0f}% of rows)")
    else:
        print("[advection] helper returned no new columns; skipping.")
else:
    print("[advection] disabled or prerequisites (grid_neighbours + wind_u_ms/wind_v_ms) "
          "missing; skipping.")

# B) Multi-month cover memory: extra lags + a trailing rolling mean of PAST cover.
_mem_lags = tuple(globals().get("WH_MEMORY_LAGS", (2, 3)))
for _k in _mem_lags:
    _col = f"wh_cover_lag{int(_k)}"
    if _col not in panel.columns:
        panel[_col] = panel.groupby("grid_id")["wh_cover"].shift(int(_k))
        _new_feat_task4.append(_col)
_roll = int(globals().get("WH_MEMORY_ROLL", 3))
if _roll >= 2:
    _rcol = f"wh_cover_roll{_roll}_lag1"
    if _rcol not in panel.columns:
        # shift(1) first so the window never includes the current month (no leakage).
        panel[_rcol] = (panel.groupby("grid_id")["wh_cover"]
                        .transform(lambda s: s.shift(1).rolling(_roll, min_periods=2).mean()))
        _new_feat_task4.append(_rcol)
print(f"[memory] cover lags/rolling added: "
      f"{[c for c in _new_feat_task4 if c.startswith('wh_cover_')]}")

# C) Driver interactions: wind-driven accumulation is amplified on open, exposed
#    fetch, so cross onshore wind with the static fetch/exposure covariates.
if globals().get("DRIVER_INTERACTIONS_ENABLED", True) and "wind_onshore_ms" in panel.columns:
    for _partner in ("openness_index", "wave_exposure_idx"):
        if _partner in panel.columns:
            _icol = f"wind_onshore_x_{_partner}"
            if _icol not in panel.columns:
                panel[_icol] = panel["wind_onshore_ms"] * panel[_partner]
                _new_feat_task4.append(_icol)
    print(f"[interactions] added: "
          f"{[c for c in _new_feat_task4 if c.startswith('wind_onshore_x_')]}")
else:
    print("[interactions] disabled or wind_onshore_ms missing; skipping.")

# Expose for the next cell so clean_feature_columns picks the new columns up.
task4_feature_cols = [c for c in _new_feat_task4 if c in panel.columns]
print(f"\n#4 features registered for the model ({len(task4_feature_cols)}): {task4_feature_cols}")


## 12. Prepare model dataset

This model is intentionally simple and robust for a test period.

It uses a **temporally blocked train/test split**, rather than a random split, because random splitting would leak information from adjacent months into the test set.

Feature columns now include the Earth Engine and CSV covariates (and their one-month lags for the time-varying ones). Rows are dropped only when the **response** or the **structural lag features** are missing; missing values in the environmental covariates themselves (for example a cloudy Sentinel-2 month, or a cell that is mostly land) are left in place and imputed inside the model pipeline, so adding many covariates does not shrink the panel.

In [ ]:
# Candidate model features.
candidate_features = [
    "month_sin",
    "month_cos",
    "time_index",
    "x_km",
    "y_km",
    "wh_cover_lag1",
    "wh_present_lag1",
]

# Time-varying covariates and their one-month lags.
candidate_features += time_varying_cols
candidate_features += [f"{c}_lag1" for c in time_varying_cols if f"{c}_lag1" in panel.columns]

# Static spatial covariates (no lag).
candidate_features += static_cols

# Bathymetry depth (static habitat covariate) and neighbour spatial-lag terms.
candidate_features += [c for c in ["depth_m"] if c in panel.columns]
candidate_features += [c for c in ["wh_cover_neigh_lag1", "wh_present_neigh_lag1"]
                       if c in panel.columns]

# Sensor indicator (Task 1d): lets the model absorb any residual S1/S2 offset in
# the cloud-gap-filled response instead of attributing it to a driver. It is
# dropped automatically by clean_feature_columns when there are no S1 gap-fills.
if globals().get("SENSOR_FEATURE_ENABLED", False) and "sensor_is_s1" in panel.columns:
    candidate_features += ["sensor_is_s1"]

# Exogenous driver covariates built in §11b (effective_depth_m,
# wind_onshore_ms) and their one-month lags. Appended here so the §49
# GAM-frame builder turns each into a forcing s() smooth automatically
# (no change to the R formula strings).
candidate_features += [
    c for c in ["effective_depth_m", "effective_depth_m_lag1",
                "wind_onshore_ms", "wind_onshore_ms_lag1"]
    if c in panel.columns and c not in candidate_features
]
# Under the per-cell random intercept s(grid_id, bs='re') in §13, the static
# depth_m is absorbed by the cell effect (concurvity ~1) and adds no driver
# signal; effective_depth_m is its time-varying replacement, so drop depth_m
# when the cell RE is on (GAM_INCLUDE_CELL_RE defaults to True in §49).
if globals().get("GAM_INCLUDE_CELL_RE", True) and "depth_m" in candidate_features:
    candidate_features.remove("depth_m")

# #4 wind-advection / multi-month memory / interaction features built in §11c.
candidate_features += [c for c in globals().get("task4_feature_cols", [])
                       if c in panel.columns and c not in candidate_features]

feature_cols = clean_feature_columns(panel, candidate_features)

print(f"Model feature columns ({len(feature_cols)}):")
for c in feature_cols:
    print(" -", c)

model_df = panel.dropna(subset=["wh_cover", "wh_present"]).copy()

# Drop rows only when the response or the structural lag features are missing.
# For a very short test period this may remove the first month (lag1 is NaN there).
# Environmental covariates are intentionally NOT in this list: their missing
# values are imputed inside the model pipeline (median imputer), so adding many
# sparse covariates (e.g. cloudy Sentinel-2 months) does not collapse the panel.
# The WH lag features are intentionally NOT required (#5): with calendar-aware t-1
# lags, first-month and post-cloud-gap rows have NaN lags that the model imputes
# (and the persistence baseline treats as 0). Requiring them here would drop every
# post-gap month and bias the panel toward densely-observed periods.
structural_required = [
    "wh_cover", "wh_present", "grid_id", "month", "valid_area_m2",
]
structural_required = [c for c in structural_required if c in model_df.columns]
model_df = model_df.dropna(subset=structural_required).copy()

if len(model_df) == 0:
    raise RuntimeError("No model rows remain after dropping missing response/lag values.")

months_sorted = np.array(sorted(model_df["month"].unique()))
split_idx = max(1, int(len(months_sorted) * TRAIN_FRACTION))
split_month = months_sorted[split_idx]

train = model_df[model_df["month"] < split_month].copy()
test = model_df[model_df["month"] >= split_month].copy()

# Split the TRAIN period into fitting and validation months. The validation
# months are used only to tune the presence probability threshold; the TEST
# months are never used for tuning and remain unchanged.
train_months = np.array(sorted(train["month"].unique()))
if len(train_months) > 1:
    n_val_months = int(round(len(train_months) * VALIDATION_FRACTION))
    n_val_months = max(1, min(n_val_months, len(train_months) - 1))
else:
    n_val_months = 0

if n_val_months > 0:
    val_months = train_months[-n_val_months:]
    fit_months = train_months[:-n_val_months]
else:
    val_months = np.array([], dtype=train_months.dtype)
    fit_months = train_months

train_fit = train[train["month"].isin(fit_months)].copy()
train_val = train[train["month"].isin(val_months)].copy()

print(
    f"Fitting months:    {pd.Timestamp(fit_months.min()).date()} to {pd.Timestamp(fit_months.max()).date()}"
    if len(fit_months) else "Fitting months:    (none)"
)
print(
    f"Validation months: {pd.Timestamp(val_months.min()).date()} to {pd.Timestamp(val_months.max()).date()}"
    if len(val_months) else "Validation months: (none - threshold tuning falls back to training)"
)
print(f"Testing months:    {test['month'].min().date()} to {test['month'].max().date()}")
print(f"Fit rows: {len(train_fit):,} | Val rows: {len(train_val):,} | "
      f"Train(total) rows: {len(train):,} | Test rows: {len(test):,}")
print(f"Train WH-present rate: {train['wh_present'].mean():.4f}")
print(f"Val   WH-present rate: {train_val['wh_present'].mean():.4f}" if len(train_val) else "Val   WH-present rate: (n/a)")
print(f"Test  WH-present rate: {test['wh_present'].mean():.4f}")

X_train = train[feature_cols]
X_test = test[feature_cols]
X_fit = train_fit[feature_cols]
X_val = train_val[feature_cols]

y_train_presence = train["wh_present"].astype(int)
y_test_presence = test["wh_present"].astype(int)
y_fit_presence = train_fit["wh_present"].astype(int)
y_val_presence = train_val["wh_present"].astype(int)

### Performance review and improvement roadmap — one-stage GAM + the two-track redesign

*Appraisal of the §13 model as it now runs (the `mgcv::bam` **Beta** GAM — `GAM_FAMILY="betar"`, logit link — of confidence-weighted WH cover), plus the two models added below it. All figures are read off the fit summary, `k.check`, concurvity, residual-dependence and block-CV outputs in §13–§13-CV.*

#### 1. What the `betar` switch fixed (vs the earlier Tweedie run)

- **Predictions are bounded again.** Under `betar`/logit, fitted cover tops out at **0.078** (q99 = 0.021) instead of the tens-of-thousands the log-link Tweedie produced for a [0,1] response. `R-sq.(adj)` is back to a small **positive** value (≈0.008).
- **The deviance inversion is gone.** Full and forcing-only models both report **34.5%** deviance explained (previously the submodel absurdly reported *more* than its superset).
- **Effect sizes are physically plausible.** The largest forcing effect is `wave_exposure_idx` at **−0.53** on the logit link (was ±11, a factor of ~60,000).
- **Basis adequacy improved.** `k.check` k-index is now 0.87–0.99 (was 0.36–0.47).

#### 2. What is still broken in the one-stage GAM (from the current outputs)

1. **It does not beat the baselines.** By the §13-CV rule (*useful only where it beats BOTH persistence and climatology on Spearman AND RMSE*): seasonal **climatology wins on RMSE everywhere** (spatial 0.056, temporal 0.057 vs the GAM's 0.093 / 0.108), **persistence recovers area almost perfectly** in spatial blocks (1.00 vs the GAM's 4.55×). The GAM's only clear win is forward-in-time *rank* (temporal Spearman 0.238 vs persistence 0.161).
2. **Spatial skill does not transfer.** Spatial Spearman mean 0.219 but **sd 0.31**: folds 1–2 are 0.54/0.56, folds 3–5 collapse to 0.04/0.07/**−0.12**, with area over-recovery up to **13×**. `te(x,y)` memorises location and has nothing to say over a held-out block.
3. **Concurvity ≈ 1.** `te(x,y)` worst-case is huge and dozens of smooths sit at 0.95–1.00, so individual partial-dependence curves are **not** safely interpretable (watch `wh_present_neigh_lag1` coming out **negative** — an artefact of spatial confounding, not ecology).
4. **Residual dependence got worse / mis-corrected.** On the AR1-whitened residuals, spatial **Moran's I = 0.74** (p = 0.001, up from 0.40) and within-cell **lag-1 = −0.42** — the AR1 (ρ = 0.62) **over-corrected** on the gappy time axis, flipping positive autocorrelation to negative.
5. **Under-prediction of the upper tail.** Observed cover reaches 1.0 (q99 = 0.27) but the model never predicts above 0.078 — a single bounded mean cannot fit 87.5% zeros *and* the rare dense mats.

#### 3. The redesign: two models, two jobs (do not ask one model to do both)

The failure is structural: a single interpretable GAM cannot simultaneously be an accurate forecaster of a zero-inflated, autoregressive, spatially-clustered, drifting invasive. So the notebook now carries two tracks, evaluated on **identical** spatial/temporal block folds.

- **Track A — predictive workhorse (accuracy).** A **two-stage hurdle** (gradient-boosted classifier for P(present) × gradient-boosted regressor for E(cover | present)) and a single **Tweedie-objective booster**, both fed the autoregressive/neighbour/seasonal features. The hurdle stops the 87.5% zeros dragging predictions to the mean; trees are immune to the concurvity that shrinks the GAM's drivers. Scored against the same persistence/climatology baselines, with stage-1 presence AUC/precision reported (the "where will it bloom" question, which predicts better than exact cover). **Lead any forecasting claim on the TEMPORAL block skill; report SPATIAL block skill separately as the harder cold-transfer test.**
- **Track B — parsimonious causal model (inference).** The §13 fit cell now breaks the AR1 segments at within-cell month gaps (fixing the −0.42 over-correction), and a **causal contrast** fits the forcing-only model with vs without the fine `te(x,y)`, reporting how much the spatial term confounds the drivers (worst-smooth concurvity) against the residual spatial autocorrelation it buys back. Report the spatially-controlled version for "net-of-location" claims and the drivers-carry-space version for interpretable effect sizes — and quote the classifier's own accuracy/κ once that metrics CSV is located.

#### 4. Status of this commit

All additions are **written but not executed here** (no Colab / Drive / Earth Engine / R runtime — outputs were cleared). Re-run **§12 → §13 → Track B → §13-CV → Track A** top-to-bottom in Colab to regenerate them. Everything is feature-flagged: `RUN_ML_WORKHORSE`, `GAM_RUN_CAUSAL_CONTRAST` (both default `True`); set them `False` to recover the previous notebook exactly. Track A prefers **LightGBM** (added to the §1 install cell) and falls back to scikit-learn `HistGradientBoosting` (Tweedie → Poisson) if LightGBM is unavailable.

> **Bottom line.** The `betar` switch made the §13 model *honest* (bounded, plausible effects) but not *accurate* — climatology still beats it on error. The path to "decent accuracy" is Track A's hurdle + boosting on the autoregressive structure, evaluated forward-in-time; Track B keeps a defensible driver-inference story separate from it. Judge each track on its own metric: block-CV skill vs baselines for Track A, effect sizes + concurvity + residual SAC for Track B.


## 15b. Track A — predictive workhorse (two-stage hurdle + Tweedie gradient boosting)

The §13 GAM is the causal/inference model and a weak predictor. This section builds the *accuracy* model on the same panel and the same block-CV folds: a two-stage hurdle (P(present) × E(cover | present)) and a single Tweedie-objective booster, scored against the same persistence/climatology baselines. See the performance-review note above for the rationale.

### Improvements applied — honest spatial transfer (#1) and area calibration (#3)

Two leakage-free, toggleable changes to the Track A workhorse:

**#1 — Drop coordinate memorisation.** `x_km`/`y_km` were the top-two features by gain, so the trees were partly learning *where* mats sit — precisely what leave-block-out spatial CV holds out. The raw coordinates are removed from the model features (`ML_DROP_RAW_COORDS`) and replaced by a **leakage-safe per-cell climatology** (`ML_ADD_CELL_CLIMATOLOGY`): mean cover over each fold's *training* rows only. Under spatial block-CV the held-out cells are unseen, so their climatology is `NaN` (LightGBM handles it natively) — an honest cold-transfer test; under temporal CV the cell is seen in earlier months, so the feature is informative and uses no future information. Set both flags to `False` to reproduce the with-coordinates baseline for a direct ablation.

**#3 — Per-fold area calibration.** The Tweedie booster recovered only ~75% (spatial) / ~47% (temporal) of observed area. Each model now also emits a calibrated variant `<model>_cal` scaled by `k = sum(y_train) / sum(pred_train)`, estimated on **training** predictions only and applied to the test predictions (no test-label leakage; `k` clipped to `ML_AREA_FACTOR_CLIP`). The scaling is monotone, so Spearman ranking is unchanged — only RMSE and area recovery move — letting the calibrated rows fix the aggregate-area bias without disturbing hotspot ranking.

Both run on the same spatial/temporal block folds as before, so the new numbers are directly comparable to the previous run.


### Further improvements — habitat-similarity prior (#1) and wind-advection + memory features (#4)

Two more leakage-free, toggleable changes to the Track A workhorse, aimed at the two weaknesses the last run exposed: the spatial-block **magnitude collapse** (area recovery ~9%) and the **under-used autoregressive structure** (`wh_cover_lag1` barely registered in tree gain).

**#1 — Hierarchical habitat-similarity prior.** The per-cell climatology (`cell_clim`) is `NaN` for a held-out spatial block, so the trees lose any anchor for that region's baseline level and predictions collapse toward zero (only ~9% of observed area recovered under leave-block-out CV). A second, *transferable* prior `env_clim` is added: cells are stratified by **static habitat covariates** (depth, distance-to-shore, wave exposure, openness) into quantile bins, and each fold maps a held-out cell to the **training-only** mean cover of its stratum. Because the binning uses covariates (not the response) and the mean uses training rows only, an unseen block still receives an honest, non-zero magnitude anchor from environmentally-similar cells elsewhere in the lake — restoring level without re-memorising location. `ML_ADD_HIER_PRIOR` (default on). An optional coarser-grained variant `ML_ADD_COARSE_SPATIAL_PRIOR` (default **off**) pools by 20 km macro-block, but that touches the CV block boundary (an adjacent training block informs the held-out one), so it is left off to keep the cold-transfer test honest.

**#4 — Wind-advection and bloom-memory features.** WH floats and drifts downwind, so a cell gains biomass from its *upwind* neighbours. `wh_adv_upwind_{flux,mean}_lag1` sum last month's neighbour cover weighted by `max(0, wind · direction-to-cell)` — a directional colonisation term the isotropic neighbour-lag cannot express. Multi-month lags (`wh_cover_lag2/3`) and a trailing rolling mean (`wh_cover_roll3_lag1`) capture bloom *build-up* rather than one-step persistence, and `wind_onshore_ms × {openness_index, wave_exposure_idx}` encode wind-driven leeward accumulation on open fetch. Every feature uses only t-1 information (the advection summaries are shifted one month exactly like the existing neighbour lag), so all are leakage-free under both block-CV designs.

Both are feature-flagged (`WIND_ADVECTION_ENABLED`, `DRIVER_INTERACTIONS_ENABLED`, `WH_MEMORY_LAGS`, `ML_ADD_HIER_PRIOR`); set them to `False`/`()` to reproduce the previous run for a direct ablation. Re-run **§11c → §12 → Track A** in Colab to regenerate the numbers (outputs cleared here — no Colab/Drive/EE runtime).

---

### This round — S2-only response, new AOI, and three accuracy changes

Applied on top of the improvements above (all leakage-free, all toggleable):

- **AOI updated to the agreed main-lake polygon** (`aoi/winam_gulf_main_lake_aoi.geojson`): western edge clipped east to lon 34.2047, dropping the small western ponds so the panel is the main Winam Gulf water body (matches the classifier's main-lake mask). The gridded-raster cache is force-refreshed once because it is keyed by raster path, not grid geometry.
- **Sentinel-1 gap-fill removed** (`ENABLE_S1_GAPFILL = False`, `CLASSIFIER_SENSOR_FILTER = ["S2"]`). The robust S1->S2 cover calibration collapsed to a near-constant floor (slope ~2x10^-4 on Pearson r~0.30), so the ~18% of S1-derived cell-months carried no usable cover *magnitude* and fed a flat, non-informative response. Dropping them also removes the sensor-alternation corruption of the autoregressive lags. Cost: the panel loses the S1-only months (~99 vs 114).
- **#5 Calendar-aware lags** (`LAG_CALENDAR_AWARE = True`): `wh_cover_lag1`/`wh_present_lag1` are now the *true* t-1 calendar month (NaN across a cloud gap, then imputed) instead of a positional shift that silently reached across gaps.
- **#1 Confidence weighting** (`ML_CONFIDENCE_WEIGHTING`): training rows are weighted by the classifier's mean winning-class confidence (`wh_conf_mean`) with a floor, so the ~46% low-confidence WH cell-months contribute less label noise.
- **#4 Isotonic recalibration** (`ML_ISOTONIC_CALIBRATION`): a monotone fit of observed cover on the training predictions (`<model>_iso`) -- a full-curve level correction that preserves the Spearman ranking -- reported next to the scalar `<model>_cal`.
- **#3 Stacked blend** (`ML_STACK_BLEND`): a non-negative (NNLS) blend of the Tweedie booster with the climatology and persistence baselines, weights fit on an inner hold-out of each fold's training rows (`stack_blend`), combining the booster's ranking with the baselines' magnitude to target the spatial-block area collapse.


---

### This round (CV honesty) — folds, buffers, and skill scores

Three leakage-free, toggleable changes to the Track A **evaluation** (the model, features and calibrators are unchanged), aimed at the two ways the previous CV over-stated skill. Re-run **Track A** in Colab to regenerate the numbers (outputs cleared here).

- **#1 Spatially-coherent folds + buffer/embargo.** The spatial folds were assigned round-robin over lexicographically-sorted 10 km blocks (`i % n_folds`), which scatters *adjacent* blocks into *different* folds — so every held-out block was ringed by training blocks, and the 500 m `wh_cover_neigh_lag1` / `wh_adv_upwind_*` features read cover straight across the held-out boundary. Folds are now **contiguous super-blocks** (k-means on block centroids, `ML_SPATIAL_FOLD_MODE="contiguous"`), so each held-out region is one compact patch; a **buffer dead-zone** (`ML_SPATIAL_BUFFER_KM`, default 2 km) drops training cells within that distance of any held-out cell so the lag features cannot peek; and temporal folds get a matching **month embargo** (`ML_TEMPORAL_EMBARGO_MONTHS`, default 1) because `wh_cover_lag1` straddles the train/test cut. Set `ML_SPATIAL_FOLD_MODE="roundrobin"`, `ML_SPATIAL_BUFFER_KM=0`, `ML_TEMPORAL_EMBARGO_MONTHS=0` to reproduce the previous (leaky) geometry for a direct ablation.
- **#2 Anomaly-based skill scores.** Raw pooled RMSE/Spearman mostly grade the static seasonal-mean field, which is exactly why seasonal climatology wins on RMSE. Each fold now also reports a squared-error **skill score vs climatology** (`msss_clim = 1 − MSE_model/MSE_clim`; > 0 beats climatology — climatology's own value is 0 by construction) and splits ranking skill into a **within-month** spatial component (`spearman_within_month`, ranking cells inside each test month) and a **temporal-anomaly** component (`spearman_anom`, ranking each cell's departure from its training mean). The anomaly score is the real test of dynamic skill; it is `NaN` under spatial CV where held-out cells are wholly unseen (no per-cell reference), so read it on the **temporal** folds. Lead with `msss_clim` and `spearman_anom` for forecasting claims.
- **#3 Structure-aware internal splits.** LightGBM early-stopping used a *random* 10% eval slice and the stacked blend a *random* 25% inner hold-out; under the panel's spatio-temporal autocorrelation a random split places eval rows next to near-duplicate training rows, so early stopping fires late and the NNLS weights over-fit. Both now hold out the **latest training months** (`ML_STRUCTURE_AWARE_EARLYSTOP`, `ML_STRUCTURE_AWARE_INNER_SPLIT`, default on), an out-of-time slice consistent with the outer CV.


---

### This round — spatial magnitude collapse + `mean_x`/`mean_y` data-integrity fix

Two targeted changes from the last performance review; both leakage-free and toggleable.

**Spatial magnitude collapse (area recovery ~0.45x) — regime-matched calibration (`ML_REGIME_MATCHED_CALIBRATION`, default on).** Under leave-block-out spatial CV the held-out cells are unseen, so their per-cell anchor `cell_clim` is `NaN` and the boosters under-recover total held-out area. The existing area calibration could not fix it: `k = sum(y_train)/sum(pred_train)` was estimated on the *training* predictions, which were formed **with** `cell_clim` present, so `k` came out ~1.0 (see the near-1.0 spatial `k` in the last run) and corrected nothing. The fix recomputes the calibration base on training predictions with `cell_clim` **masked to `NaN`**, so the calibration population matches the unseen-cell test regime; the scalar-`k` (`_cal`) and isotonic (`_iso`) level corrections then actually lift held-out area toward 1.0. It is monotone (Spearman unchanged), needs no refit and no test labels, and engages only on the spatial folds — temporal test cells are seen, so their calibration base is left exactly as before. A synthetic leave-block-out check reproduces the collapse (area 0.18) and the fix (0.91); the milder real collapse (0.45) stays inside the `ML_AREA_FACTOR_CLIP` guard. **Read the improvement on `tweedie_gbm_cal` / `_iso` spatial area recovery and RMSE; Spearman and the temporal rows should be unchanged.**

**`mean_x` / `mean_y` were mislabeled Earth-Engine reducer outputs, not real covariates.** `reduceRegions` on a **single-band** image names its output after the *reducer* (`"mean"`), not the band; two such per-cell/month layers (MODIS water-surface temperature, plus a second single-band layer carried in an older Drive cache) then collided into pandas' default `mean_x`/`mean_y` suffixes on merge and were fed to the model as two anonymous features (2.8% / 2.1% gain). A third, bare `mean` column is the JRC GSW water-fraction *static* mask, which — under the name `mean` — slipped past the author's mask-only exclusion and leaked into the features as a predictor. Two fixes: (1) `reduce_image_over_cells` now renames a single-band reducer output back to its band name **at the source**, so a fresh cache (`EE_FORCE_REFRESH=True`) carries every layer under its real name (e.g. `gsw_water_fraction`, `water_temp_c`) and never produces the collision again; (2) for an existing cache, a feature-engineering repair drops the un-provenanced `mean_x`/`mean_y` columns by default (`ML_KEEP_MEAN_REDUCER_COLS=False`) since their physical identity is not recoverable from code, with a rebuild recommended as the clean resolution. Set `ML_KEEP_MEAN_REDUCER_COLS=True` to keep them (renaming `mean_x` -> `water_temp_c` on the positional inference that it is the MODIS water-temperature slot).

Re-run **S4 (EE helpers) -> S11c -> S12 -> Track A** in Colab to regenerate the numbers (outputs cleared here — no Colab/Drive/Earth-Engine runtime). To restore `water_temp_c` as a named per-cell covariate, additionally set `EE_FORCE_REFRESH=True` once so the corrected single-band naming rewrites the cache.

In [ ]:
# =====================================================================
# TRACK A — Predictive workhorse: two-stage hurdle + Tweedie gradient boosting
# =====================================================================
# The §13 GAM is the *causal / inference* model. It is a weak *predictor*: the
# bounded Beta mean reverts (fitted cover tops out ~0.08 against an observed max
# of 1.0), concurvity shrinks correlated drivers toward zero, and on block-CV it
# does not beat persistence / climatology on RMSE. This section builds the
# *accuracy* model on the SAME panel and the SAME spatial / temporal block folds
# as §13-CV, so the two tracks are directly comparable.
#
# Why this design fits the data-generating process (87.5% zeros, rare dense mats,
# strong autocorrelation, a drifting invasive):
#   1. Hurdle (two-stage): E[cover] = P(present) x E(cover | present). Fitting the
#      intensity stage on the ~12.5% positive rows only stops the 87.5% zeros
#      dragging every prediction to the mean (the GAM's fitted-max ~0.08 problem).
#   2. Gradient-boosted trees: immune to the concurvity that shrinks the GAM's
#      collinear drivers -- trees just split on whichever correlated feature helps.
#   3. Lean on the autoregressive + neighbour + seasonal structure
#      (wh_cover_lag1, wh_present_lag1, wh_cover_neigh_lag1, month_*) -- that is
#      where the predictable signal lives for a wind-drifted, fragmenting invasive.
# A single Tweedie-objective booster (native zero-inflation, one model) is fit as a
# second workhorse and compared. Everything is scored against persistence + seasonal
# climatology on identical folds; a model earns its keep only where it beats BOTH
# baselines on Spearman AND RMSE (the same rule as the §13-CV cell).

RUN_ML_WORKHORSE   = True
ML_TWEEDIE_POWER   = 1.3      # LightGBM tweedie_variance_power in (1,2): 1=Poisson, 2=Gamma
ML_N_ESTIMATORS    = 700
ML_LEARNING_RATE   = 0.05
ML_NUM_LEAVES      = 63
ML_MIN_CHILD_SAMP  = 200
ML_SUBSAMPLE       = 0.8
ML_COLSAMPLE       = 0.8
ML_EARLY_STOP      = 50       # LightGBM early-stopping rounds on an internal validation slice
ML_RANDOM_STATE    = int(globals().get("RANDOM_STATE", 42))
# Match the §13-CV blocking so the ML skill is comparable to the GAM skill.
ML_SPATIAL_BLOCK_KM = float(globals().get("GAM_SPATIAL_BLOCK_KM", 10.0))
ML_N_SPATIAL_FOLDS  = int(globals().get("GAM_N_SPATIAL_FOLDS", 5))
ML_N_TEMPORAL_FOLDS = int(globals().get("GAM_N_TEMPORAL_FOLDS", 4))

# --- Improvements (this notebook): honest spatial transfer + area calibration ----
# #1 Coordinate memorisation: x_km/y_km were the top-2 features by gain, so the
#    trees partly learn *where* mats sit -- exactly what leave-block-out spatial CV
#    holds out. Drop the raw coordinates from the feature set and, in their place,
#    add a LEAKAGE-SAFE per-cell climatology (mean cover over the TRAINING rows of
#    each fold only). Under spatial block-CV the held-out cells are unseen, so their
#    climatology is NaN (LightGBM handles it) -- an honest cold-transfer test; under
#    temporal CV the cell is seen in earlier months, so it is informative and leak-free.
#    Set ML_DROP_RAW_COORDS/ML_ADD_CELL_CLIMATOLOGY = False to reproduce the
#    with-coordinates baseline for a direct ablation.
# #3 Area under-forecast: the Tweedie booster recovered only ~75% (spatial) / ~47%
#    (temporal) of observed area. Add a per-fold MULTIPLICATIVE area calibration whose
#    factor k = sum(y_train)/sum(pred_train) is estimated on TRAINING predictions only
#    (no test-label leakage) and applied to the test predictions. Calibrated variants
#    are scored alongside the raw models as "<model>_cal". Scaling is monotone, so
#    calibration moves RMSE / area recovery but leaves the Spearman ranking unchanged.
ML_DROP_RAW_COORDS      = True
ML_ADD_CELL_CLIMATOLOGY = True
ML_AREA_CALIBRATION     = True
ML_AREA_FACTOR_CLIP     = (0.2, 5.0)   # guard against tiny-denominator blow-ups
# Regime-matched calibration (spatial magnitude-collapse fix). Under leave-block-out
# SPATIAL CV the held-out cells are unseen, so their cell_clim anchor is NaN and the
# raw predictions under-recover total area (~0.45x). The scalar k = sum(y)/sum(pred)
# was estimated on TRAINING predictions formed WITH cell_clim -> k~1.0 -> it corrected
# nothing (the near-1.0 spatial k in the last run is the tell). Estimate the calibration
# base on training predictions with cell_clim MASKED to NaN so the calibration regime
# matches the unseen-cell test block; the scalar-k and isotonic level corrections then
# lift held-out area toward 1.0. Monotone -> Spearman unchanged. Spatial folds only
# (temporal test cells are seen, so their base is left as-is). Set False to revert.
ML_REGIME_MATCHED_CALIBRATION = True

# --- #1 (refined): hierarchical habitat-similarity prior ---------------------
# cell_clim is NaN for a held-out spatial block, so magnitude collapses there
# (last run recovered ~9% of area). Add a transferable env_clim: stratify cells
# by STATIC habitat covariates and map each held-out cell to the TRAINING-only
# mean cover of its stratum -- an honest, non-zero level anchor for unseen regions
# that uses no coordinates and no test labels. The optional coarse 20 km
# macro-block prior is off by default (it touches the CV block boundary).
ML_ADD_HIER_PRIOR           = True
ML_ENV_STRATA_COLS          = ["depth_m", "dist_shore_m", "wave_exposure_idx", "openness_index"]
ML_ENV_STRATA_Q             = 4        # quantile bins per stratifying covariate
ML_ADD_COARSE_SPATIAL_PRIOR = False    # 20 km macro-block prior (boundary-adjacent)
ML_HIER_COARSE_KM           = 2.0 * ML_SPATIAL_BLOCK_KM

# --- #1 Response confidence weighting (this round) ---------------------------
# The classifier's per-pixel winning-class confidence enters the panel as
# wh_conf_mean (mean over WH pixels in a cell-month); ~46% of WH-bearing cell-months
# sit below 0.60, i.e. the response carries real measurement error. Weight every
# TRAINING row by that confidence (absence rows have no WH pixels, so they keep
# weight 1.0) with a floor so no row is fully discarded -- reduces the label-noise
# contribution without dropping data. Leakage-free: the weight uses the response
# *confidence*, never the cover value the model predicts.
ML_CONFIDENCE_WEIGHTING = True
ML_CONF_COL             = "wh_conf_mean"
ML_CONF_WEIGHT_MIN      = 0.30

# --- #4 Isotonic (monotone) recalibration (this round) -----------------------
# The scalar area factor k only rescales the mean; an isotonic fit of observed
# cover on the TRAINING predictions is a full monotone recalibration (preserves the
# Spearman ranking, corrects the level/curve). Emitted as "<model>_iso" alongside
# the scalar "<model>_cal" so the two are directly comparable.
ML_ISOTONIC_CALIBRATION = True

# --- #3 Stacked blend (this round) -------------------------------------------
# The Tweedie booster ranks well but under-recovers area on held-out spatial blocks
# (~10%); persistence/climatology recover area almost perfectly there but rank
# poorly. A non-negative (NNLS) blend of the three -- weights fit on an INNER
# hold-out of each fold's training rows (out-of-sample, so no leakage) -- combines
# the booster's ranking with the baselines' magnitude. Emitted as "stack_blend".
ML_STACK_BLEND          = True
ML_STACK_INNER_FRACTION = 0.25   # inner hold-out share of training used to fit blend weights

# --- #1/#2/#3 (this round): honest CV geometry + skill scoring ---------------
# The three fixes below target the EVALUATION rather than the model:
#   #1 Spatial folds were assigned round-robin over lexicographically-sorted 10 km
#      blocks, so ADJACENT blocks landed in DIFFERENT folds -- every held-out block
#      ended up ringed by training blocks, and the 500 m neighbour-lag / advection
#      features then read cover straight across that boundary. Assign spatially
#      CONTIGUOUS super-blocks (k-means on block centroids) and drop training cells
#      within a buffer (dead-zone) of any held-out cell so the lag features cannot
#      peek. Temporal folds get a matching month embargo (wh_cover_lag1 straddles
#      the split). Set ML_SPATIAL_FOLD_MODE="roundrobin", ML_SPATIAL_BUFFER_KM=0,
#      ML_TEMPORAL_EMBARGO_MONTHS=0 to reproduce the previous (leaky) geometry.
#   #2 Raw pooled RMSE/Spearman mostly grade the static mean field (which is why
#      seasonal climatology wins on RMSE). Report a squared-error SKILL SCORE vs
#      climatology (MSSS = 1 - MSE_model/MSE_clim; >0 beats climatology) and split
#      ranking skill into a WITHIN-MONTH spatial component (rank cells inside each
#      test month) and a TEMPORAL-ANOMALY component (rank departures from each
#      cell's training mean) -- the anomaly score is the real test of dynamic skill.
#   #3 Early-stopping and the stacked-blend inner hold-out used RANDOM row subsets,
#      which sit temporally/spatially next to their kept counterparts under the
#      panel's autocorrelation (early stopping fires late; blend weights over-fit).
#      Use STRUCTURE-AWARE holdouts (the latest training months) instead.
ML_SPATIAL_FOLD_MODE           = "contiguous"  # "contiguous" (k-means super-blocks) | "roundrobin" (legacy)
ML_SPATIAL_BUFFER_KM           = 2.0           # dead-zone: drop train cells within this of any held-out cell (0=off)
ML_TEMPORAL_EMBARGO_MONTHS     = 1             # drop this many months just before each temporal test fold (0=off)
ML_STRUCTURE_AWARE_EARLYSTOP   = True          # early-stopping eval = latest training months (not a random 10%)
ML_EARLYSTOP_EVAL_FRACTION     = 0.15          # share of training MONTHS held out for early-stopping eval
ML_STRUCTURE_AWARE_INNER_SPLIT = True          # stacked-blend inner hold-out = latest training months (not random)
ML_MIN_FOLD_TRAIN_ROWS         = 200           # skip a fold if buffer/embargo leaves fewer training rows than this

if not RUN_ML_WORKHORSE:
    print("RUN_ML_WORKHORSE is False; skipping the Track A predictive workhorse.")
else:
    import numpy as np
    import pandas as pd
    from scipy.stats import spearmanr
    from scipy.optimize import nnls
    from scipy.spatial import cKDTree
    from sklearn.cluster import KMeans
    from sklearn.isotonic import IsotonicRegression
    from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

    # --- backend: LightGBM if available, else a scikit-learn fallback -------------
    try:
        import lightgbm as lgb
        _HAVE_LGB = True
        print("ML backend: LightGBM", lgb.__version__)
    except Exception as _e:
        _HAVE_LGB = False
        from sklearn.ensemble import (HistGradientBoostingClassifier,
                                       HistGradientBoostingRegressor)
        print("ML backend: scikit-learn HistGradientBoosting (LightGBM not importable:",
              f"{_e}).\n  NOTE: the single-model Tweedie objective falls back to Poisson (p=1).")

    def _make_classifier():
        if _HAVE_LGB:
            return lgb.LGBMClassifier(
                n_estimators=ML_N_ESTIMATORS, learning_rate=ML_LEARNING_RATE,
                num_leaves=ML_NUM_LEAVES, min_child_samples=ML_MIN_CHILD_SAMP,
                subsample=ML_SUBSAMPLE, subsample_freq=1, colsample_bytree=ML_COLSAMPLE,
                random_state=ML_RANDOM_STATE, n_jobs=-1, verbose=-1)
        return HistGradientBoostingClassifier(
            max_iter=ML_N_ESTIMATORS, learning_rate=ML_LEARNING_RATE,
            max_leaf_nodes=ML_NUM_LEAVES, min_samples_leaf=ML_MIN_CHILD_SAMP,
            early_stopping=True, random_state=ML_RANDOM_STATE)

    def _make_regressor(objective="regression"):
        if _HAVE_LGB:
            params = dict(
                n_estimators=ML_N_ESTIMATORS, learning_rate=ML_LEARNING_RATE,
                num_leaves=ML_NUM_LEAVES, min_child_samples=ML_MIN_CHILD_SAMP,
                subsample=ML_SUBSAMPLE, subsample_freq=1, colsample_bytree=ML_COLSAMPLE,
                random_state=ML_RANDOM_STATE, n_jobs=-1, verbose=-1)
            if objective == "tweedie":
                params.update(objective="tweedie", tweedie_variance_power=ML_TWEEDIE_POWER)
            return lgb.LGBMRegressor(**params)
        # scikit-learn fallback: Tweedie -> Poisson (p=1) approximation for the single
        # model; squared error for the hurdle intensity stage.
        loss = "poisson" if objective == "tweedie" else "squared_error"
        return HistGradientBoostingRegressor(
            max_iter=ML_N_ESTIMATORS, learning_rate=ML_LEARNING_RATE,
            max_leaf_nodes=ML_NUM_LEAVES, min_samples_leaf=ML_MIN_CHILD_SAMP,
            loss=loss, early_stopping=True, random_state=ML_RANDOM_STATE)

    def _earlystop_mask(t):
        # #3: STRUCTURE-AWARE early-stopping eval slice = the latest training MONTHS
        # (t = time_rank), not a random 10%. A random split places eval rows next to
        # their training counterparts under the panel's spatio-temporal
        # autocorrelation, so the loss keeps improving on rows that are near-duplicates
        # of the fit rows and early stopping fires late (mild overfit). Holding out the
        # trailing months keeps the eval out-of-time and consistent with the outer CV.
        # Returns a boolean mask (True = eval row) or None to fall back to a random slice.
        if t is None or not ML_STRUCTURE_AWARE_EARLYSTOP:
            return None
        t = np.asarray(t, float)
        ur = np.unique(t[np.isfinite(t)])
        if ur.size < 3:
            return None
        k = max(1, int(round(ur.size * ML_EARLYSTOP_EVAL_FRACTION)))
        thr = ur[-k]
        return t >= thr

    def _fit(model, X, y, w=None, t=None):
        # LightGBM handles NaN natively; pass raw features. Early-stopping needs an
        # eval slice: use a STRUCTURE-AWARE trailing-months holdout when a time vector
        # `t` (time_rank, aligned to X rows) is supplied (#3), else a random 10%.
        # sample_weight (w) carries the #1 response-confidence weighting when set.
        y = np.asarray(y)
        w = None if w is None else np.asarray(w, float)
        if _HAVE_LGB and ML_EARLY_STOP:
            n = len(X)
            va = _earlystop_mask(t)
            if va is None:
                rng = np.random.RandomState(ML_RANDOM_STATE)
                va = rng.rand(n) < 0.1
            if va.sum() >= 50 and (~va).sum() >= 50 and (np.unique(y[~va]).size > 1):
                fit_kw = {} if w is None else dict(
                    sample_weight=w[~va], eval_sample_weight=[w[va]])
                model.fit(X[~va], y[~va],
                          eval_set=[(X[va], y[va])],
                          callbacks=[lgb.early_stopping(ML_EARLY_STOP, verbose=False),
                                     lgb.log_evaluation(0)],
                          **fit_kw)
                return model
        model.fit(X, y, **({} if w is None else dict(sample_weight=w)))
        return model

    # --- assemble the ML frame + reconstruct the §13-CV folds exactly -------------
    _feat = [c for c in feature_cols if c in model_df.columns]
    # Keep x_km/y_km in ml_df (needed for CV blocking + area calibration) but exclude
    # them from the MODEL features when ML_DROP_RAW_COORDS is on (#1).
    _coord_cols = [c for c in ("x_km", "y_km") if c in _feat]
    _feat_model = [c for c in _feat if c not in _coord_cols] if ML_DROP_RAW_COORDS else list(_feat)
    _CELL_CLIM = "cell_clim"
    ml_df = model_df.dropna(subset=["wh_cover", "x_km", "y_km", "month_num"]).copy()
    ml_df = ml_df.reset_index(drop=True)

    # #1: spatial CV blocks. The 10 km blocks match §13-CV; how the blocks are
    # GROUPED into folds is what changed. "contiguous" clusters the block centroids
    # with k-means so each fold is one spatially-coherent super-block (the held-out
    # region is a compact patch, not a scatter ringed by training cells); "roundrobin"
    # is the legacy interleaved assignment kept for an ablation.
    _bx = (ml_df["x_km"] // ML_SPATIAL_BLOCK_KM).astype(int)
    _by = (ml_df["y_km"] // ML_SPATIAL_BLOCK_KM).astype(int)
    ml_df["_block"] = _bx.astype(str) + "_" + _by.astype(str)
    _bcent = ml_df.groupby("_block")[["x_km", "y_km"]].mean()
    if ML_SPATIAL_FOLD_MODE == "contiguous" and len(_bcent) > ML_N_SPATIAL_FOLDS:
        _km = KMeans(n_clusters=ML_N_SPATIAL_FOLDS, n_init=10, random_state=ML_RANDOM_STATE)
        _lab = _km.fit_predict(_bcent[["x_km", "y_km"]].to_numpy())
        _bmap = {b: int(l) + 1 for b, l in zip(_bcent.index, _lab)}
        _fold_mode_used = "contiguous (k-means super-blocks)"
    else:
        _bmap = {b: i % ML_N_SPATIAL_FOLDS + 1 for i, b in enumerate(sorted(_bcent.index))}
        _fold_mode_used = "roundrobin (legacy interleaved)"
    ml_df["fold_sp"] = ml_df["_block"].map(_bmap).astype(int)

    _months = np.sort(ml_df["month"].unique())
    ml_df["time_rank"] = ml_df["month"].map({m: i for i, m in enumerate(_months)}).astype(int)
    _n_tm = int(min(ML_N_TEMPORAL_FOLDS, max(0, len(_months) - 1)))
    _tm_map = ({r: j + 1 for j, r in enumerate(range(len(_months) - _n_tm, len(_months)))}
               if _n_tm > 0 else {})
    ml_df["fold_tm"] = ml_df["time_rank"].map(lambda r: _tm_map.get(int(r), 0)).astype(int)

    # #1: static, response-free stratifications for the hierarchical priors. The
    # bin edges use only covariates (not wh_cover), so labelling a held-out cell
    # leaks nothing; the per-fold group MEAN of wh_cover (computed below on the
    # training rows only) supplies the transferable magnitude anchor.
    _ENV_CLIM, _COARSE_CLIM = "env_clim", "coarse_clim"
    _hier_feats = []
    if ML_ADD_HIER_PRIOR:
        _strat_src = [c for c in ML_ENV_STRATA_COLS if c in ml_df.columns
                      and pd.api.types.is_numeric_dtype(ml_df[c])]
        if _strat_src:
            _lab = pd.DataFrame(index=ml_df.index)
            for _c in _strat_src:
                _b = pd.qcut(ml_df[_c], ML_ENV_STRATA_Q, labels=False, duplicates="drop")
                _lab[_c] = np.where(_b.notna(), _b.fillna(0).astype(int).astype(str), "na")
            ml_df["env_stratum"] = _lab.agg("_".join, axis=1)
        else:
            ml_df["env_stratum"] = "all"
        _hier_feats.append((_ENV_CLIM, "env_stratum"))
        print(f"Hierarchical prior: env_clim over {ml_df['env_stratum'].nunique():,} "
              f"habitat strata from {_strat_src or ['(none -> global mean)']}")
        if ML_ADD_COARSE_SPATIAL_PRIOR:
            _ck = float(ML_HIER_COARSE_KM)
            ml_df["macro_block"] = ((ml_df["x_km"] // _ck).astype(int).astype(str) + "_" +
                                    (ml_df["y_km"] // _ck).astype(int).astype(str))
            _hier_feats.append((_COARSE_CLIM, "macro_block"))

    _lag_zero = ml_df["wh_cover_lag1"].fillna(0.0).to_numpy() if "wh_cover_lag1" in ml_df else None
    _n_model_feat = (len(_feat_model) + (1 if ML_ADD_CELL_CLIMATOLOGY else 0)
                     + len(_hier_feats))
    print(f"ML frame: {len(ml_df):,} rows x {_n_model_feat} model features "
          f"(raw coords {'DROPPED' if ML_DROP_RAW_COORDS else 'kept'}, "
          f"cell-climatology {'ON' if ML_ADD_CELL_CLIMATOLOGY else 'off'}, "
          f"area-calibration {'ON' if ML_AREA_CALIBRATION else 'off'}, "
          f"hier-prior {'ON' if ML_ADD_HIER_PRIOR else 'off'}); "
          f"{ml_df['grid_id'].nunique():,} cells; "
          f"spatial folds {sorted(ml_df['fold_sp'].unique())} [{_fold_mode_used}] | "
          f"temporal folds {sorted(r for r in ml_df['fold_tm'].unique() if r > 0) or 'NONE'}")
    print(f"CV geometry: spatial buffer {ML_SPATIAL_BUFFER_KM} km | "
          f"temporal embargo {ML_TEMPORAL_EMBARGO_MONTHS} month(s) | "
          f"structure-aware early-stop {'ON' if ML_STRUCTURE_AWARE_EARLYSTOP else 'off'} | "
          f"structure-aware inner-blend {'ON' if ML_STRUCTURE_AWARE_INNER_SPLIT else 'off'}")

    # #1: per-row TRAINING weights from the classifier's response confidence.
    # Absence / no-WH cell-months carry no WH-pixel confidence, so they keep weight
    # 1.0; WH-bearing cell-months are down-weighted toward ML_CONF_WEIGHT_MIN by
    # their mean winning-class confidence. Leakage-free (uses response *confidence*,
    # never the cover value being predicted).
    if ML_CONFIDENCE_WEIGHTING and ML_CONF_COL in ml_df.columns:
        _wc = pd.to_numeric(ml_df[ML_CONF_COL], errors="coerce")
        ml_df["_ml_w"] = np.where(_wc.notna(),
                                  _wc.clip(ML_CONF_WEIGHT_MIN, 1.0), 1.0).astype(float)
        _wmean = float(ml_df.loc[_wc.notna(), "_ml_w"].mean()) if _wc.notna().any() else float("nan")
        print(f"#1 confidence weighting ON: {int(_wc.notna().sum()):,} WH-bearing rows "
              f"down-weighted (mean weight {_wmean:.3f}, floor {ML_CONF_WEIGHT_MIN}); "
              f"absence rows weight 1.0.")
    else:
        ml_df["_ml_w"] = 1.0
        if ML_CONFIDENCE_WEIGHTING:
            print(f"#1 confidence weighting requested but '{ML_CONF_COL}' not in frame; "
                  f"using uniform weights.")

    def _metrics(obs, pred):
        pred = np.clip(np.asarray(pred, float), 0, 1)
        obs = np.asarray(obs, float)
        sp = spearmanr(obs, pred).correlation if np.nanstd(pred) > 0 else np.nan
        rmse = float(np.sqrt(np.mean((obs - pred) ** 2)))
        mae = float(np.mean(np.abs(obs - pred)))
        ar = float(pred.sum() / obs.sum()) if obs.sum() > 0 else np.nan
        return sp, rmse, mae, ar

    def _row(model, kind, fold, ntr, nte, obs, pred):
        sp, rmse, mae, ar = _metrics(obs, pred)
        return dict(model=model, kind=kind, fold=fold, n_train=ntr, n_test=nte,
                    spearman=sp, rmse=rmse, mae=mae, area_recovery=ar)

    def _extra_skill(te, pred, clim_pred, cell_mean_tr):
        # #2: skill scores that go beyond the pooled RMSE/Spearman (which mostly grade
        # the static mean field). All three references are leakage-free (training rows
        # only): clim_pred = training per-month_num mean mapped to test; cell_mean_tr =
        # each cell's training mean cover.
        #   msss_clim  = 1 - MSE_model/MSE_clim  (squared-error skill vs climatology;
        #                >0 means the model beats climatology, the baseline that wins RMSE)
        #   sp_within  = mean over test MONTHS of the within-month Spearman (pure spatial
        #                hotspot ranking, isolated from the seasonal mean field)
        #   sp_anom    = Spearman of ANOMALIES (obs/pred minus each cell's training mean)
        #                -- the real test of dynamic skill; NaN for wholly-unseen cells
        #                (spatial folds), which have no per-cell reference.
        obs = te["wh_cover"].to_numpy(float)
        pred = np.clip(np.asarray(pred, float), 0, 1)
        cp = np.asarray(clim_pred, float)
        mse_m = float(np.mean((obs - pred) ** 2))
        mse_c = float(np.mean((obs - cp) ** 2))
        msss = float(1.0 - mse_m / mse_c) if mse_c > 0 else np.nan
        _df = pd.DataFrame({"month": te["month"].to_numpy(), "o": obs, "p": pred})
        sps = []
        for _, g in _df.groupby("month"):
            if len(g) >= 5 and g["o"].std(ddof=0) > 0 and g["p"].std(ddof=0) > 0:
                _c = spearmanr(g["o"], g["p"]).correlation
                if np.isfinite(_c):
                    sps.append(_c)
        sp_within = float(np.mean(sps)) if sps else np.nan
        cm = te["grid_id"].map(cell_mean_tr).to_numpy(float)
        ok = np.isfinite(cm)
        if ok.sum() >= 5:
            oa = obs[ok] - cm[ok]
            pa = pred[ok] - cm[ok]
            sp_anom = (spearmanr(oa, pa).correlation
                       if np.nanstd(oa) > 0 and np.nanstd(pa) > 0 else np.nan)
        else:
            sp_anom = np.nan
        return msss, sp_within, sp_anom

    def _fold_features(tr, te):
        # Per-fold model design matrices. Raw coords are already excluded from
        # _feat_model (#1); the leakage-safe cell climatology is estimated on the
        # TRAINING rows of THIS fold only and mapped onto both splits. Unseen cells
        # (whole held-out blocks under spatial CV) map to NaN, which LightGBM handles.
        Xtr = tr[_feat_model].copy()
        Xte = te[_feat_model].copy()
        if ML_ADD_CELL_CLIMATOLOGY:
            clim = tr.groupby("grid_id")["wh_cover"].mean()
            Xtr[_CELL_CLIM] = tr["grid_id"].map(clim).to_numpy()
            Xte[_CELL_CLIM] = te["grid_id"].map(clim).to_numpy()
        # #1: hierarchical habitat / coarse-spatial priors. Each is a TRAINING-only
        # group mean of cover over a STATIC, response-free stratification, so a
        # held-out spatial block inherits a non-zero level anchor from similar cells
        # (env strata) instead of the NaN that collapses cell_clim. Unseen strata
        # fall back to the training global mean.
        if ML_ADD_HIER_PRIOR:
            _gm = float(tr["wh_cover"].mean())
            for _feat, _idc in _hier_feats:
                _m = tr.groupby(_idc)["wh_cover"].mean()
                Xtr[_feat] = tr[_idc].map(_m).fillna(_gm).to_numpy()
                Xte[_feat] = te[_idc].map(_m).fillna(_gm).to_numpy()
        return Xtr, Xte

    def _area_factor(y_tr, pred_tr):
        # #3 multiplicative bias correction estimated on TRAINING predictions only.
        s = float(np.sum(pred_tr))
        if not np.isfinite(s) or s <= 0:
            return 1.0
        lo, hi = ML_AREA_FACTOR_CLIP
        return float(np.clip(float(np.sum(y_tr)) / s, lo, hi))

    def _predict_hurdle(Xtr, Xte, tr, w_tr=None):
        t_all = tr["time_rank"].to_numpy() if "time_rank" in tr.columns else None
        clf = _fit(_make_classifier(), Xtr, tr["wh_present"].astype(int), w=w_tr, t=t_all)
        p_pres_te = clf.predict_proba(Xte)[:, 1]
        p_pres_tr = clf.predict_proba(Xtr)[:, 1]
        pos = tr["wh_present"].astype(int).to_numpy() == 1
        if pos.sum() >= 50:
            w_pos = None if w_tr is None else np.asarray(w_tr, float)[pos]
            t_pos = None if t_all is None else np.asarray(t_all)[pos]
            reg = _fit(_make_regressor("regression"), Xtr[pos], tr.loc[pos, "wh_cover"],
                       w=w_pos, t=t_pos)
            cover_te = np.clip(reg.predict(Xte), 0, 1)
            cover_tr = np.clip(reg.predict(Xtr), 0, 1)
        else:
            cover_te = np.full(len(Xte), tr["wh_cover"].mean())
            cover_tr = np.full(len(Xtr), tr["wh_cover"].mean())
            reg = None
        return (np.clip(p_pres_te * cover_te, 0, 1),
                np.clip(p_pres_tr * cover_tr, 0, 1), p_pres_te, clf, reg)

    def _predict_tweedie(Xtr, Xte, tr, w_tr=None):
        t_all = tr["time_rank"].to_numpy() if "time_rank" in tr.columns else None
        reg = _fit(_make_regressor("tweedie"), Xtr, tr["wh_cover"], w=w_tr, t=t_all)
        return (np.clip(reg.predict(Xte), 0, 1),
                np.clip(reg.predict(Xtr), 0, 1), reg)

    def _isotonic_map(y_tr, pred_tr, w_tr=None):
        # #4: monotone recalibration fit on TRAINING (pred -> observed cover). Being
        # monotone non-decreasing it leaves the Spearman ranking intact and only
        # corrects the level/curve. Returns a fitted mapper or None on failure.
        try:
            iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
            iso.fit(np.clip(np.asarray(pred_tr, float), 0, 1), np.asarray(y_tr, float),
                    sample_weight=None if w_tr is None else np.asarray(w_tr, float))
            return iso
        except Exception:
            return None

    def _nnls_blend(B, y, sw=None):
        # #3: non-negative least-squares blend weights (>=0, not forced to sum to 1,
        # so the blend can also scale magnitude). Sample weights fold in via row
        # scaling by sqrt(w). Falls back to tweedie-only on any failure.
        A = np.asarray(B, float); b = np.asarray(y, float)
        if sw is not None:
            rw = np.sqrt(np.clip(np.asarray(sw, float), 0, None))
            A = A * rw[:, None]; b = b * rw
        try:
            w, _ = nnls(A, b)
        except Exception:
            w = np.array([1.0, 0.0, 0.0])
        if (not np.isfinite(w).all()) or float(np.sum(w)) <= 0:
            w = np.array([1.0, 0.0, 0.0])
        return w

    def _inner_holdout_mask(tr):
        # #3: STRUCTURE-AWARE inner hold-out for the stacked blend = the latest
        # training MONTHS (out-of-time), not a random subset. A random subset leaks
        # under autocorrelation exactly like the random early-stopping split, so the
        # NNLS weights over-fit. Returns a boolean mask (True = inner hold-out).
        if ML_STRUCTURE_AWARE_INNER_SPLIT and "time_rank" in tr.columns:
            _t = tr["time_rank"].to_numpy(float)
            ur = np.unique(_t[np.isfinite(_t)])
            if ur.size >= 3:
                k = max(1, int(round(ur.size * ML_STACK_INNER_FRACTION)))
                return _t >= ur[-k]
        rng = np.random.RandomState(ML_RANDOM_STATE + 7)
        return rng.rand(len(tr)) < ML_STACK_INNER_FRACTION

    def _stack_blend(tr, te, p_tw_te):
        # #3: fit blend weights on an inner hold-out of the training rows so the base
        # predictions feeding the blend are out-of-sample (no leakage). The hold-out is
        # the latest training months (structure-aware) unless disabled. Tweedie is
        # refit on the inner-train slice; climatology/persistence are recomputed on it
        # too. The learned weights are then applied to the OUTER test predictions
        # (tweedie already supplied; clim/persistence recomputed on the full train).
        m = _inner_holdout_mask(tr)
        if int(m.sum()) < 200 or int((~m).sum()) < 200:
            return None
        tr_in, tr_ho = tr[~m], tr[m]
        Xin, Xho = _fold_features(tr_in, tr_ho)
        w_in = tr_in["_ml_w"].to_numpy() if ML_CONFIDENCE_WEIGHTING else None
        p_tw_ho, _, _ = _predict_tweedie(Xin, Xho, tr_in, w_tr=w_in)
        clim_in = tr_in.groupby("month_num")["wh_cover"].mean()
        p_cl_ho = tr_ho["month_num"].map(clim_in).fillna(tr_in["wh_cover"].mean()).to_numpy()
        p_pe_ho = (tr_ho["wh_cover_lag1"].fillna(0.0).to_numpy()
                   if "wh_cover_lag1" in tr_ho.columns else np.zeros(len(tr_ho)))
        B_ho = np.column_stack([p_tw_ho, p_cl_ho, p_pe_ho])
        sw = tr_ho["_ml_w"].to_numpy() if ML_CONFIDENCE_WEIGHTING else None
        wts = _nnls_blend(B_ho, tr_ho["wh_cover"].to_numpy(), sw=sw)
        clim_full = tr.groupby("month_num")["wh_cover"].mean()
        p_cl_te = te["month_num"].map(clim_full).fillna(tr["wh_cover"].mean()).to_numpy()
        p_pe_te = (te["wh_cover_lag1"].fillna(0.0).to_numpy()
                   if "wh_cover_lag1" in te.columns else np.zeros(len(te)))
        B_te = np.column_stack([np.asarray(p_tw_te, float), p_cl_te, p_pe_te])
        return np.clip(B_te @ wts, 0, 1), [float(x) for x in wts]

    def _baselines(tr, te):
        # Persistence + seasonal climatology, both with the #2 skill scores attached.
        # climatology's own msss_clim is 0 by construction (a useful sanity check).
        clim = tr.groupby("month_num")["wh_cover"].mean()
        pc = te["month_num"].map(clim).fillna(tr["wh_cover"].mean()).to_numpy()
        cell_mean_tr = tr.groupby("grid_id")["wh_cover"].mean()
        def _mk(name, pred):
            r = _row(name, None, None, len(tr), len(te), te["wh_cover"], pred)
            msss, sp_w, sp_a = _extra_skill(te, pred, pc, cell_mean_tr)
            r["msss_clim"], r["spearman_within_month"], r["spearman_anom"] = msss, sp_w, sp_a
            return r
        out = []
        if _lag_zero is not None:
            pp = te["wh_cover_lag1"].fillna(0.0).to_numpy()
            out.append(_mk("persistence", pp))
        out.append(_mk("climatology", pc))
        return out

    cover_rows, pres_rows = [], []

    def _run_fold(tr, te, kind, fold):
        Xtr, Xte = _fold_features(tr, te)
        y_tr = tr["wh_cover"].to_numpy()
        w_tr = tr["_ml_w"].to_numpy() if ML_CONFIDENCE_WEIGHTING else None
        # #2: leakage-free references for the skill scores (training rows only).
        clim_tr = tr.groupby("month_num")["wh_cover"].mean()
        clim_pred = te["month_num"].map(clim_tr).fillna(tr["wh_cover"].mean()).to_numpy()
        cell_mean_tr = tr.groupby("grid_id")["wh_cover"].mean()

        def _add(name, pred_te, extra=None):
            r = _row(name, kind, fold, len(tr), len(te), te["wh_cover"], pred_te)
            msss, sp_w, sp_a = _extra_skill(te, pred_te, clim_pred, cell_mean_tr)
            r["msss_clim"], r["spearman_within_month"], r["spearman_anom"] = msss, sp_w, sp_a
            if extra:
                r.update(extra)
            cover_rows.append(r)

        # cover models: raw test predictions + train predictions (for #3/#4 calibration)
        p_hur, p_hur_tr, p_pres, clf, reg_hur = _predict_hurdle(Xtr, Xte, tr, w_tr=w_tr)
        p_tw, p_tw_tr, reg_tw = _predict_tweedie(Xtr, Xte, tr, w_tr=w_tr)
        # Regime-matched calibration base: on the SPATIAL folds the held-out cells
        # have cell_clim = NaN, so recompute the TRAINING predictions with cell_clim
        # masked to NaN and calibrate on those -- the calibration regime then matches
        # the unseen-cell test block instead of the (seen) training rows. Leakage-free
        # (no refit, no test labels); temporal folds keep the ordinary train base.
        cal_hur, cal_tw = p_hur_tr, p_tw_tr
        if (ML_REGIME_MATCHED_CALIBRATION and kind == "spatial"
                and _CELL_CLIM in Xtr.columns):
            Xtr_m = Xtr.copy()
            Xtr_m[_CELL_CLIM] = np.nan
            cal_tw = np.clip(reg_tw.predict(Xtr_m), 0, 1)
            if reg_hur is not None:
                cal_hur = np.clip(clf.predict_proba(Xtr_m)[:, 1]
                                  * np.clip(reg_hur.predict(Xtr_m), 0, 1), 0, 1)
        for name, pred_te, cal_tr in [("hurdle_gbm", p_hur, cal_hur),
                                      ("tweedie_gbm", p_tw, cal_tw)]:
            _add(name, pred_te)
            if ML_AREA_CALIBRATION:
                k = _area_factor(y_tr, cal_tr)
                _add(name + "_cal", np.clip(k * pred_te, 0, 1), extra={"area_factor": k})
            if ML_ISOTONIC_CALIBRATION:                       # #4 monotone recalibration
                iso = _isotonic_map(y_tr, cal_tr, w_tr)
                if iso is not None:
                    _add(name + "_iso",
                         np.clip(iso.predict(np.clip(pred_te, 0, 1)), 0, 1))
        if ML_STACK_BLEND:                                    # #3 stacked blend
            sb = _stack_blend(tr, te, p_tw)
            if sb is not None:
                pred_stack, wts = sb
                _add("stack_blend", pred_stack,
                     extra={"w_tweedie": wts[0], "w_clim": wts[1], "w_persist": wts[2]})
        for b in _baselines(tr, te):
            b["kind"], b["fold"] = kind, fold; cover_rows.append(b)
        # stage-1 presence classification skill (where "will it bloom" lives)
        yte = te["wh_present"].astype(int).to_numpy()
        if np.unique(yte).size > 1:
            pres_rows.append(dict(kind=kind, fold=fold, n_test=len(te),
                                  prevalence=float(yte.mean()),
                                  auc=float(roc_auc_score(yte, p_pres)),
                                  avg_precision=float(average_precision_score(yte, p_pres)),
                                  brier=float(brier_score_loss(yte, p_pres))))

    # ---- leave-block-out spatial folds, with a buffer dead-zone (#1) -------------
    # Contiguous folds mean each held-out block is a compact patch; the buffer then
    # drops training cells within ML_SPATIAL_BUFFER_KM of ANY held-out cell so the
    # 500 m neighbour-lag / wind-advection features cannot read cover across the
    # held-out boundary. Cell centroids (x_km, y_km) are per-row, so the KD-tree
    # query on every training row's centroid excludes whole buffer cells at once.
    _cell_te_key = ["x_km", "y_km"]
    for f in sorted(ml_df["fold_sp"].unique()):
        te = ml_df[ml_df["fold_sp"] == f]
        tr = ml_df[ml_df["fold_sp"] != f]
        if ML_SPATIAL_BUFFER_KM and ML_SPATIAL_BUFFER_KM > 0 and len(te):
            _te_xy = te.groupby("grid_id")[_cell_te_key].first().to_numpy()
            _tree = cKDTree(_te_xy)
            _d, _ = _tree.query(tr[_cell_te_key].to_numpy(), k=1)
            _keep = _d > ML_SPATIAL_BUFFER_KM
            _n_drop = int((~_keep).sum())
            tr = tr[_keep]
        else:
            _n_drop = 0
        if len(tr) < ML_MIN_FOLD_TRAIN_ROWS or len(te) == 0:
            print(f"  [spatial fold {int(f)}] skipped "
                  f"(train {len(tr):,} < {ML_MIN_FOLD_TRAIN_ROWS} or empty test after buffer)")
            continue
        print(f"  [spatial fold {int(f)}] buffer dropped {_n_drop:,} boundary train rows; "
              f"train {len(tr):,}, test {len(te):,}")
        _run_fold(tr, te, "spatial", int(f))

    # ---- rolling-origin temporal folds, with a month embargo (#1) ----------------
    # wh_cover_lag1 / the rolling means straddle the train/test cut, so the month(s)
    # immediately before each test fold are dropped from training (embargo) to stop
    # the lag features peeking one step across the split.
    for f in sorted(r for r in ml_df["fold_tm"].unique() if r > 0):
        te = ml_df[ml_df["fold_tm"] == f]
        cut = int(ml_df.loc[ml_df["fold_tm"] == f, "time_rank"].min())
        emb = int(ML_TEMPORAL_EMBARGO_MONTHS)
        tr = ml_df[ml_df["time_rank"] < cut - emb]
        if len(tr) < ML_MIN_FOLD_TRAIN_ROWS or len(te) == 0:
            print(f"  [temporal fold {int(f)}] skipped "
                  f"(train {len(tr):,} < {ML_MIN_FOLD_TRAIN_ROWS} or empty test after embargo)")
            continue
        print(f"  [temporal fold {int(f)}] embargo {emb} month(s): train up to time_rank "
              f"{cut - emb - 1} ({len(tr):,} rows), test at time_rank {int(te['time_rank'].min())}+ "
              f"({len(te):,} rows)")
        _run_fold(tr, te, "temporal", int(f))

    ml_cv = pd.DataFrame(cover_rows)
    ml_presence_cv = pd.DataFrame(pres_rows)

    pd.set_option("display.float_format", lambda v: f"{v:.4f}")
    print("\n== Cover skill by block type (mean +/- sd across folds) ==")
    print(ml_cv.groupby(["kind", "model"])[["spearman", "rmse", "mae", "area_recovery"]]
              .agg(["mean", "std"]).round(4))

    # #2: skill scores that go beyond the pooled RMSE/Spearman. msss_clim > 0 means
    # the model beats seasonal climatology in squared error (the baseline that wins
    # raw RMSE); spearman_within_month is pure spatial hotspot ranking inside each
    # test month; spearman_anom ranks departures from each cell's training mean (the
    # real dynamic-skill test, NaN-heavy under spatial CV where cells are unseen).
    _skill_cols = ["msss_clim", "spearman_within_month", "spearman_anom"]
    if set(_skill_cols).issubset(ml_cv.columns):
        print("\n== Skill scores vs climatology + decomposed ranking (#2) ==")
        print("   msss_clim>0 beats climatology (MSE); within-month = spatial hotspot "
              "ranking; anom = dynamic skill")
        print(ml_cv.groupby(["kind", "model"])[_skill_cols].mean().round(4).to_string())

    print("\n== Per-fold cover skill ==")
    print(ml_cv.sort_values(["kind", "fold", "model"]).to_string(index=False))
    if len(ml_presence_cv):
        print("\n== Stage-1 presence classification skill (AUC / avg-precision / Brier) ==")
        print(ml_presence_cv.groupby("kind")[["auc", "avg_precision", "brier"]]
                  .agg(["mean", "std"]).round(4))

    # --- #3: area recovery, raw vs per-fold calibrated ---------------------------
    if ML_AREA_CALIBRATION:
        _cal_models = ["hurdle_gbm", "hurdle_gbm_cal", "hurdle_gbm_iso",
                       "tweedie_gbm", "tweedie_gbm_cal", "tweedie_gbm_iso", "stack_blend"]
        print("\n== Area recovery: raw vs per-fold calibrated (target = 1.0) ==")
        _ar = (ml_cv[ml_cv["model"].isin(_cal_models)]
                    .groupby(["kind", "model"])["area_recovery"].mean().unstack("model"))
        print(_ar.round(3).to_string())
        if "area_factor" in ml_cv.columns:
            print("\n== Mean applied area-calibration factor k (per fold, from training) ==")
            print(ml_cv.dropna(subset=["area_factor"])
                       .groupby(["kind", "model"])["area_factor"].mean().round(3).to_string())

    # --- verdict: does a workhorse beat BOTH baselines on Spearman AND RMSE? ------
    # Reported alongside the #2 climatology skill score (msss_clim): the RMSE rule is
    # the historical one; msss_clim>0 is the same test expressed as a skill score.
    print("\n== Verdict (model useful only where it beats BOTH baselines) ==")
    _mean = ml_cv.groupby(["kind", "model"])[["spearman", "rmse", "msss_clim",
                                              "spearman_within_month", "spearman_anom"]].mean()
    _verdict_models = ["hurdle_gbm", "hurdle_gbm_cal", "hurdle_gbm_iso",
                       "tweedie_gbm", "tweedie_gbm_cal", "tweedie_gbm_iso", "stack_blend"]
    for kind in [k for k in ["spatial", "temporal"] if k in ml_cv["kind"].unique()]:
        base_sp = _mean.loc[(kind,), "spearman"].reindex(["persistence", "climatology"]).max()
        base_rm = _mean.loc[(kind,), "rmse"].reindex(["persistence", "climatology"]).min()
        for m in [m for m in _verdict_models if (kind, m) in _mean.index]:
            sp, rm = _mean.loc[(kind, m), "spearman"], _mean.loc[(kind, m), "rmse"]
            msss = _mean.loc[(kind, m), "msss_clim"]
            win = "YES" if (sp > base_sp and rm < base_rm) else "no"
            print(f"  {kind:8s} {m:14s}: Spearman {sp:.3f} (vs best baseline {base_sp:.3f}) | "
                  f"RMSE {rm:.4f} (vs best baseline {base_rm:.4f}) | MSSS-clim {msss:+.3f} "
                  f"-> beats both: {win}")

    # --- #3: learned stacked-blend weights (NNLS; mean over folds) ----------------
    if ML_STACK_BLEND and {"w_tweedie", "w_clim", "w_persist"}.issubset(ml_cv.columns):
        print("\n== Stacked blend weights (NNLS on inner train hold-out; mean over folds) ==")
        print(ml_cv.dropna(subset=["w_tweedie"])
                   .groupby("kind")[["w_tweedie", "w_clim", "w_persist"]]
                   .mean().round(3).to_string())

    # --- save ---------------------------------------------------------------------
    _ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    _cell = int(globals().get("CELL_SIZE_M", 500))
    ml_cv.to_csv(OUTPUT_DIR / f"ml_workhorse_cv_{_cell}m_{_ts}.csv", index=False)
    if len(ml_presence_cv):
        ml_presence_cv.to_csv(OUTPUT_DIR / f"ml_workhorse_presence_cv_{_cell}m_{_ts}.csv", index=False)
    print(f"\nSaved ML workhorse CV tables to {OUTPUT_DIR} (suffix {_ts}).")


In [ ]:
# --- Track A: feature importance + skill figures ------------------------------
# Fit the hurdle on the FULL panel (all folds together) to read which features the
# trees actually use. Unlike the GAM's per-driver effects, tree gain is robust to
# concurvity: correlated features simply share the gain rather than exploding.
if not globals().get("RUN_ML_WORKHORSE", False):
    print("RUN_ML_WORKHORSE is False; skipping Track A importance / figures.")
else:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # NOWCAST: this Track-A importance/figures cell uses the contemporaneous
    # feature set. The forecast (t-1 cutoff) evaluation is in §15c-§15k.
    print("[NOWCAST] Track A importance/figures use the contemporaneous feature "
          "set; see §15c-§15k for the forecast (t-1 cutoff) evaluation.")

    # Match the CV model features: raw coords dropped (#1) and the cell climatology
    # added. Here the climatology is computed on the WHOLE panel purely to read
    # feature gain -- it is descriptive only and is NOT used for any scored prediction.
    _feat_imp = list(_feat_model)
    X_full = ml_df[_feat_model].copy()
    if ML_ADD_CELL_CLIMATOLOGY:
        _clim_full = ml_df.groupby("grid_id")["wh_cover"].mean()
        X_full[_CELL_CLIM] = ml_df["grid_id"].map(_clim_full).to_numpy()
        _feat_imp = _feat_imp + [_CELL_CLIM]
    # #1: mirror the hierarchical priors so their gain shows up here too. Computed
    # on the whole panel purely to READ importance -- NOT used for any scored prediction.
    if globals().get("ML_ADD_HIER_PRIOR", False) and globals().get("_hier_feats"):
        _gm_full = float(ml_df["wh_cover"].mean())
        for _feat, _idc in _hier_feats:
            if _idc in ml_df.columns:
                _m = ml_df.groupby(_idc)["wh_cover"].mean()
                X_full[_feat] = ml_df[_idc].map(_m).fillna(_gm_full).to_numpy()
                _feat_imp = _feat_imp + [_feat]

    _w_full = (ml_df["_ml_w"].to_numpy()
               if globals().get("ML_CONFIDENCE_WEIGHTING", False) and "_ml_w" in ml_df.columns
               else None)
    clf_full = _fit(_make_classifier(), X_full, ml_df["wh_present"].astype(int), w=_w_full)
    _pos = ml_df["wh_present"].astype(int).to_numpy() == 1
    reg_full = _fit(_make_regressor("regression"), X_full[_pos], ml_df.loc[_pos, "wh_cover"],
                    w=(None if _w_full is None else _w_full[_pos]))

    def _importance(model):
        imp = getattr(model, "feature_importances_", None)
        if imp is None:  # sklearn HistGB has no native importance -> permutation is too slow here
            return pd.Series(np.nan, index=_feat_imp)
        return pd.Series(np.asarray(imp, float), index=_feat_imp)

    imp_pres = _importance(clf_full)
    imp_cover = _importance(reg_full)
    fi = (pd.DataFrame({"presence_gain": imp_pres, "cover_gain": imp_cover})
            .fillna(0.0))
    for c in ["presence_gain", "cover_gain"]:
        s = fi[c].sum()
        fi[c + "_pct"] = 100 * fi[c] / s if s > 0 else np.nan
    fi = fi.sort_values("presence_gain", ascending=False)
    print("== Feature importance (gain %); top 20 by presence stage ==")
    print(fi[["presence_gain_pct", "cover_gain_pct"]].head(20).round(2).to_string())

    _ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    _cell = int(globals().get("CELL_SIZE_M", 500))
    fi.to_csv(OUTPUT_DIR / f"ml_workhorse_feature_importance_{_cell}m_{_ts}.csv")

    # ---- Figure 1: CV Spearman by model x block vs baselines --------------------
    try:
        piv = (ml_cv.groupby(["kind", "model"])["spearman"].mean().unstack("model"))
        order = [m for m in ["hurdle_gbm", "tweedie_gbm", "persistence", "climatology"]
                 if m in piv.columns]
        piv = piv[order]
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        ax0 = axes[0]
        piv.plot(kind="bar", ax=ax0, width=0.8)
        ax0.axhline(0, color="k", lw=0.8)
        ax0.set_ylabel("Spearman rho (held-out)"); ax0.set_xlabel("CV block type")
        ax0.set_title("NOWCAST cover-ranking skill vs baselines (higher = better)")
        ax0.legend(fontsize=8, ncol=2); ax0.tick_params(axis="x", rotation=0)

        # ---- Figure 2: stage-1 presence PR curve on the last temporal fold ------
        ax1 = axes[1]
        if "wh_cover_lag1" in ml_df and len(_months) > ML_N_TEMPORAL_FOLDS:
            from sklearn.metrics import precision_recall_curve, average_precision_score
            last = ml_df["fold_tm"].max()
            cut = ml_df.loc[ml_df["fold_tm"] == last, "time_rank"].min()
            tr = ml_df[ml_df["time_rank"] < cut]; te = ml_df[ml_df["fold_tm"] == last]
            _Xtr, _Xte = _fold_features(tr, te)
            _w_pr = (tr["_ml_w"].to_numpy()
                     if globals().get("ML_CONFIDENCE_WEIGHTING", False) and "_ml_w" in tr.columns
                     else None)
            _, _, p_pres, _, _ = _predict_hurdle(_Xtr, _Xte, tr, w_tr=_w_pr)
            yte = te["wh_present"].astype(int).to_numpy()
            if np.unique(yte).size > 1:
                pr, rc, _ = precision_recall_curve(yte, p_pres)
                ap = average_precision_score(yte, p_pres)
                ax1.plot(rc, pr, lw=2, label=f"hurdle stage-1 (AP={ap:.2f})")
                ax1.axhline(yte.mean(), color="grey", ls="--", lw=1,
                            label=f"prevalence={yte.mean():.2f}")
                ax1.set_xlabel("Recall"); ax1.set_ylabel("Precision")
                ax1.set_title("Presence detection, last temporal fold")
                ax1.set_ylim(0, 1); ax1.legend(fontsize=8)
        fig.tight_layout()
        _png = OUTPUT_DIR / f"ml_workhorse_skill_{_cell}m_{_ts}.png"
        fig.savefig(_png, dpi=130, bbox_inches="tight")
        plt.show()
        print(f"Saved skill figure -> {_png}")
    except Exception as _e:
        print("Figure step skipped:", _e)

    print("\nTrack A read: compare hurdle_gbm / tweedie_gbm to the §13 GAM CV. Lead any\n"
          "forecasting claim on the TEMPORAL block skill; report SPATIAL block skill\n"
          "separately as the harder cold-transfer test.")


## 15c. Predictive-validity redesign — nowcast vs forecast, honest CV, area metrics

*This block (§15c–§15k) is **additive**. The §15b Track-A workhorse above is the untouched **baseline**: its
legacy confidence weighting and its saved CV tables are unchanged. Everything below is namespaced `nf_*`,
feature-flagged, and leakage-checked, and every output is tagged **NOWCAST** or **FORECAST**.*

### Why these changes

The §15b evaluation feeds the model **contemporaneous** month-*t* drivers (rainfall, wind, temperature,
water quality), so its "skill" is really a **nowcast** — it assumes the current month's environment is
already known. A dissertation forecasting claim needs the harder question answered separately: *given only
what is observable now, what will next month look like?* §15d–§15k separate the two and put the temporal
evaluation on an honest footing.

### The forecast task, defined precisely

A one-month-ahead **forecast** for target month *t* is **issued at t−1**, information **cutoff = end of t−1**.
Two distinct cutoffs are enforced (embargo semantics):

- **`predictor_cutoff = t−1`** — latest month whose *observed* values may be used as **predictors**.
  Observed lags for *t* (e.g. `wh_cover_lag1` from *t−1*) are available at the issue date.
- **`label_cutoff = t−1−embargo`** — latest month whose *response labels* may enter model **training**. The
  embargo drops the row(s) immediately before *t* from **fitting** so their autoregressive features cannot
  straddle the split — but it does **not** forbid using *t−1*'s observed cover as a *predictor* for *t*.

**Forecast-safe** features = static covariates, calendar terms for *t*, and strictly-lagged terms
(`*_lag1/2/3`, `*_roll*_lag1`, neighbour/advection lags). **Nowcast-only** = the contemporaneous month-*t*
drivers (`time_varying_cols`, `effective_depth_m`, `wind_onshore_ms`, `wind_onshore_x_*`).

### Repeated one-month-ahead origins → multi-month windows

Temporal validation uses **repeated one-step-ahead origins**: for each target month *t*, train on months
`≤ label_cutoff` and predict the single month *t*. Multi-month "test windows" are then formed **by grouping
the one-step predictions for reporting** — never by holding out a multi-month block from a single origin.
This prevents autoregressive leakage: inside a multi-month held-out block the lag for a later month would be
the *observed* cover of an earlier held-out month. An optional **recursive** multi-step mode
(`NF_FORECAST_RECURSIVE`, default off) feeds the *predicted* cover back as the lag and is labelled
"recursive forecast" wherever it appears.

### The rest of the redesign

- **§15d** builds the nowcast/forecast feature sets and runs the feature-availability, forecast-timing and
  lag-alignment **validation checks**.
- **§15e** runs the repeated one-step evaluation for **both** modes, saving **all out-of-fold predictions**
  and reporting **pooled**, **per-origin/fold** and **multi-month-window** metrics, plus **valid-area-weighted
  monthly WH-area error & bias** (hectares). The old cover-sum ratio is kept only as
  `legacy_cover_sum_ratio`. Every learned component (climatologies, priors, imputers, thresholds, blend
  weights, calibrators) is fit **fold-locally** on training rows.
- **§15f** compares the absolute-cover model with a **Δcover** change model and an **ecological-transition**
  classifier.
- **§15g** fits the cover and presence-probability calibrators on **nested out-of-fold** training predictions.
- **§15h/§15i/§15j** (opt-in, default off) add **blocked hyperparameter tuning**, **held-out permutation +
  SHAP importance**, and **block-bootstrap model/estimation uncertainty**, applied to the selected final model.
- **§15k** summarises every new output with its NOWCAST/FORECAST tag.

### Confidence / class weighting (#5)

§15b down-weights only WH-present rows toward a 0.30 floor while absences stay at 1.0, so uncertain positives
are disproportionately suppressed. The new sections default to a **balanced** scheme (`NF_CONF_WEIGHT_MODE`)
that keeps the relative confidence ordering among present rows but rescales them so the positive class carries
the same **total** training influence as under uniform weights (`sum(w_present) = count(present)`, mean weight
= 1.0), with a raised floor. `"legacy"` reproduces the §15b scheme for a direct ablation.


In [ ]:
# =====================================================================
# 15c (helpers). Nowcast/forecast predictive-evaluation utilities
# =====================================================================
# These helpers power the new §15d-§15k predictive-evaluation sections. They are
# ADDITIVE: the §15b Track-A baseline (cell above) is left exactly as-is, and all
# names here are namespaced `nf_*` (nowcast/forecast) so nothing in §15b changes.
#
# They REUSE the §15b generics rather than re-implement them: `_fit`,
# `_make_classifier`, `_make_regressor`, `_predict_hurdle`, `_predict_tweedie`,
# `_metrics`, `_extra_skill`, `_area_factor`, `_isotonic_map`. Every learned or
# target-derived component (cell/stratum climatologies, imputers, thresholds,
# blend weights, calibrators) is fit on TRAINING rows only, inside each fold.
if not globals().get("RUN_ML_WORKHORSE", False):
    print("RUN_ML_WORKHORSE is False; skipping the nowcast/forecast helpers (§15c-§15k).")
else:
    import numpy as np
    import pandas as pd
    from scipy.stats import spearmanr

    _NF_CLIM = globals().get("_CELL_CLIM", "cell_clim")

    # -- forecast-task configuration (issue/target/cutoff + embargo semantics) ----
    # A one-month-ahead FORECAST for target month t is issued at t-1 with an
    # information cutoff at the end of t-1. Two DISTINCT cutoffs (embargo semantics):
    #   label_cutoff     = t-1-embargo : latest month whose RESPONSE labels may enter
    #                      model training (the embargo drops the rows just before t so
    #                      their autoregressive features cannot straddle the split).
    #   predictor_cutoff = t-1         : latest month whose OBSERVED values may be used
    #                      as PREDICTORS for target t (e.g. wh_cover_lag1 from t-1 is
    #                      available at the issue date even though the t-1 ROW is
    #                      embargoed out of training).
    NF_EMBARGO_MONTHS       = int(globals().get("ML_TEMPORAL_EMBARGO_MONTHS", 1))
    NF_MIN_TRAIN_MONTHS     = 6      # need at least this many available training months
    NF_EVAL_WINDOW_MONTHS   = 3      # group one-step results into multi-month report windows
    NF_FORECAST_RECURSIVE   = False  # multi-step: feed PREDICTED cover back as the lag
    NF_RECURSIVE_HORIZON    = 3      # months ahead for the (optional) recursive forecast

    # -- confidence / class weighting (#5) ----------------------------------------
    # The §15b baseline down-weights ONLY WH-present rows toward a 0.30 floor while
    # absences keep weight 1.0, so uncertain positives are disproportionately
    # suppressed (a de-facto class weight that biases toward under-prediction). The
    # "balanced" mode below decouples label-noise weighting from class balance: it
    # keeps the RELATIVE confidence ordering among present rows but rescales them so
    # sum(w_present) == count(present) -- the positive class carries the same TOTAL
    # training influence as under uniform weights, and the overall mean weight stays
    # 1.0 (absences are 1.0). "legacy" reproduces the §15b asymmetric scheme for a
    # direct ablation. Leakage-free: uses the response CONFIDENCE, never the cover.
    NF_CONF_WEIGHT_MODE = "balanced"   # "balanced" (#5 fix, default) | "legacy" | "uniform"
    NF_CONF_FLOOR       = 0.50         # raised floor (was 0.30) so positives are suppressed less
    NF_CONF_COL         = globals().get("ML_CONF_COL", "wh_conf_mean")

    def nf_row_weights(tr, mode=None):
        """Per-row TRAINING weights, fit on the given training rows only.

        balanced: raw = conf.clip(floor,1) for WH-bearing rows (1.0 for absence / NaN),
                  then present rows are rescaled so sum(w_present) == count(present) and
                  the overall mean weight == 1.0 (see note above).
        legacy:   the §15b scheme (present rows down-weighted, absences 1.0, no rescale).
        uniform:  all ones.
        """
        mode = NF_CONF_WEIGHT_MODE if mode is None else mode
        n = len(tr)
        if mode == "uniform" or NF_CONF_COL not in tr.columns:
            return np.ones(n, float)
        conf = pd.to_numeric(tr[NF_CONF_COL], errors="coerce").to_numpy()
        has = np.isfinite(conf)
        floor = NF_CONF_FLOOR if mode == "balanced" else float(globals().get("ML_CONF_WEIGHT_MIN", 0.30))
        raw = np.where(has, np.clip(conf, floor, 1.0), 1.0).astype(float)
        if mode == "legacy":
            return raw
        present = (tr["wh_present"].astype(float).to_numpy() == 1)
        w = raw.copy()
        s = float(w[present].sum())
        k = int(present.sum())
        if s > 0 and k > 0:
            w[present] *= (k / s)   # sum(w_present) == k  ->  positive class not net-suppressed
        return w

    # -- feature-availability split: nowcast (contemporaneous) vs forecast (t-1) ---
    def nf_build_feature_sets(model_feats, time_varying_cols, task4_feature_cols):
        """Return (nowcast_feats, forecast_feats, contemporaneous_set).

        NOWCAST uses the same features the §15b model uses (contemporaneous month-t
        drivers included). FORECAST drops every feature not observable by the issue
        cutoff t-1: the raw time-varying covariates and the contemporaneous exogenous
        / interaction terms. All strictly-lagged terms (*_lag1/2/3, *_roll*_lag1,
        neighbour/advection lags) and static/calendar terms are forecast-safe.
        """
        contemporaneous = set(time_varying_cols)
        contemporaneous |= {"effective_depth_m", "wind_onshore_ms"}
        contemporaneous |= {c for c in model_feats if c.startswith("wind_onshore_x_")}
        nowcast = list(model_feats)
        forecast = [c for c in model_feats if c not in contemporaneous]
        return nowcast, forecast, contemporaneous

    def nf_assert_forecast_safe(forecast_feats, contemporaneous_set):
        """Feature-availability check: no forecast feature may be contemporaneous."""
        bad = [c for c in forecast_feats if c in contemporaneous_set]
        assert not bad, f"forecast feature set contains contemporaneous features: {bad}"
        return True

    def nf_check_forecast_timing(origin, feat_list, contemporaneous_set, embargo=None,
                                 nowcast_ok=False):
        """Explicit issue-month / target-month / information-cutoff check.

        Asserts issue_month == target-1 (calendar), predictor_cutoff == issue_month,
        label_cutoff == issue_month - embargo, and (for a forecast) that no feature is
        contemporaneous. Distinguishes the LABEL cutoff from the PREDICTOR cutoff.
        """
        embargo = NF_EMBARGO_MONTHS if embargo is None else int(embargo)
        t = pd.Timestamp(origin["target_month"]).to_period("M")
        iss = pd.Timestamp(origin["issue_month"]).to_period("M")
        assert iss == t - 1, f"issue_month {iss} != target-1 {t - 1}"
        assert pd.Timestamp(origin["predictor_cutoff"]).to_period("M") == iss, \
            "predictor_cutoff != issue_month"
        assert pd.Timestamp(origin["label_cutoff"]).to_period("M") == iss - embargo, \
            "label_cutoff != issue_month - embargo"
        if not nowcast_ok:
            nf_assert_forecast_safe(feat_list, contemporaneous_set)
        return True

    # -- repeated one-month-ahead origins (grouped into windows for reporting) -----
    def nf_forecast_origins(months, min_train_months=None, embargo=None):
        """Calendar-based repeated one-step-ahead origins.

        For each target month t with >= min_train_months available training months at
        or before label_cutoff = t-(1+embargo), yield an origin dict. Calendar (not
        positional) arithmetic, so cloud-gap months are handled correctly: training
        uses the AVAILABLE months <= label_cutoff and the target's lag predictors use
        the observed value at predictor_cutoff = t-1 (NaN across a gap, then imputed).
        """
        min_train_months = NF_MIN_TRAIN_MONTHS if min_train_months is None else int(min_train_months)
        embargo = NF_EMBARGO_MONTHS if embargo is None else int(embargo)
        mm = pd.DatetimeIndex(sorted(pd.to_datetime(pd.unique(months))))
        origins = []
        for t in mm:
            tp = t.to_period("M")
            issue = (tp - 1).to_timestamp()
            label_cut = (tp - (1 + embargo)).to_timestamp()
            train_months = mm[mm <= label_cut]
            if len(train_months) < min_train_months:
                continue
            origins.append(dict(target_month=t, issue_month=issue,
                                 predictor_cutoff=issue, label_cutoff=label_cut,
                                 train_months=train_months.values))
        return origins

    def nf_assign_windows(target_months, window_months=None):
        """Map each target month to a consecutive multi-month reporting window id.

        Windows GROUP the one-step predictions for reporting only; they never define a
        held-out block, so no autoregressive value from another month inside a window
        is ever used as a feature (that would be leakage). Returns a Series id per row.
        """
        window_months = NF_EVAL_WINDOW_MONTHS if window_months is None else int(window_months)
        tm = pd.to_datetime(pd.Series(target_months).reset_index(drop=True))
        order = {m: i for i, m in enumerate(sorted(tm.unique()))}
        idx = tm.map(order)
        return (idx // window_months).astype(int).to_numpy()

    # -- fold-local design matrices (target-derived anchors fit on train only) -----
    def nf_build_design(tr, te, feat_list, add_clim=True, add_hier=True):
        """Design matrices for an explicit feature list (generalizes §15b _fold_features).

        The per-cell climatology and the hierarchical habitat/coarse priors are
        target-derived, so they are computed on the TRAINING rows of THIS fold only and
        mapped onto both splits (unseen keys -> NaN / training global mean). LightGBM /
        HistGradientBoosting handle NaN natively, so no separate imputer is needed; when
        one is used it is fit on tr only. This keeps every learned component fold-local.
        """
        Xtr = tr[feat_list].copy()
        Xte = te[feat_list].copy()
        if add_clim and globals().get("ML_ADD_CELL_CLIMATOLOGY", False):
            clim = tr.groupby("grid_id")["wh_cover"].mean()      # train-only anchor
            Xtr[_NF_CLIM] = tr["grid_id"].map(clim).to_numpy()
            Xte[_NF_CLIM] = te["grid_id"].map(clim).to_numpy()
        if add_hier and globals().get("ML_ADD_HIER_PRIOR", False) and globals().get("_hier_feats"):
            gm = float(tr["wh_cover"].mean())
            for _feat, _idc in _hier_feats:
                if _idc in tr.columns:
                    m = tr.groupby(_idc)["wh_cover"].mean()      # train-only stratum mean
                    Xtr[_feat] = tr[_idc].map(m).fillna(gm).to_numpy()
                    Xte[_feat] = te[_idc].map(m).fillna(gm).to_numpy()
        return Xtr, Xte

    # -- valid-area-weighted WH-area metrics (#4) ---------------------------------
    def nf_monthly_area_metrics(df, obs_col="obs", pred_col="pred", area_col="valid_area_m2"):
        """Valid-area-weighted monthly WH-area error & bias (hectares).

        WH area (ha) per cell-month = cover * valid_area_m2 / 1e4; monthly totals sum
        over cells. Returns per-month observed/predicted area plus the summary metrics.
        This REPLACES the old cover-sum ratio (kept only as legacy_cover_sum_ratio),
        which ignored valid_area_m2 and so was not a valid area error.
        """
        d = df[[ "month", obs_col, pred_col, area_col]].copy()
        d["obs_ha"]  = d[obs_col].clip(0, 1)  * d[area_col] / 1e4
        d["pred_ha"] = d[pred_col].clip(0, 1) * d[area_col] / 1e4
        g = d.groupby("month", as_index=False)[["obs_ha", "pred_ha"]].sum()
        g["err_ha"] = g["pred_ha"] - g["obs_ha"]
        with np.errstate(invalid="ignore", divide="ignore"):
            g["pct_err"] = 100.0 * g["err_ha"] / g["obs_ha"].replace(0, np.nan)
        obs_tot = float(g["obs_ha"].sum())
        summary = dict(
            n_months=int(len(g)),
            area_mae_ha=float(g["err_ha"].abs().mean()),
            area_rmse_ha=float(np.sqrt((g["err_ha"] ** 2).mean())),
            area_bias_ha=float(g["err_ha"].mean()),                 # mean signed error
            area_bias_pct=float(100.0 * g["err_ha"].sum() / obs_tot) if obs_tot > 0 else np.nan,
            area_ratio=float(g["pred_ha"].sum() / obs_tot) if obs_tot > 0 else np.nan,
        )
        return g, summary

    def nf_area_sanity_check(df, area_col="valid_area_m2", stored_ha_col="wh_area_ha",
                             cover_col="wh_cover", tol_ha=1e-6):
        """Area-calculation check: reconstructed cover*valid_area/1e4 ~ stored wh_area_ha."""
        if stored_ha_col not in df.columns:
            return None
        recon = df[cover_col].clip(0, 1) * df[area_col] / 1e4
        diff = float(np.nanmax(np.abs(recon.to_numpy() - df[stored_ha_col].to_numpy())))
        ok = np.isnan(diff) or diff <= max(tol_ha, 1e-3 * float(np.nanmean(df[stored_ha_col])))
        return dict(max_abs_diff_ha=diff, ok=bool(ok))

    print("§15c helpers ready: nf_row_weights, nf_build_feature_sets, nf_assert_forecast_safe, "
          "nf_check_forecast_timing, nf_forecast_origins, nf_assign_windows, nf_build_design, "
          "nf_monthly_area_metrics, nf_area_sanity_check.")
    print(f"  weighting mode={NF_CONF_WEIGHT_MODE} (floor {NF_CONF_FLOOR}); "
          f"embargo={NF_EMBARGO_MONTHS} month(s); min-train={NF_MIN_TRAIN_MONTHS} months; "
          f"report window={NF_EVAL_WINDOW_MONTHS} months; recursive={NF_FORECAST_RECURSIVE}.")


In [ ]:
# =====================================================================
# 15d. Nowcast vs forecast feature sets + temporal-alignment checks (#1,#2,#3)
# =====================================================================
if not globals().get("RUN_ML_WORKHORSE", False):
    print("RUN_ML_WORKHORSE is False; skipping §15d.")
else:
    import numpy as np
    import pandas as pd

    # The §15b model features (raw coords already dropped when ML_DROP_RAW_COORDS is
    # on). NOWCAST reuses these exactly; FORECAST drops the contemporaneous ones.
    _nf_model_feats = list(_feat_model)
    nowcast_feature_cols, forecast_feature_cols, nf_contemporaneous_set = \
        nf_build_feature_sets(_nf_model_feats, time_varying_cols,
                              globals().get("task4_feature_cols", []))

    # (a) feature-availability check: no forecast feature may be contemporaneous.
    nf_assert_forecast_safe(forecast_feature_cols, nf_contemporaneous_set)

    _drop = sorted(c for c in nowcast_feature_cols if c not in forecast_feature_cols)
    print(f"NOWCAST feature set: {len(nowcast_feature_cols)} features (contemporaneous drivers included).")
    print(f"FORECAST feature set: {len(forecast_feature_cols)} features "
          f"(observable by the t-1 issue cutoff).")
    print(f"\nDropped for FORECAST ({len(_drop)} contemporaneous / same-month features):")
    for c in _drop:
        print("   -", c)
    _fc_show = [c for c in forecast_feature_cols][:40]
    print(f"\nForecast-safe features kept ({len(forecast_feature_cols)}; showing up to 40):")
    for c in _fc_show:
        print("   +", c)

    # (b) forecast-timing check on a sample origin (issue/target/label/predictor cutoffs).
    _origins_probe = nf_forecast_origins(ml_df["month"])
    if _origins_probe:
        _o = _origins_probe[len(_origins_probe) // 2]
        nf_check_forecast_timing(_o, forecast_feature_cols, nf_contemporaneous_set, nowcast_ok=False)
        nf_check_forecast_timing(_o, nowcast_feature_cols, nf_contemporaneous_set, nowcast_ok=True)
        print(f"\n[timing check OK] sample origin: target {pd.Timestamp(_o['target_month']).date()} | "
              f"issue {pd.Timestamp(_o['issue_month']).date()} | "
              f"predictor_cutoff {pd.Timestamp(_o['predictor_cutoff']).date()} | "
              f"label_cutoff {pd.Timestamp(_o['label_cutoff']).date()} "
              f"(embargo {NF_EMBARGO_MONTHS} mo) | "
              f"{len(_o['train_months'])} training months.")
        print(f"[origins] {len(_origins_probe)} repeated one-step-ahead target months available "
              f"(min-train {NF_MIN_TRAIN_MONTHS}).")
    else:
        print("\n[timing check] too few months for any one-step-ahead origin; "
              "temporal forecast evaluation in §15e will be skipped.")

    # (c) lag-alignment check: wh_cover_lag1 must equal the TRUE previous-calendar-month
    #     observed cover (or NaN across a cloud gap), never a positional shift across a gap.
    _al = ml_df[["grid_id", "month", "wh_cover", "wh_cover_lag1"]].dropna(subset=["wh_cover_lag1"]).copy()
    if len(_al):
        _prev = (_al["month"].dt.to_period("M") - 1).dt.to_timestamp()
        _truth = _al.assign(_pm=_prev).merge(
            ml_df[["grid_id", "month", "wh_cover"]].rename(
                columns={"month": "_pm", "wh_cover": "_true_prev"}),
            on=["grid_id", "_pm"], how="left")
        _bad = int((~np.isclose(_truth["wh_cover_lag1"].to_numpy(),
                                _truth["_true_prev"].to_numpy(), equal_nan=True)).sum())
        print(f"\n[lag-alignment check] {len(_al):,} non-null wh_cover_lag1 rows; "
              f"{_bad} disagree with the true previous-calendar-month cover "
              f"({'OK' if _bad == 0 else 'INVESTIGATE'}).")

    # (d) area-calculation check: reconstructed cover*valid_area/1e4 ~ stored wh_area_ha.
    _asan = nf_area_sanity_check(ml_df)
    if _asan is not None:
        print(f"[area check] max |reconstructed - stored| wh_area_ha = "
              f"{_asan['max_abs_diff_ha']:.3e} ha ({'OK' if _asan['ok'] else 'INVESTIGATE'}).")


In [ ]:
# =====================================================================
# 15e. Repeated one-month-ahead predictive evaluation (NOWCAST + FORECAST)
# =====================================================================
# For each mode (nowcast = contemporaneous features; forecast = t-1 cutoff features)
# run the repeated one-step-ahead temporal origins AND the leave-block-out spatial CV
# (reused from §15b), scoring the §15b hurdle + Tweedie boosters and the
# persistence/climatology baselines. Every learned/target-derived component is fit
# fold-locally on training rows (an inline assert guards this). ALL out-of-fold
# predictions are saved; pooled, per-fold and multi-month-window metrics are reported
# alongside VALID-AREA-WEIGHTED monthly WH-area error & bias (hectares). Outputs are
# labelled NOWCAST / FORECAST.  (#2 #3 #4 #6-fold-local #12)
NF_EVAL_RUN      = True
NF_EVAL_SPATIAL  = True     # also run the leave-block-out spatial CV per feature mode

if not (globals().get("RUN_ML_WORKHORSE", False) and NF_EVAL_RUN):
    print("RUN_ML_WORKHORSE / NF_EVAL_RUN is False; skipping §15e.")
else:
    import numpy as np
    import pandas as pd
    from scipy.spatial import cKDTree
    from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

    _NF_COVER_MODELS = ["hurdle_gbm", "tweedie_gbm"]
    _NF_BASELINES    = ["persistence", "climatology"]

    def nf_eval_predict(tr, te, feat_list, w_mode=None):
        """Fold-local predictions for the hurdle + Tweedie boosters and the baselines.

        Reuses the §15b generics (_predict_hurdle / _predict_tweedie); the design
        matrices, weights, climatology and persistence are all built from `tr` only.
        Returns (preds_by_model, p_pres_te).
        """
        Xtr, Xte = nf_build_design(tr, te, feat_list)
        w_tr = nf_row_weights(tr, w_mode)
        p_hur, _, p_pres, _clf, _reg = _predict_hurdle(Xtr, Xte, tr, w_tr=w_tr)
        p_tw, _, _reg_tw = _predict_tweedie(Xtr, Xte, tr, w_tr=w_tr)
        preds = {"hurdle_gbm": np.asarray(p_hur, float),
                 "tweedie_gbm": np.asarray(p_tw, float)}
        clim = tr.groupby("month_num")["wh_cover"].mean()          # train-only climatology
        preds["climatology"] = te["month_num"].map(clim).fillna(tr["wh_cover"].mean()).to_numpy()
        if "wh_cover_lag1" in te.columns:                          # persistence (t-1 observed)
            preds["persistence"] = te["wh_cover_lag1"].fillna(0.0).to_numpy()
        return preds, np.asarray(p_pres, float)

    _oof_records = []   # long-form per-row OOF predictions
    _metric_rows = []   # per (mode, kind, fold, model) skill row
    _pres_rows   = []   # per (mode, kind, fold) stage-1 presence skill

    def nf_score_fold(tr, te, kind, mode, fold, preds, p_pres):
        # #6-fold-local references for the skill scores (training rows only).
        clim_tr = tr.groupby("month_num")["wh_cover"].mean()
        clim_pred = te["month_num"].map(clim_tr).fillna(tr["wh_cover"].mean()).to_numpy()
        cell_mean_tr = tr.groupby("grid_id")["wh_cover"].mean()
        obs = te["wh_cover"].to_numpy(float)
        tmon = te["month"].to_numpy()
        gid = te["grid_id"].to_numpy()
        varea = te["valid_area_m2"].to_numpy(float)
        for name, pred in preds.items():
            pred = np.clip(np.asarray(pred, float), 0, 1)
            sp, rmse, mae, ratio = _metrics(obs, pred)   # ratio = OLD cover-sum ratio
            msss, sp_w, sp_a = _extra_skill(te, pred, clim_pred, cell_mean_tr)
            _metric_rows.append(dict(
                mode=mode, kind=kind, fold=fold, model=name,
                n_train=len(tr), n_test=len(te),
                spearman=sp, rmse=rmse, mae=mae,
                msss_clim=msss, spearman_within_month=sp_w, spearman_anom=sp_a,
                legacy_cover_sum_ratio=ratio))     # continuity only; NOT a valid area metric
            for j in range(len(te)):
                _oof_records.append((int(gid[j]), tmon[j], kind, mode, fold, name,
                                     float(obs[j]), float(pred[j]), float(varea[j])))
        yte = te["wh_present"].astype(int).to_numpy()
        if np.unique(yte).size > 1:
            _pres_rows.append(dict(mode=mode, kind=kind, fold=fold, n_test=len(te),
                                   prevalence=float(yte.mean()),
                                   auc=float(roc_auc_score(yte, p_pres)),
                                   avg_precision=float(average_precision_score(yte, p_pres)),
                                   brier=float(brier_score_loss(yte, p_pres))))

    _modes = [("nowcast", nowcast_feature_cols, True),
              ("forecast", forecast_feature_cols, False)]

    # ---- repeated one-step-ahead TEMPORAL origins -------------------------------
    _origins = nf_forecast_origins(ml_df["month"])
    for mode, feats, nowcast_ok in _modes:
        n_done = 0
        for _o in _origins:
            nf_check_forecast_timing(_o, feats, nf_contemporaneous_set, nowcast_ok=nowcast_ok)
            tr = ml_df[ml_df["month"].isin(_o["train_months"])]
            te = ml_df[ml_df["month"] == _o["target_month"]]
            if len(tr) < ML_MIN_FOLD_TRAIN_ROWS or len(te) == 0:
                continue
            # fold-local assert: the training climatology must not see any test month.
            assert te["month"].min() > pd.Timestamp(_o["label_cutoff"]), \
                "temporal leakage: test month <= label_cutoff"
            preds, p_pres = nf_eval_predict(tr, te, feats)
            nf_score_fold(tr, te, "temporal", mode, pd.Timestamp(_o["target_month"]).strftime("%Y-%m"),
                          preds, p_pres)
            n_done += 1
        print(f"[{mode.upper():8s}] temporal: scored {n_done} one-step-ahead target months.")

    # ---- leave-block-out SPATIAL CV (reuse §15b contiguous folds + buffer) -------
    if NF_EVAL_SPATIAL:
        _cell_key = ["x_km", "y_km"]
        for mode, feats, nowcast_ok in _modes:
            n_done = 0
            for f in sorted(ml_df["fold_sp"].unique()):
                te = ml_df[ml_df["fold_sp"] == f]
                tr = ml_df[ml_df["fold_sp"] != f]
                if ML_SPATIAL_BUFFER_KM and ML_SPATIAL_BUFFER_KM > 0 and len(te):
                    _te_xy = te.groupby("grid_id")[_cell_key].first().to_numpy()
                    _d, _ = cKDTree(_te_xy).query(tr[_cell_key].to_numpy(), k=1)
                    tr = tr[_d > ML_SPATIAL_BUFFER_KM]
                if len(tr) < ML_MIN_FOLD_TRAIN_ROWS or len(te) == 0:
                    continue
                if not nowcast_ok:
                    nf_assert_forecast_safe(feats, nf_contemporaneous_set)
                preds, p_pres = nf_eval_predict(tr, te, feats)
                nf_score_fold(tr, te, "spatial", mode, int(f), preds, p_pres)
                n_done += 1
            print(f"[{mode.upper():8s}] spatial: scored {n_done} leave-block-out folds.")

    # ---- assemble OOF predictions + metric tables -------------------------------
    nf_oof = pd.DataFrame(_oof_records, columns=[
        "grid_id", "month", "kind", "mode", "fold", "model", "obs", "pred", "valid_area_m2"])
    nf_cv = pd.DataFrame(_metric_rows)
    nf_presence_cv = pd.DataFrame(_pres_rows)
    nf_oof["month"] = pd.to_datetime(nf_oof["month"])
    nf_oof["window"] = np.nan
    # multi-month reporting windows are assigned within each (mode, kind).
    for (mode, kind), sub in nf_oof.groupby(["mode", "kind"]):
        if kind == "temporal":
            nf_oof.loc[sub.index, "window"] = nf_assign_windows(sub["month"])

    pd.set_option("display.float_format", lambda v: f"{v:.4f}")
    _skill_cols = ["spearman", "rmse", "mae", "msss_clim",
                   "spearman_within_month", "spearman_anom", "legacy_cover_sum_ratio"]

    # (1) FOLD-LEVEL summaries (mean +/- sd across folds/origins), per mode.
    print("\n" + "=" * 78)
    print("== §15e cover skill — FOLD-LEVEL (mean +/- sd across folds), by mode & block ==")
    print("   NOWCAST = contemporaneous features | FORECAST = t-1 cutoff features")
    print(nf_cv.groupby(["mode", "kind", "model"])[_skill_cols[:6]]
              .agg(["mean", "std"]).round(4).to_string())

    # (2) POOLED metrics over ALL out-of-fold rows, per mode & block.
    def _pooled(df):
        o = df["obs"].to_numpy(float); p = df["pred"].clip(0, 1).to_numpy()
        from scipy.stats import spearmanr
        sp = spearmanr(o, p).correlation if np.nanstd(p) > 0 else np.nan
        return pd.Series(dict(n=len(df),
                              spearman=sp,
                              rmse=float(np.sqrt(np.mean((o - p) ** 2))),
                              mae=float(np.mean(np.abs(o - p)))))
    print("\n== §15e cover skill — POOLED over all OOF rows, by mode & block ==")
    print(nf_oof.groupby(["mode", "kind", "model"]).apply(_pooled).round(4).to_string())

    # (3) VALID-AREA-WEIGHTED monthly WH-area error & bias (hectares) — the #4 metric.
    _area_summ = []
    for (mode, kind, model), sub in nf_oof.groupby(["mode", "kind", "model"]):
        _, s = nf_monthly_area_metrics(sub)
        s.update(mode=mode, kind=kind, model=model)
        _area_summ.append(s)
    nf_area_summary = pd.DataFrame(_area_summ)[
        ["mode", "kind", "model", "n_months", "area_mae_ha", "area_rmse_ha",
         "area_bias_ha", "area_bias_pct", "area_ratio"]]
    print("\n== §15e VALID-AREA-WEIGHTED monthly WH-area error & bias (ha) — LEAD METRIC (#4) ==")
    print("   (legacy_cover_sum_ratio is reported above for continuity only)")
    print(nf_area_summary.sort_values(["kind", "mode", "model"]).round(3).to_string(index=False))

    # (4) MULTI-MONTH WINDOW summaries (temporal only): area error per report window.
    _win = nf_oof[nf_oof["kind"] == "temporal"].dropna(subset=["window"])
    if len(_win):
        _wrows = []
        for (mode, model, w), sub in _win.groupby(["mode", "model", "window"]):
            _, s = nf_monthly_area_metrics(sub)
            _wrows.append(dict(mode=mode, model=model, window=int(w),
                               months=sub["month"].dt.strftime("%Y-%m").nunique(),
                               area_mae_ha=s["area_mae_ha"], area_bias_ha=s["area_bias_ha"]))
        print("\n== §15e FORECAST/NOWCAST multi-month WINDOW area error (temporal one-step, grouped) ==")
        print(pd.DataFrame(_wrows).sort_values(["mode", "model", "window"]).round(3).to_string(index=False))

    # (5) stage-1 presence skill.
    if len(nf_presence_cv):
        print("\n== §15e stage-1 presence skill (AUC / avg-precision / Brier), by mode & block ==")
        print(nf_presence_cv.groupby(["mode", "kind"])[["auc", "avg_precision", "brier"]]
                  .agg(["mean", "std"]).round(4).to_string())

    # ---- save -------------------------------------------------------------------
    _ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    _cell = int(globals().get("CELL_SIZE_M", 500))
    nf_oof.to_csv(OUTPUT_DIR / f"nf_oof_predictions_{_cell}m_{_ts}.csv", index=False)
    nf_cv.to_csv(OUTPUT_DIR / f"nf_cv_foldlevel_{_cell}m_{_ts}.csv", index=False)
    nf_area_summary.to_csv(OUTPUT_DIR / f"nf_area_metrics_{_cell}m_{_ts}.csv", index=False)
    if len(nf_presence_cv):
        nf_presence_cv.to_csv(OUTPUT_DIR / f"nf_presence_cv_{_cell}m_{_ts}.csv", index=False)
    print(f"\nSaved OOF predictions + fold/area/presence tables to {OUTPUT_DIR} (suffix {_ts}).")
    print("READ: lead forecasting claims on the FORECAST temporal rows "
          "(msss_clim, spearman_anom, area_mae_ha); NOWCAST is the easier "
          "contemporaneous-information bound.")


In [ ]:
# =====================================================================
# 15f. Absolute-cover vs temporal-CHANGE and ecological-TRANSITION models (#7)
# =====================================================================
# The §15b/§15e models predict ABSOLUTE cover. Here two dynamics-focused
# alternatives are fit on the FORECAST feature set and the SAME repeated one-step
# origins, then compared head-to-head:
#   (a) delta-cover: predict d = wh_cover_t - wh_cover_lag1 and reconstruct
#       cover = clip(wh_cover_lag1 + d_hat, 0, 1)  -- targets the month-over-month
#       change directly (the real test of dynamic skill);
#   (b) ecological-transition classifier: 4 states from (wh_present_lag1, wh_present)
#       -- absent / colonisation / persistence / loss -- scored with macro-F1,
#       balanced accuracy, per-class precision/recall and a confusion matrix
#       (classes are imbalanced, so pooled accuracy alone would mislead).
# FORECAST-labelled throughout.
NF_CHANGE_RUN = True
if not (globals().get("RUN_ML_WORKHORSE", False) and NF_CHANGE_RUN):
    print("RUN_ML_WORKHORSE / NF_CHANGE_RUN is False; skipping §15f.")
elif not globals().get("_origins", None) and not nf_forecast_origins(ml_df["month"]):
    print("No one-step-ahead origins available; skipping §15f change/transition models.")
else:
    import numpy as np
    import pandas as pd
    from scipy.stats import spearmanr
    from sklearn.metrics import (f1_score, balanced_accuracy_score,
                                 precision_recall_fscore_support, confusion_matrix)

    _origins_ch = nf_forecast_origins(ml_df["month"])
    _feats_ch = forecast_feature_cols

    # ---- (a) delta-cover change model, compared with the absolute-cover forecast --
    _rows = []   # pooled OOF: one row per (target-month, cell) with valid lag
    for _o in _origins_ch:
        tr = ml_df[ml_df["month"].isin(_o["train_months"])]
        te = ml_df[ml_df["month"] == _o["target_month"]]
        # restrict to rows with an observed t-1 lag (the change target is defined there).
        tr = tr[tr["wh_cover_lag1"].notna()]
        te_valid = te[te["wh_cover_lag1"].notna()]
        if len(tr) < ML_MIN_FOLD_TRAIN_ROWS or len(te_valid) == 0:
            continue
        Xtr, Xte = nf_build_design(tr, te_valid, _feats_ch)
        w_tr = nf_row_weights(tr)
        # absolute-cover forecast reference (hurdle) on the same rows.
        p_abs, _, _p_pres, _clf, _reg = _predict_hurdle(Xtr, Xte, tr, w_tr=w_tr)
        # delta model: squared-error booster on d = cover_t - cover_lag1.
        d_tr = (tr["wh_cover"] - tr["wh_cover_lag1"]).to_numpy(float)
        reg_d = _fit(_make_regressor("regression"), Xtr, d_tr, w=w_tr,
                     t=tr["time_rank"].to_numpy() if "time_rank" in tr.columns else None)
        d_hat = reg_d.predict(Xte)
        cover_delta = np.clip(te_valid["wh_cover_lag1"].to_numpy(float) + d_hat, 0, 1)  # #7 clip [0,1]
        for gid, m, o, pa, pd_, va, lag in zip(
                te_valid["grid_id"], te_valid["month"], te_valid["wh_cover"],
                np.clip(p_abs, 0, 1), cover_delta, te_valid["valid_area_m2"],
                te_valid["wh_cover_lag1"]):
            _rows.append((int(gid), m, float(o), float(pa), float(pd_), float(va), float(lag)))

    nf_change_oof = pd.DataFrame(_rows, columns=[
        "grid_id", "month", "obs", "pred_abs_cover", "pred_delta_cover", "valid_area_m2", "lag1"])
    if len(nf_change_oof):
        nf_change_oof["month"] = pd.to_datetime(nf_change_oof["month"])
        o = nf_change_oof["obs"].to_numpy()
        # anomaly vs the persistence baseline (obs - lag1) -- the dynamic-skill target.
        oa = o - nf_change_oof["lag1"].to_numpy()
        def _cmp(col):
            p = nf_change_oof[col].to_numpy()
            pa = p - nf_change_oof["lag1"].to_numpy()
            sp = spearmanr(o, p).correlation if np.nanstd(p) > 0 else np.nan
            sp_anom = (spearmanr(oa, pa).correlation
                       if np.nanstd(oa) > 0 and np.nanstd(pa) > 0 else np.nan)
            rmse = float(np.sqrt(np.mean((o - p) ** 2)))
            _, s = nf_monthly_area_metrics(
                nf_change_oof.rename(columns={col: "pred"})[["month", "obs", "pred", "valid_area_m2"]])
            return dict(spearman=sp, spearman_anom=sp_anom, rmse=rmse,
                        area_mae_ha=s["area_mae_ha"], area_bias_ha=s["area_bias_ha"])
        _tbl = pd.DataFrame({
            "absolute_cover (hurdle)": _cmp("pred_abs_cover"),
            "delta_cover (change)":    _cmp("pred_delta_cover"),
        }).T
        print("== §15f FORECAST: absolute-cover vs delta-cover change model ==")
        print("   (same one-step rows with an observed t-1 lag; spearman_anom = dynamic skill)")
        print(_tbl.round(4).to_string())
        _ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        _cell = int(globals().get("CELL_SIZE_M", 500))
        nf_change_oof.to_csv(OUTPUT_DIR / f"nf_change_oof_{_cell}m_{_ts}.csv", index=False)
    else:
        print("§15f delta-cover: no eligible one-step rows with an observed lag; skipped.")

    # ---- (b) ecological-transition classifier -----------------------------------
    _STATE = {(0, 0): "absent", (0, 1): "colonisation", (1, 1): "persistence", (1, 0): "loss"}
    _CODES = {"absent": 0, "colonisation": 1, "persistence": 2, "loss": 3}
    _INV = {v: k for k, v in _CODES.items()}
    def _states(df):
        lag = df["wh_present_lag1"]; cur = df["wh_present"].astype(int)
        ok = lag.notna()
        s = pd.Series(index=df.index, dtype=object)
        for (l, c), name in _STATE.items():
            s[ok & (lag == l) & (cur == c)] = name
        return s

    _yte_all, _yhat_all = [], []
    for _o in _origins_ch:
        tr = ml_df[ml_df["month"].isin(_o["train_months"])].copy()
        te = ml_df[ml_df["month"] == _o["target_month"]].copy()
        tr["_state"] = _states(tr); te["_state"] = _states(te)
        tr = tr[tr["_state"].notna()]; te = te[te["_state"].notna()]
        if len(tr) < ML_MIN_FOLD_TRAIN_ROWS or len(te) == 0 or tr["_state"].nunique() < 2:
            continue
        Xtr, Xte = nf_build_design(tr, te, _feats_ch)
        ytr = tr["_state"].map(_CODES).astype(int).to_numpy()
        clf = _fit(_make_classifier(), Xtr, ytr, w=nf_row_weights(tr),
                   t=tr["time_rank"].to_numpy() if "time_rank" in tr.columns else None)
        yhat = clf.predict(Xte)
        _yte_all.append(te["_state"].map(_CODES).astype(int).to_numpy())
        _yhat_all.append(np.asarray(yhat, int))

    if _yte_all:
        yte = np.concatenate(_yte_all); yhat = np.concatenate(_yhat_all)
        labels = sorted(set(yte) | set(yhat))
        names = [_INV[i] for i in labels]
        macro_f1 = f1_score(yte, yhat, labels=labels, average="macro", zero_division=0)
        bal_acc = balanced_accuracy_score(yte, yhat)
        pr, rc, f1c, sup = precision_recall_fscore_support(
            yte, yhat, labels=labels, zero_division=0)
        print("\n== §15f FORECAST: ecological-transition classifier (pooled one-step OOF) ==")
        print(f"   macro-F1 = {macro_f1:.3f} | balanced accuracy = {bal_acc:.3f} "
              f"(imbalanced classes -> read per-class, not raw accuracy)")
        print(pd.DataFrame({"precision": pr, "recall": rc, "f1": f1c, "support": sup},
                           index=names).round(3).to_string())
        print("   confusion matrix (rows = true, cols = predicted):")
        print(pd.DataFrame(confusion_matrix(yte, yhat, labels=labels),
                           index=names, columns=names).to_string())
    else:
        print("\n§15f transition classifier: too few labelled transitions; skipped.")


In [ ]:
# =====================================================================
# 15g. Nested out-of-fold calibration (cover + presence probability) (#6)
# =====================================================================
# The §15b calibrators (scalar-k, isotonic) were fit on IN-FOLD training predictions
# (the model scoring its own fit rows), which is optimistic. Here every calibrator is
# fit on NESTED, out-of-fold predictions: inside each outer training set an inner
# out-of-time hold-out (latest training months, with the same embargo) yields
# inner-OOF predictions, and the calibrators are fit on those only, then applied to the
# outer test. For the presence probability we fit a simple 1-D logistic (Platt) or
# isotonic map DIRECTLY on the inner-OOF probabilities -> observed presence, rather
# than CalibratedClassifierCV, so the nested logic is explicit and the base classifier
# is never accidentally refit under an unsuitable internal CV scheme.
NF_CALIB_RUN            = True
NF_PRESENCE_CALIBRATOR  = "isotonic"   # "isotonic" | "logistic" (Platt), fit on inner-OOF probs
NF_CALIB_INNER_FRACTION = 0.25         # share of training MONTHS used as the inner out-of-time hold-out
if not (globals().get("RUN_ML_WORKHORSE", False) and NF_CALIB_RUN):
    print("RUN_ML_WORKHORSE / NF_CALIB_RUN is False; skipping §15g.")
else:
    import numpy as np
    import pandas as pd
    from sklearn.isotonic import IsotonicRegression
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import brier_score_loss

    def _fit_presence_calibrator(p_inner, y_inner):
        """1-D calibrator mapping raw P(present) -> calibrated P(present), fit on the
        inner-OOF probabilities only. Returns a callable prob-vector -> prob-vector."""
        p = np.clip(np.asarray(p_inner, float), 1e-6, 1 - 1e-6)
        y = np.asarray(y_inner, int)
        if np.unique(y).size < 2:
            return lambda z: np.clip(np.asarray(z, float), 0, 1)
        if NF_PRESENCE_CALIBRATOR == "logistic":
            lr = LogisticRegression(C=1e6, solver="lbfgs")
            lr.fit(np.log(p / (1 - p)).reshape(-1, 1), y)             # Platt on the logit
            def _apply(z):
                z = np.clip(np.asarray(z, float), 1e-6, 1 - 1e-6)
                return lr.predict_proba(np.log(z / (1 - z)).reshape(-1, 1))[:, 1]
            return _apply
        iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
        iso.fit(p, y)
        return lambda z: np.clip(iso.predict(np.clip(np.asarray(z, float), 0, 1)), 0, 1)

    def _inner_oof(tr, feats):
        """Inner out-of-time hold-out -> inner-OOF hurdle predictions + presence probs."""
        months = sorted(pd.to_datetime(tr["month"].unique()))
        k = max(1, int(round(len(months) * NF_CALIB_INNER_FRACTION)))
        ho_months = months[-k:]
        cut = (pd.Timestamp(ho_months[0]).to_period("M") - NF_EMBARGO_MONTHS).to_timestamp()
        tr_in = tr[tr["month"] < cut]
        tr_ho = tr[tr["month"].isin(ho_months)]
        if len(tr_in) < ML_MIN_FOLD_TRAIN_ROWS or len(tr_ho) == 0:
            return None
        Xin, Xho = nf_build_design(tr_in, tr_ho, feats)
        p_hur, _, p_pres, _clf, _reg = _predict_hurdle(Xin, Xho, tr_in, w_tr=nf_row_weights(tr_in))
        return dict(obs=tr_ho["wh_cover"].to_numpy(float),
                    present=tr_ho["wh_present"].astype(int).to_numpy(),
                    hur=np.clip(p_hur, 0, 1), p_pres=np.asarray(p_pres, float))

    _origins_cal = nf_forecast_origins(ml_df["month"])
    _cal_rows, _cal_oof = [], []
    for mode, feats, nowcast_ok in [("nowcast", nowcast_feature_cols, True),
                                    ("forecast", forecast_feature_cols, False)]:
        recs, pres_y, pres_raw, pres_cal = [], [], [], []
        for _o in _origins_cal:
            tr = ml_df[ml_df["month"].isin(_o["train_months"])]
            te = ml_df[ml_df["month"] == _o["target_month"]]
            if len(tr) < ML_MIN_FOLD_TRAIN_ROWS or len(te) == 0:
                continue
            inner = _inner_oof(tr, feats)
            if inner is None:
                continue
            iso = _isotonic_map(inner["obs"], inner["hur"])          # cover isotonic (nested)
            kfac = _area_factor(inner["obs"], inner["hur"])          # cover scalar-k (nested)
            cal_pres = _fit_presence_calibrator(inner["p_pres"], inner["present"])
            Xtr, Xte = nf_build_design(tr, te, feats)
            p_hur, _, p_pres, _clf, _reg = _predict_hurdle(Xtr, Xte, tr, w_tr=nf_row_weights(tr))
            p_hur = np.clip(p_hur, 0, 1)
            m = te["month"].to_numpy(); va = te["valid_area_m2"].to_numpy(float)
            obs = te["wh_cover"].to_numpy(float)
            for j in range(len(te)):
                recs.append((m[j], obs[j], p_hur[j],
                             float(np.clip(iso.predict([p_hur[j]])[0], 0, 1)) if iso is not None else p_hur[j],
                             float(np.clip(kfac * p_hur[j], 0, 1)), va[j]))
            pres_y.append(te["wh_present"].astype(int).to_numpy())
            pres_raw.append(np.asarray(p_pres, float)); pres_cal.append(cal_pres(p_pres))
        if not recs:
            print(f"[{mode.upper()}] §15g: no eligible nested folds; skipped.")
            continue
        cdf = pd.DataFrame(recs, columns=["month", "obs", "raw", "iso", "scalar_k", "valid_area_m2"])
        cdf["month"] = pd.to_datetime(cdf["month"]); cdf["mode"] = mode
        _cal_oof.append(cdf)
        row = dict(mode=mode)
        for col in ["raw", "iso", "scalar_k"]:
            _, s = nf_monthly_area_metrics(cdf.rename(columns={col: "pred"}))
            row[f"area_mae_ha_{col}"] = s["area_mae_ha"]; row[f"area_bias_ha_{col}"] = s["area_bias_ha"]
        yb = np.concatenate(pres_y)
        row["brier_pres_raw"] = float(brier_score_loss(yb, np.clip(np.concatenate(pres_raw), 0, 1)))
        row["brier_pres_cal"] = float(brier_score_loss(yb, np.clip(np.concatenate(pres_cal), 0, 1)))
        _cal_rows.append(row)

    if _cal_rows:
        nf_calibration = pd.DataFrame(_cal_rows)
        pd.set_option("display.float_format", lambda v: f"{v:.4f}")
        print("== §15g nested out-of-fold calibration — cover area error & presence Brier ==")
        print("   cover monthly-area error (ha): raw vs nested isotonic vs nested scalar-k")
        print(nf_calibration[["mode", "area_mae_ha_raw", "area_mae_ha_iso", "area_mae_ha_scalar_k",
                              "area_bias_ha_raw", "area_bias_ha_iso", "area_bias_ha_scalar_k"]]
              .round(3).to_string(index=False))
        print("\n   presence probability Brier: raw vs nested calibrator "
              f"({NF_PRESENCE_CALIBRATOR}); lower = better")
        print(nf_calibration[["mode", "brier_pres_raw", "brier_pres_cal"]].round(4).to_string(index=False))
        _ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        _cell = int(globals().get("CELL_SIZE_M", 500))
        pd.concat(_cal_oof, ignore_index=True).to_csv(
            OUTPUT_DIR / f"nf_calibration_oof_{_cell}m_{_ts}.csv", index=False)
        nf_calibration.to_csv(OUTPUT_DIR / f"nf_calibration_summary_{_cell}m_{_ts}.csv", index=False)
        print(f"\nSaved nested-calibration OOF + summary to {OUTPUT_DIR} (suffix {_ts}).")


In [ ]:
# =====================================================================
# 15h. Blocked hyperparameter tuning (opt-in; default OFF) (#8)
# =====================================================================
# A small SEEDED randomized search over the Tweedie booster's hyperparameters,
# scored on the SAME blocked CV as the evaluation and on an objective ALIGNED with
# the three tasks that matter: within-month spatial ranking (spatial folds),
# temporal-change / anomaly ranking (one-step origins) and monthly HECTARE area
# error. Default OFF (heavy); the chosen config is exported for the analyst to fold
# into the §15b/§15e settings for the final model.
NF_TUNE_HYPERPARAMS = False
NF_TUNE_N_CONFIGS   = 20
NF_TUNE_WEIGHTS     = dict(within_month=1.0, anom=1.0, area=1.0)   # composite objective weights
if not (globals().get("RUN_ML_WORKHORSE", False) and NF_TUNE_HYPERPARAMS):
    print("NF_TUNE_HYPERPARAMS is False; skipping §15h (set True to run the blocked HP search).")
else:
    import numpy as np
    import pandas as pd
    from scipy.spatial import cKDTree
    try:
        import lightgbm as lgb
        _HAVE_LGB_H = True
    except Exception:
        _HAVE_LGB_H = False
        from sklearn.ensemble import HistGradientBoostingRegressor

    def _make_reg_cfg(p):
        if _HAVE_LGB_H:
            return lgb.LGBMRegressor(
                objective="tweedie", tweedie_variance_power=p["tw_power"],
                n_estimators=p["n_estimators"], learning_rate=p["lr"],
                num_leaves=p["num_leaves"], min_child_samples=p["min_child"],
                subsample=ML_SUBSAMPLE, subsample_freq=1, colsample_bytree=ML_COLSAMPLE,
                random_state=ML_RANDOM_STATE, n_jobs=-1, verbose=-1)
        return HistGradientBoostingRegressor(
            loss="poisson", max_iter=p["n_estimators"], learning_rate=p["lr"],
            max_leaf_nodes=p["num_leaves"], min_samples_leaf=p["min_child"],
            early_stopping=True, random_state=ML_RANDOM_STATE)

    rng = np.random.RandomState(ML_RANDOM_STATE)
    grid = dict(num_leaves=[31, 63, 95, 127], min_child=[100, 200, 400, 800],
                lr=[0.02, 0.03, 0.05, 0.08], n_estimators=[400, 700, 1000],
                tw_power=[1.1, 1.3, 1.5, 1.7])
    configs = [{k: rng.choice(v) for k, v in grid.items()} for _ in range(NF_TUNE_N_CONFIGS)]

    _origins_t = nf_forecast_origins(ml_df["month"])
    _cell_key = ["x_km", "y_km"]

    def _fit_pred(tr, te, params):
        Xtr, Xte = nf_build_design(tr, te, forecast_feature_cols)
        reg = _fit(_make_reg_cfg(params), Xtr, tr["wh_cover"], w=nf_row_weights(tr),
                   t=tr["time_rank"].to_numpy() if "time_rank" in tr.columns else None)
        return np.clip(reg.predict(Xte), 0, 1)

    rows = []
    for ci, params in enumerate(configs):
        within, anoms, arec = [], [], []
        # spatial folds -> within-month spatial ranking
        for f in sorted(ml_df["fold_sp"].unique()):
            te = ml_df[ml_df["fold_sp"] == f]; tr = ml_df[ml_df["fold_sp"] != f]
            if ML_SPATIAL_BUFFER_KM > 0 and len(te):
                _d, _ = cKDTree(te.groupby("grid_id")[_cell_key].first().to_numpy()).query(
                    tr[_cell_key].to_numpy(), k=1)
                tr = tr[_d > ML_SPATIAL_BUFFER_KM]
            if len(tr) < ML_MIN_FOLD_TRAIN_ROWS or len(te) == 0:
                continue
            pred = _fit_pred(tr, te, params)
            clim = tr.groupby("month_num")["wh_cover"].mean()
            cp = te["month_num"].map(clim).fillna(tr["wh_cover"].mean()).to_numpy()
            _, sp_w, _ = _extra_skill(te, pred, cp, tr.groupby("grid_id")["wh_cover"].mean())
            within.append(sp_w)
        # one-step temporal origins -> anomaly ranking + hectare area error
        arows = []
        for _o in _origins_t:
            tr = ml_df[ml_df["month"].isin(_o["train_months"])]; te = ml_df[ml_df["month"] == _o["target_month"]]
            if len(tr) < ML_MIN_FOLD_TRAIN_ROWS or len(te) == 0:
                continue
            pred = _fit_pred(tr, te, params)
            clim = tr.groupby("month_num")["wh_cover"].mean()
            cp = te["month_num"].map(clim).fillna(tr["wh_cover"].mean()).to_numpy()
            _, _, sp_a = _extra_skill(te, pred, cp, tr.groupby("grid_id")["wh_cover"].mean())
            anoms.append(sp_a)
            arows.append(pd.DataFrame({"month": te["month"].to_numpy(), "obs": te["wh_cover"].to_numpy(),
                                       "pred": pred, "valid_area_m2": te["valid_area_m2"].to_numpy()}))
        area_mae = np.nan
        if arows:
            _, s = nf_monthly_area_metrics(pd.concat(arows, ignore_index=True))
            area_mae = s["area_mae_ha"]
        rows.append(dict(config=ci, **{k: (float(v) if k != "n_estimators" else int(v))
                                       for k, v in params.items()},
                         within_month=float(np.nanmean(within)) if within else np.nan,
                         anom=float(np.nanmean(anoms)) if anoms else np.nan,
                         area_mae_ha=area_mae))

    tune = pd.DataFrame(rows)
    # composite objective (normalise area error to [0,1] across configs so it is comparable).
    _amax = np.nanmax(tune["area_mae_ha"]) if tune["area_mae_ha"].notna().any() else 1.0
    tune["objective"] = (NF_TUNE_WEIGHTS["within_month"] * tune["within_month"].fillna(0)
                         + NF_TUNE_WEIGHTS["anom"] * tune["anom"].fillna(0)
                         - NF_TUNE_WEIGHTS["area"] * (tune["area_mae_ha"] / (_amax or 1.0)).fillna(1.0))
    tune = tune.sort_values("objective", ascending=False).reset_index(drop=True)
    pd.set_option("display.float_format", lambda v: f"{v:.4f}")
    print("== §15h blocked HP tuning (FORECAST features; composite = within-month + anom - area) ==")
    print(tune.head(10).to_string(index=False))
    _best = tune.iloc[0]
    print(f"\nBest config: num_leaves={int(_best.num_leaves)}, min_child={int(_best.min_child)}, "
          f"lr={_best.lr:.3f}, n_estimators={int(_best.n_estimators)}, tw_power={_best.tw_power:.2f} "
          f"| objective={_best.objective:.4f}")
    nf_best_hp = {k: (int(_best[k]) if k in ("num_leaves", "min_child", "n_estimators") else float(_best[k]))
                  for k in ("num_leaves", "min_child", "lr", "n_estimators", "tw_power")}
    _ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    tune.to_csv(OUTPUT_DIR / f"nf_hp_tuning_{int(globals().get('CELL_SIZE_M',500))}m_{_ts}.csv", index=False)
    print(f"Saved HP-tuning leaderboard to {OUTPUT_DIR} (suffix {_ts}). "
          "Fold nf_best_hp into ML_* for the final model.")


In [ ]:
# =====================================================================
# 15i. Held-out permutation + SHAP importance (opt-in; default OFF) (#9)
# =====================================================================
# §15b/cell-57 read BUILT-IN tree gain (in-sample). Here we add HELD-OUT permutation
# importance (permute one feature at a time in each held-out fold and measure the drop
# in Spearman / rise in RMSE -- an out-of-sample importance) and, where available,
# SHAP (TreeExplainer on the LightGBM booster, mean|SHAP| on a subsample). Applied to
# the SELECTED FINAL model on the FORECAST feature set. Default OFF (slow).
NF_PERMUTATION_IMPORTANCE = False
NF_SHAP                   = False
NF_SHAP_SAMPLE            = 4000
if not (globals().get("RUN_ML_WORKHORSE", False)
        and (NF_PERMUTATION_IMPORTANCE or NF_SHAP)):
    print("NF_PERMUTATION_IMPORTANCE / NF_SHAP are False; skipping §15i.")
else:
    import numpy as np
    import pandas as pd
    from scipy.stats import spearmanr

    _feats_imp = list(forecast_feature_cols)
    _origins_i = nf_forecast_origins(ml_df["month"])

    # ---- held-out permutation importance on the cover (Tweedie) model -----------
    if NF_PERMUTATION_IMPORTANCE and _origins_i:
        rng = np.random.RandomState(ML_RANDOM_STATE)
        base_sp, drops = [], {c: [] for c in _feats_imp}
        for _o in _origins_i[-min(4, len(_origins_i)):]:   # last few origins keep it affordable
            tr = ml_df[ml_df["month"].isin(_o["train_months"])]; te = ml_df[ml_df["month"] == _o["target_month"]]
            if len(tr) < ML_MIN_FOLD_TRAIN_ROWS or len(te) < 20:
                continue
            Xtr, Xte = nf_build_design(tr, te, _feats_imp)
            reg = _fit(_make_regressor("tweedie"), Xtr, tr["wh_cover"], w=nf_row_weights(tr),
                       t=tr["time_rank"].to_numpy() if "time_rank" in tr.columns else None)
            obs = te["wh_cover"].to_numpy(float)
            base = np.clip(reg.predict(Xte), 0, 1)
            b_sp = spearmanr(obs, base).correlation if np.nanstd(base) > 0 else np.nan
            base_sp.append(b_sp)
            for c in _feats_imp:
                if c not in Xte.columns:
                    continue
                Xp = Xte.copy(); Xp[c] = rng.permutation(Xp[c].to_numpy())
                pp = np.clip(reg.predict(Xp), 0, 1)
                sp_p = spearmanr(obs, pp).correlation if np.nanstd(pp) > 0 else np.nan
                if np.isfinite(b_sp) and np.isfinite(sp_p):
                    drops[c].append(b_sp - sp_p)     # positive = feature helped (perm hurt skill)
        perm = (pd.Series({c: (np.mean(v) if v else np.nan) for c, v in drops.items()},
                          name="perm_spearman_drop")
                .sort_values(ascending=False))
        print("== §15i held-out permutation importance (FORECAST; mean Spearman drop; top 20) ==")
        print(f"   baseline held-out Spearman = {np.nanmean(base_sp):.3f}")
        print(perm.head(20).round(4).to_string())
        _ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        perm.to_csv(OUTPUT_DIR / f"nf_permutation_importance_{int(globals().get('CELL_SIZE_M',500))}m_{_ts}.csv")

    # ---- SHAP on the final cover model ------------------------------------------
    if NF_SHAP:
        try:
            import shap
            tr = ml_df; Xall, _ = nf_build_design(tr, tr.head(1), _feats_imp)
            reg = _fit(_make_regressor("tweedie"), Xall, tr["wh_cover"], w=nf_row_weights(tr))
            Xs = Xall.sample(min(NF_SHAP_SAMPLE, len(Xall)), random_state=ML_RANDOM_STATE)
            expl = shap.TreeExplainer(reg)
            sv = expl.shap_values(Xs)
            msv = pd.Series(np.abs(sv).mean(axis=0), index=Xs.columns,
                            name="mean_abs_shap").sort_values(ascending=False)
            print("\n== §15i SHAP mean|value| (FORECAST cover model; top 20) ==")
            print(msv.head(20).round(5).to_string())
            _ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
            msv.to_csv(OUTPUT_DIR / f"nf_shap_importance_{int(globals().get('CELL_SIZE_M',500))}m_{_ts}.csv")
        except Exception as _e:
            print("\n§15i SHAP skipped:", _e,
                  "\n  (install shap and use the LightGBM backend for TreeExplainer support).")


In [ ]:
# =====================================================================
# 15j. Predictive uncertainty — block bootstrap (opt-in; default OFF) (#10)
# =====================================================================
# Refit the FINAL cover model on B block-bootstrap resamples of the TRAINING data
# (resampling whole grid CELLS, so spatial autocorrelation is respected) and take the
# per-row spread of the resampled predictions as p10/p50/p90. Evaluated on the
# FORECAST one-step origins.
#
# IMPORTANT (wording): these intervals capture MODEL / ESTIMATION (parameter)
# uncertainty only -- the spread of the fitted prediction across resamples. They are
# NOT full predictive intervals unless observation/residual uncertainty is added. An
# optional residual-inflation step (NF_UNC_RESIDUAL_INFLATION) widens them by the
# fold residual spread for a fuller predictive interval; coverage is reported for both.
NF_UNCERTAINTY              = False
NF_UNC_B                    = 20
NF_UNC_RESIDUAL_INFLATION   = False
if not (globals().get("RUN_ML_WORKHORSE", False) and NF_UNCERTAINTY):
    print("NF_UNCERTAINTY is False; skipping §15j (set True for block-bootstrap intervals).")
else:
    import numpy as np
    import pandas as pd

    _origins_u = nf_forecast_origins(ml_df["month"])
    rng = np.random.RandomState(ML_RANDOM_STATE)
    recs = []
    for _o in _origins_u[-min(4, len(_origins_u)):]:
        tr = ml_df[ml_df["month"].isin(_o["train_months"])]; te = ml_df[ml_df["month"] == _o["target_month"]]
        if len(tr) < ML_MIN_FOLD_TRAIN_ROWS or len(te) == 0:
            continue
        cells = tr["grid_id"].unique()
        boot_preds = []
        for b in range(NF_UNC_B):
            samp = rng.choice(cells, size=len(cells), replace=True)     # block bootstrap over cells
            tr_b = pd.concat([tr[tr["grid_id"] == g] for g in samp], ignore_index=True)
            if len(tr_b) < ML_MIN_FOLD_TRAIN_ROWS:
                continue
            Xtr, Xte = nf_build_design(tr_b, te, forecast_feature_cols)
            reg = _fit(_make_regressor("tweedie"), Xtr, tr_b["wh_cover"], w=nf_row_weights(tr_b))
            boot_preds.append(np.clip(reg.predict(Xte), 0, 1))
        if not boot_preds:
            continue
        P = np.vstack(boot_preds)
        p10, p50, p90 = np.percentile(P, [10, 50, 90], axis=0)
        if NF_UNC_RESIDUAL_INFLATION:
            # widen by the in-sample residual spread (adds observation/residual uncertainty).
            resid_sd = float(np.std(tr["wh_cover"].to_numpy() - tr["wh_cover"].mean()))
            p10 = np.clip(p10 - 1.28 * resid_sd, 0, 1); p90 = np.clip(p90 + 1.28 * resid_sd, 0, 1)
        obs = te["wh_cover"].to_numpy(float)
        for j in range(len(te)):
            recs.append((int(te["grid_id"].iloc[j]), te["month"].iloc[j], float(obs[j]),
                         float(p10[j]), float(p50[j]), float(p90[j])))
    if recs:
        nf_uncertainty = pd.DataFrame(recs, columns=["grid_id", "month", "obs", "p10", "p50", "p90"])
        cov = float(((nf_uncertainty["obs"] >= nf_uncertainty["p10"]) &
                     (nf_uncertainty["obs"] <= nf_uncertainty["p90"])).mean())
        width = float((nf_uncertainty["p90"] - nf_uncertainty["p10"]).mean())
        kind = "full predictive (residual-inflated)" if NF_UNC_RESIDUAL_INFLATION else "model/estimation"
        print(f"== §15j FORECAST block-bootstrap uncertainty (B={NF_UNC_B}; {kind}) ==")
        print(f"   empirical 80% interval coverage = {cov:.3f} (target 0.80); "
              f"mean interval width = {width:.4f} cover units")
        print("   NOTE: model/estimation intervals understate full predictive uncertainty unless "
              "NF_UNC_RESIDUAL_INFLATION is on.")
        _ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        nf_uncertainty.to_csv(
            OUTPUT_DIR / f"nf_uncertainty_{int(globals().get('CELL_SIZE_M',500))}m_{_ts}.csv", index=False)
        print(f"Saved bootstrap intervals to {OUTPUT_DIR} (suffix {_ts}).")
    else:
        print("§15j: no eligible one-step origins for bootstrap; skipped.")


## 15k. Summary — new predictive-validity outputs, labelled

Every table/figure below is tagged **NOWCAST** (contemporaneous month-*t* drivers used) or **FORECAST**
(only information observable by the *t−1* issue cutoff). **Lead any forecasting claim on the FORECAST temporal
rows** — `msss_clim`, `spearman_anom` and the valid-area-weighted `area_mae_ha` — because they answer *"given
only what is known now, what will next month look like?"* NOWCAST is the easier upper bound that assumes the
current month's environment is already known.

| Section | Requirement | Output (files prefixed `nf_…`) | Label |
|---|---|---|---|
| §15d | #1 #2 #3 | nowcast/forecast feature split; forecast-timing, feature-availability, lag-alignment, area checks | both |
| §15e | #2 #3 #4 #6 #12 | **all OOF predictions** (`nf_oof_predictions`), pooled + per-fold + multi-month-window metrics, **valid-area hectare area error & bias** (`nf_area_metrics`), presence skill | NOWCAST + FORECAST |
| §15f | #7 | absolute-cover vs **Δcover** change model; **ecological-transition** classifier (macro-F1, balanced acc, per-class P/R, confusion matrix) | FORECAST |
| §15g | #6 | **nested out-of-fold** cover + presence calibration (`nf_calibration_*`) | NOWCAST + FORECAST |
| §15h | #8 | blocked HP tuning leaderboard (`nf_hp_tuning`), opt-in | FORECAST |
| §15i | #9 | held-out **permutation** + **SHAP** importance, opt-in | FORECAST |
| §15j | #10 | **block-bootstrap** model/estimation uncertainty (`nf_uncertainty`), opt-in | FORECAST |

### Reading guide & caveats

- **Nowcast ≠ forecast.** A high NOWCAST score with a weak FORECAST score means the skill lives in the
  contemporaneous drivers, not in genuine lead-time predictability. Report both.
- **Area, not cover-sum.** The lead accuracy metric is valid-area-weighted **hectare** error/bias
  (`area_mae_ha`, `area_bias_ha`). `legacy_cover_sum_ratio` is retained only for continuity with the §15b
  baseline and is **not** a valid area metric (it ignores `valid_area_m2`).
- **No autoregressive leakage.** Multi-month windows are *groupings of one-step predictions*; within a window
  no month's observed cover is ever used as a feature for another month unless the explicitly-labelled
  `NF_FORECAST_RECURSIVE` mode is on (which feeds *predicted* cover back as the lag).
- **Fold-local everything.** Climatologies, hierarchical priors, imputers, presence thresholds, blend weights
  and calibrators are all fit on within-fold training rows; calibration (§15g) additionally uses nested
  out-of-fold predictions.
- **Uncertainty scope.** §15j intervals are **model/estimation** uncertainty. They are not full predictive
  intervals unless `NF_UNC_RESIDUAL_INFLATION` adds residual/observation spread.
- **Baseline preserved.** The §15b Track-A workhorse and its saved CV tables are unchanged; §15c–§15k are
  purely additive. The only edits to existing cells were the `shap` install line and the cell-57
  `_predict_hurdle` unpacking fix.

*As with the other feature-flagged additions in this notebook, §15c–§15k are written but not executed here
(no Colab/Drive/Earth-Engine/LightGBM runtime; outputs cleared). Re-run **§12 → §15b (Track A) → §15c…§15k**
top-to-bottom in Colab to regenerate the tables; §15h/§15i/§15j are opt-in via their `NF_*` flags.*


## 16. Map mean observed WH cover by grid cell

This checks whether the spatial pattern looks ecologically sensible.

Cells with consistently high cover should generally correspond to sheltered littoral/embayment zones if the classification and grid design are working.

In [ ]:
cell_summary = (
    panel.groupby("grid_id", as_index=False)
    .agg(
        mean_wh_cover=("wh_cover", "mean"),
        max_wh_cover=("wh_cover", "max"),
        occurrence_rate=("wh_present", "mean"),
        mean_wh_area_ha=("wh_area_ha", "mean"),
        n_months=("month", "nunique"),
    )
)

grid_summary = grid.merge(cell_summary, on="grid_id", how="left")

ax = grid_summary.to_crs("EPSG:4326").plot(
    column="mean_wh_cover",
    figsize=(9, 7),
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "No valid observations"},
)
ax.set_title("Mean observed WH cover by grid cell")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

ax = grid_summary.to_crs("EPSG:4326").plot(
    column="occurrence_rate",
    figsize=(9, 7),
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "No valid observations"},
)
ax.set_title("WH occurrence rate by grid cell")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

## 17. Export outputs

In addition to the panel, predictions, monthly summary, coefficients and grid summary, the raw Earth Engine covariate tables (`ee_monthly_cov`, `ee_cellmonth_cov`, `ee_static_cov`) are saved when they were produced, so the extracted drivers can be inspected or reused without re-querying Earth Engine.

In [ ]:
panel_csv = OUTPUT_DIR / f"wh_spatial_panel_{CELL_SIZE_M}m_{TEST_START}_to_{TEST_END}.csv"
monthly_summary_csv = OUTPUT_DIR / f"wh_monthly_summary_{CELL_SIZE_M}m_{TEST_START}_to_{TEST_END}.csv"
grid_summary_gpkg = OUTPUT_DIR / f"wh_grid_summary_{CELL_SIZE_M}m_{TEST_START}_to_{TEST_END}.gpkg"

panel.to_csv(panel_csv, index=False)
monthly_summary.to_csv(monthly_summary_csv, index=False)
grid_summary.to_file(grid_summary_gpkg, layer="grid_summary", driver="GPKG")

saved_paths = [panel_csv, monthly_summary_csv, grid_summary_gpkg]

# The one-stage GAM driver-model tables are saved next to these in §13 (driver_gam_tweedie_*).
# Also save the raw Earth Engine covariate tables when they were produced.
ee_covariate_tables = {
    "ee_monthly_covariates": ee_monthly_cov,
    "ee_cellmonth_covariates": ee_cellmonth_cov,
    "ee_static_covariates": ee_static_cov,
}
for name, table in ee_covariate_tables.items():
    if table is not None and len(table) > 0:
        out_csv = OUTPUT_DIR / f"{name}_{CELL_SIZE_M}m_{TEST_START}_to_{TEST_END}.csv"
        table.to_csv(out_csv, index=False)
        saved_paths.append(out_csv)

print("Saved:")
for p in saved_paths:
    print(p)


## 18. Interpretation checklist

Use this checklist before treating the results as environmental inference.

### Raster/classification checks

- Confirm that `WH_CLASS_VALUES` correctly identifies water hyacinth.
- Inspect several classified rasters visually.
- Check that missing/cloudy months are not being interpreted as true WH absence.
- **Implemented:** the response can use a **confidence-weighted** WH cover from the classifier probability rasters (`USE_PROBABILITY_RESPONSE`); the classifier-accuracy/confidence cell reports the measurement error entering the response (Priority 2).
- Compare S1-derived and S2-derived classifications separately if both are present.

### Earth Engine covariate checks

- Confirm Earth Engine initialised on the expected project (`EE_PROJECT`) and that section 8 reported the covariate columns you expected.
- The reanalysis wind & air-temperature layers (ERA5/ERA5-Land) are **AOI-mean monthly** values, so they vary in time but not between cells; treat them as regional forcing. CHIRPS rainfall (`EE_RAIN_PER_CELL`) and MODIS LST water-surface temperature (`EE_WATER_TEMP_PER_CELL`) are **per cell** by default, so they carry within-gulf gradients.
- Sentinel-2 turbidity (NDTI + red reflectance) is a **proxy**, not calibrated turbidity/TSS; cloudy months legitimately come back as missing and are imputed in the model.
- `water_temp_c` (MODIS LST over water) and the water-quality proxies are masked to JRC permanent water at `EE_WATER_OCCURRENCE_THRESHOLD`; cells that are mostly land will be missing.
- Chlorophyll-a now comes solely from Sentinel-3 OLCI (MCI/MPH). MODIS-Aqua chl-a is off by default (`chl_modis=False`; it ends 2022-02-28 and is unreliable in turbid inland water) and the Sentinel-2 NDCI chl-a proxy is off (`EE_S2_PRODUCTS={'turbidity'}`). Add 'chla' to EE_S2_PRODUCTS or set chl_modis=True to re-enable those cross-checks.
- Sentinel-3 chl-a (`chl_mci_s3`, `chl_mph_s3`) follows Kravitz et al. (2020): MCI/MPH band-difference algorithms on OLCI **TOA reflectance**, because GEE's OLCI asset lacks the geometry/meteorology needed for the paper's BRR/6SV corrections. The MPH calibration is BRR-derived applied to TOA reflectance, and none of the calibrations are tuned for Winam Gulf, so treat the mg/m³ values as indicative and locally re-calibrate against in-situ chl-a where possible.
- Check the bay-axis assumption: `EE_BAY_AXIS_BEARING_DEG` should match the orientation of the gulf you are modelling.

### Panel-design checks

- Test 500 m vs 1 km cells (1 km is much faster for the per-cell Earth Engine reductions).
- **Implemented:** the grid is restricted to a water/littoral mask via `WATER_MASK_SOURCE` (default `bathymetry`, with `MIN_WATER_FRACTION`); check the dropped-cell counts printed by the habitat filter and tune the threshold.
- Avoid pixel-level modelling because this creates extreme pseudo-replication.
- Check whether high-cover cells correspond to plausible embayments, sheltered shores or river-mouth zones.

### Modelling checks

- Use temporally blocked validation.
- **Implemented:** rolling-origin temporal CV **and** spatial block CV are in §13, on the one-stage GAM (Roberts et al. 2017; Ploton et al. 2020); report the spatial-block scores, not just the single split.
- **Implemented:** residual autocorrelation is checked on the one-stage GAM — spatial (Moran's I) and temporal (within-cell lag-1); if it stays significant, raise the `te(x_km, y_km)` basis, lean on the neighbour spatial-lag term (`wh_cover_neigh_lag1`), or add a within-cell AR1 term (Dormann et al. 2007).
- **Implemented:** a queen/rook neighbour spatial-lag of the lagged response (`NEIGHBOUR_LAG_ENABLED`) captures drift/colonisation (autologistic-style; Augustin et al. 1996).
- Check collinearity among rainfall, antecedent rainfall, turbidity, chlorophyll-a, temperature and lake level (the Earth Engine layers add several correlated predictors).
- Treat coefficients as associations, not proof of causation.
- The main model is now a one-stage Tweedie spatial GAM (§13); for the final dissertation, an ordered-Beta or Bayesian spatio-temporal model is the natural next step if time allows.

### Layers still to add from non-Earth-Engine sources

The Earth Engine extraction now covers rainfall, wind, temperature, water-quality proxies, river/shore geometry and catchment pressure. The remaining recommended drivers have no standard Earth Engine asset and should be supplied through `ENV_MONTHLY_CSV` / `SPATIAL_COVARIATES_CSV`:

- lake-level anomaly (DAHITI, Hydroweb, G-REALM or Jason/Sentinel altimetry);
- ENSO/IOD indices (ONI/Niño 3.4, Dipole Mode Index) for temporal models;
- in-situ nutrients (TN, TP, DIN, SRP) as mechanistic/confounder variables.

**Bathymetric depth is now included** as a static habitat covariate (`depth_m`, sampled from the Lake Victoria analytical bathymetry raster), which also defines the water mask. Adding **lake level** next — and a level×depth interaction — would complete the move from a technical prototype to a genuinely ecological spatio-temporal model.

**Key references:** Kravitz et al. (2020), *Remote Sensing of Environment* 237:111562 (Sentinel-3 OLCI chl-a, atmospheric-correction comparison); Matthews et al. (2012), *Remote Sensing of Environment* 124:637–652 (MPH algorithm). Methodological additions in this version: Barve et al. (2011), *Ecol. Modelling* 222:1810 (accessible-area/background restriction); Roberts et al. (2017), *Ecography* 40:913 and Ploton et al. (2020), *Nat. Commun.* 11:4540 (spatial/temporal block cross-validation); Dormann et al. (2007), *Ecography* 30:609 (residual spatial autocorrelation); Augustin et al. (1996), *J. Appl. Ecol.* 33:339 (autologistic neighbour term); Hirzel et al. (2006), *Ecol. Modelling* 199:142 (Boyce index); Lobo et al. (2008), *Glob. Ecol. Biogeogr.* 17:145 (limits of AUC).

## Appendix: superseded models (kept for the record — not run)

The cells below are the **earlier modelling attempts that were superseded by the one-stage
Tweedie GAM in §13**, kept here for transparency but **commented out so they do not execute**:

1. **Two-stage sklearn hurdle** (old §13 presence + §14 cover + §15 combined/area + §15b CV) —
   a regularised-logistic presence stage feeding a linear cover stage. On the held-out test block
   it reached only ROC-AUC ≈ 0.64, recall ≈ 0.09 and recovered ≈ 31 % of observed WH area, so it
   was retired in favour of modelling cover directly.
2. **`pygam` two-part GAM** (old §14d) — the first GAM attempt, limited by `pygam` having no
   Tweedie/Beta family and no true random effect (hence a logit-Gaussian cover stage and a
   cluster-bootstrap work-around). The §13 `mgcv` model supersedes it.
3. The old **model-diagnosis** summary for the hurdle.

To revisit any of these, un-comment the cell(s). They depend on objects built in §12; some also
reference each other, so re-run them in order.

##### (superseded) 13. Stage 1 model: WH presence/absence

This estimates whether each grid cell contains any mapped WH in each month.

For the final dissertation, you may want to compare this with a GAM, GLMM or Bayesian model. This notebook uses regularised logistic regression because it is fast, transparent and easy to test.

In [ ]:
# if y_train_presence.nunique() < 2:
#     raise RuntimeError(
#         "The training set contains only one presence/absence class. "
#         "Try a longer test period, larger AOI, different class codes, or coarser grid."
#     )
#
#
# def find_best_threshold(y_true, y_prob, metric="prevalence_match"):
#     """Sweep probability thresholds in [0.01, 0.99] and pick the best by ``metric``.
#
#     Supported metrics:
#       - "prevalence_match": pick the threshold whose predicted positive rate is
#         closest to the observed positive rate of ``y_true``. With calibrated
#         probabilities this matches the predicted number of present cells to the
#         expected number, which de-biases total predicted WH extent (the dominant
#         source of the area over-prediction). Recommended for extent estimation.
#       - "f1" / "precision" / "recall": classic operating-point metrics.
#
#     Returns (best, sweep):
#       - best:  pandas Series with threshold, precision, recall, f1, pred_pos_rate
#       - sweep: DataFrame of those metrics across all thresholds
#     """
#     y_true = np.asarray(y_true).astype(int)
#     y_prob = np.asarray(y_prob, dtype=float)
#     obs_pos_rate = float(y_true.mean())
#     thresholds = np.round(np.arange(0.01, 0.991, 0.01), 2)
#     rows = []
#     for t in thresholds:
#         pred = (y_prob >= t).astype(int)
#         pred_pos_rate = float(pred.mean())
#         rows.append({
#             "threshold": t,
#             "precision": precision_score(y_true, pred, zero_division=0),
#             "recall": recall_score(y_true, pred, zero_division=0),
#             "f1": f1_score(y_true, pred, zero_division=0),
#             "pred_pos_rate": pred_pos_rate,
#             "abs_prevalence_gap": abs(pred_pos_rate - obs_pos_rate),
#         })
#     sweep = pd.DataFrame(rows)
#     if metric == "prevalence_match":
#         # Smallest gap between predicted and observed positive rate; ties broken
#         # toward the higher threshold (fewer false positives).
#         best = sweep.sort_values(
#             ["abs_prevalence_gap", "threshold"], ascending=[True, False]
#         ).iloc[0].copy()
#     else:
#         score_col = metric if metric in {"f1", "precision", "recall"} else "f1"
#         best = sweep.loc[sweep[score_col].idxmax()].copy()
#     return best, sweep
#
#
# def make_base_presence_model():
#     """Uncalibrated presence pipeline.
#
#     class_weight="balanced" is intentionally NOT used: balancing inflates the
#     predicted probabilities away from the true (~17-20%) WH prevalence and was the
#     main driver of the WH-area over-prediction. The raw probabilities are instead
#     calibrated below so that their sum reflects the true expected number of
#     present cells.
#     """
#     return Pipeline(steps=[
#         ("imputer", SimpleImputer(strategy="median")),
#         ("scaler", StandardScaler()),
#         ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
#     ])
#
#
# # --- Fit and calibrate the presence model ------------------------------------
# # Preferred path: fit the base model on the fitting months, then calibrate the
# # probabilities on the held-out validation months (cv="prefit"). Calibrating on
# # the most recent pre-test months also partly counters the downward prevalence
# # drift into the test period. The presence threshold is then chosen on the same
# # calibrated validation probabilities.
# use_validation = (
#     len(train_val) > 0
#     and y_val_presence.nunique() == 2
#     and len(train_fit) > 0
#     and y_fit_presence.nunique() == 2
# )
#
# if use_validation:
#     base_fit_model = make_base_presence_model()
#     base_fit_model.fit(X_fit, y_fit_presence)
#     presence_model = CalibratedClassifierCV(
#         base_fit_model, method=PRESENCE_CALIBRATION_METHOD, cv="prefit"
#     )
#     presence_model.fit(X_val, y_val_presence)
#     val_presence_prob = presence_model.predict_proba(X_val)[:, 1]
#     best_threshold_row, threshold_sweep = find_best_threshold(
#         y_val_presence, val_presence_prob, metric=PRESENCE_THRESHOLD_METRIC
#     )
#     threshold_source = "validation months (calibrated)"
#     threshold_target_rate = float(y_val_presence.mean())
# else:
#     # Fallback: no usable validation block. Calibrate with internal CV on the
#     # full training period and tune the threshold on the training predictions.
#     presence_model = CalibratedClassifierCV(
#         make_base_presence_model(), method=PRESENCE_CALIBRATION_METHOD, cv=5
#     )
#     presence_model.fit(X_train, y_train_presence)
#     train_presence_prob = presence_model.predict_proba(X_train)[:, 1]
#     best_threshold_row, threshold_sweep = find_best_threshold(
#         y_train_presence, train_presence_prob, metric=PRESENCE_THRESHOLD_METRIC
#     )
#     threshold_source = "training predictions (OPTIMISTIC: no usable validation block)"
#     threshold_target_rate = float(y_train_presence.mean())
#
# selected_threshold = float(best_threshold_row["threshold"])
#
# # Higher-recall alternative: highest-recall threshold whose precision stays above
# # PRESENCE_MIN_PRECISION_FOR_RECALL.
# acceptable = threshold_sweep[threshold_sweep["precision"] >= PRESENCE_MIN_PRECISION_FOR_RECALL]
# if len(acceptable) > 0:
#     high_recall_row = acceptable.sort_values(["recall", "threshold"], ascending=[False, True]).iloc[0]
# else:
#     high_recall_row = None
#
# # --- Final predictions on the test period ------------------------------------
# # The calibrated model is the final predictor (the base is NOT refit on the full
# # train, so the validation months stay a genuine calibration hold-out).
# test_presence_prob = presence_model.predict_proba(X_test)[:, 1]
# test_presence_pred = test_presence_prob >= selected_threshold      # tuned threshold
# test_presence_pred_05 = test_presence_prob >= 0.5                  # legacy threshold
#
# # Uncalibrated model refit on the full training period, used only to report
# # interpretable standardised logistic-regression coefficients.
# coef_model = make_base_presence_model()
# coef_model.fit(X_train, y_train_presence)
#
# print(f"Presence model: LogisticRegression (no class balancing) + "
#       f"{PRESENCE_CALIBRATION_METHOD} probability calibration")
# print(f"Threshold tuned on: {threshold_source}")
# print(f"Threshold selection metric: {PRESENCE_THRESHOLD_METRIC}")
# print(f"Selected probability threshold: {selected_threshold:.2f}")
# print(f"  threshold-set precision={best_threshold_row['precision']:.3f} "
#       f"recall={best_threshold_row['recall']:.3f} f1={best_threshold_row['f1']:.3f} "
#       f"pred_pos_rate={best_threshold_row['pred_pos_rate']:.3f} "
#       f"(target observed rate={threshold_target_rate:.3f})")
# if high_recall_row is not None:
#     print(f"Higher-recall option (precision >= {PRESENCE_MIN_PRECISION_FOR_RECALL}): "
#           f"threshold={high_recall_row['threshold']:.2f} "
#           f"precision={high_recall_row['precision']:.3f} "
#           f"recall={high_recall_row['recall']:.3f} f1={high_recall_row['f1']:.3f}")
# else:
#     print(f"No threshold kept precision >= {PRESENCE_MIN_PRECISION_FOR_RECALL}; "
#           f"using the {PRESENCE_THRESHOLD_METRIC} threshold only.")
#
# print(f"\nPresence/absence classification report (TEST, tuned threshold = {selected_threshold:.2f}):")
# print(classification_report(y_test_presence, test_presence_pred, digits=3))
#
# print("Presence/absence classification report (TEST, legacy threshold = 0.50):")
# print(classification_report(y_test_presence, test_presence_pred_05, digits=3))
#
# if y_test_presence.nunique() == 2:
#     print("ROC AUC:", round(roc_auc_score(y_test_presence, test_presence_prob), 4))
#     print("Average precision (PR AUC):", round(average_precision_score(y_test_presence, test_presence_prob), 4))
#     print("Brier score:", round(brier_score_loss(y_test_presence, test_presence_prob), 4))
# else:
#     print("Test set contains only one class, so ROC AUC and average precision are undefined.")
#
# presence_coef = pd.DataFrame({
#     "feature": coef_model.named_steps["imputer"].get_feature_names_out(feature_cols),
#     "standardised_logit_coefficient": coef_model.named_steps["model"].coef_[0],
# }).sort_values("standardised_logit_coefficient", key=np.abs, ascending=False)
#
# display(presence_coef)
#

##### (superseded) Presence model diagnostics

Fuller diagnostics for the presence stage: confusion matrix, predicted positive
rates at the legacy (0.5) and tuned thresholds, precision/recall/F1 at the
selected threshold, and the distribution of predicted probabilities for truly
absent vs truly present test cells.


In [ ]:
# print("Selected threshold:", round(selected_threshold, 2), f"(tuned on {threshold_source})")
#
# if y_test_presence.nunique() == 2:
#     print("ROC AUC:", round(roc_auc_score(y_test_presence, test_presence_prob), 4))
#     print("PR AUC (average precision):", round(average_precision_score(y_test_presence, test_presence_prob), 4))
# else:
#     print("ROC AUC / PR AUC: undefined (single class in test).")
#
# print("\nConfusion matrix (TEST, tuned threshold); rows=observed, cols=predicted:")
# cm = confusion_matrix(y_test_presence, test_presence_pred, labels=[0, 1])
# display(pd.DataFrame(cm, index=["obs_absent", "obs_present"],
#                      columns=["pred_absent", "pred_present"]))
#
# obs_pos_rate = float(y_test_presence.mean())
# pred_pos_rate_05 = float(np.mean(test_presence_pred_05))
# pred_pos_rate_tuned = float(np.mean(test_presence_pred))
# prec_t = precision_score(y_test_presence, test_presence_pred, zero_division=0)
# rec_t = recall_score(y_test_presence, test_presence_pred, zero_division=0)
# f1_t = f1_score(y_test_presence, test_presence_pred, zero_division=0)
#
# print(f"\nObserved positive rate (TEST):              {obs_pos_rate:.4f}")
# print(f"Predicted positive rate @0.50:              {pred_pos_rate_05:.4f}")
# print(f"Predicted positive rate @{selected_threshold:.2f} (tuned):       {pred_pos_rate_tuned:.4f}")
# print(f"TEST precision/recall/F1 @ tuned threshold: {prec_t:.3f} / {rec_t:.3f} / {f1_t:.3f}")
#
# # Distribution of predicted probabilities by true class.
# prob_absent = test_presence_prob[y_test_presence.values == 0]
# prob_present = test_presence_prob[y_test_presence.values == 1]
#
# quantiles = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
# prob_quantile_table = pd.DataFrame({
#     "quantile": quantiles,
#     "prob_true_absent": [np.quantile(prob_absent, q) if len(prob_absent) else np.nan for q in quantiles],
#     "prob_true_present": [np.quantile(prob_present, q) if len(prob_present) else np.nan for q in quantiles],
# })
# print("\nPredicted-probability quantiles by true class (TEST):")
# display(prob_quantile_table)
#
# fig, ax = plt.subplots(figsize=(8, 4))
# bins = np.linspace(0, 1, 21)
# if len(prob_absent):
#     ax.hist(prob_absent, bins=bins, alpha=0.5, density=True, label="True absent")
# if len(prob_present):
#     ax.hist(prob_present, bins=bins, alpha=0.5, density=True, label="True present")
# ax.axvline(0.5, color="grey", linestyle=":", label="0.50 (legacy)")
# ax.axvline(selected_threshold, color="red", linestyle="--", label=f"{selected_threshold:.2f} (tuned)")
# ax.set_title("Predicted presence probability by true class (TEST)")
# ax.set_xlabel("Predicted probability of WH presence")
# ax.set_ylabel("Density")
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.show()
#
# print("Threshold sweep (validation/optimistic), every 5th row:")
# display(threshold_sweep.iloc[::5].reset_index(drop=True))
#

##### (superseded) Presence model: Boyce index and residual spatial autocorrelation

Prevalence-robust ranking (Boyce) plus a Moran's I check on the predictive model's TEST residuals (Priorities 3 and 4).

In [ ]:
# # ---------------------------------------------------------------------
# # Presence model: Boyce index + residual spatial autocorrelation (Priority 3, 4)
# # ---------------------------------------------------------------------
# # Two structure-aware checks on the predictive presence model, both on the
# # untouched TEST period:
# #   - Boyce index (Hirzel et al. 2006): a prevalence-robust ranking score that
# #     complements ROC/PR AUC (Lobo et al. 2008 show AUC alone can mislead).
# #   - Moran's I on per-cell mean TEST residuals: is spatial structure left in the
# #     residuals that the plain x_km/y_km terms do not absorb? Significant residual
# #     autocorrelation motivates the neighbour spatial-lag term and an explicit
# #     spatial effect (Dormann et al. 2007).
#
# if y_test_presence.nunique() == 2:
#     _boyce = boyce_index(y_test_presence.values, test_presence_prob, n_bins=BOYCE_N_BINS)
#     print(f"Boyce index (TEST): {_boyce:.3f}   "
#           "(+1 = presences ranked above absences; ~0 = random)")
# else:
#     print("Boyce index: undefined (single presence class in TEST).")
#
# # Residual spatial autocorrelation on the predictive presence model.
# try:
#     from libpysal.weights import KNN
#     from esda.moran import Moran
#     _HAVE_PYSAL = True
# except Exception as _exc:
#     _HAVE_PYSAL = False
#     print("libpysal/esda not available for residual Moran's I (pip install libpysal esda):",
#           repr(_exc))
#
# if _HAVE_PYSAL:
#     _res = test[["grid_id", "x_km", "y_km"]].copy()
#     _res["resid"] = y_test_presence.values - test_presence_prob      # observed - predicted prob
#     _agg = _res.groupby("grid_id", as_index=False).agg(
#         x_km=("x_km", "mean"), y_km=("y_km", "mean"), resid=("resid", "mean"))
#     if len(_agg) >= 10:
#         _w = KNN.from_array(_agg[["x_km", "y_km"]].values, k=min(8, len(_agg) - 1))
#         _w.transform = "r"
#         _mi = Moran(_agg["resid"].values, _w, permutations=999)
#         print(f"\nPresence residuals: Moran's I = {_mi.I:.4f} (p = {_mi.p_sim:.4f}) "
#               f"over {len(_agg):,} cells.")
#         if _mi.p_sim < 0.05:
#             print("  -> residual spatial autocorrelation remains: the neighbour spatial-lag "
#                   "term and/or an explicit spatial effect are warranted.")
#         else:
#             print("  -> no significant residual spatial autocorrelation in the predictive model.")
#     else:
#         print("Too few test cells for a residual Moran's I.")
#

##### (superseded) 14. Stage 2 model: WH cover where present

This estimates proportional WH cover for cell-months where WH is present.

The response is logit-transformed WH cover. Predictions are converted back to proportional cover.

In [ ]:
# def logit_clip(x, eps=1e-5):
#     x = np.clip(np.asarray(x, dtype=float), eps, 1 - eps)
#     return np.log(x / (1 - x))
#
#
# def inv_logit(z):
#     return 1 / (1 + np.exp(-z))
#
#
# positive_train = train[train["wh_present"]].copy()
# positive_test = test[test["wh_present"]].copy()
#
# if len(positive_train) < 10:
#     raise RuntimeError(
#         "Too few positive WH training rows for the cover model. "
#         "Try a longer period, larger AOI, coarser grid, or check WH_CLASS_VALUES."
#     )
#
# X_train_cover = positive_train[feature_cols]
# y_train_cover = logit_clip(positive_train["wh_cover"])
#
# cover_model = Pipeline(steps=[
#     ("imputer", SimpleImputer(strategy="median")),
#     ("scaler", StandardScaler()),
#     ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
# ])
#
# # Optionally weight the cover regression by per-cell classification confidence
# # (Priority 2): high-confidence WH observations carry more weight in the fit.
# cover_fit_kwargs = {}
# if WEIGHT_COVER_BY_CONFIDENCE and "wh_conf_mean" in positive_train.columns \
#         and positive_train["wh_conf_mean"].notna().any():
#     w = positive_train["wh_conf_mean"].fillna(positive_train["wh_conf_mean"].median()).clip(0.01, 1.0)
#     cover_fit_kwargs["model__sample_weight"] = w.values
#     print(f"Stage-2 cover model weighted by classification confidence "
#           f"(mean weight {w.mean():.3f}).")
#
# cover_model.fit(X_train_cover, y_train_cover, **cover_fit_kwargs)
#
# # Conditional cover predictions for all test rows.
# test_cover_logit_pred = cover_model.predict(X_test)
# test_cover_conditional_pred = inv_logit(test_cover_logit_pred)
#
# # Evaluate only where WH was truly present.
# if len(positive_test) > 0:
#     X_pos_test = positive_test[feature_cols]
#     pos_pred = inv_logit(cover_model.predict(X_pos_test))
#
#     rmse = np.sqrt(mean_squared_error(positive_test["wh_cover"], pos_pred))
#     mae = mean_absolute_error(positive_test["wh_cover"], pos_pred)
#
#     print("Positive-cover model performance, evaluated only where WH is observed present:")
#     print("RMSE:", round(rmse, 4))
#     print("MAE: ", round(mae, 4))
# else:
#     print("No positive WH rows in test set, so positive-cover performance is undefined.")
#
# cover_coef = pd.DataFrame({
#     "feature": cover_model.named_steps["imputer"].get_feature_names_out(feature_cols),
#     "standardised_cover_coefficient": cover_model.named_steps["model"].coef_,
# }).sort_values("standardised_cover_coefficient", key=np.abs, ascending=False)
#
# display(cover_coef)

##### (superseded) 15. Combined expected-cover predictions

The two stages are combined as:

`expected WH cover = probability of WH presence × conditional WH cover`

This is useful for estimating expected WH area in each grid cell and month.

In [ ]:
# test_preds = test[[
#     "grid_id", "month", "valid_area_m2", "wh_area_m2", "wh_area_ha",
#     "wh_cover", "wh_present"
# ]].copy()
#
# test_preds["pred_presence_prob"] = test_presence_prob
# test_preds["pred_cover_if_present"] = test_cover_conditional_pred
#
# # Soft "expected cover" = P(present) * E(cover | present). This is a valid
# # per-cell expectation, but summing it over every cell (including the many truly
# # absent cells, each carrying a small non-zero probability) systematically
# # OVER-states total WH area. It is therefore kept for reference only; the
# # reported area uses the hurdle estimator below.
# test_preds["pred_expected_cover"] = (
#     test_preds["pred_presence_prob"] * test_preds["pred_cover_if_present"]
# )
# test_preds["pred_wh_area_ha_expected"] = (
#     test_preds["pred_expected_cover"] * test_preds["valid_area_m2"] / 10_000
# )
#
# # --- Two-stage hurdle predictions (used for the reported area) ---------------
# # Apply the conditional cover model only where presence is predicted at the given
# # threshold, and assign zero cover to predicted-absent cells.
# test_preds["pred_present_05"] = (test_preds["pred_presence_prob"] >= 0.5).astype(int)
# test_preds["pred_present_tuned"] = (test_preds["pred_presence_prob"] >= selected_threshold).astype(int)
#
# test_preds["pred_wh_area_ha_hurdle_05"] = np.where(
#     test_preds["pred_present_05"] == 1,
#     test_preds["pred_cover_if_present"] * test_preds["valid_area_m2"] / 10_000,
#     0.0,
# )
# test_preds["pred_wh_area_ha_hurdle_tuned"] = np.where(
#     test_preds["pred_present_tuned"] == 1,
#     test_preds["pred_cover_if_present"] * test_preds["valid_area_m2"] / 10_000,
#     0.0,
# )
#
# # Primary reported WH-area prediction = hurdle at the tuned threshold.
# test_preds["pred_wh_area_ha"] = test_preds["pred_wh_area_ha_hurdle_tuned"]
#
# display(test_preds.head())
#
# monthly_pred_summary = (
#     test_preds.groupby("month", as_index=False)
#     .agg(
#         observed_wh_area_ha=("wh_area_ha", "sum"),
#         predicted_wh_area_ha=("pred_wh_area_ha_hurdle_tuned", "sum"),
#         predicted_wh_area_ha_expected=("pred_wh_area_ha_expected", "sum"),
#         observed_mean_cover=("wh_cover", "mean"),
#     )
# )
#
# display(monthly_pred_summary)
#
# # Test-period totals: reported hurdle estimate vs the biased expected-cover sum.
# obs_total = monthly_pred_summary["observed_wh_area_ha"].sum()
# hurdle_total = monthly_pred_summary["predicted_wh_area_ha"].sum()
# soft_total = monthly_pred_summary["predicted_wh_area_ha_expected"].sum()
# print(f"Test-period WH area (ha): observed = {obs_total:,.1f}")
# if obs_total > 0:
#     print(f"  reported (hurdle @ {selected_threshold:.2f} threshold): "
#           f"{hurdle_total:,.1f}  ({hurdle_total / obs_total:.0%} of observed)")
#     print(f"  reference expected-cover sum (biased high): "
#           f"{soft_total:,.1f}  ({soft_total / obs_total:.0%} of observed)")
# else:
#     print(f"  reported (hurdle): {hurdle_total:,.1f} | expected-cover sum: {soft_total:,.1f}")
#
# fig, ax = plt.subplots(figsize=(10, 4))
# ax.plot(monthly_pred_summary["month"], monthly_pred_summary["observed_wh_area_ha"],
#         marker="o", label="Observed")
# ax.plot(monthly_pred_summary["month"], monthly_pred_summary["predicted_wh_area_ha"],
#         marker="o", label=f"Predicted (hurdle @ {selected_threshold:.2f})")
# ax.plot(monthly_pred_summary["month"], monthly_pred_summary["predicted_wh_area_ha_expected"],
#         marker="o", linestyle=":", alpha=0.5, label="Expected-cover sum (biased high)")
# ax.set_title("Observed vs predicted WH area in test months")
# ax.set_xlabel("Month")
# ax.set_ylabel("WH area (ha)")
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.show()
#
# fig, ax = plt.subplots(figsize=(5, 5))
# ax.scatter(test_preds["wh_cover"], test_preds["pred_expected_cover"], s=8, alpha=0.4)
# ax.plot([0, 1], [0, 1], linestyle="--")
# ax.set_title("Observed vs predicted expected WH cover (per cell)")
# ax.set_xlabel("Observed WH cover")
# ax.set_ylabel("Predicted expected WH cover")
# ax.grid(True, alpha=0.3)
# plt.show()
#

##### (superseded) Test-month area comparison: legacy 0.5 vs tuned threshold

Observed WH area per test month compared with the hurdle prediction using the
legacy 0.5 threshold and the tuned threshold, with absolute and percentage
errors. This shows whether threshold tuning reduces the severe under-prediction
in the later test months.


In [ ]:
# area_compare = (
#     test_preds.groupby("month", as_index=False)
#     .agg(
#         observed_wh_area_ha=("wh_area_ha", "sum"),
#         pred_area_ha_05=("pred_wh_area_ha_hurdle_05", "sum"),
#         pred_area_ha_tuned=("pred_wh_area_ha_hurdle_tuned", "sum"),
#     )
# )
#
#
# def _pct_err(pred, obs):
#     return np.where(obs > 0, 100.0 * (pred - obs) / obs, np.nan)
#
#
# area_compare["abs_err_05_ha"] = (area_compare["pred_area_ha_05"] - area_compare["observed_wh_area_ha"]).abs()
# area_compare["pct_err_05"] = _pct_err(area_compare["pred_area_ha_05"].values, area_compare["observed_wh_area_ha"].values)
# area_compare["abs_err_tuned_ha"] = (area_compare["pred_area_ha_tuned"] - area_compare["observed_wh_area_ha"]).abs()
# area_compare["pct_err_tuned"] = _pct_err(area_compare["pred_area_ha_tuned"].values, area_compare["observed_wh_area_ha"].values)
#
# # Rounded copy for display.
# area_compare_display = area_compare.copy()
# for c in ["observed_wh_area_ha", "pred_area_ha_05", "pred_area_ha_tuned", "abs_err_05_ha", "abs_err_tuned_ha"]:
#     area_compare_display[c] = area_compare_display[c].round(2)
# for c in ["pct_err_05", "pct_err_tuned"]:
#     area_compare_display[c] = area_compare_display[c].round(1)
#
# print("Test-month WH area: observed vs hurdle predictions (0.5 vs tuned threshold)")
# display(area_compare_display)
#
# fig, ax = plt.subplots(figsize=(10, 4))
# ax.plot(area_compare["month"], area_compare["observed_wh_area_ha"], marker="o", label="Observed")
# ax.plot(area_compare["month"], area_compare["pred_area_ha_05"], marker="o", label="Predicted (hurdle @0.5)")
# ax.plot(area_compare["month"], area_compare["pred_area_ha_tuned"], marker="o",
#         label=f"Predicted (hurdle @{selected_threshold:.2f})")
# ax.set_title("Observed vs hurdle-predicted WH area by test month")
# ax.set_xlabel("Month")
# ax.set_ylabel("WH area (ha)")
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.show()
#
# print(f"Test-period totals (ha): observed={area_compare['observed_wh_area_ha'].sum():.1f}, "
#       f"hurdle@0.5={area_compare['pred_area_ha_05'].sum():.1f}, "
#       f"hurdle@tuned={area_compare['pred_area_ha_tuned'].sum():.1f}")
#

##### (superseded) 15b. Structure-aware cross-validation (spatial + temporal blocking)

Rolling-origin temporal CV and spatial block CV give honest, dependence-aware skill estimates (Roberts et al. 2017; Ploton et al. 2020).

In [ ]:
# # ---------------------------------------------------------------------
# # Structure-aware cross-validation: rolling-origin + spatial block (Priority 4)
# # ---------------------------------------------------------------------
# # A single train/test split gives one, high-variance skill estimate. Here the
# # hurdle (presence + conditional-cover) is re-evaluated under two blocking
# # schemes that respect the panel's dependence structure (Roberts et al. 2017):
# #   - rolling-origin temporal CV: expanding-window folds that always predict the
# #     future, so temporal autocorrelation cannot leak backwards;
# #   - spatial block CV: leave-block-out over coarse spatial blocks, testing
# #     transfer to unseen areas (the optimistic bias non-spatial CV hides;
# #     Ploton et al. 2020).
# # Each fold reports ROC/PR AUC, Brier, Boyce and total-area recovery.
#
# def _make_cover_pipeline():
#     return Pipeline(steps=[
#         ("imputer", SimpleImputer(strategy="median")),
#         ("scaler", StandardScaler()),
#         ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
#     ])
#
#
# def _cv_fit_eval(tr, te, cols):
#     """Fit presence + conditional-cover on `tr`, evaluate on `te`. Returns a dict."""
#     if len(tr) == 0 or len(te) == 0 or tr["wh_present"].nunique() < 2:
#         return None
#     ytr = tr["wh_present"].astype(int)
#     yte = te["wh_present"].astype(int)
#     pm = make_base_presence_model().fit(tr[cols], ytr)
#     p_te = pm.predict_proba(te[cols])[:, 1]
#     p_tr = pm.predict_proba(tr[cols])[:, 1]
#     thr = float(find_best_threshold(ytr, p_tr, metric=PRESENCE_THRESHOLD_METRIC)[0]["threshold"])
#     pred_present = (p_te >= thr).astype(int)
#
#     out = {"n_train": len(tr), "n_test": len(te),
#            "test_prev": float(yte.mean()), "threshold": round(thr, 2)}
#     if yte.nunique() == 2:
#         out["roc_auc"] = roc_auc_score(yte, p_te)
#         out["pr_auc"] = average_precision_score(yte, p_te)
#         out["brier"] = brier_score_loss(yte, p_te)
#         out["boyce"] = boyce_index(yte.values, p_te, n_bins=BOYCE_N_BINS)
#
#     pos = tr[tr["wh_present"]]
#     if len(pos) >= 10:
#         cm = _make_cover_pipeline().fit(pos[cols], logit_clip(pos["wh_cover"]))
#         cover_te = inv_logit(cm.predict(te[cols]))
#         area_obs = float((te["wh_cover"] * te["valid_area_m2"] / 1e4).sum())
#         area_pred = float(np.where(pred_present == 1, cover_te * te["valid_area_m2"] / 1e4, 0.0).sum())
#         out["area_obs_ha"] = round(area_obs, 1)
#         out["area_pred_ha"] = round(area_pred, 1)
#         out["area_recovery"] = round(area_pred / area_obs, 3) if area_obs > 0 else np.nan
#     return out
#
#
# def _summarise_cv(rows, label):
#     if not rows:
#         print(f"{label}: no evaluable folds.")
#         return None
#     df = pd.DataFrame(rows)
#     print(f"\n{label}: {len(df)} folds")
#     display(df)
#     _num = [c for c in ["roc_auc", "pr_auc", "brier", "boyce", "area_recovery"] if c in df.columns]
#     if _num:
#         print(f"{label} mean +/- sd:")
#         display(pd.concat([df[_num].mean().rename("mean"),
#                            df[_num].std().rename("sd")], axis=1).round(3))
#     return df
#
# cv_cols = feature_cols
# _all_months = np.array(sorted(model_df["month"].unique()))
#
# # --- Rolling-origin (expanding-window) temporal CV ---
# temporal_cv = None
# if CV_TEMPORAL_ENABLED and len(_all_months) >= CV_TEMPORAL_MIN_TRAIN_MONTHS + CV_TEMPORAL_N_FOLDS:
#     rows = []
#     first_cut = max(CV_TEMPORAL_MIN_TRAIN_MONTHS, len(_all_months) - CV_TEMPORAL_N_FOLDS)
#     cut_points = range(first_cut, len(_all_months))
#     for k, cut in enumerate(cut_points, start=1):
#         tr = model_df[model_df["month"] < _all_months[cut]]
#         te = model_df[model_df["month"] == _all_months[cut]]
#         res = _cv_fit_eval(tr, te, cv_cols)
#         if res is not None:
#             res = {"fold": k, "test_month": pd.Timestamp(_all_months[cut]).date(), **res}
#             rows.append(res)
#     temporal_cv = _summarise_cv(rows, "Rolling-origin temporal CV")
# else:
#     print("Rolling-origin temporal CV skipped (not enough months).")
#
# # --- Spatial block CV (leave-block-out) ---
# spatial_cv = None
# if CV_SPATIAL_ENABLED:
#     bx = np.floor(model_df["x_km"].values / CV_SPATIAL_BLOCK_KM).astype(int)
#     by = np.floor(model_df["y_km"].values / CV_SPATIAL_BLOCK_KM).astype(int)
#     block_id = pd.Series(list(zip(bx, by)), index=model_df.index)
#     uniq_blocks = pd.unique(block_id)
#     rng = np.random.default_rng(RANDOM_STATE)
#     fold_of_block = {b: (i % CV_SPATIAL_N_FOLDS)
#                      for i, b in enumerate(rng.permutation(uniq_blocks))}
#     fold_assign = block_id.map(fold_of_block)
#     rows = []
#     for f in range(CV_SPATIAL_N_FOLDS):
#         te_mask = (fold_assign == f).values
#         tr = model_df[~te_mask]
#         te = model_df[te_mask]
#         res = _cv_fit_eval(tr, te, cv_cols)
#         if res is not None:
#             res = {"fold": f + 1, "n_test_cells": int(te["grid_id"].nunique()), **res}
#             rows.append(res)
#     spatial_cv = _summarise_cv(rows, f"Spatial block CV ({CV_SPATIAL_BLOCK_KM:g} km blocks)")
# else:
#     print("Spatial block CV skipped (CV_SPATIAL_ENABLED = False).")
#
# print("\nInterpretation: large drops from the temporal to the spatial-block scores indicate the "
#       "model leans on local spatial structure and transfers poorly to unseen areas -- expected "
#       "for a drifting invasive, and the motivation for the neighbour-lag and bathymetry terms.")
#

##### (superseded) Model diagnosis after threshold and target-definition changes

A concise summary of the presence-definition and threshold changes and their
effect on predicted WH extent.


In [ ]:
# print("=" * 70)
# print("MODEL DIAGNOSIS AFTER THRESHOLD AND TARGET-DEFINITION CHANGES")
# print("=" * 70)
#
# # 1. Presence definition.
# print(f"\nPresence definition: {presence_definition}")
#
# # 2. Selected probability threshold.
# print(f"Selected probability threshold: {selected_threshold:.2f} "
#       f"(metric={PRESENCE_THRESHOLD_METRIC}, tuned on {threshold_source})")
#
#
# # 3. Train / validation / test months.
# def _months_range(arr):
#     arr = list(arr)
#     if len(arr) == 0:
#         return "(none)"
#     return f"{pd.Timestamp(min(arr)).date()} to {pd.Timestamp(max(arr)).date()} ({len(arr)} months)"
#
#
# print(f"\nFitting months:    {_months_range(fit_months)}")
# print(f"Validation months: {_months_range(val_months)}")
# print(f"Test months:       {_months_range(test['month'].unique())}")
#
# # 4. Presence model precision / recall / F1 on TEST.
# prec_t = precision_score(y_test_presence, test_presence_pred, zero_division=0)
# rec_t = recall_score(y_test_presence, test_presence_pred, zero_division=0)
# f1_t = f1_score(y_test_presence, test_presence_pred, zero_division=0)
# prec_0 = precision_score(y_test_presence, test_presence_pred_05, zero_division=0)
# rec_0 = recall_score(y_test_presence, test_presence_pred_05, zero_division=0)
# f1_0 = f1_score(y_test_presence, test_presence_pred_05, zero_division=0)
# print(f"\nPresence model (TEST) @ tuned {selected_threshold:.2f}: "
#       f"precision={prec_t:.3f}, recall={rec_t:.3f}, F1={f1_t:.3f}")
# print(f"Presence model (TEST) @ legacy 0.50: "
#       f"precision={prec_0:.3f}, recall={rec_0:.3f}, F1={f1_0:.3f}")
#
# # 5. Observed vs predicted area by test month.
# print("\nTest-month observed vs predicted WH area (ha):")
# display(area_compare_display)
#
# # 6. Notes on whether the model still under-predicts.
# obs_total = area_compare["observed_wh_area_ha"].sum()
# pred_total_tuned = area_compare["pred_area_ha_tuned"].sum()
# pred_total_05 = area_compare["pred_area_ha_05"].sum()
# ratio_tuned = (pred_total_tuned / obs_total) if obs_total > 0 else np.nan
# ratio_05 = (pred_total_05 / obs_total) if obs_total > 0 else np.nan
#
# print("\nNotes:")
# print(f"- Tuned threshold recovers {ratio_tuned:.0%} of observed test-period WH area "
#       f"(legacy 0.5 recovered {ratio_05:.0%})." if np.isfinite(ratio_tuned)
#       else "- Observed test-period WH area is zero, so recovery ratios are undefined.")
# if np.isfinite(ratio_tuned) and ratio_tuned < 0.8:
#     print("- The model still UNDER-predicts total WH extent (< 80% of observed). "
#           "Consider a lower presence threshold, the higher-recall probability "
#           "threshold, or additional habitat covariates.")
# elif np.isfinite(ratio_tuned) and ratio_tuned > 1.2:
#     print("- The model now OVER-predicts total WH extent (> 120% of observed). "
#           "Consider a higher presence/probability threshold.")
# else:
#     print("- Total predicted WH extent is broadly consistent with observed (within +/-20%).")
# print(f"- Presence recall changed from {rec_0:.2f} (legacy 0.5) to {rec_t:.2f} (tuned).")
#